#**Step 2**

#**Step 2: Light GBM**

In [ ]:
# -*- coding: utf-8 -*-
"""
Step 2 dynamic postoperative PROM analysis with LightGBM
========================================================

Input
-----
/content/Step 2_ODI_Cohort.csv

Target
------
final_reop_step2
    1 = reoperation from postoperative day 91 through day 365
    0 = no reoperation from postoperative day 91 through day 365

Design
------
This script runs paired baseline-only and dynamic PROM-expanded LightGBM
models for delayed lumbar reoperation prediction. The baseline-only model
uses the 35 baseline variables used in Step 1. The dynamic PROM-expanded
model includes the same baseline variables plus preoperative PROM,
postoperative PROM, PROM change, PROM change rate, relative MCID status,
and timing of postoperative PROM assessment. Paired models use identical
patient-level train/calibration/test splits and identical group-aware
cross-validation fold construction. Hyperparameter tuning is performed
exclusively within the training split using cross-validated average
precision as the primary selection metric. Probability calibration and
threshold selection are performed only on the calibration split. The
held-out test set is reserved until the model-development pipeline is
locked.
"""

# ============================================================
# 0) Install/import
# ============================================================

import os
import sys
import json
import math
import time
import zipfile
import platform
import subprocess
import warnings
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
    import lightgbm as lgb
    from lightgbm import LGBMClassifier

try:
    import shap
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
    import shap

try:
    import joblib
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "joblib"])
    import joblib

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedGroupKFold, ParameterSampler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_recall_curve,
    roc_curve,
    confusion_matrix,
    f1_score,
)
from sklearn.isotonic import IsotonicRegression

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)


# ============================================================
# 1) User configuration
# ============================================================

BASE_DIR = "/content"
INPUT_CSV = os.path.join(BASE_DIR, "Step 2_ODI_Cohort.csv")

OUTPUT_DIR = os.path.join(BASE_DIR, "Step2_DynamicPROM_LightGBM_outputs")
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

TARGET_COL = "final_reop_step2"
GROUP_COL = "PersonKey"

RANDOM_STATE = 20260524

# Split design: approximately 64% train, 16% calibration, 20% test
TEST_FRACTION = 0.20
CALIBRATION_FRACTION_OF_REMAINING = 0.20

# CV and tuning
N_CV_FOLDS = 5
N_RANDOM_COMBINATIONS = 300

# Calibration and threshold
CALIBRATION_METHOD = "isotonic"
THRESHOLD_STRATEGY = "max_f1"  # options: max_f1, youden, fixed, top_percent
FIXED_THRESHOLD = 0.50
THRESHOLD_TOP_PERCENT = 0.05

# Test-set uncertainty
N_BOOTSTRAPS = 2000
ECE_N_BINS = 10

# LightGBM runtime
N_JOBS = -1

# Early stopping is used only inside cross-validation to estimate a stable
# boosting length. The locked selected model is refit on the full training split
# using the median best iteration from CV folds.
USE_EARLY_STOPPING_IN_CV = True
EARLY_STOPPING_ROUNDS = 100
MIN_FINAL_N_ESTIMATORS = 50

# SHAP and threshold analysis
SHAP_MAX_EXPLAIN_ROWS = None
SHAP_BAR_MAX_DISPLAY = 30
SHAP_BEESWARM_MAX_DISPLAY = 30
SHAP_THRESHOLD_TOP_N_NUMERIC = 20
SHAP_THRESHOLD_MAX_BINS = 35

# Colors matched to the standard SHAP beeswarm palette:
# blue = lower/protective contribution, pink = higher/risk-increasing contribution.
SHAP_BEESWARM_BLUE = "#008BFB"
SHAP_BEESWARM_PINK = "#FF0051"

# Primary model selection is by cross-validated AP only.
# Test-set metrics are reported but are not used to choose the selected model.
PRIMARY_MODEL_SELECTION_RULE = "highest mean group-aware CV AP on training split; test metrics are diagnostic only and never used for model selection"

# Output archive options
AUTO_DOWNLOAD_ZIP = False
CREATE_COLAB_DOWNLOAD_LINK = True
ZIP_COMPRESSION_LEVEL = 1


# ============================================================
# 2) Step 2 feature set
# ============================================================

BASE_FEATURES = [
    "finaldx_degenerative",
    "finaldx_radicular",
    "finaldx_stenosis",
    "finaldx_deformity_instability",
    "finaldx_other_diagnosis",
    "age",
    "sex",
    "race",
    "ethnicity",
    "cancer_status",
    "chronic_pulmonary_disease",
    "congestive_heart_failure",
    "connective_tissue_rheumatic_disease",
    "diabetes_status",
    "myocardial_infarction",
    "renal_disease",
    "institution_type",
    "institution_size",
    "institution_region",
    "asa",
    "bmi",
    "payer_status",
    "alif_llif",
    "corpectomy",
    "discectomy",
    "foraminotomy",
    "instrumentation",
    "laminectomy_posterior_decompression",
    "pelvic_fixation",
    "plf",
    "tlif_plif",
    "other_lumbar_procedure",
    "number_operated_levels",
    "operative_region_extent",
    "PatTobaccoUse",
]

# Dynamic ODI features are derived explicitly from preop_ODI, postop_ODI, and days_between_PROMs below.
# The input input column ODI_MCID_binary, if present, is not used as a feature;
# it is retained only for audit and reproducibility checks.
RELATIVE_ODI_MCID_CUTOFF = 0.30
INPUT_ODI_MCID_COL = "ODI_MCID_binary"
PROM_CHANGE_RATE_COL = "ODI_change_rate"
RELATIVE_ODI_MCID_COL = "ODI_relative_MCID_binary"
DAYS_BETWEEN_PROM_COL = "days_between_PROMs"

STEP2_ODI_FEATURES = [
    "preop_ODI",
    "postop_ODI",
    "ODI_change",
    PROM_CHANGE_RATE_COL,
    RELATIVE_ODI_MCID_COL,
    "postop_ODI_day",
]

FEATURES = BASE_FEATURES + STEP2_ODI_FEATURES

# Excluded deliberately
EXCLUDED_FEATURES = {
    "reop",
    "reoptime",
    "final_reop",
    "final_reop_step2",
    "death_within_90d",
    "death_within_180d",
    "death_within_365d",
    "death_after_index_surgery",
    "death_before_or_on_index_surgery",
    "PersonDeathDate",
    "days_to_death_from_index_surgery",
    "removal_hardware",
    "any_arthroplasty",
    "final_diagnosis_complexity",
    "procedure_complexity_score",
}

bad_features = sorted(set(FEATURES) & EXCLUDED_FEATURES)
if bad_features:
    raise ValueError(f"Excluded/leakage-prone features were accidentally included: {bad_features}")

CONTINUOUS_FEATURES = [
    "age",
    "bmi",
    "preop_ODI",
    "postop_ODI",
    "ODI_change",
    PROM_CHANGE_RATE_COL,
    "postop_ODI_day",
]

BINARY_FEATURES = [
    "finaldx_degenerative",
    "finaldx_radicular",
    "finaldx_stenosis",
    "finaldx_deformity_instability",
    "finaldx_other_diagnosis",
    "sex",
    "ethnicity",
    "cancer_status",
    "chronic_pulmonary_disease",
    "congestive_heart_failure",
    "connective_tissue_rheumatic_disease",
    "myocardial_infarction",
    "renal_disease",
    "institution_type",
    "alif_llif",
    "corpectomy",
    "discectomy",
    "foraminotomy",
    "instrumentation",
    "laminectomy_posterior_decompression",
    "pelvic_fixation",
    "plf",
    "tlif_plif",
    "other_lumbar_procedure",
    "operative_region_extent",
    RELATIVE_ODI_MCID_COL,
]

ORDINAL_FEATURES = [
    "diabetes_status",
    "institution_size",
    "asa",
    "number_operated_levels",
]

NOMINAL_FEATURES = [
    "race",
    "institution_region",
    "payer_status",
    "PatTobaccoUse",
]


# ============================================================
# 3) Hyperparameter search space
# ============================================================

# positive_weight_multiplier is converted to sample weights:
# positive_weight = n_negative / n_positive * positive_weight_multiplier
LGBM_SEARCH_SPACE = {
    # Wider final search space. Candidate selection remains based only on group-aware CV AP.
    "n_estimators": [200, 400, 700, 1000, 1400, 1800, 2200],
    "learning_rate": [0.003, 0.005, 0.01, 0.02, 0.03, 0.05, 0.08],
    "num_leaves": [7, 15, 31, 63, 127],
    "max_depth": [-1, 2, 3, 5, 7, 9],
    "min_child_samples": [10, 20, 50, 100, 200, 400],
    "subsample": [0.60, 0.75, 0.90, 1.00],
    "subsample_freq": [0, 1, 2],
    "colsample_bytree": [0.60, 0.75, 0.90, 1.00],
    "reg_alpha": [0.0, 0.001, 0.01, 0.05, 0.10, 0.50, 1.00, 2.00],
    "reg_lambda": [0.0, 0.001, 0.01, 0.05, 0.10, 0.50, 1.00, 2.00, 5.00],
    "min_split_gain": [0.0, 0.001, 0.005, 0.01, 0.05, 0.10],
    "max_bin": [63, 127, 255],
    # Class imbalance handling is treated as a tuned hyperparameter.
    # The base positive weight is n_negative / n_positive in each training fold.
    "positive_weight_multiplier": [0.25, 0.50, 0.75, 1.00, 1.50, 2.00, 3.00, 4.00, 6.00, 8.00],
}

LGBM_INT_PARAMS = {
    "n_estimators",
    "num_leaves",
    "max_depth",
    "min_child_samples",
    "subsample_freq",
    "max_bin",
}

LGBM_FLOAT_PARAMS = {
    "learning_rate",
    "subsample",
    "colsample_bytree",
    "reg_alpha",
    "reg_lambda",
    "min_split_gain",
    "positive_weight_multiplier",
}


def sanitize_lgbm_params(params: Dict[str, Any]) -> Dict[str, Any]:
    """Convert LightGBM params to native Python types, especially integer params."""
    clean: Dict[str, Any] = {}

    for k, v in params.items():
        if k in LGBM_INT_PARAMS:
            clean[k] = int(round(float(v)))
        elif k in LGBM_FLOAT_PARAMS:
            clean[k] = float(v)
        else:
            if isinstance(v, (np.integer,)):
                clean[k] = int(v)
            elif isinstance(v, (np.floating,)):
                clean[k] = float(v)
            else:
                clean[k] = v

    return clean


# ============================================================
# 4) General helpers
# ============================================================

MISSING_STRINGS = {
    "",
    " ",
    "na",
    "n/a",
    "nan",
    "none",
    "null",
    ".",
    "missing",
    "<na>",
}


def json_native(obj: Any) -> Any:
    """Recursively convert numpy/pandas objects to JSON-serializable Python objects."""
    if isinstance(obj, dict):
        return {str(k): json_native(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [json_native(v) for v in obj]
    if isinstance(obj, tuple):
        return [json_native(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj


def clean_scalar(x: Any) -> Any:
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().replace("≥", ">=")
        if s.lower() in MISSING_STRINGS:
            return np.nan
        return s
    return x


def norm_text(x: Any) -> Optional[str]:
    x = clean_scalar(x)
    if pd.isna(x):
        return None
    return str(x).strip().replace("≥", ">=").lower()


def to_binary_target(x: Any) -> float:
    sx = norm_text(x)
    if sx is None:
        return np.nan
    if sx in {"1", "1.0", "yes", "y", "true", "t"}:
        return 1.0
    if sx in {"0", "0.0", "no", "n", "false", "f"}:
        return 0.0
    try:
        v = float(sx)
        if v in (0.0, 1.0):
            return float(v)
    except Exception:
        pass
    return np.nan


def count_pct(n: int, denom: int, digits: int = 2) -> str:
    if denom == 0:
        return f"{int(n):,} (NA)"
    return f"{int(n):,} ({100 * n / denom:.{digits}f}%)"


def safe_average_precision(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(average_precision_score(y_true, y_prob))


def safe_roc_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, y_prob))


def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10) -> float:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(y_true)

    if n == 0:
        return np.nan

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i == n_bins - 1:
            mask = (y_prob >= lo) & (y_prob <= hi)
        else:
            mask = (y_prob >= lo) & (y_prob < hi)

        if np.any(mask):
            bin_conf = float(np.mean(y_prob[mask]))
            bin_acc = float(np.mean(y_true[mask]))
            ece += (np.sum(mask) / n) * abs(bin_acc - bin_conf)

    return float(ece)


def classification_metrics_at_threshold(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: float,
) -> Dict[str, Any]:
    y_true = np.asarray(y_true).astype(int)
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    f1 = f1_score(y_true, y_pred, zero_division=0)

    return {
        "threshold": float(threshold),
        "F1": float(f1),
        "Sensitivity": float(sensitivity),
        "Specificity": float(specificity),
        "PPV": float(ppv),
        "NPV": float(npv),
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "Predicted_positive_rate": float(np.mean(y_pred)),
    }


def top_percent_metrics(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    top_fraction: float = 0.05,
) -> Dict[str, Any]:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    n = len(y_true)
    k = max(1, int(math.ceil(n * top_fraction)))

    order = np.argsort(-y_prob)
    top_idx = order[:k]

    prevalence = float(np.mean(y_true)) if n > 0 else np.nan
    top_event_rate = float(np.mean(y_true[top_idx])) if k > 0 else np.nan
    lift = top_event_rate / prevalence if prevalence > 0 else np.nan
    captured = float(np.sum(y_true[top_idx]) / np.sum(y_true)) if np.sum(y_true) > 0 else np.nan

    return {
        "Top_5pct_n": int(k),
        "Top_5pct_event_rate": top_event_rate,
        "Top_5pct_lift": float(lift),
        "Top_5pct_captured_events": captured,
    }


def select_threshold(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    strategy: str = "max_f1",
    fixed_threshold: float = 0.50,
    top_percent: float = 0.05,
) -> Tuple[float, Dict[str, Any]]:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    if strategy == "fixed":
        threshold = float(fixed_threshold)

    elif strategy == "top_percent":
        threshold = float(np.quantile(y_prob, 1.0 - top_percent))

    elif strategy == "youden":
        fpr, tpr, thresholds = roc_curve(y_true, y_prob)
        youden = tpr - fpr
        threshold = float(thresholds[int(np.nanargmax(youden))])

    elif strategy == "max_f1":
        precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
        if len(thresholds) == 0:
            threshold = 0.50
        else:
            precision = precision[:-1]
            recall = recall[:-1]
            f1_values = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
            threshold = float(thresholds[int(np.nanargmax(f1_values))])

    else:
        raise ValueError(f"Unknown threshold strategy: {strategy}")

    return threshold, classification_metrics_at_threshold(y_true, y_prob, threshold)


def make_positive_sample_weights(y: np.ndarray, multiplier: float) -> np.ndarray:
    y = np.asarray(y).astype(int)
    n_pos = int(np.sum(y == 1))
    n_neg = int(np.sum(y == 0))

    if n_pos == 0:
        raise ValueError("No positive events in training fold.")

    base_weight = n_neg / max(n_pos, 1)
    pos_weight = base_weight * float(multiplier)

    return np.where(y == 1, pos_weight, 1.0).astype(float)


def actual_positive_weight(y: np.ndarray, multiplier: float) -> float:
    y = np.asarray(y).astype(int)
    n_pos = int(np.sum(y == 1))
    n_neg = int(np.sum(y == 0))
    return float((n_neg / max(n_pos, 1)) * multiplier)


# ============================================================
# 5) Preprocessor
# ============================================================

BINARY_MAPS = {
    "sex": {
        "female": 0,
        "f": 0,
        "male": 1,
        "m": 1,
    },
    "ethnicity": {
        "non-hispanic": 0,
        "non hispanic": 0,
        "hispanic": 1,
    },
    "cancer_status": {
        "no cancer": 0,
        "no": 0,
        "none": 0,
        "any cancer": 1,
        "yes": 1,
        "cancer": 1,
    },
    "institution_type": {
        "hospital": 0,
        "non-hospital": 1,
        "non hospital": 1,
        "nonhospital": 1,
    },
    "operative_region_extent": {
        "lumbar only": 0,
        "extended_region_involvement": 1,
        "extended region involvement": 1,
        "extended": 1,
    },
}

ORDINAL_MAPS = {
    "diabetes_status": {
        "no": 0,
        "none": 0,
        "0": 0,
        "without comp": 1,
        "without complication": 1,
        "without complications": 1,
        "1": 1,
        "with comp": 2,
        "with complication": 2,
        "with complications": 2,
        "2": 2,
    },
    "institution_size": {
        "between 1-99 beds": 0,
        "1-99": 0,
        "between 100-399 beds": 1,
        "100-399": 1,
        ">= 400 beds": 2,
        ">=400 beds": 2,
        ">=400": 2,
        ">= 400": 2,
    },
    "asa": {
        "1": 1,
        "i": 1,
        "2": 2,
        "ii": 2,
        "3": 3,
        "iii": 3,
        "4": 4,
        "iv": 4,
        ">=4": 4,
        ">= 4": 4,
        "5": 4,
        "v": 4,
    },
    "number_operated_levels": {
        "0": 0,
        "1": 1,
        "2": 2,
        "3": 3,
        "4": 4,
        ">=4": 4,
        ">= 4": 4,
        "5": 4,
        "6": 4,
        "7": 4,
        "8": 4,
        "9": 4,
        "10": 4,
    },
}

PREFERRED_NOMINAL_LEVELS = {
    "race": ["White", "Black", "Other"],
    "institution_region": ["South", "North East", "West", "Midwest"],
    "payer_status": ["Medicare", "Commercial/Private", "Other", "Medicaid/Public/Government"],
    "PatTobaccoUse": ["Unknown/Not reported/Multiple", "Never", "Former", "Current"],
}

FEATURE_LABELS = {
    "finaldx_degenerative": "Degenerative diagnosis",
    "finaldx_radicular": "Radiculopathy diagnosis",
    "finaldx_stenosis": "Spinal stenosis diagnosis",
    "finaldx_deformity_instability": "Deformity or instability diagnosis",
    "finaldx_other_diagnosis": "Other lumbar diagnosis",
    "age": "Age",
    "sex": "Sex",
    "race": "Race",
    "ethnicity": "Ethnicity",
    "cancer_status": "Cancer status",
    "chronic_pulmonary_disease": "Chronic pulmonary disease",
    "congestive_heart_failure": "Congestive heart failure",
    "connective_tissue_rheumatic_disease": "Connective tissue/rheumatic disease",
    "diabetes_status": "Diabetes status",
    "myocardial_infarction": "Myocardial infarction",
    "renal_disease": "Renal disease",
    "institution_type": "Institution type",
    "institution_size": "Institution size",
    "institution_region": "Institution region",
    "asa": "ASA physical status",
    "bmi": "Body mass index",
    "payer_status": "Primary payer",
    "alif_llif": "Anterior/lateral lumbar interbody fusion",
    "corpectomy": "Corpectomy",
    "discectomy": "Discectomy",
    "foraminotomy": "Foraminotomy",
    "instrumentation": "Instrumentation",
    "laminectomy_posterior_decompression": "Posterior decompression or laminectomy",
    "pelvic_fixation": "Pelvic fixation",
    "plf": "Posterolateral fusion",
    "tlif_plif": "Posterior/transforaminal lumbar interbody fusion",
    "other_lumbar_procedure": "Other lumbar procedure",
    "number_operated_levels": "Number of operated levels",
    "operative_region_extent": "Operative region extent",
    "PatTobaccoUse": "Tobacco use",
    "preop_ODI": "Preoperative ODI",
    "postop_ODI": "Postoperative ODI",
    "ODI_change": "Change in ODI",
    PROM_CHANGE_RATE_COL: "ODI change rate",
    RELATIVE_ODI_MCID_COL: "Relative ODI MCID",
    "postop_ODI_day": "Timing of postoperative ODI assessment",
}

def pretty_feature_name(feature: str) -> str:
    return FEATURE_LABELS.get(feature, feature.replace("_", " "))


@dataclass
class Step2Preprocessor:
    continuous_features: List[str]
    binary_features: List[str]
    ordinal_features: List[str]
    nominal_features: List[str]
    preferred_nominal_levels: Dict[str, List[str]]

    def __post_init__(self):
        self.continuous_imputer: Optional[SimpleImputer] = None
        self.discrete_imputer: Optional[SimpleImputer] = None
        self.nominal_imputer: Optional[SimpleImputer] = None
        self.onehot: Optional[OneHotEncoder] = None
        self.numeric_feature_names_: List[str] = []
        self.output_feature_names_: List[str] = []

    def _parse_binary(self, x: Any, feature: str) -> float:
        sx = norm_text(x)
        if sx is None:
            return np.nan

        if feature in BINARY_MAPS and sx in BINARY_MAPS[feature]:
            return float(BINARY_MAPS[feature][sx])

        if sx in {"1", "1.0", "yes", "y", "true", "t", "present", "positive"}:
            return 1.0
        if sx in {"0", "0.0", "no", "n", "false", "f", "absent", "negative"}:
            return 0.0

        try:
            v = float(sx)
            if v in (0.0, 1.0):
                return float(v)
        except Exception:
            pass

        return np.nan

    def _parse_ordinal(self, x: Any, feature: str) -> float:
        sx = norm_text(x)
        if sx is None:
            return np.nan

        if feature in ORDINAL_MAPS and sx in ORDINAL_MAPS[feature]:
            return float(ORDINAL_MAPS[feature][sx])

        try:
            v = float(sx)
            if feature == "asa":
                return float(min(max(int(round(v)), 1), 4))
            if feature == "number_operated_levels":
                return float(min(max(int(round(v)), 0), 4))
            if feature == "diabetes_status":
                return float(min(max(int(round(v)), 0), 2))
            if feature == "institution_size":
                return float(min(max(int(round(v)), 0), 2))
            return float(v)
        except Exception:
            return np.nan

    def _canonical_nominal(self, feature: str, x: Any) -> Any:
        x = clean_scalar(x)
        if pd.isna(x):
            return np.nan

        s = str(x).strip()
        sl = s.lower()

        if feature == "race":
            if sl == "white":
                return "White"
            if sl == "black":
                return "Black"
            return "Other"

        if feature == "institution_region":
            mapping = {
                "south": "South",
                "north east": "North East",
                "northeast": "North East",
                "north-east": "North East",
                "west": "West",
                "midwest": "Midwest",
                "mid west": "Midwest",
            }
            return mapping.get(sl, s)

        if feature == "payer_status":
            if sl == "medicare":
                return "Medicare"
            if sl in {"commercial/private", "commercial", "private", "commercial private"}:
                return "Commercial/Private"
            if sl in {"medicaid/public/government", "medicaid", "public", "government", "public/government"}:
                return "Medicaid/Public/Government"
            return "Other"

        if feature == "PatTobaccoUse":
            if sl == "never":
                return "Never"
            if sl == "former":
                return "Former"
            if sl == "current":
                return "Current"
            return "Unknown/Not reported/Multiple"

        return s

    def _make_parts(self, X: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        cont = pd.DataFrame(index=X.index)
        for c in self.continuous_features:
            cont[c] = pd.to_numeric(X[c].map(clean_scalar), errors="coerce")

        discrete = pd.DataFrame(index=X.index)
        for c in self.binary_features:
            discrete[c] = X[c].map(lambda z: self._parse_binary(z, c)).astype(float)
        for c in self.ordinal_features:
            discrete[c] = X[c].map(lambda z: self._parse_ordinal(z, c)).astype(float)

        nominal = pd.DataFrame(index=X.index)
        for c in self.nominal_features:
            nominal[c] = X[c].map(lambda z: self._canonical_nominal(c, z)).astype("object")

        return cont, discrete, nominal

    def fit(self, X: pd.DataFrame):
        cont, discrete, nominal = self._make_parts(X)

        self.continuous_imputer = SimpleImputer(strategy="median")
        self.discrete_imputer = SimpleImputer(strategy="most_frequent")
        self.nominal_imputer = SimpleImputer(strategy="constant", fill_value="Missing")

        self.continuous_imputer.fit(cont)
        self.discrete_imputer.fit(discrete)

        nominal_imp = self.nominal_imputer.fit_transform(nominal)
        nominal_imp = pd.DataFrame(nominal_imp, columns=self.nominal_features)

        categories = []
        for c in self.nominal_features:
            preferred = list(self.preferred_nominal_levels.get(c, []))
            observed = nominal_imp[c].astype(str).unique().tolist()
            final_cats = preferred + sorted([x for x in observed if x not in preferred])
            categories.append(final_cats)

        try:
            self.onehot = OneHotEncoder(categories=categories, handle_unknown="ignore", sparse_output=False)
        except TypeError:
            self.onehot = OneHotEncoder(categories=categories, handle_unknown="ignore", sparse=False)

        self.onehot.fit(nominal_imp.astype(str))

        self.numeric_feature_names_ = self.continuous_features + self.binary_features + self.ordinal_features
        self.output_feature_names_ = (
            list(self.numeric_feature_names_)
            + self.onehot.get_feature_names_out(self.nominal_features).tolist()
        )
        return self

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        if self.continuous_imputer is None or self.discrete_imputer is None or self.nominal_imputer is None or self.onehot is None:
            raise RuntimeError("Preprocessor is not fitted.")

        cont, discrete, nominal = self._make_parts(X)

        cont_imp = self.continuous_imputer.transform(cont)
        discrete_imp = self.discrete_imputer.transform(discrete)

        nominal_imp = self.nominal_imputer.transform(nominal)
        nominal_imp = pd.DataFrame(nominal_imp, columns=self.nominal_features)
        nominal_oh = self.onehot.transform(nominal_imp.astype(str))

        return np.concatenate([cont_imp, discrete_imp, nominal_oh], axis=1).astype(float)

    def fit_transform(self, X: pd.DataFrame) -> np.ndarray:
        self.fit(X)
        return self.transform(X)

    @property
    def output_feature_names(self) -> List[str]:
        return self.output_feature_names_


# ============================================================
# 6) Splitting, tuning, fitting
# ============================================================

def patient_level_train_cal_test_split(
    df: pd.DataFrame,
    target_col: str,
    group_col: str,
    test_fraction: float,
    calibration_fraction_of_remaining: float,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    group_df = df.groupby(group_col, dropna=False)[target_col].max().reset_index()
    y_group = group_df[target_col].astype(int).to_numpy()
    groups = group_df[group_col].to_numpy()

    if len(np.unique(y_group)) < 2:
        raise ValueError("Only one class at patient level; stratified split impossible.")

    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_fraction, random_state=seed)
    train_cal_idx, test_idx = next(sss1.split(groups, y_group))

    groups_train_cal = groups[train_cal_idx]
    y_train_cal = y_group[train_cal_idx]

    sss2 = StratifiedShuffleSplit(
        n_splits=1,
        test_size=calibration_fraction_of_remaining,
        random_state=seed + 1,
    )
    train_idx_rel, cal_idx_rel = next(sss2.split(groups_train_cal, y_train_cal))

    train_groups = set(groups_train_cal[train_idx_rel])
    cal_groups = set(groups_train_cal[cal_idx_rel])
    test_groups = set(groups[test_idx])

    assert train_groups.isdisjoint(cal_groups)
    assert train_groups.isdisjoint(test_groups)
    assert cal_groups.isdisjoint(test_groups)

    return (
        df[group_col].isin(train_groups).to_numpy(),
        df[group_col].isin(cal_groups).to_numpy(),
        df[group_col].isin(test_groups).to_numpy(),
    )


def cv_fold_count(y: np.ndarray, groups: np.ndarray, requested_folds: int) -> int:
    group_df = pd.DataFrame({"group": groups, "y": y}).groupby("group")["y"].max().reset_index()
    n_pos_groups = int((group_df["y"] == 1).sum())
    n_neg_groups = int((group_df["y"] == 0).sum())
    n_folds = min(requested_folds, n_pos_groups, n_neg_groups)

    if n_folds < 2:
        raise ValueError("Not enough positive/negative patient groups for group-aware CV.")

    return int(n_folds)


def make_lgbm_model(
    params: Dict[str, Any],
    seed: int,
    override_n_estimators: Optional[int] = None,
) -> LGBMClassifier:
    params = sanitize_lgbm_params(params)
    model_params = {k: v for k, v in params.items() if k != "positive_weight_multiplier"}
    if override_n_estimators is not None:
        model_params["n_estimators"] = int(max(MIN_FINAL_N_ESTIMATORS, override_n_estimators))

    return LGBMClassifier(
        objective="binary",
        boosting_type="gbdt",
        metric="average_precision",
        random_state=seed,
        n_jobs=N_JOBS,
        verbosity=-1,
        force_col_wise=True,
        **model_params,
    )


def fit_model_pipeline(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    params: Dict[str, Any],
    seed: int,
    eval_set: Optional[Tuple[pd.DataFrame, np.ndarray]] = None,
    use_early_stopping: bool = False,
    override_n_estimators: Optional[int] = None,
) -> Tuple[Step2Preprocessor, LGBMClassifier, Optional[int]]:
    params = sanitize_lgbm_params(params)

    pre = Step2Preprocessor(
        continuous_features=CONTINUOUS_FEATURES,
        binary_features=BINARY_FEATURES,
        ordinal_features=ORDINAL_FEATURES,
        nominal_features=NOMINAL_FEATURES,
        preferred_nominal_levels=PREFERRED_NOMINAL_LEVELS,
    )

    Xp_train = pre.fit_transform(X_train)

    weights = make_positive_sample_weights(
        y_train,
        multiplier=float(params["positive_weight_multiplier"]),
    )

    model = make_lgbm_model(params, seed=seed, override_n_estimators=override_n_estimators)
    best_iteration: Optional[int] = None

    if eval_set is not None and use_early_stopping:
        X_val, y_val = eval_set
        Xp_val = pre.transform(X_val)
        callbacks = [
            lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
            lgb.log_evaluation(period=0),
        ]
        try:
            model.fit(
                Xp_train,
                y_train,
                sample_weight=weights,
                eval_set=[(Xp_val, y_val)],
                eval_metric="average_precision",
                callbacks=callbacks,
            )
            if hasattr(model, "best_iteration_") and model.best_iteration_ is not None:
                best_iteration = int(model.best_iteration_)
        except Exception:
            # Conservative fallback for older LightGBM builds.
            model.fit(Xp_train, y_train, sample_weight=weights)
    else:
        model.fit(Xp_train, y_train, sample_weight=weights)

    return pre, model, best_iteration


def predict_pipeline(pre: Step2Preprocessor, model: LGBMClassifier, X: pd.DataFrame) -> np.ndarray:
    Xp = pre.transform(X)
    return model.predict_proba(Xp)[:, 1]


def tune_hyperparameters(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    groups_train: np.ndarray,
    seed: int,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    n_folds = cv_fold_count(y_train, groups_train, N_CV_FOLDS)
    cv = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    sampler = list(ParameterSampler(
        LGBM_SEARCH_SPACE,
        n_iter=N_RANDOM_COMBINATIONS,
        random_state=seed,
    ))

    candidate_rows = []
    fold_rows = []

    for i, raw_params in enumerate(sampler, start=1):
        params = sanitize_lgbm_params(raw_params)
        fold_aps = []
        fold_aucs = []
        fold_train_aps = []
        fold_train_aucs = []
        fold_best_iterations = []
        t0 = time.time()

        for fold_id, (tr_idx, va_idx) in enumerate(cv.split(X_train, y_train, groups_train), start=1):
            X_tr = X_train.iloc[tr_idx].reset_index(drop=True)
            y_tr = y_train[tr_idx]

            X_va = X_train.iloc[va_idx].reset_index(drop=True)
            y_va = y_train[va_idx]

            pre, model, best_iter = fit_model_pipeline(
                X_tr,
                y_tr,
                params=params,
                seed=seed + i * 1000 + fold_id,
                eval_set=(X_va, y_va),
                use_early_stopping=USE_EARLY_STOPPING_IN_CV,
            )

            p_tr = predict_pipeline(pre, model, X_tr)
            p_va = predict_pipeline(pre, model, X_va)

            train_ap = safe_average_precision(y_tr, p_tr)
            train_auc = safe_roc_auc(y_tr, p_tr)
            ap = safe_average_precision(y_va, p_va)
            auc = safe_roc_auc(y_va, p_va)

            fold_train_aps.append(train_ap)
            fold_train_aucs.append(train_auc)
            fold_aps.append(ap)
            fold_aucs.append(auc)
            if best_iter is not None and best_iter > 0:
                fold_best_iterations.append(best_iter)

            fold_rows.append({
                "candidate_id": i,
                "fold": fold_id,
                "fold_train_n": int(len(y_tr)),
                "fold_train_events": int(np.sum(y_tr)),
                "fold_validation_n": int(len(y_va)),
                "fold_validation_events": int(np.sum(y_va)),
                "fold_validation_event_rate": float(np.mean(y_va)),
                "fold_train_AP": train_ap,
                "fold_train_ROC_AUC": train_auc,
                "fold_validation_AP": ap,
                "fold_validation_ROC_AUC": auc,
                "fold_train_minus_validation_AP_gap": train_ap - ap if np.isfinite(train_ap) and np.isfinite(ap) else np.nan,
                "fold_train_minus_validation_ROC_AUC_gap": train_auc - auc if np.isfinite(train_auc) and np.isfinite(auc) else np.nan,
                "fold_best_iteration": best_iter,
                "positive_weight_used": actual_positive_weight(y_tr, params["positive_weight_multiplier"]),
                **params,
            })

        elapsed = time.time() - t0
        locked_n_estimators = int(np.median(fold_best_iterations)) if fold_best_iterations else int(params["n_estimators"])

        row = {
            "candidate_id": i,
            "cv_folds": n_folds,
            "cv_train_AP_mean": float(np.nanmean(fold_train_aps)),
            "cv_train_AP_SD": float(np.nanstd(fold_train_aps, ddof=1)),
            "cv_train_ROC_AUC_mean": float(np.nanmean(fold_train_aucs)),
            "cv_train_ROC_AUC_SD": float(np.nanstd(fold_train_aucs, ddof=1)),
            "cv_AP_mean": float(np.nanmean(fold_aps)),
            "cv_AP_SD": float(np.nanstd(fold_aps, ddof=1)),
            "cv_ROC_AUC_mean": float(np.nanmean(fold_aucs)),
            "cv_ROC_AUC_SD": float(np.nanstd(fold_aucs, ddof=1)),
            "cv_train_minus_validation_AP_gap": float(np.nanmean(fold_train_aps) - np.nanmean(fold_aps)),
            "cv_train_minus_validation_ROC_AUC_gap": float(np.nanmean(fold_train_aucs) - np.nanmean(fold_aucs)),
            "mean_cv_best_iteration": float(np.nanmean(fold_best_iterations)) if fold_best_iterations else np.nan,
            "locked_n_estimators_from_cv": locked_n_estimators,
            "elapsed_seconds": float(elapsed),
            **params,
        }
        candidate_rows.append(row)

        print(
            f"Candidate {i:03d}/{len(sampler)} | "
            f"CV AP={row['cv_AP_mean']:.5f} ± {row['cv_AP_SD']:.5f} | "
            f"Train-CV AP gap={row['cv_train_minus_validation_AP_gap']:.5f} | "
            f"CV AUC={row['cv_ROC_AUC_mean']:.5f} | "
            f"locked_n={locked_n_estimators} | "
            f"pos_mult={params['positive_weight_multiplier']}"
        )

    candidates = (
        pd.DataFrame(candidate_rows)
        .sort_values("cv_AP_mean", ascending=False)
        .reset_index(drop=True)
    )
    folds = pd.DataFrame(fold_rows)

    return candidates, folds


# ============================================================
# 7) Evaluation and bootstrap
# ============================================================

def evaluate_predictions(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: Optional[float] = None,
    prefix: str = "",
) -> Dict[str, Any]:
    out = {
        f"{prefix}AP": safe_average_precision(y_true, y_prob),
        f"{prefix}ROC_AUC": safe_roc_auc(y_true, y_prob),
        f"{prefix}Brier_score": float(brier_score_loss(y_true, y_prob)),
        f"{prefix}ECE": expected_calibration_error(y_true, y_prob, n_bins=ECE_N_BINS),
        f"{prefix}N": int(len(y_true)),
        f"{prefix}Events": int(np.sum(y_true)),
        f"{prefix}Prevalence": float(np.mean(y_true)),
    }

    if threshold is not None:
        cls = classification_metrics_at_threshold(y_true, y_prob, threshold)
        out.update({f"{prefix}{k}": v for k, v in cls.items()})

        top = top_percent_metrics(y_true, y_prob, top_fraction=0.05)
        out.update({f"{prefix}{k}": v for k, v in top.items()})

    return out


def patient_level_stratified_bootstrap_ci(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    groups: np.ndarray,
    metric_name: str,
    threshold: Optional[float] = None,
    n_bootstraps: int = 2000,
    seed: int = 123,
) -> Tuple[float, float]:
    rng = np.random.default_rng(seed)

    d = pd.DataFrame({
        "row_id": np.arange(len(y_true)),
        "group": groups,
        "y": np.asarray(y_true).astype(int),
        "p": np.asarray(y_prob).astype(float),
    })

    group_y = d.groupby("group")["y"].max()
    pos_groups = group_y[group_y == 1].index.to_numpy()
    neg_groups = group_y[group_y == 0].index.to_numpy()

    if len(pos_groups) == 0 or len(neg_groups) == 0:
        return np.nan, np.nan

    rows_by_group = {
        g: d.loc[d["group"].eq(g), "row_id"].to_numpy()
        for g in group_y.index
    }

    values = []

    for _ in range(n_bootstraps):
        sampled_pos = rng.choice(pos_groups, size=len(pos_groups), replace=True)
        sampled_neg = rng.choice(neg_groups, size=len(neg_groups), replace=True)
        sampled_groups = np.concatenate([sampled_pos, sampled_neg])

        idx = np.concatenate([rows_by_group[g] for g in sampled_groups])
        yy = y_true[idx]
        pp = y_prob[idx]

        if len(np.unique(yy)) < 2:
            continue

        if metric_name == "AP":
            values.append(average_precision_score(yy, pp))
        elif metric_name == "ROC_AUC":
            values.append(roc_auc_score(yy, pp))
        elif metric_name == "F1":
            if threshold is None:
                raise ValueError("threshold required for F1 bootstrap")
            yhat = (pp >= threshold).astype(int)
            values.append(f1_score(yy, yhat, zero_division=0))
        else:
            raise ValueError(f"Unsupported bootstrap metric: {metric_name}")

    if len(values) == 0:
        return np.nan, np.nan

    lo, hi = np.percentile(values, [2.5, 97.5])
    return float(lo), float(hi)


def save_pr_curve(y_true: np.ndarray, y_prob: np.ndarray, path: str, title: str) -> None:
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, linewidth=2)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{title}\nAP = {ap:.3f}")
    plt.tight_layout()
    plt.savefig(path, dpi=300)
    plt.close()


def save_roc_curve(y_true: np.ndarray, y_prob: np.ndarray, path: str, title: str) -> None:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, linewidth=2)
    plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    plt.xlabel("False-positive rate")
    plt.ylabel("True-positive rate")
    plt.title(f"{title}\nROC-AUC = {auc:.3f}")
    plt.tight_layout()
    plt.savefig(path, dpi=300)
    plt.close()


def save_calibration_plot(y_true: np.ndarray, y_prob: np.ndarray, path: str, title: str) -> None:
    """
    Calibration plot for rare-event prediction.

    The event prevalence in this task is low; therefore, most calibrated
    probabilities are expected to be close to zero. A conventional 0-1 axis can
    visually compress all calibration points into the lower-left corner. This
    plot uses quantile-based probability bins and a data-adaptive zoomed axis,
    while reporting Brier score and expected calibration error (ECE).
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    finite_mask = np.isfinite(y_prob)
    y_true = y_true[finite_mask]
    y_prob = y_prob[finite_mask]

    brier = float(brier_score_loss(y_true, y_prob))
    ece = float(expected_calibration_error(y_true, y_prob, n_bins=ECE_N_BINS))
    prevalence = float(np.mean(y_true)) if len(y_true) else np.nan

    tmp = pd.DataFrame({"y": y_true, "p": y_prob})
    n_unique = int(tmp["p"].nunique())
    n_bins = min(ECE_N_BINS, max(2, n_unique))
    tmp["bin"] = pd.qcut(tmp["p"].rank(method="first"), q=n_bins, duplicates="drop")

    cal = tmp.groupby("bin", observed=True).agg(
        mean_pred=("p", "mean"),
        observed=("y", "mean"),
        n=("y", "size"),
    ).reset_index(drop=True)

    upper_prob = float(np.nanquantile(y_prob, 0.995)) if len(y_prob) else 0.05
    upper_obs = float(max(cal["observed"].max(), cal["mean_pred"].max())) if len(cal) else 0.05
    axis_max = max(0.05, min(0.25, max(upper_prob, upper_obs, prevalence) * 1.35 + 0.005))

    fig, ax = plt.subplots(figsize=(7.2, 6.0))

    ax.plot(
        [0, axis_max],
        [0, axis_max],
        linestyle="--",
        linewidth=1.6,
        color="black",
        alpha=0.80,
        label="Perfect calibration",
    )
    ax.plot(
        cal["mean_pred"],
        cal["observed"],
        marker="o",
        linewidth=2.2,
        markersize=6.5,
        label="Model calibration",
    )
    ax.axhline(
        prevalence,
        linestyle=":",
        linewidth=1.5,
        color="gray",
        alpha=0.90,
        label=f"Observed event rate = {prevalence:.2%}",
    )

    for _, r in cal.iterrows():
        ax.annotate(
            f"n={int(r['n'])}",
            (float(r["mean_pred"]), float(r["observed"])),
            textcoords="offset points",
            xytext=(5, 4),
            fontsize=7,
            alpha=0.75,
        )

    ax.set_xlim(0, axis_max)
    ax.set_ylim(0, axis_max)
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed event rate")
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.legend(loc="upper left", frameon=True)

    text_box = (
        f"Brier score = {brier:.4f}\n"
        f"ECE = {ece:.4f}\n"
        f"N = {len(y_true):,}\n"
        f"Events = {int(np.sum(y_true)):,}"
    )
    ax.text(
        0.98,
        0.04,
        text_box,
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="gray", alpha=0.95),
    )

    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def save_learning_curve_grouped_ap(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    groups_train: np.ndarray,
    best_params: Dict[str, Any],
    path_png: str,
    path_csv: str,
    title: str,
    train_fractions: Tuple[float, ...] = (0.20, 0.40, 0.60, 0.80, 1.00),
    n_splits: int = 3,
    seed: int = RANDOM_STATE,
) -> pd.DataFrame:
    """
    Group-aware learning curve using Average Precision.

    This diagnostic is computed only inside the training split using the locked
    best hyperparameter configuration. It is not used for model selection and
    never uses the calibration or test sets.
    """
    X_train = X_train.reset_index(drop=True)
    y_train = np.asarray(y_train).astype(int)
    groups_train = np.asarray(groups_train)

    n_splits_eff = min(cv_fold_count(y_train, groups_train, n_splits), n_splits)
    cv = StratifiedGroupKFold(n_splits=n_splits_eff, shuffle=True, random_state=seed)

    rows = []
    rng = np.random.default_rng(seed)

    for frac in train_fractions:
        for fold_id, (tr_idx, va_idx) in enumerate(cv.split(X_train, y_train, groups_train), start=1):
            X_tr_full = X_train.iloc[tr_idx].reset_index(drop=True)
            y_tr_full = y_train[tr_idx]
            g_tr_full = groups_train[tr_idx]

            X_va = X_train.iloc[va_idx].reset_index(drop=True)
            y_va = y_train[va_idx]

            group_event = (
                pd.DataFrame({"group": g_tr_full, "y": y_tr_full})
                .groupby("group")["y"]
                .max()
                .reset_index()
            )
            pos_groups = group_event.loc[group_event["y"].eq(1), "group"].to_numpy()
            neg_groups = group_event.loc[group_event["y"].eq(0), "group"].to_numpy()

            n_pos_keep = max(1, int(np.ceil(len(pos_groups) * frac)))
            n_neg_keep = max(1, int(np.ceil(len(neg_groups) * frac)))

            sampled_pos = rng.choice(pos_groups, size=min(n_pos_keep, len(pos_groups)), replace=False)
            sampled_neg = rng.choice(neg_groups, size=min(n_neg_keep, len(neg_groups)), replace=False)
            sampled_groups = set(np.concatenate([sampled_pos, sampled_neg]))

            keep = np.array([g in sampled_groups for g in g_tr_full])
            X_tr = X_tr_full.loc[keep].reset_index(drop=True)
            y_tr = y_tr_full[keep]

            if len(np.unique(y_tr)) < 2 or len(np.unique(y_va)) < 2:
                continue

            pre_lc, model_lc, _ = fit_model_pipeline(
                X_tr,
                y_tr,
                params=best_params,
                seed=seed + fold_id + int(frac * 1000),
            )

            p_tr = predict_pipeline(pre_lc, model_lc, X_tr)
            p_va = predict_pipeline(pre_lc, model_lc, X_va)

            rows.append({
                "train_fraction": float(frac),
                "fold": int(fold_id),
                "train_n": int(len(y_tr)),
                "train_events": int(np.sum(y_tr)),
                "validation_n": int(len(y_va)),
                "validation_events": int(np.sum(y_va)),
                "train_AP": safe_average_precision(y_tr, p_tr),
                "validation_AP": safe_average_precision(y_va, p_va),
            })

    lc_raw = pd.DataFrame(rows)
    lc_raw.to_csv(path_csv, index=False)

    if lc_raw.empty:
        return lc_raw

    lc = (
        lc_raw.groupby("train_fraction", as_index=False)
        .agg(
            train_AP_mean=("train_AP", "mean"),
            train_AP_sd=("train_AP", "std"),
            validation_AP_mean=("validation_AP", "mean"),
            validation_AP_sd=("validation_AP", "std"),
        )
    ).fillna(0.0)

    fig, ax = plt.subplots(figsize=(7.2, 6.0))
    ax.plot(lc["train_fraction"], lc["train_AP_mean"], marker="o", linewidth=2.2, label="Training AP")
    ax.plot(lc["train_fraction"], lc["validation_AP_mean"], marker="o", linewidth=2.2, label="Validation AP")

    ax.fill_between(
        lc["train_fraction"].astype(float).to_numpy(),
        (lc["train_AP_mean"] - lc["train_AP_sd"]).astype(float).to_numpy(),
        (lc["train_AP_mean"] + lc["train_AP_sd"]).astype(float).to_numpy(),
        alpha=0.15,
    )
    ax.fill_between(
        lc["train_fraction"].astype(float).to_numpy(),
        (lc["validation_AP_mean"] - lc["validation_AP_sd"]).astype(float).to_numpy(),
        (lc["validation_AP_mean"] + lc["validation_AP_sd"]).astype(float).to_numpy(),
        alpha=0.15,
    )

    ax.set_xlabel("Fraction of training groups used")
    ax.set_ylabel("Average precision")
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xticks(list(train_fractions))
    ax.legend(loc="best", frameon=True)
    ax.grid(alpha=0.20)

    text_box = (
        "Group-aware CV learning curve\n"
        f"Folds = {n_splits_eff}\n"
        "Metric = Average precision"
    )
    ax.text(
        0.98,
        0.04,
        text_box,
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=9.5,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="gray", alpha=0.95),
    )

    fig.tight_layout()
    fig.savefig(path_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

    return lc_raw


# ============================================================
# 8) Audit helpers
# ============================================================

def build_split_audit(df: pd.DataFrame, split_col: str, target_col: str) -> pd.DataFrame:
    rows = []

    for split in ["train", "calibration", "test"]:
        sub = df[df[split_col] == split].copy()
        rows.append({
            "split": split,
            "rows": len(sub),
            "events": int(sub[target_col].sum()),
            "event_rate": float(sub[target_col].mean()) if len(sub) else np.nan,
            "unique_patients": int(sub[GROUP_COL].nunique()) if GROUP_COL in sub.columns else np.nan,
        })

    return pd.DataFrame(rows)


def build_institution_audit(df: pd.DataFrame, split_col: str) -> pd.DataFrame:
    rows = []
    cols = [c for c in ["institution_type", "institution_region"] if c in df.columns]

    for col in cols:
        ct = (
            df.groupby([split_col, col], dropna=False)
            .size()
            .reset_index(name="n")
        )
        total = df.groupby(split_col).size().rename("split_n").reset_index()
        ct = ct.merge(total, on=split_col, how="left")
        ct["percent_within_split"] = 100 * ct["n"] / ct["split_n"]
        ct.insert(0, "variable", col)
        rows.append(ct)

    if rows:
        return pd.concat(rows, ignore_index=True)

    return pd.DataFrame({"note": ["No institution columns found for audit."]})


def style_excel_workbook(writer: pd.ExcelWriter) -> None:
    try:
        from openpyxl.styles import Font, PatternFill, Alignment

        for sheet_name in writer.sheets:
            ws = writer.sheets[sheet_name]
            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions

            for cell in ws[1]:
                cell.font = Font(bold=True)
                cell.fill = PatternFill(start_color="D9EAF7", end_color="D9EAF7", fill_type="solid")
                cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

            for col_cells in ws.columns:
                max_len = 0
                col_letter = col_cells[0].column_letter

                for cell in col_cells:
                    if cell.value is not None:
                        max_len = max(max_len, len(str(cell.value)))

                ws.column_dimensions[col_letter].width = min(max(max_len + 2, 12), 60)
    except Exception:
        pass



# ============================================================
# 9) Grouped SHAP and threshold analysis helpers
# ============================================================

def parse_binary_global(x: Any, feature: str) -> float:
    sx = norm_text(x)
    if sx is None:
        return np.nan
    if feature in BINARY_MAPS and sx in BINARY_MAPS[feature]:
        return float(BINARY_MAPS[feature][sx])
    if sx in {"1", "1.0", "yes", "y", "true", "t", "present", "positive"}:
        return 1.0
    if sx in {"0", "0.0", "no", "n", "false", "f", "absent", "negative"}:
        return 0.0
    try:
        v = float(sx)
        if v in (0.0, 1.0):
            return float(v)
    except Exception:
        pass
    return np.nan


def parse_ordinal_global(x: Any, feature: str) -> float:
    sx = norm_text(x)
    if sx is None:
        return np.nan
    if feature in ORDINAL_MAPS and sx in ORDINAL_MAPS[feature]:
        return float(ORDINAL_MAPS[feature][sx])
    try:
        v = float(sx)
        if feature == "asa":
            return float(min(max(int(round(v)), 1), 4))
        if feature == "number_operated_levels":
            return float(min(max(int(round(v)), 0), 4))
        if feature == "diabetes_status":
            return float(min(max(int(round(v)), 0), 2))
        if feature == "institution_size":
            return float(min(max(int(round(v)), 0), 2))
        return float(v)
    except Exception:
        return np.nan


def raw_feature_numeric_values(X_raw: pd.DataFrame, feature: str) -> pd.Series:
    if feature in CONTINUOUS_FEATURES:
        s = pd.to_numeric(X_raw[feature].map(clean_scalar), errors="coerce")
    elif feature in BINARY_FEATURES:
        s = X_raw[feature].map(lambda z: parse_binary_global(z, feature)).astype(float)
    elif feature in ORDINAL_FEATURES:
        s = X_raw[feature].map(lambda z: parse_ordinal_global(z, feature)).astype(float)
    else:
        vals = X_raw[feature].map(lambda z: str(clean_scalar(z)) if not pd.isna(clean_scalar(z)) else "Missing")
        categories = {v: i for i, v in enumerate(sorted(vals.dropna().unique()))}
        s = vals.map(categories).astype(float)
    if s.isna().any():
        med = s.median(skipna=True)
        if pd.isna(med):
            med = 0.0
        s = s.fillna(med)
    return s.astype(float)


def encoded_to_group_mapping(pre: Step2Preprocessor) -> Dict[str, List[int]]:
    encoded_names = list(pre.output_feature_names)
    mapping: Dict[str, List[int]] = {}
    for feature in FEATURES:
        idx: List[int] = []
        if feature in encoded_names:
            idx.append(encoded_names.index(feature))
        if feature in NOMINAL_FEATURES:
            prefix = feature + "_"
            idx.extend([i for i, name in enumerate(encoded_names) if name.startswith(prefix)])
        idx = sorted(set(idx))
        if idx:
            mapping[feature] = idx
    return mapping


def compute_grouped_shap(
    pre: Step2Preprocessor,
    model: LGBMClassifier,
    X_raw: pd.DataFrame,
    y_true: np.ndarray,
    p_calibrated: np.ndarray,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, List[str]]]:
    """Compute native LightGBM TreeSHAP and aggregate one-hot variables back to raw features."""
    n = len(X_raw)
    # Explain the complete held-out test set; no row cap or sampling is applied.
    explain_idx = np.arange(n)

    X_explain_raw = X_raw.iloc[explain_idx].reset_index(drop=True)
    y_explain = np.asarray(y_true)[explain_idx]
    p_explain = np.asarray(p_calibrated)[explain_idx]
    X_explain_enc = pre.transform(X_explain_raw)
    encoded_names = list(pre.output_feature_names)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_explain_enc)
    if isinstance(shap_values, list):
        shap_values = shap_values[-1]
    shap_values = np.asarray(shap_values)
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, -1]

    mapping_idx = encoded_to_group_mapping(pre)
    grouped_values = {}
    grouped_feature_data = {}
    mapping_names: Dict[str, List[str]] = {}

    for raw_feature, idx in mapping_idx.items():
        grouped_values[pretty_feature_name(raw_feature)] = shap_values[:, idx].sum(axis=1)
        grouped_feature_data[pretty_feature_name(raw_feature)] = raw_feature_numeric_values(X_explain_raw, raw_feature).values
        mapping_names[raw_feature] = [encoded_names[i] for i in idx]

    grouped_shap_df = pd.DataFrame(grouped_values)
    grouped_data_df = pd.DataFrame(grouped_feature_data)

    importance = []
    total_abs = float(np.abs(grouped_shap_df.values).mean(axis=0).sum())
    for raw_feature in mapping_idx.keys():
        display = pretty_feature_name(raw_feature)
        mean_abs = float(np.abs(grouped_shap_df[display]).mean())
        importance.append({
            "raw_feature": raw_feature,
            "display_feature": display,
            "mean_abs_SHAP": mean_abs,
            "percent_of_grouped_SHAP": 100.0 * mean_abs / total_abs if total_abs > 0 else np.nan,
            "feature_type": (
                "Continuous" if raw_feature in CONTINUOUS_FEATURES else
                "Binary" if raw_feature in BINARY_FEATURES else
                "Ordinal" if raw_feature in ORDINAL_FEATURES else
                "Nominal" if raw_feature in NOMINAL_FEATURES else "Unknown"
            ),
            "n_encoded_columns_grouped": len(mapping_idx[raw_feature]),
        })
    importance_df = pd.DataFrame(importance).sort_values("mean_abs_SHAP", ascending=False).reset_index(drop=True)

    grouped_shap_df.insert(0, "__row_id__", explain_idx)
    grouped_shap_df.insert(1, "__y_true__", y_explain)
    grouped_shap_df.insert(2, "__p_calibrated__", p_explain)

    return grouped_shap_df, grouped_data_df, importance_df, mapping_names


def save_grouped_shap_beeswarm(grouped_shap_df: pd.DataFrame, grouped_data_df: pd.DataFrame, importance_df: pd.DataFrame) -> str:
    ordered_displays = importance_df["display_feature"].tolist()
    shap_matrix = grouped_shap_df[ordered_displays]
    data_matrix = grouped_data_df[ordered_displays]

    plt.figure(figsize=(10.5, 9.0))
    shap.summary_plot(
        shap_matrix.values,
        features=data_matrix,
        feature_names=ordered_displays,
        max_display=SHAP_BEESWARM_MAX_DISPLAY,
        show=False,
        plot_size=None,
    )
    plt.title("Step 2 ODI LightGBM: grouped SHAP beeswarm", fontsize=15, fontweight="bold")
    path = os.path.join(PLOT_DIR, "SHAP_beeswarm_GROUPED_all_features_best_model.png")
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    return path


def save_grouped_shap_bar(importance_df: pd.DataFrame) -> str:
    plot_df = importance_df.head(SHAP_BAR_MAX_DISPLAY).copy().iloc[::-1]
    fig_h = max(7.0, len(plot_df) * 0.35)
    fig, ax = plt.subplots(figsize=(10.5, fig_h))
    bars = ax.barh(plot_df["display_feature"], plot_df["mean_abs_SHAP"])
    ax.set_xlabel("Mean absolute SHAP value")
    ax.set_title("Step 2 ODI LightGBM: grouped SHAP importance", fontsize=15, fontweight="bold")
    max_x = float(plot_df["mean_abs_SHAP"].max()) if len(plot_df) else 1.0
    ax.set_xlim(0, max_x * 1.28)
    for bar, val, pct in zip(bars, plot_df["mean_abs_SHAP"], plot_df["percent_of_grouped_SHAP"]):
        ax.text(
            bar.get_width() + max_x * 0.015,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.4f} ({pct:.1f}%)",
            va="center",
            fontsize=9,
        )
    fig.tight_layout()
    path = os.path.join(PLOT_DIR, "SHAP_bar_GROUPED_all_features_best_model.png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return path


def build_binned_curve(x: np.ndarray, shap_y: np.ndarray, pred: np.ndarray, max_bins: int = 35) -> pd.DataFrame:
    d = pd.DataFrame({"x": x, "shap": shap_y, "pred": pred}).replace([np.inf, -np.inf], np.nan).dropna()
    if d.empty:
        return pd.DataFrame(columns=["x", "shap_mean", "pred_mean", "n"])
    unique_x = np.sort(d["x"].unique())
    if len(unique_x) <= min(max_bins, 20):
        curve = d.groupby("x", as_index=False).agg(shap_mean=("shap", "mean"), pred_mean=("pred", "mean"), n=("x", "size"))
    else:
        q = min(max_bins, max(8, int(np.sqrt(len(d)))))
        d["bin"] = pd.qcut(d["x"].rank(method="first"), q=q, duplicates="drop")
        curve = d.groupby("bin", observed=True).agg(x=("x", "mean"), shap_mean=("shap", "mean"), pred_mean=("pred", "mean"), n=("x", "size")).reset_index(drop=True)
    curve = curve.sort_values("x").reset_index(drop=True)
    return curve


def find_shap_threshold(curve: pd.DataFrame) -> Tuple[float, str]:
    if curve.empty:
        return np.nan, "unavailable"
    xs = curve["x"].astype(float).to_numpy()
    ys = curve["shap_mean"].astype(float).to_numpy()
    if len(xs) == 1:
        return float(xs[0]), "single_unique_value"
    crossings = []
    for i in range(len(xs) - 1):
        y1, y2 = ys[i], ys[i + 1]
        if not (np.isfinite(y1) and np.isfinite(y2)):
            continue
        if y1 == 0:
            crossings.append((abs(y2 - y1), xs[i]))
        elif y1 * y2 < 0:
            # Linear interpolation for zero crossing.
            t = abs(y1) / (abs(y1) + abs(y2))
            x_cross = xs[i] + t * (xs[i + 1] - xs[i])
            crossings.append((abs(y2 - y1), x_cross))
    if crossings:
        # Use the most pronounced zero-crossing.
        crossings = sorted(crossings, key=lambda z: z[0], reverse=True)
        return float(crossings[0][1]), "zero_crossing"
    idx = int(np.nanargmin(np.abs(ys)))
    return float(xs[idx]), "closest_to_zero_SHAP"


def save_threshold_plot_for_feature(
    raw_feature: str,
    display_feature: str,
    grouped_shap_df: pd.DataFrame,
    X_raw_all: pd.DataFrame,
    p_calibrated_all: np.ndarray,
) -> Dict[str, Any]:
    row_ids = grouped_shap_df["__row_id__"].astype(int).to_numpy()
    X_raw = X_raw_all.iloc[row_ids].reset_index(drop=True)
    p = np.asarray(p_calibrated_all)[row_ids]
    x = raw_feature_numeric_values(X_raw, raw_feature).to_numpy(dtype=float)
    shap_y = grouped_shap_df[display_feature].to_numpy(dtype=float)
    curve = build_binned_curve(x, shap_y, p, max_bins=SHAP_THRESHOLD_MAX_BINS)
    threshold, method = find_shap_threshold(curve)

    below = x < threshold
    above = x >= threshold
    risk_below = float(np.mean(p[below])) if np.any(below) else np.nan
    risk_above = float(np.mean(p[above])) if np.any(above) else np.nan
    abs_increase = risk_above - risk_below if np.isfinite(risk_below) and np.isfinite(risk_above) else np.nan
    rel_increase = (abs_increase / risk_below) if np.isfinite(abs_increase) and risk_below > 0 else np.nan

    fig, ax1 = plt.subplots(figsize=(8.5, 5.2))
    ax1.scatter(x, shap_y, s=9, alpha=0.12, color="gray", edgecolors="none")
    if not curve.empty:
        xs = curve["x"].to_numpy(dtype=float)
        ys = curve["shap_mean"].to_numpy(dtype=float)
        ax1.plot(xs, ys, color="black", linewidth=2.0)
        # Pink = risk-increasing/reoperation direction; blue = risk-decreasing/protective direction.
        ax1.fill_between(xs, 0, ys, where=ys >= 0, color=SHAP_BEESWARM_PINK, alpha=0.55, interpolate=True)
        ax1.fill_between(xs, 0, ys, where=ys < 0, color=SHAP_BEESWARM_BLUE, alpha=0.55, interpolate=True)
    ax1.axhline(0, color="black", linewidth=0.8, alpha=0.7)
    if np.isfinite(threshold):
        ax1.axvline(threshold, color="#1f77b4", linestyle=":", linewidth=1.6)
        ax1.text(threshold, ax1.get_ylim()[0], f"{threshold:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax1.set_xlabel(display_feature, fontweight="bold")
    ax1.set_ylabel(f"SHAP value for {display_feature}")
    ax1.set_title(f"SHAP threshold analysis: {display_feature}", fontsize=13, fontweight="bold")

    ax2 = ax1.twinx()
    if not curve.empty:
        ax2.plot(curve["x"], curve["pred_mean"] * 100, color=SHAP_BEESWARM_PINK, linestyle="--", linewidth=1.8, alpha=0.90)
    ax2.set_ylabel("Predicted risk of reoperation (%)")

    txt = (
        f"Threshold = {threshold:.2f}\n"
        f"Predicted risk < threshold = {risk_below * 100:.2f}%\n"
        f"Predicted risk ≥ threshold = {risk_above * 100:.2f}%\n"
        f"Absolute increase = {abs_increase * 100:.2f} points\n"
        f"Relative increase = {rel_increase * 100:.1f}%\n"
        f"Method = {method}"
    )
    ax1.text(
        0.02, 0.97, txt,
        transform=ax1.transAxes,
        ha="left", va="top", fontsize=8.5,
        bbox=dict(boxstyle="round,pad=0.30", facecolor="white", alpha=0.90, edgecolor="gray"),
    )
    fig.tight_layout()
    safe_name = raw_feature.replace("/", "_").replace(" ", "_")
    path = os.path.join(PLOT_DIR, f"SHAP_threshold_{safe_name}.png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    return {
        "raw_feature": raw_feature,
        "display_feature": display_feature,
        "threshold": threshold,
        "threshold_method": method,
        "risk_below_threshold": risk_below,
        "risk_at_or_above_threshold": risk_above,
        "absolute_risk_increase": abs_increase,
        "relative_risk_increase": rel_increase,
        "n_below_threshold": int(np.sum(below)),
        "n_at_or_above_threshold": int(np.sum(above)),
        "plot_path": path,
    }


def run_grouped_shap_and_threshold_analysis(
    pre: Step2Preprocessor,
    model: LGBMClassifier,
    calibrator: IsotonicRegression,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
    p_test_calibrated: np.ndarray,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, List[str]], List[str]]:
    print("\nRunning native LightGBM TreeSHAP for grouped features...")
    grouped_shap_df, grouped_data_df, importance_df, mapping_names = compute_grouped_shap(
        pre=pre,
        model=model,
        X_raw=X_test,
        y_true=y_test,
        p_calibrated=p_test_calibrated,
    )
    paths = []
    grouped_shap_df.to_csv(os.path.join(OUTPUT_DIR, "grouped_shap_values_test_best_model.csv"), index=False)
    grouped_data_df.to_csv(os.path.join(OUTPUT_DIR, "grouped_shap_feature_values_test_best_model.csv"), index=False)
    importance_df.to_csv(os.path.join(OUTPUT_DIR, "grouped_shap_importance_best_model.csv"), index=False)
    with open(os.path.join(OUTPUT_DIR, "grouped_shap_feature_mapping.json"), "w") as f:
        json.dump(json_native(mapping_names), f, indent=2, sort_keys=True)

    paths.append(save_grouped_shap_beeswarm(grouped_shap_df, grouped_data_df, importance_df))
    paths.append(save_grouped_shap_bar(importance_df))

    numeric_raw = set(CONTINUOUS_FEATURES + BINARY_FEATURES + ORDINAL_FEATURES)
    threshold_candidates = importance_df[importance_df["raw_feature"].isin(numeric_raw)].head(SHAP_THRESHOLD_TOP_N_NUMERIC)
    threshold_rows = []
    for _, r in threshold_candidates.iterrows():
        threshold_rows.append(save_threshold_plot_for_feature(
            raw_feature=r["raw_feature"],
            display_feature=r["display_feature"],
            grouped_shap_df=grouped_shap_df,
            X_raw_all=X_test,
            p_calibrated_all=p_test_calibrated,
        ))
    threshold_df = pd.DataFrame(threshold_rows)
    threshold_df.to_csv(os.path.join(OUTPUT_DIR, "SHAP_thresholds_top20_numeric_features.csv"), index=False)
    paths.extend(threshold_df["plot_path"].dropna().tolist() if not threshold_df.empty else [])

    return importance_df, threshold_df, mapping_names, paths


def add_dynamic_odi_features(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Derive Step 2 dynamic ODI variables from raw preoperative/postoperative ODI values.

    Derived variables
    -----------------
    PROM_change_rate:
        (postoperative ODI - preoperative ODI) / days between ODI assessments.
        This is intentionally a per-day rate and is not multiplied by 30.
        Negative values indicate ODI improvement; positive values indicate worsening.

    Relative ODI MCID:
        (preoperative ODI - postoperative ODI) / preoperative ODI >= 0.30.
        This variable is set to missing when baseline ODI is missing; when baseline ODI is 0,
        it is coded as 0 because a 30% relative reduction is not attainable.

    The input input column ODI_MCID_binary, if present, is not used as a model
    feature. It is retained only for audit purposes.
    """
    out = df.copy()
    required = ["preop_ODI", "postop_ODI", DAYS_BETWEEN_PROM_COL]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(
            "Cannot derive dynamic ODI features because required columns are missing: "
            + ", ".join(missing)
        )

    preop = pd.to_numeric(out["preop_ODI"].map(clean_scalar), errors="coerce")
    postop = pd.to_numeric(out["postop_ODI"].map(clean_scalar), errors="coerce")
    days_between = pd.to_numeric(out[DAYS_BETWEEN_PROM_COL].map(clean_scalar), errors="coerce")

    odi_change = postop - preop
    odi_improvement = preop - postop

    valid_rate = preop.notna() & postop.notna() & days_between.gt(0)
    prom_change_rate = pd.Series(np.nan, index=out.index, dtype="float")
    prom_change_rate.loc[valid_rate] = (odi_change.loc[valid_rate] / days_between.loc[valid_rate]).astype(float)

    valid_relative = preop.notna() & postop.notna() & preop.gt(0)
    zero_baseline_with_postop = preop.eq(0) & postop.notna()
    relative_fraction = odi_improvement / preop.replace(0, np.nan)
    relative_mcid = pd.Series(np.nan, index=out.index, dtype="float")
    relative_mcid.loc[valid_relative] = (
        relative_fraction.loc[valid_relative] >= RELATIVE_ODI_MCID_CUTOFF
    ).astype(float)
    # A 30% relative reduction is not attainable when baseline ODI is 0; code as not meeting relative MCID.
    relative_mcid.loc[zero_baseline_with_postop] = 0.0

    # Use formula-derived values to guarantee consistency across datasets.
    out["ODI_change"] = odi_change
    out[PROM_CHANGE_RATE_COL] = prom_change_rate
    out[RELATIVE_ODI_MCID_COL] = relative_mcid
    out["ODI_improvement_points_for_MCID_audit"] = odi_improvement
    out["ODI_relative_improvement_fraction_for_MCID_audit"] = relative_fraction

    audit_rows = [
        {"item": "PROM change rate definition", "value": "(postop_ODI - preop_ODI) / days_between_PROMs"},
        {"item": "Relative ODI MCID definition", "value": f"(preop_ODI - postop_ODI) / preop_ODI >= {RELATIVE_ODI_MCID_CUTOFF}; preop_ODI=0 coded as 0 when postop_ODI is available"},
        {"item": "Rows", "value": int(len(out))},
        {"item": "Rows with valid preop/postop ODI and positive days between PROMs", "value": int(valid_rate.sum())},
        {"item": "Rows with calculable PROM change rate", "value": int(prom_change_rate.notna().sum())},
        {"item": "Rows with valid preop > 0 and postop ODI for relative MCID", "value": int(valid_relative.sum())},
        {"item": "Rows with preop ODI = 0 and postop ODI available coded as Relative ODI MCID = 0", "value": int(zero_baseline_with_postop.sum())},
        {"item": "Rows with Relative ODI MCID = 1", "value": int((relative_mcid == 1).sum())},
        {"item": "Rows with Relative ODI MCID = 0", "value": int((relative_mcid == 0).sum())},
    ]

    if INPUT_ODI_MCID_COL in out.columns:
        input = out[INPUT_ODI_MCID_COL].map(to_binary_target)
        audit_rows.extend([
            {"item": "Input ODI_MCID_binary column present", "value": True},
            {"item": "Input ODI_MCID_binary used as model feature", "value": False},
            {"item": "Input ODI_MCID_binary note", "value": "Retained only for audit; not included in FEATURES."},
            {"item": "Rows with non-missing input ODI_MCID_binary", "value": int(input.notna().sum())},
            {"item": "Rows with input ODI_MCID_binary = 1", "value": int((input == 1).sum())},
        ])
    else:
        audit_rows.append({"item": "Input ODI_MCID_binary column present", "value": False})

    return out, pd.DataFrame(audit_rows)


def methods_rationale_table() -> pd.DataFrame:
    return pd.DataFrame([
        {"item": "Primary model", "rationale": "LightGBM was tuned from scratch for the Step 2 ODI task using patient-level group-aware train/calibration/test splitting."},
        {"item": "Patient leakage control", "rationale": "PersonKey defines groups; no PersonKey overlap is allowed across train, calibration, or test."},
        {"item": "Hyperparameter tuning", "rationale": "All LightGBM and class-weight hyperparameters are selected only by StratifiedGroupKFold CV inside the training split."},
        {"item": "Early stopping", "rationale": "Early stopping is used only within CV folds to estimate boosting length; the selected model is refit on the full training split with the locked median best iteration."},
        {"item": "Primary tuning metric", "rationale": "Average Precision is used because the positive event rate is low and PR performance is more informative than accuracy."},
        {"item": "Class imbalance", "rationale": "Positive sample weight equals the natural negative/positive ratio multiplied by a tuned multiplier; the multiplier is not hand-picked."},
        {"item": "Calibration", "rationale": "Isotonic calibration is fitted only on the calibration split and never on the test split; rare-event calibration plots use quantile bins with zoomed axes and report Brier score and ECE."},
        {"item": "Threshold selection", "rationale": "The classification threshold is selected only on the calibration split using the pre-specified max-F1 rule."},
        {"item": "Final evaluation", "rationale": "The test set is isolated until the model-development pipeline is locked."},
        {"item": "Dynamic ODI features", "rationale": "PROM change rate is derived as (postop_ODI - preop_ODI) / days_between_PROMs without multiplying by 30; Relative ODI MCID is derived as at least 30% improvement from baseline ODI. The input ODI_MCID_binary column is audited but not used as a model feature."},
        {"item": "SHAP", "rationale": "Native LightGBM TreeSHAP is computed only after model selection, and one-hot categorical variables are grouped back to their raw clinical feature."},
        {"item": "SHAP thresholds", "rationale": "Threshold plots use grouped SHAP zero-crossing when available; red shading indicates risk-increasing SHAP regions and green indicates risk-decreasing regions."},
        {"item": "Learning curve", "rationale": "A group-aware learning curve using Average Precision is generated inside the training split for the locked best configuration and is not used for model selection."},
        {"item": "Generalization-gap audit", "rationale": "Training, cross-validation, calibration, and test performance are reported for the selected model. These diagnostics are not used for test-set model selection."},
        {"item": "Reproducibility", "rationale": "Exact best parameters, split assignments, model artifact, calibrator, selected threshold, and package versions are exported."},
    ])

# ============================================================
# 10) Main
# ============================================================


# ============================================================
# 11) Paired Step 2 model comparison helpers
# ============================================================

ROOT_OUTPUT_DIR = OUTPUT_DIR
ROOT_PLOT_DIR = PLOT_DIR

BASELINE_ONLY_FEATURES = list(BASE_FEATURES)
DYNAMIC_PROM_FEATURES = list(BASE_FEATURES) + list(STEP2_ODI_FEATURES)

BASELINE_CONTINUOUS_FEATURES = [f for f in CONTINUOUS_FEATURES if f in BASELINE_ONLY_FEATURES]
BASELINE_BINARY_FEATURES = [f for f in BINARY_FEATURES if f in BASELINE_ONLY_FEATURES]
BASELINE_ORDINAL_FEATURES = [f for f in ORDINAL_FEATURES if f in BASELINE_ONLY_FEATURES]
BASELINE_NOMINAL_FEATURES = [f for f in NOMINAL_FEATURES if f in BASELINE_ONLY_FEATURES]

DYNAMIC_CONTINUOUS_FEATURES = list(CONTINUOUS_FEATURES)
DYNAMIC_BINARY_FEATURES = list(BINARY_FEATURES)
DYNAMIC_ORDINAL_FEATURES = list(ORDINAL_FEATURES)
DYNAMIC_NOMINAL_FEATURES = list(NOMINAL_FEATURES)


def activate_feature_set(model_type: str) -> List[str]:
    """Set global feature lists used by the shared preprocessing/modeling functions."""
    global FEATURES, CONTINUOUS_FEATURES, BINARY_FEATURES, ORDINAL_FEATURES, NOMINAL_FEATURES
    if model_type == "baseline_only":
        FEATURES = list(BASELINE_ONLY_FEATURES)
        CONTINUOUS_FEATURES = list(BASELINE_CONTINUOUS_FEATURES)
        BINARY_FEATURES = list(BASELINE_BINARY_FEATURES)
        ORDINAL_FEATURES = list(BASELINE_ORDINAL_FEATURES)
        NOMINAL_FEATURES = list(BASELINE_NOMINAL_FEATURES)
    elif model_type == "dynamic_PROM_expanded":
        FEATURES = list(DYNAMIC_PROM_FEATURES)
        CONTINUOUS_FEATURES = list(DYNAMIC_CONTINUOUS_FEATURES)
        BINARY_FEATURES = list(DYNAMIC_BINARY_FEATURES)
        ORDINAL_FEATURES = list(DYNAMIC_ORDINAL_FEATURES)
        NOMINAL_FEATURES = list(DYNAMIC_NOMINAL_FEATURES)
    else:
        raise ValueError(f"Unknown model_type: {model_type}")
    return FEATURES


def paired_patient_level_delta_bootstrap_ci(
    y_true: np.ndarray,
    p_baseline: np.ndarray,
    p_expanded: np.ndarray,
    groups: np.ndarray,
    metric_name: str,
    n_bootstraps: int = N_BOOTSTRAPS,
    seed: int = RANDOM_STATE,
) -> Tuple[float, float, float, float]:
    y_true = np.asarray(y_true).astype(int)
    p_baseline = np.asarray(p_baseline).astype(float)
    p_expanded = np.asarray(p_expanded).astype(float)
    groups = np.asarray(groups)

    if metric_name == "AP":
        observed = safe_average_precision(y_true, p_expanded) - safe_average_precision(y_true, p_baseline)
    elif metric_name == "ROC_AUC":
        observed = safe_roc_auc(y_true, p_expanded) - safe_roc_auc(y_true, p_baseline)
    else:
        raise ValueError(f"Unsupported paired delta metric: {metric_name}")

    d = pd.DataFrame({"row_id": np.arange(len(y_true)), "group": groups, "y": y_true})
    group_y = d.groupby("group")["y"].max()
    pos_groups = group_y[group_y == 1].index.to_numpy()
    neg_groups = group_y[group_y == 0].index.to_numpy()
    if len(pos_groups) == 0 or len(neg_groups) == 0:
        return float(observed), np.nan, np.nan, np.nan

    rows_by_group = {g: d.loc[d["group"].eq(g), "row_id"].to_numpy() for g in group_y.index}
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(n_bootstraps):
        sampled_pos = rng.choice(pos_groups, size=len(pos_groups), replace=True)
        sampled_neg = rng.choice(neg_groups, size=len(neg_groups), replace=True)
        sampled_groups = np.concatenate([sampled_pos, sampled_neg])
        idx = np.concatenate([rows_by_group[g] for g in sampled_groups])
        yy = y_true[idx]
        if len(np.unique(yy)) < 2:
            continue
        if metric_name == "AP":
            values.append(average_precision_score(yy, p_expanded[idx]) - average_precision_score(yy, p_baseline[idx]))
        else:
            values.append(roc_auc_score(yy, p_expanded[idx]) - roc_auc_score(yy, p_baseline[idx]))
    if not values:
        return float(observed), np.nan, np.nan, np.nan
    values = np.asarray(values, dtype=float)
    lo, hi = np.percentile(values, [2.5, 97.5])
    p_value = 2 * min(np.mean(values <= 0), np.mean(values >= 0))
    return float(observed), float(lo), float(hi), float(min(max(p_value, 0.0), 1.0))


def run_one_step2_model(
    model_type: str,
    work: pd.DataFrame,
    train: pd.DataFrame,
    cal: pd.DataFrame,
    test: pd.DataFrame,
    seed: int,
) -> Dict[str, Any]:
    """Tune, refit, calibrate, evaluate, and export one Step 2 model."""
    global OUTPUT_DIR, PLOT_DIR
    features = activate_feature_set(model_type)
    model_label = "Baseline-only" if model_type == "baseline_only" else "Dynamic PROM-expanded"
    model_dir = os.path.join(ROOT_OUTPUT_DIR, model_type)
    model_plot_dir = os.path.join(model_dir, "plots")
    model_artifact_dir = os.path.join(model_dir, "model_artifacts")
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(model_plot_dir, exist_ok=True)
    os.makedirs(model_artifact_dir, exist_ok=True)

    X_train = train[features].copy()
    y_train = train[TARGET_COL].to_numpy().astype(int)
    groups_train = train[GROUP_COL].to_numpy()
    X_cal = cal[features].copy()
    y_cal = cal[TARGET_COL].to_numpy().astype(int)
    X_test = test[features].copy()
    y_test = test[TARGET_COL].to_numpy().astype(int)
    groups_test = test[GROUP_COL].to_numpy()

    print("\n" + "=" * 100)
    print(f"Step 2 model: {model_label} ({model_type})")
    print(f"Features: {len(features)}")
    print("=" * 100)

    candidates, fold_metrics = tune_hyperparameters(
        X_train=X_train,
        y_train=y_train,
        groups_train=groups_train,
        seed=seed,
    )
    candidates.insert(0, "model_type", model_type)
    candidates.insert(1, "model_label", model_label)
    fold_metrics.insert(0, "model_type", model_type)
    fold_metrics.insert(1, "model_label", model_label)

    selected_cfg = candidates.iloc[0].copy()
    params = {k: selected_cfg[k] for k in LGBM_SEARCH_SPACE.keys()}
    params = sanitize_lgbm_params(params)
    locked_n_estimators = int(selected_cfg.get("locked_n_estimators_from_cv", params["n_estimators"]))
    params_for_refit = dict(params)
    params_for_refit["n_estimators"] = locked_n_estimators

    print("\nSelected configuration")
    print("Selection rule:", PRIMARY_MODEL_SELECTION_RULE)
    print(json.dumps(json_native(params_for_refit), indent=2, sort_keys=True))

    pre, model, _ = fit_model_pipeline(
        X_train,
        y_train,
        params=params,
        seed=seed + 10001,
        override_n_estimators=locked_n_estimators,
    )

    p_train_raw = predict_pipeline(pre, model, X_train)
    p_cal_raw = predict_pipeline(pre, model, X_cal)
    p_test_raw = predict_pipeline(pre, model, X_test)

    calibrator = IsotonicRegression(out_of_bounds="clip")
    calibrator.fit(p_cal_raw, y_cal)
    p_train = calibrator.transform(p_train_raw)
    p_cal = calibrator.transform(p_cal_raw)
    p_test = calibrator.transform(p_test_raw)

    selected_threshold, cal_thr_metrics = select_threshold(
        y_cal,
        p_cal,
        strategy=THRESHOLD_STRATEGY,
        fixed_threshold=FIXED_THRESHOLD,
        top_percent=THRESHOLD_TOP_PERCENT,
    )

    train_eval = evaluate_predictions(y_train, p_train, threshold=None, prefix="Train_")
    cal_eval = evaluate_predictions(y_cal, p_cal, threshold=selected_threshold, prefix="Calibration_")
    test_eval = evaluate_predictions(y_test, p_test, threshold=selected_threshold, prefix="Test_")

    ap_lo, ap_hi = patient_level_stratified_bootstrap_ci(
        y_test, p_test, groups_test, metric_name="AP", n_bootstraps=N_BOOTSTRAPS, seed=seed + 11,
    )
    auc_lo, auc_hi = patient_level_stratified_bootstrap_ci(
        y_test, p_test, groups_test, metric_name="ROC_AUC", n_bootstraps=N_BOOTSTRAPS, seed=seed + 13,
    )
    f1_lo, f1_hi = patient_level_stratified_bootstrap_ci(
        y_test, p_test, groups_test, metric_name="F1", threshold=selected_threshold, n_bootstraps=N_BOOTSTRAPS, seed=seed + 17,
    )

    pos_sample_weight_train = actual_positive_weight(y_train, multiplier=float(params["positive_weight_multiplier"]))

    summary = {
        "model_type": model_type,
        "model_label": model_label,
        "Model_selection_status": "Selected model",
        "Selection_rule": PRIMARY_MODEL_SELECTION_RULE,
        "candidate_id": int(selected_cfg["candidate_id"]),
        "Target": TARGET_COL,
        "Feature_count": len(features),
        "Train_N": int(len(y_train)),
        "Train_events": int(np.sum(y_train)),
        "Train_prevalence": float(np.mean(y_train)),
        "Calibration_N": int(len(y_cal)),
        "Calibration_events": int(np.sum(y_cal)),
        "Calibration_prevalence": float(np.mean(y_cal)),
        "Test_N": int(len(y_test)),
        "Test_events": int(np.sum(y_test)),
        "Test_prevalence": float(np.mean(y_test)),
        "Train_AP": train_eval["Train_AP"],
        "Train_ROC_AUC": train_eval["Train_ROC_AUC"],
        "Cross_validation_train_AP_mean": float(selected_cfg.get("cv_train_AP_mean", np.nan)),
        "Cross_validation_train_ROC_AUC_mean": float(selected_cfg.get("cv_train_ROC_AUC_mean", np.nan)),
        "Cross_validation_AP_mean": float(selected_cfg["cv_AP_mean"]),
        "Cross_validation_AP_SD": float(selected_cfg["cv_AP_SD"]),
        "Cross_validation_ROC_AUC_mean": float(selected_cfg["cv_ROC_AUC_mean"]),
        "Cross_validation_ROC_AUC_SD": float(selected_cfg["cv_ROC_AUC_SD"]),
        "Train_minus_CV_AP_gap": train_eval["Train_AP"] - float(selected_cfg["cv_AP_mean"]),
        "Train_minus_CV_ROC_AUC_gap": train_eval["Train_ROC_AUC"] - float(selected_cfg["cv_ROC_AUC_mean"]),
        "CV_minus_Test_AP_gap": float(selected_cfg["cv_AP_mean"]) - test_eval["Test_AP"],
        "CV_minus_Test_ROC_AUC_gap": float(selected_cfg["cv_ROC_AUC_mean"]) - test_eval["Test_ROC_AUC"],
        "CV_internal_train_minus_validation_AP_gap": float(selected_cfg.get("cv_train_minus_validation_AP_gap", np.nan)),
        "CV_internal_train_minus_validation_ROC_AUC_gap": float(selected_cfg.get("cv_train_minus_validation_ROC_AUC_gap", np.nan)),
        "locked_n_estimators_from_cv": locked_n_estimators,
        "Calibration_AP": cal_eval["Calibration_AP"],
        "Calibration_ROC_AUC": cal_eval["Calibration_ROC_AUC"],
        "Test_AP": test_eval["Test_AP"],
        "Test_AP_95CI_low_patient_bootstrap": ap_lo,
        "Test_AP_95CI_high_patient_bootstrap": ap_hi,
        "Test_ROC_AUC": test_eval["Test_ROC_AUC"],
        "Test_ROC_AUC_95CI_low_patient_bootstrap": auc_lo,
        "Test_ROC_AUC_95CI_high_patient_bootstrap": auc_hi,
        "Test_Brier_score": test_eval["Test_Brier_score"],
        "Test_ECE": test_eval["Test_ECE"],
        "Selected_threshold": selected_threshold,
        "Threshold_strategy": THRESHOLD_STRATEGY,
        "Positive_weight_multiplier": float(params["positive_weight_multiplier"]),
        "Positive_sample_weight_train": pos_sample_weight_train,
        "Test_F1": test_eval["Test_F1"],
        "Test_F1_95CI_low_patient_bootstrap": f1_lo,
        "Test_F1_95CI_high_patient_bootstrap": f1_hi,
        "Test_Sensitivity": test_eval["Test_Sensitivity"],
        "Test_Specificity": test_eval["Test_Specificity"],
        "Test_PPV": test_eval["Test_PPV"],
        "Test_NPV": test_eval["Test_NPV"],
        "Test_TP": test_eval["Test_TP"],
        "Test_FP": test_eval["Test_FP"],
        "Test_TN": test_eval["Test_TN"],
        "Test_FN": test_eval["Test_FN"],
        "Test_Top_5pct_event_rate": test_eval["Test_Top_5pct_event_rate"],
        "Test_Top_5pct_lift": test_eval["Test_Top_5pct_lift"],
        "Test_Top_5pct_captured_events": test_eval["Test_Top_5pct_captured_events"],
        "Best_params_JSON": json.dumps(json_native(params_for_refit), sort_keys=True),
    }

    calibration_row = {"model_type": model_type, "model_label": model_label, "candidate_id": int(selected_cfg["candidate_id"]), **cal_thr_metrics}

    pred_df = pd.DataFrame({
        "PersonKey": test[GROUP_COL].values,
        "row_index_in_test_split": np.arange(len(test)),
        "y_true": y_test,
        "p_test_calibrated": p_test,
        "p_test_raw": p_test_raw,
        "predicted_positive": (p_test >= selected_threshold).astype(int),
    })
    pred_df.to_csv(os.path.join(model_dir, f"test_predictions_{model_type}.csv"), index=False)

    save_pr_curve(y_test, p_test, os.path.join(model_plot_dir, f"PR_curve_{model_type}.png"), f"Step 2 LightGBM {model_label}: test set")
    save_roc_curve(y_test, p_test, os.path.join(model_plot_dir, f"ROC_curve_{model_type}.png"), f"Step 2 LightGBM {model_label}: test set")
    save_calibration_plot(y_test, p_test, os.path.join(model_plot_dir, f"Calibration_{model_type}.png"), f"Step 2 LightGBM {model_label}: test set")

    print(f"\nRunning group-aware AP learning curve for {model_label}...")
    save_learning_curve_grouped_ap(
        X_train=X_train,
        y_train=y_train,
        groups_train=groups_train,
        best_params=params_for_refit,
        path_png=os.path.join(model_plot_dir, f"Learning_curve_{model_type}.png"),
        path_csv=os.path.join(model_dir, f"learning_curve_{model_type}.csv"),
        title=f"Step 2 LightGBM {model_label}: group-aware learning curve",
    )

    artifact_path = os.path.join(model_artifact_dir, f"{model_type}_lightgbm_pipeline_artifact.joblib")
    joblib.dump({
        "model_type": model_type,
        "model_label": model_label,
        "preprocessor": pre,
        "model": model,
        "calibrator": calibrator,
        "selected_threshold": float(selected_threshold),
        "params": params_for_refit,
        "locked_n_estimators_from_cv": locked_n_estimators,
        "features": features,
        "target_col": TARGET_COL,
        "group_col": GROUP_COL,
        "random_state": seed,
    }, artifact_path)

    with open(os.path.join(model_artifact_dir, f"{model_type}_exact_settings.json"), "w") as f:
        json.dump(json_native({
            "model_type": model_type,
            "model_label": model_label,
            "selection_rule": PRIMARY_MODEL_SELECTION_RULE,
            "candidate_id": int(selected_cfg["candidate_id"]),
            "params": params_for_refit,
            "locked_n_estimators_from_cv": locked_n_estimators,
            "selected_threshold": float(selected_threshold),
            "features": features,
            "prom_change_rate_definition": "(postop_ODI - preop_ODI) / days_between_PROMs",
            "relative_mcid_definition": f"(preop_ODI - postop_ODI) / preop_ODI >= {RELATIVE_ODI_MCID_CUTOFF}; preop_ODI=0 coded as 0 when postop_ODI is available",
            "search_space": LGBM_SEARCH_SPACE,
            "n_random_combinations": N_RANDOM_COMBINATIONS,
            "n_cv_folds": N_CV_FOLDS,
            "use_early_stopping_in_cv": USE_EARLY_STOPPING_IN_CV,
            "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
            "threshold_strategy": THRESHOLD_STRATEGY,
            "calibration_method": CALIBRATION_METHOD,
        }), f, indent=2, sort_keys=True)

    # SHAP outputs are created for both models to mirror Step 1. The dynamic model
    # remains the primary interpretation model.
    old_out, old_plot = OUTPUT_DIR, PLOT_DIR
    try:
        OUTPUT_DIR = model_dir
        PLOT_DIR = model_plot_dir
        shap_importance_df, shap_threshold_df, shap_mapping, shap_plot_paths = run_grouped_shap_and_threshold_analysis(
            pre=pre,
            model=model,
            calibrator=calibrator,
            X_test=X_test,
            y_test=y_test,
            p_test_calibrated=p_test,
        )
        shap_importance_df.insert(0, "model_type", model_type)
        shap_importance_df.insert(1, "model_label", model_label)
        shap_threshold_df.insert(0, "model_type", model_type)
        shap_threshold_df.insert(1, "model_label", model_label)
    finally:
        OUTPUT_DIR, PLOT_DIR = old_out, old_plot

    print(
        f"{model_label} FINAL | CV AP={selected_cfg['cv_AP_mean']:.5f} ± {selected_cfg['cv_AP_SD']:.5f} | "
        f"Test AP={test_eval['Test_AP']:.5f} [{ap_lo:.5f}, {ap_hi:.5f}] | "
        f"Test AUC={test_eval['Test_ROC_AUC']:.5f} [{auc_lo:.5f}, {auc_hi:.5f}] | "
        f"threshold={selected_threshold:.6f}"
    )

    return {
        "model_type": model_type,
        "model_label": model_label,
        "features": features,
        "summary": summary,
        "calibration": calibration_row,
        "predictions": pred_df,
        "candidates": candidates,
        "fold_metrics": fold_metrics,
        "shap_importance": shap_importance_df,
        "shap_thresholds": shap_threshold_df,
        "artifact_path": artifact_path,
        "p_test": p_test,
        "p_test_raw": p_test_raw,
        "y_test": y_test,
        "groups_test": groups_test,
    }


def main() -> None:
    start = time.time()
    if not os.path.exists(INPUT_CSV):
        raise FileNotFoundError(f"Input file not found: {INPUT_CSV}")

    os.makedirs(ROOT_OUTPUT_DIR, exist_ok=True)
    os.makedirs(ROOT_PLOT_DIR, exist_ok=True)

    df = pd.read_csv(INPUT_CSV, low_memory=False)
    df.columns = [str(c).strip() for c in df.columns]

    print("Input:", INPUT_CSV)
    print("Input shape:", df.shape)

    df, dynamic_feature_audit = add_dynamic_odi_features(df)
    print("\nDynamic PROM feature derivation audit")
    print(dynamic_feature_audit.to_string(index=False))

    all_required_features = sorted(set(DYNAMIC_PROM_FEATURES))
    required_cols = all_required_features + [TARGET_COL, GROUP_COL]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError("Missing required columns:\n" + "\n".join(f" - {c}" for c in missing_cols))

    # The Step 2 cohort requires eligible preoperative and early postoperative PROM
    # values and a positive time interval between assessments. Relative MCID may be
    # missing when baseline PROM is zero and is handled by training-only imputation.
    preop = pd.to_numeric(df["preop_ODI"].map(clean_scalar), errors="coerce")
    postop = pd.to_numeric(df["postop_ODI"].map(clean_scalar), errors="coerce")
    days_between = pd.to_numeric(df[DAYS_BETWEEN_PROM_COL].map(clean_scalar), errors="coerce")
    eligible_prom_mask = preop.notna() & postop.notna() & days_between.gt(0)

    keep_extra_cols = [c for c in ["InstitutionName", "InstitutionNPI1"] if c in df.columns]
    work = df.loc[eligible_prom_mask, required_cols + keep_extra_cols].copy()
    work[TARGET_COL] = work[TARGET_COL].map(to_binary_target)
    before_target = len(work)
    work = work.dropna(subset=[TARGET_COL]).copy()
    work[TARGET_COL] = work[TARGET_COL].astype(int)
    dropped_target = before_target - len(work)

    if work[GROUP_COL].isna().any():
        raise ValueError(f"{int(work[GROUP_COL].isna().sum())} rows have missing {GROUP_COL}.")

    print(f"Rows after PROM eligibility filtering: {int(eligible_prom_mask.sum()):,}; dropped ineligible PROM rows: {len(df) - int(eligible_prom_mask.sum()):,}")
    print(f"Rows after target cleaning: {len(work):,}; dropped target missing: {dropped_target:,}")
    print(f"Events: {int(work[TARGET_COL].sum()):,}; prevalence: {work[TARGET_COL].mean():.5f}")

    train_mask, cal_mask, test_mask = patient_level_train_cal_test_split(
        work,
        target_col=TARGET_COL,
        group_col=GROUP_COL,
        test_fraction=TEST_FRACTION,
        calibration_fraction_of_remaining=CALIBRATION_FRACTION_OF_REMAINING,
        seed=RANDOM_STATE,
    )
    work["split"] = np.where(train_mask, "train", np.where(cal_mask, "calibration", "test"))

    train = work[work["split"] == "train"].reset_index(drop=True)
    cal = work[work["split"] == "calibration"].reset_index(drop=True)
    test = work[work["split"] == "test"].reset_index(drop=True)

    train_groups = set(train[GROUP_COL])
    cal_groups = set(cal[GROUP_COL])
    test_groups = set(test[GROUP_COL])
    assert train_groups.isdisjoint(cal_groups)
    assert train_groups.isdisjoint(test_groups)
    assert cal_groups.isdisjoint(test_groups)

    split_audit = build_split_audit(work, "split", TARGET_COL)
    institution_audit = build_institution_audit(work, "split")
    split_assignment = work[[GROUP_COL, "split", TARGET_COL]].drop_duplicates().sort_values([GROUP_COL, "split"])

    print("\nSPLIT AUDIT")
    print(split_audit.to_string(index=False))

    baseline_result = run_one_step2_model(
        model_type="baseline_only",
        work=work,
        train=train,
        cal=cal,
        test=test,
        seed=RANDOM_STATE,
    )
    dynamic_result = run_one_step2_model(
        model_type="dynamic_PROM_expanded",
        work=work,
        train=train,
        cal=cal,
        test=test,
        seed=RANDOM_STATE,
    )

    # Paired incremental value of dynamic PROM information on the same test patients.
    y_test = baseline_result["y_test"]
    groups_test = baseline_result["groups_test"]
    p_baseline = baseline_result["p_test"]
    p_dynamic = dynamic_result["p_test"]

    delta_ap = paired_patient_level_delta_bootstrap_ci(
        y_test, p_baseline, p_dynamic, groups_test, metric_name="AP", n_bootstraps=N_BOOTSTRAPS, seed=RANDOM_STATE + 3001,
    )
    delta_auc = paired_patient_level_delta_bootstrap_ci(
        y_test, p_baseline, p_dynamic, groups_test, metric_name="ROC_AUC", n_bootstraps=N_BOOTSTRAPS, seed=RANDOM_STATE + 3003,
    )

    paired_comparison = pd.DataFrame([{
        "comparison": "dynamic_PROM_expanded_minus_baseline_only",
        "baseline_candidate_id": baseline_result["summary"]["candidate_id"],
        "dynamic_candidate_id": dynamic_result["summary"]["candidate_id"],
        "baseline_Test_AP": baseline_result["summary"]["Test_AP"],
        "dynamic_Test_AP": dynamic_result["summary"]["Test_AP"],
        "Delta_AP_dynamic_minus_baseline": delta_ap[0],
        "Delta_AP_95CI_low_patient_bootstrap": delta_ap[1],
        "Delta_AP_95CI_high_patient_bootstrap": delta_ap[2],
        "Delta_AP_bootstrap_p_value": delta_ap[3],
        "baseline_Test_ROC_AUC": baseline_result["summary"]["Test_ROC_AUC"],
        "dynamic_Test_ROC_AUC": dynamic_result["summary"]["Test_ROC_AUC"],
        "Delta_ROC_AUC_dynamic_minus_baseline": delta_auc[0],
        "Delta_ROC_AUC_95CI_low_patient_bootstrap": delta_auc[1],
        "Delta_ROC_AUC_95CI_high_patient_bootstrap": delta_auc[2],
        "Delta_ROC_AUC_bootstrap_p_value": delta_auc[3],
    }])

    summary_df = pd.DataFrame([baseline_result["summary"], dynamic_result["summary"]])
    calibration_df = pd.DataFrame([baseline_result["calibration"], dynamic_result["calibration"]])
    candidates_df = pd.concat([baseline_result["candidates"], dynamic_result["candidates"]], ignore_index=True)
    folds_df = pd.concat([baseline_result["fold_metrics"], dynamic_result["fold_metrics"]], ignore_index=True)
    shap_importance_df = pd.concat([baseline_result["shap_importance"], dynamic_result["shap_importance"]], ignore_index=True)
    shap_threshold_df = pd.concat([baseline_result["shap_thresholds"], dynamic_result["shap_thresholds"]], ignore_index=True)

    features_rows = []
    for model_type, feature_list in [("baseline_only", BASELINE_ONLY_FEATURES), ("dynamic_PROM_expanded", DYNAMIC_PROM_FEATURES)]:
        if model_type == "baseline_only":
            cont, binf, ordf, nomf = BASELINE_CONTINUOUS_FEATURES, BASELINE_BINARY_FEATURES, BASELINE_ORDINAL_FEATURES, BASELINE_NOMINAL_FEATURES
        else:
            cont, binf, ordf, nomf = DYNAMIC_CONTINUOUS_FEATURES, DYNAMIC_BINARY_FEATURES, DYNAMIC_ORDINAL_FEATURES, DYNAMIC_NOMINAL_FEATURES
        for f in feature_list:
            features_rows.append({
                "model_type": model_type,
                "Feature": f,
                "Display_name": pretty_feature_name(f),
                "Feature_group": "Dynamic_PROM" if f in STEP2_ODI_FEATURES else "Baseline",
                "Feature_type": "Continuous" if f in cont else "Binary" if f in binf else "Ordinal" if f in ordf else "Nominal" if f in nomf else "Unknown",
            })
    features_df = pd.DataFrame(features_rows)

    config_df = pd.DataFrame([
        {"Parameter": "INPUT_CSV", "Value": INPUT_CSV},
        {"Parameter": "TARGET_COL", "Value": TARGET_COL},
        {"Parameter": "GROUP_COL", "Value": GROUP_COL},
        {"Parameter": "RANDOM_STATE", "Value": RANDOM_STATE},
        {"Parameter": "TEST_FRACTION", "Value": TEST_FRACTION},
        {"Parameter": "CALIBRATION_FRACTION_OF_REMAINING", "Value": CALIBRATION_FRACTION_OF_REMAINING},
        {"Parameter": "N_CV_FOLDS", "Value": N_CV_FOLDS},
        {"Parameter": "N_RANDOM_COMBINATIONS", "Value": N_RANDOM_COMBINATIONS},
        {"Parameter": "N_BOOTSTRAPS", "Value": N_BOOTSTRAPS},
        {"Parameter": "USE_EARLY_STOPPING_IN_CV", "Value": USE_EARLY_STOPPING_IN_CV},
        {"Parameter": "EARLY_STOPPING_ROUNDS", "Value": EARLY_STOPPING_ROUNDS},
        {"Parameter": "THRESHOLD_STRATEGY", "Value": THRESHOLD_STRATEGY},
        {"Parameter": "CALIBRATION_METHOD", "Value": CALIBRATION_METHOD},
        {"Parameter": "PROM_CHANGE_RATE_DEFINITION", "Value": "(postop_ODI - preop_ODI) / days_between_PROMs"},
        {"Parameter": "RELATIVE_MCID_DEFINITION", "Value": f"(preop_ODI - postop_ODI) / preop_ODI >= {RELATIVE_ODI_MCID_CUTOFF}"},
        {"Parameter": "lightgbm_version", "Value": lgb.__version__},
        {"Parameter": "sklearn_version", "Value": __import__("sklearn").__version__},
        {"Parameter": "shap_version", "Value": shap.__version__},
        {"Parameter": "python_version", "Value": platform.python_version()},
    ])

    summary_xlsx = os.path.join(ROOT_OUTPUT_DIR, "Step2_DynamicPROM_LightGBM_summary.xlsx")
    with pd.ExcelWriter(summary_xlsx, engine="openpyxl") as writer:
        methods_rationale_table().to_excel(writer, sheet_name="methods_rationale", index=False)
        dynamic_feature_audit.to_excel(writer, sheet_name="dynamic_PROM_audit", index=False)
        summary_df.to_excel(writer, sheet_name="model_performance", index=False)
        paired_comparison.to_excel(writer, sheet_name="paired_comparison", index=False)
        candidates_df.to_excel(writer, sheet_name="cv_candidates_all_models", index=False)
        folds_df.to_excel(writer, sheet_name="cv_fold_metrics_all", index=False)
        calibration_df.to_excel(writer, sheet_name="cal_thresholds", index=False)
        features_df.to_excel(writer, sheet_name="features", index=False)
        split_audit.to_excel(writer, sheet_name="split_audit", index=False)
        institution_audit.to_excel(writer, sheet_name="institution_audit", index=False)
        split_assignment.to_excel(writer, sheet_name="split_assignment", index=False)
        config_df.to_excel(writer, sheet_name="run_config", index=False)
        shap_importance_df.to_excel(writer, sheet_name="grouped_SHAP_importance", index=False)
        shap_threshold_df.to_excel(writer, sheet_name="SHAP_thresholds", index=False)
        style_excel_workbook(writer)

    summary_df.to_csv(os.path.join(ROOT_OUTPUT_DIR, "model_performance.csv"), index=False)
    paired_comparison.to_csv(os.path.join(ROOT_OUTPUT_DIR, "paired_dynamic_PROM_comparison.csv"), index=False)
    candidates_df.to_csv(os.path.join(ROOT_OUTPUT_DIR, "cv_candidates_all_models.csv"), index=False)
    folds_df.to_csv(os.path.join(ROOT_OUTPUT_DIR, "cv_fold_metrics_all.csv"), index=False)
    shap_importance_df.to_csv(os.path.join(ROOT_OUTPUT_DIR, "SHAP_importance_all_models.csv"), index=False)
    shap_threshold_df.to_csv(os.path.join(ROOT_OUTPUT_DIR, "SHAP_thresholds_all_models.csv"), index=False)

    run_manifest = {
        "input_csv": INPUT_CSV,
        "output_dir": ROOT_OUTPUT_DIR,
        "target_col": TARGET_COL,
        "group_col": GROUP_COL,
        "design": "Step 2 paired comparison: baseline-only model vs dynamic PROM-expanded model on the same cohort and same patient-level split.",
        "baseline_features": BASELINE_ONLY_FEATURES,
        "dynamic_PROM_features": DYNAMIC_PROM_FEATURES,
        "prom_change_rate_definition": "(postop_ODI - preop_ODI) / days_between_PROMs",
        "relative_mcid_definition": f"(preop_ODI - postop_ODI) / preop_ODI >= {RELATIVE_ODI_MCID_CUTOFF}; preop_ODI=0 coded as 0 when postop_ODI is available",
        "n_random_combinations_per_model": N_RANDOM_COMBINATIONS,
        "n_bootstraps": N_BOOTSTRAPS,
        "use_early_stopping_in_cv": USE_EARLY_STOPPING_IN_CV,
        "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
        "test_fraction": TEST_FRACTION,
        "calibration_fraction_of_remaining": CALIBRATION_FRACTION_OF_REMAINING,
        "threshold_strategy": THRESHOLD_STRATEGY,
        "calibration_method": CALIBRATION_METHOD,
        "primary_model_selection_rule": PRIMARY_MODEL_SELECTION_RULE,
        "runtime_minutes": float((time.time() - start) / 60),
        "summary_xlsx": summary_xlsx,
        "paired_comparison_csv": os.path.join(ROOT_OUTPUT_DIR, "paired_dynamic_PROM_comparison.csv"),
        "baseline_artifact": baseline_result["artifact_path"],
        "dynamic_artifact": dynamic_result["artifact_path"],
        "python_version": platform.python_version(),
        "lightgbm_version": lgb.__version__,
        "sklearn_version": __import__("sklearn").__version__,
        "shap_version": shap.__version__,
    }
    with open(os.path.join(ROOT_OUTPUT_DIR, "run_manifest.json"), "w") as f:
        json.dump(json_native(run_manifest), f, indent=2, sort_keys=True)

    zip_path = os.path.join(ROOT_OUTPUT_DIR, "Step2_DynamicPROM_LightGBM_outputs.zip")
    tmp_zip_path = zip_path + ".tmp"
    with open(os.path.join(ROOT_OUTPUT_DIR, "DOWNLOAD_INSTRUCTIONS.txt"), "w") as f:
        f.write("All Step 2 paired LightGBM outputs were generated successfully.\n")
        f.write(f"ZIP archive: {zip_path}\n")
        f.write("If automatic Colab download is slow or stalls, download this ZIP manually from the Colab Files panel.\n")

    for pth in [zip_path, tmp_zip_path]:
        if os.path.exists(pth):
            os.remove(pth)
    try:
        zf = zipfile.ZipFile(tmp_zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=ZIP_COMPRESSION_LEVEL)
    except TypeError:
        zf = zipfile.ZipFile(tmp_zip_path, "w", compression=zipfile.ZIP_DEFLATED)
    n_zipped_files = 0
    with zf as z:
        for root, _, file_names in os.walk(ROOT_OUTPUT_DIR):
            for name in file_names:
                full_path = os.path.join(root, name)
                if full_path in {zip_path, tmp_zip_path}:
                    continue
                rel_path = os.path.relpath(full_path, ROOT_OUTPUT_DIR)
                z.write(full_path, arcname=rel_path)
                n_zipped_files += 1
    os.replace(tmp_zip_path, zip_path)
    zip_size_mb = os.path.getsize(zip_path) / (1024 ** 2)

    print("\n" + "=" * 100)
    print("STEP 2 PAIRED DYNAMIC PROM LIGHTGBM ANALYSIS COMPLETED")
    print("Main Excel:", summary_xlsx)
    print("Model performance:", os.path.join(ROOT_OUTPUT_DIR, "model_performance.csv"))
    print("Paired comparison:", os.path.join(ROOT_OUTPUT_DIR, "paired_dynamic_PROM_comparison.csv"))
    print("Run manifest:", os.path.join(ROOT_OUTPUT_DIR, "run_manifest.json"))
    print(f"ZIP: {zip_path}")
    print(f"ZIP size: {zip_size_mb:.2f} MB; files included: {n_zipped_files}")
    print("=" * 100)

    if CREATE_COLAB_DOWNLOAD_LINK:
        try:
            from IPython.display import HTML, display
            href = "/files" + zip_path
            display(HTML(
                f'<p><b>Step 2 paired LightGBM output archive is ready.</b></p>'
                f'<p><a href="{href}" download>Click here to download the ZIP archive</a></p>'
                f'<p>Path: <code>{zip_path}</code></p>'
            ))
        except Exception as e:
            print("Download link display skipped:", e)

    if AUTO_DOWNLOAD_ZIP:
        try:
            from google.colab import files
            files.download(zip_path)
        except Exception as e:
            print("Automatic download skipped:", e)


if __name__ == "__main__":
    main()

#**Step 2: CatBoost**

In [ ]:
"""
Step 2 dynamic postoperative PROM analysis with CatBoost
========================================================

Input
-----
/content/Step 2_ODI_Cohort.csv

Target
------
final_reop_step2
    1 = reoperation from postoperative day 91 through day 365
    0 = no reoperation from postoperative day 91 through day 365

Design
------
This script runs paired baseline-only and dynamic PROM-expanded CatBoost
models for delayed lumbar reoperation prediction. The baseline-only model
uses the same 35 baseline variables as Step 1. The dynamic PROM-expanded
model includes the same baseline variables plus preoperative PROM,
postoperative PROM, PROM change, PROM change rate, relative MCID status,
and timing of postoperative PROM assessment. Paired models use identical
patient-level train/calibration/test splits and identical group-aware
cross-validation fold construction. Hyperparameter tuning is performed
exclusively within the training split using cross-validated average
precision as the primary selection metric. Probability calibration and
threshold selection are performed only on the calibration split. The
held-out test set is reserved until the model-development pipeline is
locked.
"""

# -*- coding: utf-8 -*-

import os
import re
import sys
import json
import math
import time
import zipfile
import platform
import subprocess
import warnings
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    import joblib
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'joblib'])
    import joblib

try:
    import openpyxl  # noqa
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl'])
    import openpyxl  # noqa

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedGroupKFold, ParameterSampler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_recall_curve,
    roc_curve,
    confusion_matrix,
    f1_score,
)
from sklearn.isotonic import IsotonicRegression
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')


def _ensure_catboost():
    try:
        from catboost import CatBoostClassifier
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'catboost'])
        from catboost import CatBoostClassifier
    return CatBoostClassifier


BASE_DIR = '/content'
INPUT_CSV = os.path.join(BASE_DIR, 'Step 2_ODI_Cohort.csv')
CLASSIFIER_KEY = 'CatBoost'
CLASSIFIER_DISPLAY_NAME = 'CatBoost'
SCALE_CONTINUOUS = False
OUTPUT_DIR = os.path.join(BASE_DIR, 'Step2_DynamicPROM_CatBoost_publication')
PLOT_DIR = os.path.join(OUTPUT_DIR, 'plots')
ARTIFACT_DIR = os.path.join(OUTPUT_DIR, 'model_artifacts')
for d in [OUTPUT_DIR, PLOT_DIR, ARTIFACT_DIR]:
    os.makedirs(d, exist_ok=True)

TARGET_COL = 'final_reop_step2'
GROUP_COL = 'PersonKey'
RANDOM_STATE = 20260524
TEST_FRACTION = 0.20
CALIBRATION_FRACTION_OF_REMAINING = 0.20
N_CV_FOLDS = 5
N_RANDOM_COMBINATIONS = 300
USE_EARLY_STOPPING_IN_CV = True
EARLY_STOPPING_ROUNDS = 100
MIN_FINAL_N_ESTIMATORS = 50
THRESHOLD_STRATEGY = 'max_f1'
N_BOOTSTRAPS = 2000
ECE_N_BINS = 10
N_JOBS = -1
LEARNING_CURVE_TRAIN_FRACTIONS = (0.20, 0.40, 0.60, 0.80, 1.00)
LEARNING_CURVE_CV_FOLDS = 3
ZIP_COMPRESSION_LEVEL = 1
CREATE_COLAB_DOWNLOAD_LINK = True
AUTO_DOWNLOAD_ZIP = False
RELATIVE_ODI_MCID_CUTOFF = 0.30
PROM_CHANGE_RATE_COL = 'ODI_change_rate'
RELATIVE_ODI_MCID_COL = 'ODI_relative_MCID_binary'
DAYS_BETWEEN_PROM_COL = 'days_between_PROMs'

BASELINE_FEATURES = ['finaldx_degenerative', 'finaldx_radicular', 'finaldx_stenosis', 'finaldx_deformity_instability', 'finaldx_other_diagnosis', 'age', 'sex', 'race', 'ethnicity', 'cancer_status', 'chronic_pulmonary_disease', 'congestive_heart_failure', 'connective_tissue_rheumatic_disease', 'diabetes_status', 'myocardial_infarction', 'renal_disease', 'institution_type', 'institution_size', 'institution_region', 'asa', 'bmi', 'payer_status', 'alif_llif', 'corpectomy', 'discectomy', 'foraminotomy', 'instrumentation', 'laminectomy_posterior_decompression', 'pelvic_fixation', 'plf', 'tlif_plif', 'other_lumbar_procedure', 'number_operated_levels', 'operative_region_extent', 'PatTobaccoUse']
assert len(BASELINE_FEATURES) == 35
STEP2_PROM_FEATURES = ['preop_ODI', 'postop_ODI', 'ODI_change', PROM_CHANGE_RATE_COL, RELATIVE_ODI_MCID_COL, 'postop_ODI_day']
DYNAMIC_PROM_FEATURES = BASELINE_FEATURES + STEP2_PROM_FEATURES
CONTINUOUS_BASELINE_FEATURES = ['age', 'bmi']
CONTINUOUS_FEATURES_ALL = CONTINUOUS_BASELINE_FEATURES + ['preop_ODI', 'postop_ODI', 'ODI_change', PROM_CHANGE_RATE_COL, 'postop_ODI_day']
BINARY_FEATURES = ['finaldx_degenerative', 'finaldx_radicular', 'finaldx_stenosis', 'finaldx_deformity_instability', 'finaldx_other_diagnosis', 'sex', 'ethnicity', 'cancer_status', 'chronic_pulmonary_disease', 'congestive_heart_failure', 'connective_tissue_rheumatic_disease', 'myocardial_infarction', 'renal_disease', 'institution_type', 'alif_llif', 'corpectomy', 'discectomy', 'foraminotomy', 'instrumentation', 'laminectomy_posterior_decompression', 'pelvic_fixation', 'plf', 'tlif_plif', 'other_lumbar_procedure', 'operative_region_extent', RELATIVE_ODI_MCID_COL]
ORDINAL_FEATURES = ['diabetes_status', 'institution_size', 'asa', 'number_operated_levels']
NOMINAL_FEATURES = ['race', 'institution_region', 'payer_status', 'PatTobaccoUse']

SEARCH_SPACE = {
    'iterations': [300, 500, 800, 1200, 1600, 2000],
    'learning_rate': [0.005, 0.01, 0.02, 0.03, 0.05, 0.08],
    'depth': [3, 4, 5, 6, 7, 8],
    'l2_leaf_reg': [1.0, 3.0, 5.0, 7.0, 10.0, 20.0],
    'random_strength': [0.0, 0.5, 1.0, 2.0],
    'bagging_temperature': [0.0, 0.5, 1.0, 2.0],
    'border_count': [64, 128, 254],
    'positive_weight_multiplier': [0.25, 0.50, 0.75, 1.00, 1.50, 2.00, 3.00, 4.00, 6.00, 8.00],
}
ITERATION_PARAM = 'iterations'
INT_PARAMS = {'iterations', 'depth', 'border_count'}
FLOAT_PARAMS = {'learning_rate', 'l2_leaf_reg', 'random_strength', 'bagging_temperature', 'positive_weight_multiplier'}

PARAMETER_CANDIDATES = list(ParameterSampler(SEARCH_SPACE, n_iter=N_RANDOM_COMBINATIONS, random_state=RANDOM_STATE))

MISSING_STRINGS = {'', ' ', 'na', 'n/a', 'nan', 'none', 'null', '.', 'missing', '<na>'}
BINARY_MAPS = {
    'sex': {'female': 0, 'f': 0, 'male': 1, 'm': 1},
    'ethnicity': {'non-hispanic': 0, 'non hispanic': 0, 'hispanic': 1},
    'cancer_status': {'no cancer': 0, 'no': 0, 'none': 0, 'any cancer': 1, 'yes': 1, 'cancer': 1},
    'institution_type': {'hospital': 0, 'non-hospital': 1, 'non hospital': 1, 'nonhospital': 1},
    'operative_region_extent': {'lumbar only': 0, 'extended_region_involvement': 1, 'extended region involvement': 1, 'extended': 1},
}
ORDINAL_MAPS = {
    'diabetes_status': {'no': 0, 'none': 0, '0': 0, 'without comp': 1, 'without complication': 1, 'without complications': 1, '1': 1, 'with comp': 2, 'with complication': 2, 'with complications': 2, '2': 2},
    'institution_size': {'between 1-99 beds': 0, '1-99': 0, 'between 100-399 beds': 1, '100-399': 1, '>= 400 beds': 2, '>=400 beds': 2, '>=400': 2, '>= 400': 2},
    'asa': {'1': 1, 'i': 1, '2': 2, 'ii': 2, '3': 3, 'iii': 3, '4': 4, 'iv': 4, '>=4': 4, '>= 4': 4, '5': 4, 'v': 4},
    'number_operated_levels': {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '>=4': 4, '>= 4': 4, '5': 4, '6': 4, '7': 4, '8': 4, '9': 4, '10': 4},
}
PREFERRED_NOMINAL_LEVELS = {
    'race': ['White', 'Black', 'Other'],
    'institution_region': ['South', 'North East', 'West', 'Midwest'],
    'payer_status': ['Medicare', 'Commercial/Private', 'Other', 'Medicaid/Public/Government'],
    'PatTobaccoUse': ['Unknown/Not reported/Multiple', 'Never', 'Former', 'Current'],
}
FEATURE_LABELS = {
    'age': 'Age', 'bmi': 'BMI', 'sex': 'Male sex', 'race': 'Race', 'ethnicity': 'Hispanic ethnicity',
    'cancer_status': 'Any cancer', 'diabetes_status': 'Diabetes status', 'institution_type': 'Non-hospital institution',
    'institution_size': 'Institution size', 'institution_region': 'Institution region', 'asa': 'ASA physical status',
    'payer_status': 'Primary payer', 'PatTobaccoUse': 'Tobacco use',
    'finaldx_degenerative': 'Degenerative diagnosis', 'finaldx_radicular': 'Radiculopathy diagnosis',
    'finaldx_stenosis': 'Spinal stenosis diagnosis', 'finaldx_deformity_instability': 'Deformity or instability diagnosis',
    'finaldx_other_diagnosis': 'Other lumbar diagnosis', 'chronic_pulmonary_disease': 'Chronic pulmonary disease',
    'congestive_heart_failure': 'Congestive heart failure', 'connective_tissue_rheumatic_disease': 'Connective tissue/rheumatic disease',
    'myocardial_infarction': 'Myocardial infarction', 'renal_disease': 'Renal disease',
    'alif_llif': 'Anterior/lateral lumbar interbody fusion', 'corpectomy': 'Corpectomy', 'discectomy': 'Discectomy',
    'foraminotomy': 'Foraminotomy', 'instrumentation': 'Instrumentation',
    'laminectomy_posterior_decompression': 'Posterior decompression or laminectomy', 'pelvic_fixation': 'Pelvic fixation',
    'plf': 'Posterolateral fusion', 'tlif_plif': 'Posterior/transforaminal lumbar interbody fusion',
    'other_lumbar_procedure': 'Other lumbar procedure', 'number_operated_levels': 'Number of operated levels',
    'operative_region_extent': 'Operative region extent',
}
FEATURE_LABELS.update({'preop_ODI': 'Preoperative ODI', 'postop_ODI': 'Postoperative ODI', 'ODI_change': 'Change in ODI', PROM_CHANGE_RATE_COL: 'ODI change rate', RELATIVE_ODI_MCID_COL: 'Relative ODI MCID', 'postop_ODI_day': 'Timing of postoperative ODI assessment'})

def clean_scalar(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().replace('≥', '>=')
        return np.nan if s.lower() in MISSING_STRINGS else s
    return x

def norm_text(x):
    x = clean_scalar(x)
    if pd.isna(x):
        return None
    return str(x).strip().replace('≥', '>=').lower()

def to_binary_target(x):
    sx = norm_text(x)
    if sx is None:
        return np.nan
    if sx in {'1', '1.0', 'yes', 'y', 'true', 't'}:
        return 1.0
    if sx in {'0', '0.0', 'no', 'n', 'false', 'f'}:
        return 0.0
    try:
        v = float(sx)
        return float(v) if v in (0.0, 1.0) else np.nan
    except Exception:
        return np.nan

def safe_filename(x):
    return re.sub(r'_+', '_', re.sub(r'[^A-Za-z0-9_.-]+', '_', str(x))).strip('_')[:180] or 'file'

def json_native(obj):
    if isinstance(obj, dict):
        return {str(k): json_native(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_native(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj

def sanitize_params(params):
    out = {}
    for k, v in params.items():
        if k in INT_PARAMS:
            out[k] = int(round(float(v)))
        elif k in FLOAT_PARAMS:
            out[k] = float(v)
        else:
            out[k] = v
    return out

def safe_average_precision(y, p):
    return np.nan if len(np.unique(y)) < 2 else float(average_precision_score(y, p))

def safe_roc_auc(y, p):
    return np.nan if len(np.unique(y)) < 2 else float(roc_auc_score(y, p))

def expected_calibration_error(y, p, n_bins=10):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    out = 0.0
    if len(y) == 0:
        return np.nan
    for i in range(n_bins):
        mask = (p >= bins[i]) & ((p <= bins[i + 1]) if i == n_bins - 1 else (p < bins[i + 1]))
        if np.any(mask):
            out += (np.sum(mask) / len(y)) * abs(float(np.mean(y[mask])) - float(np.mean(p[mask])))
    return float(out)

def classification_metrics(y, p, threshold):
    y = np.asarray(y).astype(int)
    pred = (np.asarray(p) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        'threshold': float(threshold),
        'F1': float(f1_score(y, pred, zero_division=0)),
        'Sensitivity': tp / (tp + fn) if tp + fn > 0 else np.nan,
        'Specificity': tn / (tn + fp) if tn + fp > 0 else np.nan,
        'PPV': tp / (tp + fp) if tp + fp > 0 else np.nan,
        'NPV': tn / (tn + fn) if tn + fn > 0 else np.nan,
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
        'Predicted_positive_rate': float(np.mean(pred)),
    }

def top5_metrics(y, p):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    n = len(y)
    k = max(1, int(math.ceil(n * 0.05)))
    idx = np.argsort(-p)[:k]
    prev = float(np.mean(y)) if n else np.nan
    rate = float(np.mean(y[idx])) if n else np.nan
    return {
        'Top_5pct_n': int(k),
        'Top_5pct_event_rate': rate,
        'Top_5pct_lift': rate / prev if prev > 0 else np.nan,
        'Top_5pct_captured_events': float(np.sum(y[idx]) / np.sum(y)) if np.sum(y) > 0 else np.nan,
    }

def select_threshold(y, p):
    precision, recall, thresholds = precision_recall_curve(y, p)
    if len(thresholds) == 0:
        threshold = 0.5
    else:
        precision = precision[:-1]
        recall = recall[:-1]
        f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
        threshold = float(thresholds[int(np.nanargmax(f1))])
    return threshold, classification_metrics(y, p, threshold)

def eval_preds(y, p, threshold=None, prefix=''):
    out = {
        f'{prefix}AP': safe_average_precision(y, p),
        f'{prefix}ROC_AUC': safe_roc_auc(y, p),
        f'{prefix}Brier_score': float(brier_score_loss(y, p)),
        f'{prefix}ECE': expected_calibration_error(y, p, ECE_N_BINS),
        f'{prefix}N': int(len(y)),
        f'{prefix}Events': int(np.sum(y)),
        f'{prefix}Prevalence': float(np.mean(y)),
    }
    if threshold is not None:
        out.update({f'{prefix}{k}': v for k, v in classification_metrics(y, p, threshold).items()})
        out.update({f'{prefix}{k}': v for k, v in top5_metrics(y, p).items()})
    return out

def make_weights(y, multiplier):
    y = np.asarray(y).astype(int)
    n_pos = int(np.sum(y == 1))
    n_neg = int(np.sum(y == 0))
    if n_pos == 0:
        raise ValueError('No positive events in training fold.')
    pos_weight = (n_neg / max(n_pos, 1)) * float(multiplier)
    return np.where(y == 1, pos_weight, 1.0).astype(float)

def actual_positive_weight(y, multiplier):
    y = np.asarray(y).astype(int)
    return float((np.sum(y == 0) / max(np.sum(y == 1), 1)) * multiplier)

@dataclass
class ModelPreprocessor:
    continuous_features: List[str]
    binary_features: List[str]
    ordinal_features: List[str]
    nominal_features: List[str]
    preferred_nominal_levels: Dict[str, List[str]]
    scale_continuous: bool = False

    def __post_init__(self):
        self.continuous_imputer = None
        self.discrete_imputer = None
        self.nominal_imputer = None
        self.scaler = None
        self.onehot = None
        self.output_feature_names_ = []

    def _parse_binary(self, x, feature):
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in BINARY_MAPS and sx in BINARY_MAPS[feature]:
            return float(BINARY_MAPS[feature][sx])
        if sx in {'1', '1.0', 'yes', 'y', 'true', 't', 'present', 'positive'}:
            return 1.0
        if sx in {'0', '0.0', 'no', 'n', 'false', 'f', 'absent', 'negative'}:
            return 0.0
        try:
            v = float(sx)
            return float(v) if v in (0.0, 1.0) else np.nan
        except Exception:
            return np.nan

    def _parse_ordinal(self, x, feature):
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in ORDINAL_MAPS and sx in ORDINAL_MAPS[feature]:
            return float(ORDINAL_MAPS[feature][sx])
        try:
            v = float(sx)
            if feature == 'asa':
                return float(min(max(int(round(v)), 1), 4))
            if feature == 'number_operated_levels':
                return float(min(max(int(round(v)), 0), 4))
            if feature == 'diabetes_status':
                return float(min(max(int(round(v)), 0), 2))
            if feature == 'institution_size':
                return float(min(max(int(round(v)), 0), 2))
            return float(v)
        except Exception:
            return np.nan

    def _nominal(self, feature, x):
        x = clean_scalar(x)
        if pd.isna(x):
            return np.nan
        s = str(x).strip()
        sl = s.lower()
        if feature == 'race':
            return 'White' if sl == 'white' else ('Black' if sl == 'black' else 'Other')
        if feature == 'institution_region':
            mapping = {'south': 'South', 'north east': 'North East', 'northeast': 'North East', 'north-east': 'North East', 'west': 'West', 'midwest': 'Midwest', 'mid west': 'Midwest'}
            return mapping.get(sl, s)
        if feature == 'payer_status':
            if sl == 'medicare':
                return 'Medicare'
            if sl in {'commercial/private', 'commercial', 'private', 'commercial private'}:
                return 'Commercial/Private'
            if sl in {'medicaid/public/government', 'medicaid', 'public', 'government', 'public/government'}:
                return 'Medicaid/Public/Government'
            return 'Other'
        if feature == 'PatTobaccoUse':
            return 'Never' if sl == 'never' else ('Former' if sl == 'former' else ('Current' if sl == 'current' else 'Unknown/Not reported/Multiple'))
        return s

    def _parts(self, X):
        cont = pd.DataFrame(index=X.index)
        disc = pd.DataFrame(index=X.index)
        nom = pd.DataFrame(index=X.index)
        for c in self.continuous_features:
            cont[c] = pd.to_numeric(X[c].map(clean_scalar), errors='coerce')
        for c in self.binary_features:
            disc[c] = X[c].map(lambda z: self._parse_binary(z, c)).astype(float)
        for c in self.ordinal_features:
            disc[c] = X[c].map(lambda z: self._parse_ordinal(z, c)).astype(float)
        for c in self.nominal_features:
            nom[c] = X[c].map(lambda z: self._nominal(c, z)).astype('object')
        return cont, disc, nom

    def fit(self, X):
        cont, disc, nom = self._parts(X)
        self.continuous_imputer = SimpleImputer(strategy='median')
        self.discrete_imputer = SimpleImputer(strategy='most_frequent')
        self.nominal_imputer = SimpleImputer(strategy='constant', fill_value='Missing')
        cont_imp = self.continuous_imputer.fit_transform(cont)
        if self.scale_continuous:
            self.scaler = StandardScaler()
            self.scaler.fit(cont_imp)
        self.discrete_imputer.fit(disc)
        nom_imp = pd.DataFrame(self.nominal_imputer.fit_transform(nom), columns=self.nominal_features)
        cats = []
        for c in self.nominal_features:
            preferred = list(self.preferred_nominal_levels.get(c, []))
            observed = nom_imp[c].astype(str).unique().tolist()
            cats.append(preferred + sorted([x for x in observed if x not in preferred]))
        try:
            self.onehot = OneHotEncoder(categories=cats, handle_unknown='ignore', sparse_output=False)
        except TypeError:
            self.onehot = OneHotEncoder(categories=cats, handle_unknown='ignore', sparse=False)
        self.onehot.fit(nom_imp.astype(str))
        self.output_feature_names_ = self.continuous_features + self.binary_features + self.ordinal_features + self.onehot.get_feature_names_out(self.nominal_features).tolist()
        return self

    def transform(self, X):
        cont, disc, nom = self._parts(X)
        cont_imp = self.continuous_imputer.transform(cont)
        if self.scale_continuous and self.scaler is not None:
            cont_imp = self.scaler.transform(cont_imp)
        disc_imp = self.discrete_imputer.transform(disc)
        nom_imp = pd.DataFrame(self.nominal_imputer.transform(nom), columns=self.nominal_features)
        nom_oh = self.onehot.transform(nom_imp.astype(str))
        return np.concatenate([cont_imp, disc_imp, nom_oh], axis=1).astype(float)

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

    @property
    def output_feature_names(self):
        return self.output_feature_names_

def raw_model_scores(model, Xp, n_iter=None):
    if CLASSIFIER_KEY == 'HistGradientBoosting' and n_iter is not None:
        selected = None
        for i, pred in enumerate(model.staged_predict_proba(Xp), start=1):
            selected = pred
            if i >= int(n_iter):
                break
        if selected is not None:
            return selected[:, 1]
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(Xp)[:, 1]
    if hasattr(model, 'decision_function'):
        return model.decision_function(Xp)
    raise AttributeError('Model provides neither predict_proba nor decision_function.')

def predict(pre, model, X, features, n_iter=None):
    return raw_model_scores(model, pre.transform(X[features]), n_iter=n_iter)

def patient_split(df, target_col, seed):
    group_df = df.groupby(GROUP_COL, dropna=False)[target_col].max().reset_index()
    y_group = group_df[target_col].astype(int).to_numpy()
    groups = group_df[GROUP_COL].to_numpy()
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=seed)
    train_cal_idx, test_idx = next(sss1.split(groups, y_group))
    groups_train_cal = groups[train_cal_idx]
    y_train_cal = y_group[train_cal_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=CALIBRATION_FRACTION_OF_REMAINING, random_state=seed + 1)
    train_idx_rel, cal_idx_rel = next(sss2.split(groups_train_cal, y_train_cal))
    train_groups = set(groups_train_cal[train_idx_rel])
    cal_groups = set(groups_train_cal[cal_idx_rel])
    test_groups = set(groups[test_idx])
    assert train_groups.isdisjoint(cal_groups) and train_groups.isdisjoint(test_groups) and cal_groups.isdisjoint(test_groups)
    return df[GROUP_COL].isin(train_groups).to_numpy(), df[GROUP_COL].isin(cal_groups).to_numpy(), df[GROUP_COL].isin(test_groups).to_numpy()

def cv_splits(y, groups, seed, n_folds=N_CV_FOLDS):
    group_df = pd.DataFrame({'group': groups, 'y': y}).groupby('group')['y'].max().reset_index()
    n_folds = min(n_folds, int(np.sum(group_df.y == 1)), int(np.sum(group_df.y == 0)))
    if n_folds < 2:
        raise ValueError('Not enough positive and negative patient groups for group-aware CV.')
    cv = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    return [(tr, va) for tr, va in cv.split(np.zeros(len(y)), y, groups)]

def feature_types(features):
    cont = [f for f in features if f in CONTINUOUS_FEATURES_ALL or (f not in BINARY_FEATURES and f not in ORDINAL_FEATURES and f not in NOMINAL_FEATURES)]
    binary = [f for f in features if f in BINARY_FEATURES]
    ordinal = [f for f in features if f in ORDINAL_FEATURES]
    nominal = [f for f in features if f in NOMINAL_FEATURES]
    return cont, binary, ordinal, nominal

def fit_pipeline(X, y, features, params, seed, eval_set=None, early=False, n_estimators_override=None):
    cont, binary, ordinal, nominal = feature_types(features)
    pre = ModelPreprocessor(cont, binary, ordinal, nominal, PREFERRED_NOMINAL_LEVELS, scale_continuous=SCALE_CONTINUOUS)
    Xp = pre.fit_transform(X[features])
    eval_transformed = None
    if eval_set is not None:
        X_val, y_val = eval_set
        eval_transformed = (pre.transform(X_val[features]), y_val)
    model, best_iter = fit_estimator(Xp, y, params, seed, eval_set=eval_transformed, use_early_stopping=early, n_estimators_override=n_estimators_override)
    return pre, model, best_iter

def tune(X_train, y_train, groups_train, features, folds, model_key, seed):
    candidate_rows = []
    fold_rows = []
    sampler = list(ParameterSampler(SEARCH_SPACE, n_iter=N_RANDOM_COMBINATIONS, random_state=seed))
    for cid, raw_params in enumerate(sampler, start=1):
        params = sanitize_params(raw_params)
        fold_train_aps, fold_train_aucs, fold_aps, fold_aucs, best_iters = [], [], [], [], []
        t0 = time.time()
        for fid, (tr_idx, va_idx) in enumerate(folds, start=1):
            X_tr = X_train.iloc[tr_idx].reset_index(drop=True)
            y_tr = y_train[tr_idx]
            X_va = X_train.iloc[va_idx].reset_index(drop=True)
            y_va = y_train[va_idx]
            pre, model, best_iter = fit_pipeline(X_tr, y_tr, features, params, seed + cid * 1000 + fid, eval_set=(X_va, y_va), early=USE_EARLY_STOPPING_IN_CV)
            p_tr = predict(pre, model, X_tr, features, n_iter=best_iter)
            p_va = predict(pre, model, X_va, features, n_iter=best_iter)
            tr_ap, tr_auc = safe_average_precision(y_tr, p_tr), safe_roc_auc(y_tr, p_tr)
            va_ap, va_auc = safe_average_precision(y_va, p_va), safe_roc_auc(y_va, p_va)
            fold_train_aps.append(tr_ap); fold_train_aucs.append(tr_auc); fold_aps.append(va_ap); fold_aucs.append(va_auc)
            if best_iter is not None and best_iter > 0:
                best_iters.append(best_iter)
            fold_rows.append({
                'model_key': model_key, 'candidate_id': cid, 'fold': fid,
                'fold_train_AP': tr_ap, 'fold_train_ROC_AUC': tr_auc,
                'fold_validation_AP': va_ap, 'fold_validation_ROC_AUC': va_auc,
                'fold_train_minus_validation_AP_gap': tr_ap - va_ap if np.isfinite(tr_ap) and np.isfinite(va_ap) else np.nan,
                'fold_train_minus_validation_ROC_AUC_gap': tr_auc - va_auc if np.isfinite(tr_auc) and np.isfinite(va_auc) else np.nan,
                'fold_best_iteration': best_iter,
                'positive_weight_used': actual_positive_weight(y_tr, params['positive_weight_multiplier']),
                **params,
            })
        locked = int(np.median(best_iters)) if best_iters and USE_EARLY_STOPPING_IN_CV else (int(params[ITERATION_PARAM]) if ITERATION_PARAM and ITERATION_PARAM in params else np.nan)
        row = {
            'model_key': model_key, 'candidate_id': cid, 'cv_folds': len(folds),
            'cv_train_AP_mean': float(np.nanmean(fold_train_aps)),
            'cv_train_ROC_AUC_mean': float(np.nanmean(fold_train_aucs)),
            'cv_AP_mean': float(np.nanmean(fold_aps)),
            'cv_AP_SD': float(np.nanstd(fold_aps, ddof=1)) if len(fold_aps) > 1 else np.nan,
            'cv_ROC_AUC_mean': float(np.nanmean(fold_aucs)),
            'cv_ROC_AUC_SD': float(np.nanstd(fold_aucs, ddof=1)) if len(fold_aucs) > 1 else np.nan,
            'mean_cv_best_iteration': float(np.mean(best_iters)) if best_iters else np.nan,
            'locked_n_estimators_from_cv': locked,
            'elapsed_seconds': float(time.time() - t0),
            **params,
        }
        candidate_rows.append(row)
        print(f"{model_key} | candidate {cid:03d}/{len(sampler)} | CV AP={row['cv_AP_mean']:.5f} | AUC={row['cv_ROC_AUC_mean']:.5f}")
    return pd.DataFrame(candidate_rows).sort_values('cv_AP_mean', ascending=False).reset_index(drop=True), pd.DataFrame(fold_rows)

def fit_final(X_train, y_train, X_cal, y_cal, X_test, features, params, locked_n, seed):
    n_override = int(locked_n) if pd.notna(locked_n) and USE_EARLY_STOPPING_IN_CV and ITERATION_PARAM else None
    pre, model, _ = fit_pipeline(X_train, y_train, features, params, seed, n_estimators_override=n_override)
    p_train_raw = predict(pre, model, X_train, features)
    p_cal_raw = predict(pre, model, X_cal, features)
    p_test_raw = predict(pre, model, X_test, features)
    calibrator = IsotonicRegression(out_of_bounds='clip')
    calibrator.fit(p_cal_raw, y_cal)
    p_train = calibrator.predict(p_train_raw)
    p_cal = calibrator.predict(p_cal_raw)
    p_test = calibrator.predict(p_test_raw)
    threshold, cal_metrics = select_threshold(y_cal, p_cal)
    return {
        'pre': pre, 'model': model, 'calibrator': calibrator,
        'p_train_raw': p_train_raw, 'p_cal_raw': p_cal_raw, 'p_test_raw': p_test_raw,
        'p_train': p_train, 'p_cal': p_cal, 'p_test': p_test,
        'threshold': threshold, 'cal_metrics': cal_metrics,
    }

def patient_boot_ci(y, p, groups, metric, threshold=None, seed=1):
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({'idx': np.arange(len(y)), 'group': groups, 'y': np.asarray(y).astype(int)})
    group_y = d.groupby('group').y.max()
    pos = group_y[group_y == 1].index.to_numpy()
    neg = group_y[group_y == 0].index.to_numpy()
    if len(pos) == 0 or len(neg) == 0:
        return np.nan, np.nan
    by_group = {g: d.loc[d.group.eq(g), 'idx'].to_numpy() for g in group_y.index}
    vals = []
    for _ in range(N_BOOTSTRAPS):
        sample_groups = np.concatenate([rng.choice(pos, len(pos), replace=True), rng.choice(neg, len(neg), replace=True)])
        sample_idx = np.concatenate([by_group[g] for g in sample_groups])
        yy = np.asarray(y)[sample_idx]
        pp = np.asarray(p)[sample_idx]
        if len(np.unique(yy)) < 2:
            continue
        if metric == 'AP':
            vals.append(average_precision_score(yy, pp))
        elif metric == 'ROC_AUC':
            vals.append(roc_auc_score(yy, pp))
        elif metric == 'F1':
            vals.append(f1_score(yy, pp >= threshold, zero_division=0))
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))) if vals else (np.nan, np.nan)

def paired_delta_boot(y, p0, p1, groups, metric, seed=1):
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({'idx': np.arange(len(y)), 'group': groups, 'y': np.asarray(y).astype(int)})
    group_y = d.groupby('group').y.max()
    pos = group_y[group_y == 1].index.to_numpy()
    neg = group_y[group_y == 0].index.to_numpy()
    by_group = {g: d.loc[d.group.eq(g), 'idx'].to_numpy() for g in group_y.index}
    observed = (safe_average_precision(y, p1) - safe_average_precision(y, p0)) if metric == 'AP' else (safe_roc_auc(y, p1) - safe_roc_auc(y, p0))
    vals = []
    for _ in range(N_BOOTSTRAPS):
        sample_groups = np.concatenate([rng.choice(pos, len(pos), replace=True), rng.choice(neg, len(neg), replace=True)])
        sample_idx = np.concatenate([by_group[g] for g in sample_groups])
        yy = np.asarray(y)[sample_idx]
        if len(np.unique(yy)) < 2:
            continue
        if metric == 'AP':
            vals.append(average_precision_score(yy, np.asarray(p1)[sample_idx]) - average_precision_score(yy, np.asarray(p0)[sample_idx]))
        else:
            vals.append(roc_auc_score(yy, np.asarray(p1)[sample_idx]) - roc_auc_score(yy, np.asarray(p0)[sample_idx]))
    if not vals:
        return observed, np.nan, np.nan, np.nan
    vals = np.asarray(vals)
    ci_low, ci_high = float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))
    p_value = float(2 * min(np.mean(vals <= 0), np.mean(vals >= 0)))
    return float(observed), ci_low, ci_high, min(p_value, 1.0)

def save_learning_curve(X_train, y_train, groups_train, features, params, locked_n, seed, out_path, title):
    group_df = pd.DataFrame({'group': groups_train, 'y': y_train}).groupby('group').y.max().reset_index()
    rng = np.random.default_rng(seed)
    rows = []
    for frac in LEARNING_CURVE_TRAIN_FRACTIONS:
        for rep in range(LEARNING_CURVE_CV_FOLDS):
            selected_groups = []
            for cls in [0, 1]:
                cls_groups = group_df.loc[group_df.y.eq(cls), 'group'].to_numpy()
                k = max(1, int(math.ceil(len(cls_groups) * frac)))
                selected_groups.extend(rng.choice(cls_groups, k, replace=False).tolist())
            mask = np.isin(groups_train, selected_groups)
            X_sub = X_train.loc[mask].reset_index(drop=True)
            y_sub = y_train[mask]
            g_sub = groups_train[mask]
            try:
                folds = cv_splits(y_sub, g_sub, seed + rep, n_folds=3)
            except Exception:
                continue
            for tr, va in folds:
                n_override = int(locked_n) if pd.notna(locked_n) and USE_EARLY_STOPPING_IN_CV and ITERATION_PARAM else None
                pre, model, _ = fit_pipeline(X_sub.iloc[tr].reset_index(drop=True), y_sub[tr], features, params, seed + rep, n_estimators_override=n_override)
                p_tr = predict(pre, model, X_sub.iloc[tr].reset_index(drop=True), features)
                p_va = predict(pre, model, X_sub.iloc[va].reset_index(drop=True), features)
                rows.append({'train_fraction': frac, 'train_AP': safe_average_precision(y_sub[tr], p_tr), 'validation_AP': safe_average_precision(y_sub[va], p_va)})
    lc = pd.DataFrame(rows)
    lc.to_csv(out_path.replace('.png', '.csv'), index=False)
    if not lc.empty:
        s = lc.groupby('train_fraction').agg(train_AP_mean=('train_AP', 'mean'), validation_AP_mean=('validation_AP', 'mean')).reset_index()
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.plot(s.train_fraction, s.train_AP_mean, marker='o', label='Training AP')
        ax.plot(s.train_fraction, s.validation_AP_mean, marker='o', label='Validation AP')
        ax.set_xlabel('Fraction of training patient groups')
        ax.set_ylabel('Average precision')
        ax.set_title(title)
        ax.legend()
        fig.tight_layout()
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close(fig)

def style_excel(writer):
    try:
        from openpyxl.styles import Font, PatternFill, Alignment
        for sheet in writer.sheets:
            ws = writer.sheets[sheet]
            ws.freeze_panes = 'A2'
            ws.auto_filter.ref = ws.dimensions
            for cell in ws[1]:
                cell.font = Font(bold=True)
                cell.fill = PatternFill(start_color='D9EAF7', end_color='D9EAF7', fill_type='solid')
                cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            for col in ws.columns:
                max_len = max([len(str(c.value)) for c in col if c.value is not None] + [12])
                ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 12), 70)
    except Exception:
        pass


def make_model(params, seed, n_estimators_override=None):
    CatBoostClassifier = _ensure_catboost()
    p = sanitize_params(params)
    if n_estimators_override is not None:
        p['iterations'] = int(max(MIN_FINAL_N_ESTIMATORS, n_estimators_override))
    p.pop('positive_weight_multiplier', None)
    return CatBoostClassifier(
        loss_function='Logloss',
        eval_metric='PRAUC',
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        thread_count=N_JOBS if isinstance(N_JOBS, int) else -1,
        **p,
    )

def fit_estimator(X_train, y_train, params, seed, eval_set=None, use_early_stopping=False, n_estimators_override=None):
    weights = make_weights(y_train, float(params['positive_weight_multiplier']))
    model = make_model(params, seed, n_estimators_override=n_estimators_override)
    best_iter = None
    if eval_set is not None and use_early_stopping:
        X_val, y_val = eval_set
        model.fit(X_train, y_train, sample_weight=weights, eval_set=(X_val, y_val), early_stopping_rounds=EARLY_STOPPING_ROUNDS, use_best_model=True)
        try:
            bi = model.get_best_iteration()
            if bi is not None and bi >= 0:
                best_iter = int(bi) + 1
        except Exception:
            best_iter = None
    else:
        model.fit(X_train, y_train, sample_weight=weights)
    return model, best_iter


def add_dynamic_odi_features(df):
    out = df.copy()
    required = ['preop_ODI', 'postop_ODI', DAYS_BETWEEN_PROM_COL]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError('Cannot derive dynamic ODI features because required columns are missing: ' + ', '.join(missing))
    preop = pd.to_numeric(out['preop_ODI'].map(clean_scalar), errors='coerce')
    postop = pd.to_numeric(out['postop_ODI'].map(clean_scalar), errors='coerce')
    days_between = pd.to_numeric(out[DAYS_BETWEEN_PROM_COL].map(clean_scalar), errors='coerce')
    odi_change = postop - preop
    valid_rate = preop.notna() & postop.notna() & days_between.gt(0)
    change_rate = pd.Series(np.nan, index=out.index, dtype='float')
    change_rate.loc[valid_rate] = (odi_change.loc[valid_rate] / days_between.loc[valid_rate]).astype(float)
    improvement = preop - postop
    rel_fraction = improvement / preop.replace(0, np.nan)
    valid_relative = preop.notna() & postop.notna() & preop.gt(0)
    zero_baseline_with_postop = preop.eq(0) & postop.notna()
    relative_mcid = pd.Series(np.nan, index=out.index, dtype='float')
    relative_mcid.loc[valid_relative] = (rel_fraction.loc[valid_relative] >= RELATIVE_ODI_MCID_CUTOFF).astype(float)
    relative_mcid.loc[zero_baseline_with_postop] = 0.0
    out['ODI_change'] = odi_change
    out[PROM_CHANGE_RATE_COL] = change_rate
    out[RELATIVE_ODI_MCID_COL] = relative_mcid
    audit = pd.DataFrame([
        {'item': 'PROM change rate definition', 'value': '(postop_ODI - preop_ODI) / days_between_PROMs'},
        {'item': 'Relative ODI MCID definition', 'value': f'(preop_ODI - postop_ODI) / preop_ODI >= {RELATIVE_ODI_MCID_CUTOFF}; preop_ODI=0 coded as 0 when postop_ODI is available'},
        {'item': 'Rows', 'value': int(len(out))},
        {'item': 'Rows with valid preop/postop ODI and positive days between PROMs', 'value': int(valid_rate.sum())},
        {'item': 'Rows with preop ODI = 0 and postop ODI available coded as Relative ODI MCID = 0', 'value': int(zero_baseline_with_postop.sum())},
    ])
    return out, audit

def run_model(model_type, work, features, folds, seed, all_candidates, all_folds):
    key = model_type
    out_dir = os.path.join(OUTPUT_DIR, model_type)
    plot_dir = os.path.join(PLOT_DIR, model_type)
    artifact_dir = os.path.join(ARTIFACT_DIR, model_type)
    for d in [out_dir, plot_dir, artifact_dir]:
        os.makedirs(d, exist_ok=True)
    train = work[work.split.eq('train')].reset_index(drop=True)
    cal = work[work.split.eq('calibration')].reset_index(drop=True)
    test = work[work.split.eq('test')].reset_index(drop=True)
    y_train = train[TARGET_COL].astype(int).to_numpy()
    y_cal = cal[TARGET_COL].astype(int).to_numpy()
    y_test = test[TARGET_COL].astype(int).to_numpy()
    candidates, fold_metrics = tune(train[features], y_train, train[GROUP_COL].to_numpy(), features, folds, key, seed)
    best_params = sanitize_params({k: candidates.loc[0, k] for k in SEARCH_SPACE.keys()})
    locked_n = candidates.loc[0, 'locked_n_estimators_from_cv']
    final = fit_final(train[features], y_train, cal[features], y_cal, test[features], y_test, features, best_params, locked_n, seed + 991)
    threshold = final['threshold']
    summary = {
        'model_type': model_type, 'classifier': CLASSIFIER_DISPLAY_NAME,
        'n_features_original': len(features), 'best_candidate_id': int(candidates.loc[0, 'candidate_id']),
        'CV_AP_mean': float(candidates.loc[0, 'cv_AP_mean']), 'CV_ROC_AUC_mean': float(candidates.loc[0, 'cv_ROC_AUC_mean']),
        'locked_n_estimators_from_cv': locked_n,
        **eval_preds(y_train, final['p_train'], threshold, prefix='Train_'),
        **eval_preds(y_cal, final['p_cal'], threshold, prefix='Calibration_'),
        **eval_preds(y_test, final['p_test'], threshold, prefix='Test_'),
    }
    summary['Train_minus_CV_AP_gap'] = summary['Train_AP'] - summary['CV_AP_mean'] if pd.notna(summary['CV_AP_mean']) else np.nan
    summary['CV_minus_Test_AP_gap'] = summary['CV_AP_mean'] - summary['Test_AP'] if pd.notna(summary['CV_AP_mean']) else np.nan
    summary['Test_AP_CI_lower'], summary['Test_AP_CI_upper'] = patient_boot_ci(y_test, final['p_test'], test[GROUP_COL].to_numpy(), 'AP', seed=seed + 10)
    summary['Test_ROC_AUC_CI_lower'], summary['Test_ROC_AUC_CI_upper'] = patient_boot_ci(y_test, final['p_test'], test[GROUP_COL].to_numpy(), 'ROC_AUC', seed=seed + 20)
    pd.DataFrame([summary]).to_csv(os.path.join(out_dir, f'performance_{{key}}.csv'), index=False)
    pd.DataFrame({'row_index_in_split': np.arange(len(test)), GROUP_COL: test[GROUP_COL].to_numpy(), 'y_true': y_test, 'p_raw': final['p_test_raw'], 'p_calibrated': final['p_test']}).to_csv(os.path.join(out_dir, f'test_predictions_{{key}}.csv'), index=False)
    candidates.to_csv(os.path.join(out_dir, f'cv_candidates_{{key}}.csv'), index=False)
    fold_metrics.to_csv(os.path.join(out_dir, f'cv_fold_metrics_{{key}}.csv'), index=False)
    joblib.dump(final, os.path.join(artifact_dir, f'final_model_{{key}}.joblib'))
    save_learning_curve(train[features], y_train, train[GROUP_COL].to_numpy(), features, best_params, locked_n, seed + 313, os.path.join(plot_dir, f'learning_curve_{{key}}.png'), key)
    all_candidates.append(candidates)
    all_folds.append(fold_metrics)
    return {'summary': summary, 'preds': pd.read_csv(os.path.join(out_dir, f'test_predictions_{{key}}.csv'))}

def split_audit(df):
    rows = []
    for split in ['train', 'calibration', 'test']:
        sub = df[df.split.eq(split)]
        rows.append({'split': split, 'n_rows': len(sub), 'n_patients': sub[GROUP_COL].nunique(), 'events': int(sub[TARGET_COL].sum()), 'prevalence': float(sub[TARGET_COL].mean())})
    return pd.DataFrame(rows)

def methods_table():
    return pd.DataFrame([
        {'item': 'Paired design', 'rationale': f'Baseline-only and dynamic PROM-expanded {{CLASSIFIER_DISPLAY_NAME}} models are tuned separately under the same protocol.'},
        {'item': 'Paired split/folds', 'rationale': 'Paired models use identical patient-level train/calibration/test splits and identical group-aware CV folds.'},
        {'item': 'Tuning metric', 'rationale': 'Average precision is the primary model-selection metric because delayed reoperation is a rare-event outcome.'},
        {'item': 'Class imbalance', 'rationale': 'Positive-class sample weighting is tuned through a multiplier applied to the natural negative-to-positive ratio.'},
        {'item': 'Calibration and threshold', 'rationale': 'Isotonic calibration and maximum-F1 threshold selection are performed only on the calibration split.'},
        {'item': 'Held-out test set', 'rationale': 'The test set is isolated until the model-development pipeline is locked.'},
    ])

def main():
    t0 = time.time()
    df = pd.read_csv(INPUT_CSV)
    df, dynamic_audit = add_dynamic_odi_features(df)
    required = sorted(set(DYNAMIC_PROM_FEATURES + [TARGET_COL, GROUP_COL, DAYS_BETWEEN_PROM_COL]))
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise ValueError('Missing required columns:\n' + '\n'.join(f' - {{c}}' for c in missing_cols))
    preop = pd.to_numeric(df['preop_ODI'].map(clean_scalar), errors='coerce')
    postop = pd.to_numeric(df['postop_ODI'].map(clean_scalar), errors='coerce')
    days = pd.to_numeric(df[DAYS_BETWEEN_PROM_COL].map(clean_scalar), errors='coerce')
    eligible = preop.notna() & postop.notna() & days.gt(0)
    work = df.loc[eligible, DYNAMIC_PROM_FEATURES + [TARGET_COL, GROUP_COL]].copy()
    work[TARGET_COL] = work[TARGET_COL].map(to_binary_target)
    work = work.dropna(subset=[TARGET_COL]).copy()
    work[TARGET_COL] = work[TARGET_COL].astype(int)
    train_mask, cal_mask, test_mask = patient_split(work, TARGET_COL, RANDOM_STATE)
    work['split'] = np.where(train_mask, 'train', np.where(cal_mask, 'calibration', 'test'))
    train = work[work.split.eq('train')].reset_index(drop=True)
    folds = cv_splits(train[TARGET_COL].astype(int).to_numpy(), train[GROUP_COL].to_numpy(), RANDOM_STATE + 10)
    all_candidates, all_folds = [], []
    baseline = run_model('baseline_only', work, BASELINE_FEATURES, folds, RANDOM_STATE + 100, all_candidates, all_folds)
    expanded = run_model('dynamic_PROM_expanded', work, DYNAMIC_PROM_FEATURES, folds, RANDOM_STATE + 200, all_candidates, all_folds)
    merged = baseline['preds'].merge(expanded['preds'], on=['row_index_in_split', GROUP_COL, 'y_true'], suffixes=('_baseline', '_expanded'), validate='one_to_one')
    y = merged.y_true.astype(int).to_numpy()
    p0 = merged.p_calibrated_baseline.astype(float).to_numpy()
    p1 = merged.p_calibrated_expanded.astype(float).to_numpy()
    groups = merged[GROUP_COL].to_numpy()
    dap = paired_delta_boot(y, p0, p1, groups, 'AP', RANDOM_STATE + 1000)
    dauc = paired_delta_boot(y, p0, p1, groups, 'ROC_AUC', RANDOM_STATE + 2000)
    comparison = pd.DataFrame([{ 'baseline_Test_AP': baseline['summary']['Test_AP'], 'expanded_Test_AP': expanded['summary']['Test_AP'], 'Delta_AP_expanded_minus_baseline': dap[0], 'Delta_AP_CI_lower': dap[1], 'Delta_AP_CI_upper': dap[2], 'Delta_AP_bootstrap_p_value': dap[3], 'baseline_Test_ROC_AUC': baseline['summary']['Test_ROC_AUC'], 'expanded_Test_ROC_AUC': expanded['summary']['Test_ROC_AUC'], 'Delta_ROC_AUC_expanded_minus_baseline': dauc[0], 'Delta_ROC_AUC_CI_lower': dauc[1], 'Delta_ROC_AUC_CI_upper': dauc[2], 'Delta_ROC_AUC_bootstrap_p_value': dauc[3]}])
    summary = pd.DataFrame([baseline['summary'], expanded['summary']])
    candidates = pd.concat(all_candidates, ignore_index=True)
    folds_df = pd.concat(all_folds, ignore_index=True)
    missingness = pd.DataFrame([{'column': c, 'n_missing_or_blank': int(work[c].isna().sum()), 'percent_missing_or_blank': 100 * float(work[c].isna().sum()) / len(work)} for c in DYNAMIC_PROM_FEATURES if c in work.columns])
    xlsx = os.path.join(OUTPUT_DIR, f'Step2_DynamicPROM_{{CLASSIFIER_KEY}}_summary.xlsx')
    with pd.ExcelWriter(xlsx, engine='openpyxl') as writer:
        methods_table().to_excel(writer, 'methods_rationale', index=False)
        dynamic_audit.to_excel(writer, 'dynamic_feature_audit', index=False)
        summary.to_excel(writer, 'model_performance', index=False)
        comparison.to_excel(writer, 'paired_PROM_comparison', index=False)
        candidates.to_excel(writer, 'cv_candidates_all_models', index=False)
        folds_df.to_excel(writer, 'cv_fold_metrics_all', index=False)
        split_audit(work).to_excel(writer, 'split_audit', index=False)
        missingness.to_excel(writer, 'missingness_audit', index=False)
        style_excel(writer)
    summary.to_csv(os.path.join(OUTPUT_DIR, 'model_performance.csv'), index=False)
    comparison.to_csv(os.path.join(OUTPUT_DIR, 'paired_PROM_comparison.csv'), index=False)
    candidates.to_csv(os.path.join(OUTPUT_DIR, 'cv_candidates_all_models.csv'), index=False)
    folds_df.to_csv(os.path.join(OUTPUT_DIR, 'cv_fold_metrics_all.csv'), index=False)
    work[[GROUP_COL, 'split']].drop_duplicates().to_csv(os.path.join(OUTPUT_DIR, 'split_assignment.csv'), index=False)
    manifest = {'classifier': CLASSIFIER_DISPLAY_NAME, 'output_dir': OUTPUT_DIR, 'design': f'paired baseline-only and dynamic PROM-expanded {{CLASSIFIER_DISPLAY_NAME}} models', 'model_selection': 'highest mean group-aware CV average precision inside training split', 'calibration': 'isotonic', 'threshold_strategy': 'maximum F1 on calibration split', 'runtime_minutes': float((time.time() - t0) / 60), 'summary_xlsx': xlsx}
    json.dump(json_native(manifest), open(os.path.join(OUTPUT_DIR, 'run_manifest.json'), 'w'), indent=2, sort_keys=True)
    zip_path = os.path.join(OUTPUT_DIR, f'Step2_DynamicPROM_{{CLASSIFIER_KEY}}_outputs.zip')
    tmp = zip_path + '.tmp'
    for p in [zip_path, tmp]:
        if os.path.exists(p):
            os.remove(p)
    with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=ZIP_COMPRESSION_LEVEL) as z:
        for root, _, files in os.walk(OUTPUT_DIR):
            for name in files:
                full = os.path.join(root, name)
                if full in {zip_path, tmp}:
                    continue
                z.write(full, os.path.relpath(full, OUTPUT_DIR))
    os.replace(tmp, zip_path)
    print('=' * 100)
    print(f'Step 2 {{CLASSIFIER_DISPLAY_NAME}} analysis completed')
    print('Summary Excel:', xlsx)
    print('ZIP archive:', zip_path)
    print('=' * 100)
    if CREATE_COLAB_DOWNLOAD_LINK:
        try:
            from IPython.display import HTML, display
            display(HTML(f'<p><b>Step 2 {{CLASSIFIER_DISPLAY_NAME}} output archive is ready.</b></p><p><a href="/files{{zip_path}}" download>Click here to download the ZIP archive</a></p><p>Path: <code>{{zip_path}}</code></p>'))
        except Exception:
            pass
    if AUTO_DOWNLOAD_ZIP:
        try:
            from google.colab import files
            files.download(zip_path)
        except Exception:
            pass

if __name__ == '__main__':
    main()


#**Step 2: Random Forest**

In [ ]:
"""
Step 2 dynamic postoperative PROM analysis with random forest
=============================================================

Input
-----
/content/Step 2_ODI_Cohort.csv

Target
------
final_reop_step2
    1 = reoperation from postoperative day 91 through day 365
    0 = no reoperation from postoperative day 91 through day 365

Design
------
This script runs paired baseline-only and dynamic PROM-expanded random forest
models for delayed lumbar reoperation prediction. The baseline-only model
uses the same 35 baseline variables as Step 1. The dynamic PROM-expanded
model includes the same baseline variables plus preoperative PROM,
postoperative PROM, PROM change, PROM change rate, relative MCID status,
and timing of postoperative PROM assessment. Paired models use identical
patient-level train/calibration/test splits and identical group-aware
cross-validation fold construction. Hyperparameter tuning is performed
exclusively within the training split using cross-validated average
precision as the primary selection metric. Probability calibration and
threshold selection are performed only on the calibration split. The
held-out test set is reserved until the model-development pipeline is
locked.
"""

# -*- coding: utf-8 -*-

import os
import re
import sys
import json
import math
import time
import zipfile
import platform
import subprocess
import warnings
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    import joblib
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'joblib'])
    import joblib

try:
    import openpyxl  # noqa
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl'])
    import openpyxl  # noqa

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedGroupKFold, ParameterSampler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_recall_curve,
    roc_curve,
    confusion_matrix,
    f1_score,
)
from sklearn.isotonic import IsotonicRegression
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')


from sklearn.ensemble import RandomForestClassifier


BASE_DIR = '/content'
INPUT_CSV = os.path.join(BASE_DIR, 'Step 2_ODI_Cohort.csv')
CLASSIFIER_KEY = 'RandomForest'
CLASSIFIER_DISPLAY_NAME = 'random forest'
SCALE_CONTINUOUS = False
OUTPUT_DIR = os.path.join(BASE_DIR, 'Step2_DynamicPROM_RandomForest_publication')
PLOT_DIR = os.path.join(OUTPUT_DIR, 'plots')
ARTIFACT_DIR = os.path.join(OUTPUT_DIR, 'model_artifacts')
for d in [OUTPUT_DIR, PLOT_DIR, ARTIFACT_DIR]:
    os.makedirs(d, exist_ok=True)

TARGET_COL = 'final_reop_step2'
GROUP_COL = 'PersonKey'
RANDOM_STATE = 20260524
TEST_FRACTION = 0.20
CALIBRATION_FRACTION_OF_REMAINING = 0.20
N_CV_FOLDS = 5
N_RANDOM_COMBINATIONS = 300
USE_EARLY_STOPPING_IN_CV = False
EARLY_STOPPING_ROUNDS = 100
MIN_FINAL_N_ESTIMATORS = 50
THRESHOLD_STRATEGY = 'max_f1'
N_BOOTSTRAPS = 2000
ECE_N_BINS = 10
N_JOBS = -1
LEARNING_CURVE_TRAIN_FRACTIONS = (0.20, 0.40, 0.60, 0.80, 1.00)
LEARNING_CURVE_CV_FOLDS = 3
ZIP_COMPRESSION_LEVEL = 1
CREATE_COLAB_DOWNLOAD_LINK = True
AUTO_DOWNLOAD_ZIP = False
RELATIVE_ODI_MCID_CUTOFF = 0.30
PROM_CHANGE_RATE_COL = 'ODI_change_rate'
RELATIVE_ODI_MCID_COL = 'ODI_relative_MCID_binary'
DAYS_BETWEEN_PROM_COL = 'days_between_PROMs'

BASELINE_FEATURES = ['finaldx_degenerative', 'finaldx_radicular', 'finaldx_stenosis', 'finaldx_deformity_instability', 'finaldx_other_diagnosis', 'age', 'sex', 'race', 'ethnicity', 'cancer_status', 'chronic_pulmonary_disease', 'congestive_heart_failure', 'connective_tissue_rheumatic_disease', 'diabetes_status', 'myocardial_infarction', 'renal_disease', 'institution_type', 'institution_size', 'institution_region', 'asa', 'bmi', 'payer_status', 'alif_llif', 'corpectomy', 'discectomy', 'foraminotomy', 'instrumentation', 'laminectomy_posterior_decompression', 'pelvic_fixation', 'plf', 'tlif_plif', 'other_lumbar_procedure', 'number_operated_levels', 'operative_region_extent', 'PatTobaccoUse']
assert len(BASELINE_FEATURES) == 35
STEP2_PROM_FEATURES = ['preop_ODI', 'postop_ODI', 'ODI_change', PROM_CHANGE_RATE_COL, RELATIVE_ODI_MCID_COL, 'postop_ODI_day']
DYNAMIC_PROM_FEATURES = BASELINE_FEATURES + STEP2_PROM_FEATURES
CONTINUOUS_BASELINE_FEATURES = ['age', 'bmi']
CONTINUOUS_FEATURES_ALL = CONTINUOUS_BASELINE_FEATURES + ['preop_ODI', 'postop_ODI', 'ODI_change', PROM_CHANGE_RATE_COL, 'postop_ODI_day']
BINARY_FEATURES = ['finaldx_degenerative', 'finaldx_radicular', 'finaldx_stenosis', 'finaldx_deformity_instability', 'finaldx_other_diagnosis', 'sex', 'ethnicity', 'cancer_status', 'chronic_pulmonary_disease', 'congestive_heart_failure', 'connective_tissue_rheumatic_disease', 'myocardial_infarction', 'renal_disease', 'institution_type', 'alif_llif', 'corpectomy', 'discectomy', 'foraminotomy', 'instrumentation', 'laminectomy_posterior_decompression', 'pelvic_fixation', 'plf', 'tlif_plif', 'other_lumbar_procedure', 'operative_region_extent', RELATIVE_ODI_MCID_COL]
ORDINAL_FEATURES = ['diabetes_status', 'institution_size', 'asa', 'number_operated_levels']
NOMINAL_FEATURES = ['race', 'institution_region', 'payer_status', 'PatTobaccoUse']

SEARCH_SPACE = {
    'n_estimators': [300, 500, 800, 1200, 1600],
    'max_depth': [None, 5, 10, 20, 40],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10, 20],
    'max_features': ['sqrt', 'log2', 0.25, 0.50, 0.75],
    'bootstrap': [True],
    'positive_weight_multiplier': [0.25, 0.50, 0.75, 1.00, 1.50, 2.00, 3.00, 4.00, 6.00, 8.00],
}
ITERATION_PARAM = 'n_estimators'
INT_PARAMS = {'n_estimators', 'min_samples_split', 'min_samples_leaf'}
FLOAT_PARAMS = {'positive_weight_multiplier'}

PARAMETER_CANDIDATES = list(ParameterSampler(SEARCH_SPACE, n_iter=N_RANDOM_COMBINATIONS, random_state=RANDOM_STATE))

MISSING_STRINGS = {'', ' ', 'na', 'n/a', 'nan', 'none', 'null', '.', 'missing', '<na>'}
BINARY_MAPS = {
    'sex': {'female': 0, 'f': 0, 'male': 1, 'm': 1},
    'ethnicity': {'non-hispanic': 0, 'non hispanic': 0, 'hispanic': 1},
    'cancer_status': {'no cancer': 0, 'no': 0, 'none': 0, 'any cancer': 1, 'yes': 1, 'cancer': 1},
    'institution_type': {'hospital': 0, 'non-hospital': 1, 'non hospital': 1, 'nonhospital': 1},
    'operative_region_extent': {'lumbar only': 0, 'extended_region_involvement': 1, 'extended region involvement': 1, 'extended': 1},
}
ORDINAL_MAPS = {
    'diabetes_status': {'no': 0, 'none': 0, '0': 0, 'without comp': 1, 'without complication': 1, 'without complications': 1, '1': 1, 'with comp': 2, 'with complication': 2, 'with complications': 2, '2': 2},
    'institution_size': {'between 1-99 beds': 0, '1-99': 0, 'between 100-399 beds': 1, '100-399': 1, '>= 400 beds': 2, '>=400 beds': 2, '>=400': 2, '>= 400': 2},
    'asa': {'1': 1, 'i': 1, '2': 2, 'ii': 2, '3': 3, 'iii': 3, '4': 4, 'iv': 4, '>=4': 4, '>= 4': 4, '5': 4, 'v': 4},
    'number_operated_levels': {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '>=4': 4, '>= 4': 4, '5': 4, '6': 4, '7': 4, '8': 4, '9': 4, '10': 4},
}
PREFERRED_NOMINAL_LEVELS = {
    'race': ['White', 'Black', 'Other'],
    'institution_region': ['South', 'North East', 'West', 'Midwest'],
    'payer_status': ['Medicare', 'Commercial/Private', 'Other', 'Medicaid/Public/Government'],
    'PatTobaccoUse': ['Unknown/Not reported/Multiple', 'Never', 'Former', 'Current'],
}
FEATURE_LABELS = {
    'age': 'Age', 'bmi': 'BMI', 'sex': 'Male sex', 'race': 'Race', 'ethnicity': 'Hispanic ethnicity',
    'cancer_status': 'Any cancer', 'diabetes_status': 'Diabetes status', 'institution_type': 'Non-hospital institution',
    'institution_size': 'Institution size', 'institution_region': 'Institution region', 'asa': 'ASA physical status',
    'payer_status': 'Primary payer', 'PatTobaccoUse': 'Tobacco use',
    'finaldx_degenerative': 'Degenerative diagnosis', 'finaldx_radicular': 'Radiculopathy diagnosis',
    'finaldx_stenosis': 'Spinal stenosis diagnosis', 'finaldx_deformity_instability': 'Deformity or instability diagnosis',
    'finaldx_other_diagnosis': 'Other lumbar diagnosis', 'chronic_pulmonary_disease': 'Chronic pulmonary disease',
    'congestive_heart_failure': 'Congestive heart failure', 'connective_tissue_rheumatic_disease': 'Connective tissue/rheumatic disease',
    'myocardial_infarction': 'Myocardial infarction', 'renal_disease': 'Renal disease',
    'alif_llif': 'Anterior/lateral lumbar interbody fusion', 'corpectomy': 'Corpectomy', 'discectomy': 'Discectomy',
    'foraminotomy': 'Foraminotomy', 'instrumentation': 'Instrumentation',
    'laminectomy_posterior_decompression': 'Posterior decompression or laminectomy', 'pelvic_fixation': 'Pelvic fixation',
    'plf': 'Posterolateral fusion', 'tlif_plif': 'Posterior/transforaminal lumbar interbody fusion',
    'other_lumbar_procedure': 'Other lumbar procedure', 'number_operated_levels': 'Number of operated levels',
    'operative_region_extent': 'Operative region extent',
}
FEATURE_LABELS.update({'preop_ODI': 'Preoperative ODI', 'postop_ODI': 'Postoperative ODI', 'ODI_change': 'Change in ODI', PROM_CHANGE_RATE_COL: 'ODI change rate', RELATIVE_ODI_MCID_COL: 'Relative ODI MCID', 'postop_ODI_day': 'Timing of postoperative ODI assessment'})

def clean_scalar(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().replace('≥', '>=')
        return np.nan if s.lower() in MISSING_STRINGS else s
    return x

def norm_text(x):
    x = clean_scalar(x)
    if pd.isna(x):
        return None
    return str(x).strip().replace('≥', '>=').lower()

def to_binary_target(x):
    sx = norm_text(x)
    if sx is None:
        return np.nan
    if sx in {'1', '1.0', 'yes', 'y', 'true', 't'}:
        return 1.0
    if sx in {'0', '0.0', 'no', 'n', 'false', 'f'}:
        return 0.0
    try:
        v = float(sx)
        return float(v) if v in (0.0, 1.0) else np.nan
    except Exception:
        return np.nan

def safe_filename(x):
    return re.sub(r'_+', '_', re.sub(r'[^A-Za-z0-9_.-]+', '_', str(x))).strip('_')[:180] or 'file'

def json_native(obj):
    if isinstance(obj, dict):
        return {str(k): json_native(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_native(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj

def sanitize_params(params):
    out = {}
    for k, v in params.items():
        if k in INT_PARAMS:
            out[k] = int(round(float(v)))
        elif k in FLOAT_PARAMS:
            out[k] = float(v)
        else:
            out[k] = v
    return out

def safe_average_precision(y, p):
    return np.nan if len(np.unique(y)) < 2 else float(average_precision_score(y, p))

def safe_roc_auc(y, p):
    return np.nan if len(np.unique(y)) < 2 else float(roc_auc_score(y, p))

def expected_calibration_error(y, p, n_bins=10):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    out = 0.0
    if len(y) == 0:
        return np.nan
    for i in range(n_bins):
        mask = (p >= bins[i]) & ((p <= bins[i + 1]) if i == n_bins - 1 else (p < bins[i + 1]))
        if np.any(mask):
            out += (np.sum(mask) / len(y)) * abs(float(np.mean(y[mask])) - float(np.mean(p[mask])))
    return float(out)

def classification_metrics(y, p, threshold):
    y = np.asarray(y).astype(int)
    pred = (np.asarray(p) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        'threshold': float(threshold),
        'F1': float(f1_score(y, pred, zero_division=0)),
        'Sensitivity': tp / (tp + fn) if tp + fn > 0 else np.nan,
        'Specificity': tn / (tn + fp) if tn + fp > 0 else np.nan,
        'PPV': tp / (tp + fp) if tp + fp > 0 else np.nan,
        'NPV': tn / (tn + fn) if tn + fn > 0 else np.nan,
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
        'Predicted_positive_rate': float(np.mean(pred)),
    }

def top5_metrics(y, p):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    n = len(y)
    k = max(1, int(math.ceil(n * 0.05)))
    idx = np.argsort(-p)[:k]
    prev = float(np.mean(y)) if n else np.nan
    rate = float(np.mean(y[idx])) if n else np.nan
    return {
        'Top_5pct_n': int(k),
        'Top_5pct_event_rate': rate,
        'Top_5pct_lift': rate / prev if prev > 0 else np.nan,
        'Top_5pct_captured_events': float(np.sum(y[idx]) / np.sum(y)) if np.sum(y) > 0 else np.nan,
    }

def select_threshold(y, p):
    precision, recall, thresholds = precision_recall_curve(y, p)
    if len(thresholds) == 0:
        threshold = 0.5
    else:
        precision = precision[:-1]
        recall = recall[:-1]
        f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
        threshold = float(thresholds[int(np.nanargmax(f1))])
    return threshold, classification_metrics(y, p, threshold)

def eval_preds(y, p, threshold=None, prefix=''):
    out = {
        f'{prefix}AP': safe_average_precision(y, p),
        f'{prefix}ROC_AUC': safe_roc_auc(y, p),
        f'{prefix}Brier_score': float(brier_score_loss(y, p)),
        f'{prefix}ECE': expected_calibration_error(y, p, ECE_N_BINS),
        f'{prefix}N': int(len(y)),
        f'{prefix}Events': int(np.sum(y)),
        f'{prefix}Prevalence': float(np.mean(y)),
    }
    if threshold is not None:
        out.update({f'{prefix}{k}': v for k, v in classification_metrics(y, p, threshold).items()})
        out.update({f'{prefix}{k}': v for k, v in top5_metrics(y, p).items()})
    return out

def make_weights(y, multiplier):
    y = np.asarray(y).astype(int)
    n_pos = int(np.sum(y == 1))
    n_neg = int(np.sum(y == 0))
    if n_pos == 0:
        raise ValueError('No positive events in training fold.')
    pos_weight = (n_neg / max(n_pos, 1)) * float(multiplier)
    return np.where(y == 1, pos_weight, 1.0).astype(float)

def actual_positive_weight(y, multiplier):
    y = np.asarray(y).astype(int)
    return float((np.sum(y == 0) / max(np.sum(y == 1), 1)) * multiplier)

@dataclass
class ModelPreprocessor:
    continuous_features: List[str]
    binary_features: List[str]
    ordinal_features: List[str]
    nominal_features: List[str]
    preferred_nominal_levels: Dict[str, List[str]]
    scale_continuous: bool = False

    def __post_init__(self):
        self.continuous_imputer = None
        self.discrete_imputer = None
        self.nominal_imputer = None
        self.scaler = None
        self.onehot = None
        self.output_feature_names_ = []

    def _parse_binary(self, x, feature):
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in BINARY_MAPS and sx in BINARY_MAPS[feature]:
            return float(BINARY_MAPS[feature][sx])
        if sx in {'1', '1.0', 'yes', 'y', 'true', 't', 'present', 'positive'}:
            return 1.0
        if sx in {'0', '0.0', 'no', 'n', 'false', 'f', 'absent', 'negative'}:
            return 0.0
        try:
            v = float(sx)
            return float(v) if v in (0.0, 1.0) else np.nan
        except Exception:
            return np.nan

    def _parse_ordinal(self, x, feature):
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in ORDINAL_MAPS and sx in ORDINAL_MAPS[feature]:
            return float(ORDINAL_MAPS[feature][sx])
        try:
            v = float(sx)
            if feature == 'asa':
                return float(min(max(int(round(v)), 1), 4))
            if feature == 'number_operated_levels':
                return float(min(max(int(round(v)), 0), 4))
            if feature == 'diabetes_status':
                return float(min(max(int(round(v)), 0), 2))
            if feature == 'institution_size':
                return float(min(max(int(round(v)), 0), 2))
            return float(v)
        except Exception:
            return np.nan

    def _nominal(self, feature, x):
        x = clean_scalar(x)
        if pd.isna(x):
            return np.nan
        s = str(x).strip()
        sl = s.lower()
        if feature == 'race':
            return 'White' if sl == 'white' else ('Black' if sl == 'black' else 'Other')
        if feature == 'institution_region':
            mapping = {'south': 'South', 'north east': 'North East', 'northeast': 'North East', 'north-east': 'North East', 'west': 'West', 'midwest': 'Midwest', 'mid west': 'Midwest'}
            return mapping.get(sl, s)
        if feature == 'payer_status':
            if sl == 'medicare':
                return 'Medicare'
            if sl in {'commercial/private', 'commercial', 'private', 'commercial private'}:
                return 'Commercial/Private'
            if sl in {'medicaid/public/government', 'medicaid', 'public', 'government', 'public/government'}:
                return 'Medicaid/Public/Government'
            return 'Other'
        if feature == 'PatTobaccoUse':
            return 'Never' if sl == 'never' else ('Former' if sl == 'former' else ('Current' if sl == 'current' else 'Unknown/Not reported/Multiple'))
        return s

    def _parts(self, X):
        cont = pd.DataFrame(index=X.index)
        disc = pd.DataFrame(index=X.index)
        nom = pd.DataFrame(index=X.index)
        for c in self.continuous_features:
            cont[c] = pd.to_numeric(X[c].map(clean_scalar), errors='coerce')
        for c in self.binary_features:
            disc[c] = X[c].map(lambda z: self._parse_binary(z, c)).astype(float)
        for c in self.ordinal_features:
            disc[c] = X[c].map(lambda z: self._parse_ordinal(z, c)).astype(float)
        for c in self.nominal_features:
            nom[c] = X[c].map(lambda z: self._nominal(c, z)).astype('object')
        return cont, disc, nom

    def fit(self, X):
        cont, disc, nom = self._parts(X)
        self.continuous_imputer = SimpleImputer(strategy='median')
        self.discrete_imputer = SimpleImputer(strategy='most_frequent')
        self.nominal_imputer = SimpleImputer(strategy='constant', fill_value='Missing')
        cont_imp = self.continuous_imputer.fit_transform(cont)
        if self.scale_continuous:
            self.scaler = StandardScaler()
            self.scaler.fit(cont_imp)
        self.discrete_imputer.fit(disc)
        nom_imp = pd.DataFrame(self.nominal_imputer.fit_transform(nom), columns=self.nominal_features)
        cats = []
        for c in self.nominal_features:
            preferred = list(self.preferred_nominal_levels.get(c, []))
            observed = nom_imp[c].astype(str).unique().tolist()
            cats.append(preferred + sorted([x for x in observed if x not in preferred]))
        try:
            self.onehot = OneHotEncoder(categories=cats, handle_unknown='ignore', sparse_output=False)
        except TypeError:
            self.onehot = OneHotEncoder(categories=cats, handle_unknown='ignore', sparse=False)
        self.onehot.fit(nom_imp.astype(str))
        self.output_feature_names_ = self.continuous_features + self.binary_features + self.ordinal_features + self.onehot.get_feature_names_out(self.nominal_features).tolist()
        return self

    def transform(self, X):
        cont, disc, nom = self._parts(X)
        cont_imp = self.continuous_imputer.transform(cont)
        if self.scale_continuous and self.scaler is not None:
            cont_imp = self.scaler.transform(cont_imp)
        disc_imp = self.discrete_imputer.transform(disc)
        nom_imp = pd.DataFrame(self.nominal_imputer.transform(nom), columns=self.nominal_features)
        nom_oh = self.onehot.transform(nom_imp.astype(str))
        return np.concatenate([cont_imp, disc_imp, nom_oh], axis=1).astype(float)

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

    @property
    def output_feature_names(self):
        return self.output_feature_names_

def raw_model_scores(model, Xp, n_iter=None):
    if CLASSIFIER_KEY == 'HistGradientBoosting' and n_iter is not None:
        selected = None
        for i, pred in enumerate(model.staged_predict_proba(Xp), start=1):
            selected = pred
            if i >= int(n_iter):
                break
        if selected is not None:
            return selected[:, 1]
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(Xp)[:, 1]
    if hasattr(model, 'decision_function'):
        return model.decision_function(Xp)
    raise AttributeError('Model provides neither predict_proba nor decision_function.')

def predict(pre, model, X, features, n_iter=None):
    return raw_model_scores(model, pre.transform(X[features]), n_iter=n_iter)

def patient_split(df, target_col, seed):
    group_df = df.groupby(GROUP_COL, dropna=False)[target_col].max().reset_index()
    y_group = group_df[target_col].astype(int).to_numpy()
    groups = group_df[GROUP_COL].to_numpy()
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=seed)
    train_cal_idx, test_idx = next(sss1.split(groups, y_group))
    groups_train_cal = groups[train_cal_idx]
    y_train_cal = y_group[train_cal_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=CALIBRATION_FRACTION_OF_REMAINING, random_state=seed + 1)
    train_idx_rel, cal_idx_rel = next(sss2.split(groups_train_cal, y_train_cal))
    train_groups = set(groups_train_cal[train_idx_rel])
    cal_groups = set(groups_train_cal[cal_idx_rel])
    test_groups = set(groups[test_idx])
    assert train_groups.isdisjoint(cal_groups) and train_groups.isdisjoint(test_groups) and cal_groups.isdisjoint(test_groups)
    return df[GROUP_COL].isin(train_groups).to_numpy(), df[GROUP_COL].isin(cal_groups).to_numpy(), df[GROUP_COL].isin(test_groups).to_numpy()

def cv_splits(y, groups, seed, n_folds=N_CV_FOLDS):
    group_df = pd.DataFrame({'group': groups, 'y': y}).groupby('group')['y'].max().reset_index()
    n_folds = min(n_folds, int(np.sum(group_df.y == 1)), int(np.sum(group_df.y == 0)))
    if n_folds < 2:
        raise ValueError('Not enough positive and negative patient groups for group-aware CV.')
    cv = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    return [(tr, va) for tr, va in cv.split(np.zeros(len(y)), y, groups)]

def feature_types(features):
    cont = [f for f in features if f in CONTINUOUS_FEATURES_ALL or (f not in BINARY_FEATURES and f not in ORDINAL_FEATURES and f not in NOMINAL_FEATURES)]
    binary = [f for f in features if f in BINARY_FEATURES]
    ordinal = [f for f in features if f in ORDINAL_FEATURES]
    nominal = [f for f in features if f in NOMINAL_FEATURES]
    return cont, binary, ordinal, nominal

def fit_pipeline(X, y, features, params, seed, eval_set=None, early=False, n_estimators_override=None):
    cont, binary, ordinal, nominal = feature_types(features)
    pre = ModelPreprocessor(cont, binary, ordinal, nominal, PREFERRED_NOMINAL_LEVELS, scale_continuous=SCALE_CONTINUOUS)
    Xp = pre.fit_transform(X[features])
    eval_transformed = None
    if eval_set is not None:
        X_val, y_val = eval_set
        eval_transformed = (pre.transform(X_val[features]), y_val)
    model, best_iter = fit_estimator(Xp, y, params, seed, eval_set=eval_transformed, use_early_stopping=early, n_estimators_override=n_estimators_override)
    return pre, model, best_iter

def tune(X_train, y_train, groups_train, features, folds, model_key, seed):
    candidate_rows = []
    fold_rows = []
    sampler = list(ParameterSampler(SEARCH_SPACE, n_iter=N_RANDOM_COMBINATIONS, random_state=seed))
    for cid, raw_params in enumerate(sampler, start=1):
        params = sanitize_params(raw_params)
        fold_train_aps, fold_train_aucs, fold_aps, fold_aucs, best_iters = [], [], [], [], []
        t0 = time.time()
        for fid, (tr_idx, va_idx) in enumerate(folds, start=1):
            X_tr = X_train.iloc[tr_idx].reset_index(drop=True)
            y_tr = y_train[tr_idx]
            X_va = X_train.iloc[va_idx].reset_index(drop=True)
            y_va = y_train[va_idx]
            pre, model, best_iter = fit_pipeline(X_tr, y_tr, features, params, seed + cid * 1000 + fid, eval_set=(X_va, y_va), early=USE_EARLY_STOPPING_IN_CV)
            p_tr = predict(pre, model, X_tr, features, n_iter=best_iter)
            p_va = predict(pre, model, X_va, features, n_iter=best_iter)
            tr_ap, tr_auc = safe_average_precision(y_tr, p_tr), safe_roc_auc(y_tr, p_tr)
            va_ap, va_auc = safe_average_precision(y_va, p_va), safe_roc_auc(y_va, p_va)
            fold_train_aps.append(tr_ap); fold_train_aucs.append(tr_auc); fold_aps.append(va_ap); fold_aucs.append(va_auc)
            if best_iter is not None and best_iter > 0:
                best_iters.append(best_iter)
            fold_rows.append({
                'model_key': model_key, 'candidate_id': cid, 'fold': fid,
                'fold_train_AP': tr_ap, 'fold_train_ROC_AUC': tr_auc,
                'fold_validation_AP': va_ap, 'fold_validation_ROC_AUC': va_auc,
                'fold_train_minus_validation_AP_gap': tr_ap - va_ap if np.isfinite(tr_ap) and np.isfinite(va_ap) else np.nan,
                'fold_train_minus_validation_ROC_AUC_gap': tr_auc - va_auc if np.isfinite(tr_auc) and np.isfinite(va_auc) else np.nan,
                'fold_best_iteration': best_iter,
                'positive_weight_used': actual_positive_weight(y_tr, params['positive_weight_multiplier']),
                **params,
            })
        locked = int(np.median(best_iters)) if best_iters and USE_EARLY_STOPPING_IN_CV else (int(params[ITERATION_PARAM]) if ITERATION_PARAM and ITERATION_PARAM in params else np.nan)
        row = {
            'model_key': model_key, 'candidate_id': cid, 'cv_folds': len(folds),
            'cv_train_AP_mean': float(np.nanmean(fold_train_aps)),
            'cv_train_ROC_AUC_mean': float(np.nanmean(fold_train_aucs)),
            'cv_AP_mean': float(np.nanmean(fold_aps)),
            'cv_AP_SD': float(np.nanstd(fold_aps, ddof=1)) if len(fold_aps) > 1 else np.nan,
            'cv_ROC_AUC_mean': float(np.nanmean(fold_aucs)),
            'cv_ROC_AUC_SD': float(np.nanstd(fold_aucs, ddof=1)) if len(fold_aucs) > 1 else np.nan,
            'mean_cv_best_iteration': float(np.mean(best_iters)) if best_iters else np.nan,
            'locked_n_estimators_from_cv': locked,
            'elapsed_seconds': float(time.time() - t0),
            **params,
        }
        candidate_rows.append(row)
        print(f"{model_key} | candidate {cid:03d}/{len(sampler)} | CV AP={row['cv_AP_mean']:.5f} | AUC={row['cv_ROC_AUC_mean']:.5f}")
    return pd.DataFrame(candidate_rows).sort_values('cv_AP_mean', ascending=False).reset_index(drop=True), pd.DataFrame(fold_rows)

def fit_final(X_train, y_train, X_cal, y_cal, X_test, features, params, locked_n, seed):
    n_override = int(locked_n) if pd.notna(locked_n) and USE_EARLY_STOPPING_IN_CV and ITERATION_PARAM else None
    pre, model, _ = fit_pipeline(X_train, y_train, features, params, seed, n_estimators_override=n_override)
    p_train_raw = predict(pre, model, X_train, features)
    p_cal_raw = predict(pre, model, X_cal, features)
    p_test_raw = predict(pre, model, X_test, features)
    calibrator = IsotonicRegression(out_of_bounds='clip')
    calibrator.fit(p_cal_raw, y_cal)
    p_train = calibrator.predict(p_train_raw)
    p_cal = calibrator.predict(p_cal_raw)
    p_test = calibrator.predict(p_test_raw)
    threshold, cal_metrics = select_threshold(y_cal, p_cal)
    return {
        'pre': pre, 'model': model, 'calibrator': calibrator,
        'p_train_raw': p_train_raw, 'p_cal_raw': p_cal_raw, 'p_test_raw': p_test_raw,
        'p_train': p_train, 'p_cal': p_cal, 'p_test': p_test,
        'threshold': threshold, 'cal_metrics': cal_metrics,
    }

def patient_boot_ci(y, p, groups, metric, threshold=None, seed=1):
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({'idx': np.arange(len(y)), 'group': groups, 'y': np.asarray(y).astype(int)})
    group_y = d.groupby('group').y.max()
    pos = group_y[group_y == 1].index.to_numpy()
    neg = group_y[group_y == 0].index.to_numpy()
    if len(pos) == 0 or len(neg) == 0:
        return np.nan, np.nan
    by_group = {g: d.loc[d.group.eq(g), 'idx'].to_numpy() for g in group_y.index}
    vals = []
    for _ in range(N_BOOTSTRAPS):
        sample_groups = np.concatenate([rng.choice(pos, len(pos), replace=True), rng.choice(neg, len(neg), replace=True)])
        sample_idx = np.concatenate([by_group[g] for g in sample_groups])
        yy = np.asarray(y)[sample_idx]
        pp = np.asarray(p)[sample_idx]
        if len(np.unique(yy)) < 2:
            continue
        if metric == 'AP':
            vals.append(average_precision_score(yy, pp))
        elif metric == 'ROC_AUC':
            vals.append(roc_auc_score(yy, pp))
        elif metric == 'F1':
            vals.append(f1_score(yy, pp >= threshold, zero_division=0))
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))) if vals else (np.nan, np.nan)

def paired_delta_boot(y, p0, p1, groups, metric, seed=1):
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({'idx': np.arange(len(y)), 'group': groups, 'y': np.asarray(y).astype(int)})
    group_y = d.groupby('group').y.max()
    pos = group_y[group_y == 1].index.to_numpy()
    neg = group_y[group_y == 0].index.to_numpy()
    by_group = {g: d.loc[d.group.eq(g), 'idx'].to_numpy() for g in group_y.index}
    observed = (safe_average_precision(y, p1) - safe_average_precision(y, p0)) if metric == 'AP' else (safe_roc_auc(y, p1) - safe_roc_auc(y, p0))
    vals = []
    for _ in range(N_BOOTSTRAPS):
        sample_groups = np.concatenate([rng.choice(pos, len(pos), replace=True), rng.choice(neg, len(neg), replace=True)])
        sample_idx = np.concatenate([by_group[g] for g in sample_groups])
        yy = np.asarray(y)[sample_idx]
        if len(np.unique(yy)) < 2:
            continue
        if metric == 'AP':
            vals.append(average_precision_score(yy, np.asarray(p1)[sample_idx]) - average_precision_score(yy, np.asarray(p0)[sample_idx]))
        else:
            vals.append(roc_auc_score(yy, np.asarray(p1)[sample_idx]) - roc_auc_score(yy, np.asarray(p0)[sample_idx]))
    if not vals:
        return observed, np.nan, np.nan, np.nan
    vals = np.asarray(vals)
    ci_low, ci_high = float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))
    p_value = float(2 * min(np.mean(vals <= 0), np.mean(vals >= 0)))
    return float(observed), ci_low, ci_high, min(p_value, 1.0)

def save_learning_curve(X_train, y_train, groups_train, features, params, locked_n, seed, out_path, title):
    group_df = pd.DataFrame({'group': groups_train, 'y': y_train}).groupby('group').y.max().reset_index()
    rng = np.random.default_rng(seed)
    rows = []
    for frac in LEARNING_CURVE_TRAIN_FRACTIONS:
        for rep in range(LEARNING_CURVE_CV_FOLDS):
            selected_groups = []
            for cls in [0, 1]:
                cls_groups = group_df.loc[group_df.y.eq(cls), 'group'].to_numpy()
                k = max(1, int(math.ceil(len(cls_groups) * frac)))
                selected_groups.extend(rng.choice(cls_groups, k, replace=False).tolist())
            mask = np.isin(groups_train, selected_groups)
            X_sub = X_train.loc[mask].reset_index(drop=True)
            y_sub = y_train[mask]
            g_sub = groups_train[mask]
            try:
                folds = cv_splits(y_sub, g_sub, seed + rep, n_folds=3)
            except Exception:
                continue
            for tr, va in folds:
                n_override = int(locked_n) if pd.notna(locked_n) and USE_EARLY_STOPPING_IN_CV and ITERATION_PARAM else None
                pre, model, _ = fit_pipeline(X_sub.iloc[tr].reset_index(drop=True), y_sub[tr], features, params, seed + rep, n_estimators_override=n_override)
                p_tr = predict(pre, model, X_sub.iloc[tr].reset_index(drop=True), features)
                p_va = predict(pre, model, X_sub.iloc[va].reset_index(drop=True), features)
                rows.append({'train_fraction': frac, 'train_AP': safe_average_precision(y_sub[tr], p_tr), 'validation_AP': safe_average_precision(y_sub[va], p_va)})
    lc = pd.DataFrame(rows)
    lc.to_csv(out_path.replace('.png', '.csv'), index=False)
    if not lc.empty:
        s = lc.groupby('train_fraction').agg(train_AP_mean=('train_AP', 'mean'), validation_AP_mean=('validation_AP', 'mean')).reset_index()
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.plot(s.train_fraction, s.train_AP_mean, marker='o', label='Training AP')
        ax.plot(s.train_fraction, s.validation_AP_mean, marker='o', label='Validation AP')
        ax.set_xlabel('Fraction of training patient groups')
        ax.set_ylabel('Average precision')
        ax.set_title(title)
        ax.legend()
        fig.tight_layout()
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close(fig)

def style_excel(writer):
    try:
        from openpyxl.styles import Font, PatternFill, Alignment
        for sheet in writer.sheets:
            ws = writer.sheets[sheet]
            ws.freeze_panes = 'A2'
            ws.auto_filter.ref = ws.dimensions
            for cell in ws[1]:
                cell.font = Font(bold=True)
                cell.fill = PatternFill(start_color='D9EAF7', end_color='D9EAF7', fill_type='solid')
                cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            for col in ws.columns:
                max_len = max([len(str(c.value)) for c in col if c.value is not None] + [12])
                ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 12), 70)
    except Exception:
        pass


def make_model(params, seed, n_estimators_override=None):
    p = sanitize_params(params)
    p.pop('positive_weight_multiplier', None)
    return RandomForestClassifier(
        random_state=seed,
        n_jobs=N_JOBS,
        **p,
    )

def fit_estimator(X_train, y_train, params, seed, eval_set=None, use_early_stopping=False, n_estimators_override=None):
    weights = make_weights(y_train, float(params['positive_weight_multiplier']))
    model = make_model(params, seed)
    model.fit(X_train, y_train, sample_weight=weights)
    return model, None


def add_dynamic_odi_features(df):
    out = df.copy()
    required = ['preop_ODI', 'postop_ODI', DAYS_BETWEEN_PROM_COL]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError('Cannot derive dynamic ODI features because required columns are missing: ' + ', '.join(missing))
    preop = pd.to_numeric(out['preop_ODI'].map(clean_scalar), errors='coerce')
    postop = pd.to_numeric(out['postop_ODI'].map(clean_scalar), errors='coerce')
    days_between = pd.to_numeric(out[DAYS_BETWEEN_PROM_COL].map(clean_scalar), errors='coerce')
    odi_change = postop - preop
    valid_rate = preop.notna() & postop.notna() & days_between.gt(0)
    change_rate = pd.Series(np.nan, index=out.index, dtype='float')
    change_rate.loc[valid_rate] = (odi_change.loc[valid_rate] / days_between.loc[valid_rate]).astype(float)
    improvement = preop - postop
    rel_fraction = improvement / preop.replace(0, np.nan)
    valid_relative = preop.notna() & postop.notna() & preop.gt(0)
    zero_baseline_with_postop = preop.eq(0) & postop.notna()
    relative_mcid = pd.Series(np.nan, index=out.index, dtype='float')
    relative_mcid.loc[valid_relative] = (rel_fraction.loc[valid_relative] >= RELATIVE_ODI_MCID_CUTOFF).astype(float)
    relative_mcid.loc[zero_baseline_with_postop] = 0.0
    out['ODI_change'] = odi_change
    out[PROM_CHANGE_RATE_COL] = change_rate
    out[RELATIVE_ODI_MCID_COL] = relative_mcid
    audit = pd.DataFrame([
        {'item': 'PROM change rate definition', 'value': '(postop_ODI - preop_ODI) / days_between_PROMs'},
        {'item': 'Relative ODI MCID definition', 'value': f'(preop_ODI - postop_ODI) / preop_ODI >= {RELATIVE_ODI_MCID_CUTOFF}; preop_ODI=0 coded as 0 when postop_ODI is available'},
        {'item': 'Rows', 'value': int(len(out))},
        {'item': 'Rows with valid preop/postop ODI and positive days between PROMs', 'value': int(valid_rate.sum())},
        {'item': 'Rows with preop ODI = 0 and postop ODI available coded as Relative ODI MCID = 0', 'value': int(zero_baseline_with_postop.sum())},
    ])
    return out, audit

def run_model(model_type, work, features, folds, seed, all_candidates, all_folds):
    key = model_type
    out_dir = os.path.join(OUTPUT_DIR, model_type)
    plot_dir = os.path.join(PLOT_DIR, model_type)
    artifact_dir = os.path.join(ARTIFACT_DIR, model_type)
    for d in [out_dir, plot_dir, artifact_dir]:
        os.makedirs(d, exist_ok=True)
    train = work[work.split.eq('train')].reset_index(drop=True)
    cal = work[work.split.eq('calibration')].reset_index(drop=True)
    test = work[work.split.eq('test')].reset_index(drop=True)
    y_train = train[TARGET_COL].astype(int).to_numpy()
    y_cal = cal[TARGET_COL].astype(int).to_numpy()
    y_test = test[TARGET_COL].astype(int).to_numpy()
    candidates, fold_metrics = tune(train[features], y_train, train[GROUP_COL].to_numpy(), features, folds, key, seed)
    best_params = sanitize_params({k: candidates.loc[0, k] for k in SEARCH_SPACE.keys()})
    locked_n = candidates.loc[0, 'locked_n_estimators_from_cv']
    final = fit_final(train[features], y_train, cal[features], y_cal, test[features], y_test, features, best_params, locked_n, seed + 991)
    threshold = final['threshold']
    summary = {
        'model_type': model_type, 'classifier': CLASSIFIER_DISPLAY_NAME,
        'n_features_original': len(features), 'best_candidate_id': int(candidates.loc[0, 'candidate_id']),
        'CV_AP_mean': float(candidates.loc[0, 'cv_AP_mean']), 'CV_ROC_AUC_mean': float(candidates.loc[0, 'cv_ROC_AUC_mean']),
        'locked_n_estimators_from_cv': locked_n,
        **eval_preds(y_train, final['p_train'], threshold, prefix='Train_'),
        **eval_preds(y_cal, final['p_cal'], threshold, prefix='Calibration_'),
        **eval_preds(y_test, final['p_test'], threshold, prefix='Test_'),
    }
    summary['Train_minus_CV_AP_gap'] = summary['Train_AP'] - summary['CV_AP_mean'] if pd.notna(summary['CV_AP_mean']) else np.nan
    summary['CV_minus_Test_AP_gap'] = summary['CV_AP_mean'] - summary['Test_AP'] if pd.notna(summary['CV_AP_mean']) else np.nan
    summary['Test_AP_CI_lower'], summary['Test_AP_CI_upper'] = patient_boot_ci(y_test, final['p_test'], test[GROUP_COL].to_numpy(), 'AP', seed=seed + 10)
    summary['Test_ROC_AUC_CI_lower'], summary['Test_ROC_AUC_CI_upper'] = patient_boot_ci(y_test, final['p_test'], test[GROUP_COL].to_numpy(), 'ROC_AUC', seed=seed + 20)
    pd.DataFrame([summary]).to_csv(os.path.join(out_dir, f'performance_{{key}}.csv'), index=False)
    pd.DataFrame({'row_index_in_split': np.arange(len(test)), GROUP_COL: test[GROUP_COL].to_numpy(), 'y_true': y_test, 'p_raw': final['p_test_raw'], 'p_calibrated': final['p_test']}).to_csv(os.path.join(out_dir, f'test_predictions_{{key}}.csv'), index=False)
    candidates.to_csv(os.path.join(out_dir, f'cv_candidates_{{key}}.csv'), index=False)
    fold_metrics.to_csv(os.path.join(out_dir, f'cv_fold_metrics_{{key}}.csv'), index=False)
    joblib.dump(final, os.path.join(artifact_dir, f'final_model_{{key}}.joblib'))
    save_learning_curve(train[features], y_train, train[GROUP_COL].to_numpy(), features, best_params, locked_n, seed + 313, os.path.join(plot_dir, f'learning_curve_{{key}}.png'), key)
    all_candidates.append(candidates)
    all_folds.append(fold_metrics)
    return {'summary': summary, 'preds': pd.read_csv(os.path.join(out_dir, f'test_predictions_{{key}}.csv'))}

def split_audit(df):
    rows = []
    for split in ['train', 'calibration', 'test']:
        sub = df[df.split.eq(split)]
        rows.append({'split': split, 'n_rows': len(sub), 'n_patients': sub[GROUP_COL].nunique(), 'events': int(sub[TARGET_COL].sum()), 'prevalence': float(sub[TARGET_COL].mean())})
    return pd.DataFrame(rows)

def methods_table():
    return pd.DataFrame([
        {'item': 'Paired design', 'rationale': f'Baseline-only and dynamic PROM-expanded {{CLASSIFIER_DISPLAY_NAME}} models are tuned separately under the same protocol.'},
        {'item': 'Paired split/folds', 'rationale': 'Paired models use identical patient-level train/calibration/test splits and identical group-aware CV folds.'},
        {'item': 'Tuning metric', 'rationale': 'Average precision is the primary model-selection metric because delayed reoperation is a rare-event outcome.'},
        {'item': 'Class imbalance', 'rationale': 'Positive-class sample weighting is tuned through a multiplier applied to the natural negative-to-positive ratio.'},
        {'item': 'Calibration and threshold', 'rationale': 'Isotonic calibration and maximum-F1 threshold selection are performed only on the calibration split.'},
        {'item': 'Held-out test set', 'rationale': 'The test set is isolated until the model-development pipeline is locked.'},
    ])

def main():
    t0 = time.time()
    df = pd.read_csv(INPUT_CSV)
    df, dynamic_audit = add_dynamic_odi_features(df)
    required = sorted(set(DYNAMIC_PROM_FEATURES + [TARGET_COL, GROUP_COL, DAYS_BETWEEN_PROM_COL]))
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise ValueError('Missing required columns:\n' + '\n'.join(f' - {{c}}' for c in missing_cols))
    preop = pd.to_numeric(df['preop_ODI'].map(clean_scalar), errors='coerce')
    postop = pd.to_numeric(df['postop_ODI'].map(clean_scalar), errors='coerce')
    days = pd.to_numeric(df[DAYS_BETWEEN_PROM_COL].map(clean_scalar), errors='coerce')
    eligible = preop.notna() & postop.notna() & days.gt(0)
    work = df.loc[eligible, DYNAMIC_PROM_FEATURES + [TARGET_COL, GROUP_COL]].copy()
    work[TARGET_COL] = work[TARGET_COL].map(to_binary_target)
    work = work.dropna(subset=[TARGET_COL]).copy()
    work[TARGET_COL] = work[TARGET_COL].astype(int)
    train_mask, cal_mask, test_mask = patient_split(work, TARGET_COL, RANDOM_STATE)
    work['split'] = np.where(train_mask, 'train', np.where(cal_mask, 'calibration', 'test'))
    train = work[work.split.eq('train')].reset_index(drop=True)
    folds = cv_splits(train[TARGET_COL].astype(int).to_numpy(), train[GROUP_COL].to_numpy(), RANDOM_STATE + 10)
    all_candidates, all_folds = [], []
    baseline = run_model('baseline_only', work, BASELINE_FEATURES, folds, RANDOM_STATE + 100, all_candidates, all_folds)
    expanded = run_model('dynamic_PROM_expanded', work, DYNAMIC_PROM_FEATURES, folds, RANDOM_STATE + 200, all_candidates, all_folds)
    merged = baseline['preds'].merge(expanded['preds'], on=['row_index_in_split', GROUP_COL, 'y_true'], suffixes=('_baseline', '_expanded'), validate='one_to_one')
    y = merged.y_true.astype(int).to_numpy()
    p0 = merged.p_calibrated_baseline.astype(float).to_numpy()
    p1 = merged.p_calibrated_expanded.astype(float).to_numpy()
    groups = merged[GROUP_COL].to_numpy()
    dap = paired_delta_boot(y, p0, p1, groups, 'AP', RANDOM_STATE + 1000)
    dauc = paired_delta_boot(y, p0, p1, groups, 'ROC_AUC', RANDOM_STATE + 2000)
    comparison = pd.DataFrame([{ 'baseline_Test_AP': baseline['summary']['Test_AP'], 'expanded_Test_AP': expanded['summary']['Test_AP'], 'Delta_AP_expanded_minus_baseline': dap[0], 'Delta_AP_CI_lower': dap[1], 'Delta_AP_CI_upper': dap[2], 'Delta_AP_bootstrap_p_value': dap[3], 'baseline_Test_ROC_AUC': baseline['summary']['Test_ROC_AUC'], 'expanded_Test_ROC_AUC': expanded['summary']['Test_ROC_AUC'], 'Delta_ROC_AUC_expanded_minus_baseline': dauc[0], 'Delta_ROC_AUC_CI_lower': dauc[1], 'Delta_ROC_AUC_CI_upper': dauc[2], 'Delta_ROC_AUC_bootstrap_p_value': dauc[3]}])
    summary = pd.DataFrame([baseline['summary'], expanded['summary']])
    candidates = pd.concat(all_candidates, ignore_index=True)
    folds_df = pd.concat(all_folds, ignore_index=True)
    missingness = pd.DataFrame([{'column': c, 'n_missing_or_blank': int(work[c].isna().sum()), 'percent_missing_or_blank': 100 * float(work[c].isna().sum()) / len(work)} for c in DYNAMIC_PROM_FEATURES if c in work.columns])
    xlsx = os.path.join(OUTPUT_DIR, f'Step2_DynamicPROM_{{CLASSIFIER_KEY}}_summary.xlsx')
    with pd.ExcelWriter(xlsx, engine='openpyxl') as writer:
        methods_table().to_excel(writer, 'methods_rationale', index=False)
        dynamic_audit.to_excel(writer, 'dynamic_feature_audit', index=False)
        summary.to_excel(writer, 'model_performance', index=False)
        comparison.to_excel(writer, 'paired_PROM_comparison', index=False)
        candidates.to_excel(writer, 'cv_candidates_all_models', index=False)
        folds_df.to_excel(writer, 'cv_fold_metrics_all', index=False)
        split_audit(work).to_excel(writer, 'split_audit', index=False)
        missingness.to_excel(writer, 'missingness_audit', index=False)
        style_excel(writer)
    summary.to_csv(os.path.join(OUTPUT_DIR, 'model_performance.csv'), index=False)
    comparison.to_csv(os.path.join(OUTPUT_DIR, 'paired_PROM_comparison.csv'), index=False)
    candidates.to_csv(os.path.join(OUTPUT_DIR, 'cv_candidates_all_models.csv'), index=False)
    folds_df.to_csv(os.path.join(OUTPUT_DIR, 'cv_fold_metrics_all.csv'), index=False)
    work[[GROUP_COL, 'split']].drop_duplicates().to_csv(os.path.join(OUTPUT_DIR, 'split_assignment.csv'), index=False)
    manifest = {'classifier': CLASSIFIER_DISPLAY_NAME, 'output_dir': OUTPUT_DIR, 'design': f'paired baseline-only and dynamic PROM-expanded {{CLASSIFIER_DISPLAY_NAME}} models', 'model_selection': 'highest mean group-aware CV average precision inside training split', 'calibration': 'isotonic', 'threshold_strategy': 'maximum F1 on calibration split', 'runtime_minutes': float((time.time() - t0) / 60), 'summary_xlsx': xlsx}
    json.dump(json_native(manifest), open(os.path.join(OUTPUT_DIR, 'run_manifest.json'), 'w'), indent=2, sort_keys=True)
    zip_path = os.path.join(OUTPUT_DIR, f'Step2_DynamicPROM_{{CLASSIFIER_KEY}}_outputs.zip')
    tmp = zip_path + '.tmp'
    for p in [zip_path, tmp]:
        if os.path.exists(p):
            os.remove(p)
    with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=ZIP_COMPRESSION_LEVEL) as z:
        for root, _, files in os.walk(OUTPUT_DIR):
            for name in files:
                full = os.path.join(root, name)
                if full in {zip_path, tmp}:
                    continue
                z.write(full, os.path.relpath(full, OUTPUT_DIR))
    os.replace(tmp, zip_path)
    print('=' * 100)
    print(f'Step 2 {{CLASSIFIER_DISPLAY_NAME}} analysis completed')
    print('Summary Excel:', xlsx)
    print('ZIP archive:', zip_path)
    print('=' * 100)
    if CREATE_COLAB_DOWNLOAD_LINK:
        try:
            from IPython.display import HTML, display
            display(HTML(f'<p><b>Step 2 {{CLASSIFIER_DISPLAY_NAME}} output archive is ready.</b></p><p><a href="/files{{zip_path}}" download>Click here to download the ZIP archive</a></p><p>Path: <code>{{zip_path}}</code></p>'))
        except Exception:
            pass
    if AUTO_DOWNLOAD_ZIP:
        try:
            from google.colab import files
            files.download(zip_path)
        except Exception:
            pass

if __name__ == '__main__':
    main()


#**Step 2: HistGradientBoosting**

In [ ]:
"""
Step 2 dynamic postoperative PROM analysis with HistGradientBoosting
====================================================================

Input
-----
/content/Step 2_ODI_Cohort.csv

Target
------
final_reop_step2
    1 = reoperation from postoperative day 91 through day 365
    0 = no reoperation from postoperative day 91 through day 365

Design
------
This script runs paired baseline-only and dynamic PROM-expanded HistGradientBoosting
models for delayed lumbar reoperation prediction. The baseline-only model
uses the same 35 baseline variables as Step 1. The dynamic PROM-expanded
model includes the same baseline variables plus preoperative PROM,
postoperative PROM, PROM change, PROM change rate, relative MCID status,
and timing of postoperative PROM assessment. Paired models use identical
patient-level train/calibration/test splits and identical group-aware
cross-validation fold construction. Hyperparameter tuning is performed
exclusively within the training split using cross-validated average
precision as the primary selection metric. Probability calibration and
threshold selection are performed only on the calibration split. The
held-out test set is reserved until the model-development pipeline is
locked.
"""

# -*- coding: utf-8 -*-

import os
import re
import sys
import json
import math
import time
import zipfile
import platform
import subprocess
import warnings
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    import joblib
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'joblib'])
    import joblib

try:
    import openpyxl  # noqa
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl'])
    import openpyxl  # noqa

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedGroupKFold, ParameterSampler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_recall_curve,
    roc_curve,
    confusion_matrix,
    f1_score,
)
from sklearn.isotonic import IsotonicRegression
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')


from sklearn.ensemble import HistGradientBoostingClassifier


BASE_DIR = '/content'
INPUT_CSV = os.path.join(BASE_DIR, 'Step 2_ODI_Cohort.csv')
CLASSIFIER_KEY = 'HistGradientBoosting'
CLASSIFIER_DISPLAY_NAME = 'HistGradientBoosting'
SCALE_CONTINUOUS = False
OUTPUT_DIR = os.path.join(BASE_DIR, 'Step2_DynamicPROM_HistGradientBoosting_publication')
PLOT_DIR = os.path.join(OUTPUT_DIR, 'plots')
ARTIFACT_DIR = os.path.join(OUTPUT_DIR, 'model_artifacts')
for d in [OUTPUT_DIR, PLOT_DIR, ARTIFACT_DIR]:
    os.makedirs(d, exist_ok=True)

TARGET_COL = 'final_reop_step2'
GROUP_COL = 'PersonKey'
RANDOM_STATE = 20260524
TEST_FRACTION = 0.20
CALIBRATION_FRACTION_OF_REMAINING = 0.20
N_CV_FOLDS = 5
N_RANDOM_COMBINATIONS = 300
USE_EARLY_STOPPING_IN_CV = True
EARLY_STOPPING_ROUNDS = 100
MIN_FINAL_N_ESTIMATORS = 50
THRESHOLD_STRATEGY = 'max_f1'
N_BOOTSTRAPS = 2000
ECE_N_BINS = 10
N_JOBS = -1
LEARNING_CURVE_TRAIN_FRACTIONS = (0.20, 0.40, 0.60, 0.80, 1.00)
LEARNING_CURVE_CV_FOLDS = 3
ZIP_COMPRESSION_LEVEL = 1
CREATE_COLAB_DOWNLOAD_LINK = True
AUTO_DOWNLOAD_ZIP = False
RELATIVE_ODI_MCID_CUTOFF = 0.30
PROM_CHANGE_RATE_COL = 'ODI_change_rate'
RELATIVE_ODI_MCID_COL = 'ODI_relative_MCID_binary'
DAYS_BETWEEN_PROM_COL = 'days_between_PROMs'

BASELINE_FEATURES = ['finaldx_degenerative', 'finaldx_radicular', 'finaldx_stenosis', 'finaldx_deformity_instability', 'finaldx_other_diagnosis', 'age', 'sex', 'race', 'ethnicity', 'cancer_status', 'chronic_pulmonary_disease', 'congestive_heart_failure', 'connective_tissue_rheumatic_disease', 'diabetes_status', 'myocardial_infarction', 'renal_disease', 'institution_type', 'institution_size', 'institution_region', 'asa', 'bmi', 'payer_status', 'alif_llif', 'corpectomy', 'discectomy', 'foraminotomy', 'instrumentation', 'laminectomy_posterior_decompression', 'pelvic_fixation', 'plf', 'tlif_plif', 'other_lumbar_procedure', 'number_operated_levels', 'operative_region_extent', 'PatTobaccoUse']
assert len(BASELINE_FEATURES) == 35
STEP2_PROM_FEATURES = ['preop_ODI', 'postop_ODI', 'ODI_change', PROM_CHANGE_RATE_COL, RELATIVE_ODI_MCID_COL, 'postop_ODI_day']
DYNAMIC_PROM_FEATURES = BASELINE_FEATURES + STEP2_PROM_FEATURES
CONTINUOUS_BASELINE_FEATURES = ['age', 'bmi']
CONTINUOUS_FEATURES_ALL = CONTINUOUS_BASELINE_FEATURES + ['preop_ODI', 'postop_ODI', 'ODI_change', PROM_CHANGE_RATE_COL, 'postop_ODI_day']
BINARY_FEATURES = ['finaldx_degenerative', 'finaldx_radicular', 'finaldx_stenosis', 'finaldx_deformity_instability', 'finaldx_other_diagnosis', 'sex', 'ethnicity', 'cancer_status', 'chronic_pulmonary_disease', 'congestive_heart_failure', 'connective_tissue_rheumatic_disease', 'myocardial_infarction', 'renal_disease', 'institution_type', 'alif_llif', 'corpectomy', 'discectomy', 'foraminotomy', 'instrumentation', 'laminectomy_posterior_decompression', 'pelvic_fixation', 'plf', 'tlif_plif', 'other_lumbar_procedure', 'operative_region_extent', RELATIVE_ODI_MCID_COL]
ORDINAL_FEATURES = ['diabetes_status', 'institution_size', 'asa', 'number_operated_levels']
NOMINAL_FEATURES = ['race', 'institution_region', 'payer_status', 'PatTobaccoUse']

SEARCH_SPACE = {
    'max_iter': [200, 400, 700, 1000, 1400, 1800],
    'learning_rate': [0.005, 0.01, 0.02, 0.03, 0.05, 0.08],
    'max_leaf_nodes': [7, 15, 31, 63],
    'max_depth': [2, 3, 5, 7, None],
    'min_samples_leaf': [10, 20, 50, 100, 200],
    'l2_regularization': [0.0, 0.001, 0.01, 0.10, 0.50, 1.00, 2.00],
    'positive_weight_multiplier': [0.25, 0.50, 0.75, 1.00, 1.50, 2.00, 3.00, 4.00, 6.00, 8.00],
}
ITERATION_PARAM = 'max_iter'
INT_PARAMS = {'max_iter', 'max_leaf_nodes', 'min_samples_leaf'}
FLOAT_PARAMS = {'learning_rate', 'l2_regularization', 'positive_weight_multiplier'}

PARAMETER_CANDIDATES = list(ParameterSampler(SEARCH_SPACE, n_iter=N_RANDOM_COMBINATIONS, random_state=RANDOM_STATE))

MISSING_STRINGS = {'', ' ', 'na', 'n/a', 'nan', 'none', 'null', '.', 'missing', '<na>'}
BINARY_MAPS = {
    'sex': {'female': 0, 'f': 0, 'male': 1, 'm': 1},
    'ethnicity': {'non-hispanic': 0, 'non hispanic': 0, 'hispanic': 1},
    'cancer_status': {'no cancer': 0, 'no': 0, 'none': 0, 'any cancer': 1, 'yes': 1, 'cancer': 1},
    'institution_type': {'hospital': 0, 'non-hospital': 1, 'non hospital': 1, 'nonhospital': 1},
    'operative_region_extent': {'lumbar only': 0, 'extended_region_involvement': 1, 'extended region involvement': 1, 'extended': 1},
}
ORDINAL_MAPS = {
    'diabetes_status': {'no': 0, 'none': 0, '0': 0, 'without comp': 1, 'without complication': 1, 'without complications': 1, '1': 1, 'with comp': 2, 'with complication': 2, 'with complications': 2, '2': 2},
    'institution_size': {'between 1-99 beds': 0, '1-99': 0, 'between 100-399 beds': 1, '100-399': 1, '>= 400 beds': 2, '>=400 beds': 2, '>=400': 2, '>= 400': 2},
    'asa': {'1': 1, 'i': 1, '2': 2, 'ii': 2, '3': 3, 'iii': 3, '4': 4, 'iv': 4, '>=4': 4, '>= 4': 4, '5': 4, 'v': 4},
    'number_operated_levels': {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '>=4': 4, '>= 4': 4, '5': 4, '6': 4, '7': 4, '8': 4, '9': 4, '10': 4},
}
PREFERRED_NOMINAL_LEVELS = {
    'race': ['White', 'Black', 'Other'],
    'institution_region': ['South', 'North East', 'West', 'Midwest'],
    'payer_status': ['Medicare', 'Commercial/Private', 'Other', 'Medicaid/Public/Government'],
    'PatTobaccoUse': ['Unknown/Not reported/Multiple', 'Never', 'Former', 'Current'],
}
FEATURE_LABELS = {
    'age': 'Age', 'bmi': 'BMI', 'sex': 'Male sex', 'race': 'Race', 'ethnicity': 'Hispanic ethnicity',
    'cancer_status': 'Any cancer', 'diabetes_status': 'Diabetes status', 'institution_type': 'Non-hospital institution',
    'institution_size': 'Institution size', 'institution_region': 'Institution region', 'asa': 'ASA physical status',
    'payer_status': 'Primary payer', 'PatTobaccoUse': 'Tobacco use',
    'finaldx_degenerative': 'Degenerative diagnosis', 'finaldx_radicular': 'Radiculopathy diagnosis',
    'finaldx_stenosis': 'Spinal stenosis diagnosis', 'finaldx_deformity_instability': 'Deformity or instability diagnosis',
    'finaldx_other_diagnosis': 'Other lumbar diagnosis', 'chronic_pulmonary_disease': 'Chronic pulmonary disease',
    'congestive_heart_failure': 'Congestive heart failure', 'connective_tissue_rheumatic_disease': 'Connective tissue/rheumatic disease',
    'myocardial_infarction': 'Myocardial infarction', 'renal_disease': 'Renal disease',
    'alif_llif': 'Anterior/lateral lumbar interbody fusion', 'corpectomy': 'Corpectomy', 'discectomy': 'Discectomy',
    'foraminotomy': 'Foraminotomy', 'instrumentation': 'Instrumentation',
    'laminectomy_posterior_decompression': 'Posterior decompression or laminectomy', 'pelvic_fixation': 'Pelvic fixation',
    'plf': 'Posterolateral fusion', 'tlif_plif': 'Posterior/transforaminal lumbar interbody fusion',
    'other_lumbar_procedure': 'Other lumbar procedure', 'number_operated_levels': 'Number of operated levels',
    'operative_region_extent': 'Operative region extent',
}
FEATURE_LABELS.update({'preop_ODI': 'Preoperative ODI', 'postop_ODI': 'Postoperative ODI', 'ODI_change': 'Change in ODI', PROM_CHANGE_RATE_COL: 'ODI change rate', RELATIVE_ODI_MCID_COL: 'Relative ODI MCID', 'postop_ODI_day': 'Timing of postoperative ODI assessment'})

def clean_scalar(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().replace('≥', '>=')
        return np.nan if s.lower() in MISSING_STRINGS else s
    return x

def norm_text(x):
    x = clean_scalar(x)
    if pd.isna(x):
        return None
    return str(x).strip().replace('≥', '>=').lower()

def to_binary_target(x):
    sx = norm_text(x)
    if sx is None:
        return np.nan
    if sx in {'1', '1.0', 'yes', 'y', 'true', 't'}:
        return 1.0
    if sx in {'0', '0.0', 'no', 'n', 'false', 'f'}:
        return 0.0
    try:
        v = float(sx)
        return float(v) if v in (0.0, 1.0) else np.nan
    except Exception:
        return np.nan

def safe_filename(x):
    return re.sub(r'_+', '_', re.sub(r'[^A-Za-z0-9_.-]+', '_', str(x))).strip('_')[:180] or 'file'

def json_native(obj):
    if isinstance(obj, dict):
        return {str(k): json_native(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_native(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj

def sanitize_params(params):
    out = {}
    for k, v in params.items():
        if k in INT_PARAMS:
            out[k] = int(round(float(v)))
        elif k in FLOAT_PARAMS:
            out[k] = float(v)
        else:
            out[k] = v
    return out

def safe_average_precision(y, p):
    return np.nan if len(np.unique(y)) < 2 else float(average_precision_score(y, p))

def safe_roc_auc(y, p):
    return np.nan if len(np.unique(y)) < 2 else float(roc_auc_score(y, p))

def expected_calibration_error(y, p, n_bins=10):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    out = 0.0
    if len(y) == 0:
        return np.nan
    for i in range(n_bins):
        mask = (p >= bins[i]) & ((p <= bins[i + 1]) if i == n_bins - 1 else (p < bins[i + 1]))
        if np.any(mask):
            out += (np.sum(mask) / len(y)) * abs(float(np.mean(y[mask])) - float(np.mean(p[mask])))
    return float(out)

def classification_metrics(y, p, threshold):
    y = np.asarray(y).astype(int)
    pred = (np.asarray(p) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        'threshold': float(threshold),
        'F1': float(f1_score(y, pred, zero_division=0)),
        'Sensitivity': tp / (tp + fn) if tp + fn > 0 else np.nan,
        'Specificity': tn / (tn + fp) if tn + fp > 0 else np.nan,
        'PPV': tp / (tp + fp) if tp + fp > 0 else np.nan,
        'NPV': tn / (tn + fn) if tn + fn > 0 else np.nan,
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
        'Predicted_positive_rate': float(np.mean(pred)),
    }

def top5_metrics(y, p):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    n = len(y)
    k = max(1, int(math.ceil(n * 0.05)))
    idx = np.argsort(-p)[:k]
    prev = float(np.mean(y)) if n else np.nan
    rate = float(np.mean(y[idx])) if n else np.nan
    return {
        'Top_5pct_n': int(k),
        'Top_5pct_event_rate': rate,
        'Top_5pct_lift': rate / prev if prev > 0 else np.nan,
        'Top_5pct_captured_events': float(np.sum(y[idx]) / np.sum(y)) if np.sum(y) > 0 else np.nan,
    }

def select_threshold(y, p):
    precision, recall, thresholds = precision_recall_curve(y, p)
    if len(thresholds) == 0:
        threshold = 0.5
    else:
        precision = precision[:-1]
        recall = recall[:-1]
        f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
        threshold = float(thresholds[int(np.nanargmax(f1))])
    return threshold, classification_metrics(y, p, threshold)

def eval_preds(y, p, threshold=None, prefix=''):
    out = {
        f'{prefix}AP': safe_average_precision(y, p),
        f'{prefix}ROC_AUC': safe_roc_auc(y, p),
        f'{prefix}Brier_score': float(brier_score_loss(y, p)),
        f'{prefix}ECE': expected_calibration_error(y, p, ECE_N_BINS),
        f'{prefix}N': int(len(y)),
        f'{prefix}Events': int(np.sum(y)),
        f'{prefix}Prevalence': float(np.mean(y)),
    }
    if threshold is not None:
        out.update({f'{prefix}{k}': v for k, v in classification_metrics(y, p, threshold).items()})
        out.update({f'{prefix}{k}': v for k, v in top5_metrics(y, p).items()})
    return out

def make_weights(y, multiplier):
    y = np.asarray(y).astype(int)
    n_pos = int(np.sum(y == 1))
    n_neg = int(np.sum(y == 0))
    if n_pos == 0:
        raise ValueError('No positive events in training fold.')
    pos_weight = (n_neg / max(n_pos, 1)) * float(multiplier)
    return np.where(y == 1, pos_weight, 1.0).astype(float)

def actual_positive_weight(y, multiplier):
    y = np.asarray(y).astype(int)
    return float((np.sum(y == 0) / max(np.sum(y == 1), 1)) * multiplier)

@dataclass
class ModelPreprocessor:
    continuous_features: List[str]
    binary_features: List[str]
    ordinal_features: List[str]
    nominal_features: List[str]
    preferred_nominal_levels: Dict[str, List[str]]
    scale_continuous: bool = False

    def __post_init__(self):
        self.continuous_imputer = None
        self.discrete_imputer = None
        self.nominal_imputer = None
        self.scaler = None
        self.onehot = None
        self.output_feature_names_ = []

    def _parse_binary(self, x, feature):
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in BINARY_MAPS and sx in BINARY_MAPS[feature]:
            return float(BINARY_MAPS[feature][sx])
        if sx in {'1', '1.0', 'yes', 'y', 'true', 't', 'present', 'positive'}:
            return 1.0
        if sx in {'0', '0.0', 'no', 'n', 'false', 'f', 'absent', 'negative'}:
            return 0.0
        try:
            v = float(sx)
            return float(v) if v in (0.0, 1.0) else np.nan
        except Exception:
            return np.nan

    def _parse_ordinal(self, x, feature):
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in ORDINAL_MAPS and sx in ORDINAL_MAPS[feature]:
            return float(ORDINAL_MAPS[feature][sx])
        try:
            v = float(sx)
            if feature == 'asa':
                return float(min(max(int(round(v)), 1), 4))
            if feature == 'number_operated_levels':
                return float(min(max(int(round(v)), 0), 4))
            if feature == 'diabetes_status':
                return float(min(max(int(round(v)), 0), 2))
            if feature == 'institution_size':
                return float(min(max(int(round(v)), 0), 2))
            return float(v)
        except Exception:
            return np.nan

    def _nominal(self, feature, x):
        x = clean_scalar(x)
        if pd.isna(x):
            return np.nan
        s = str(x).strip()
        sl = s.lower()
        if feature == 'race':
            return 'White' if sl == 'white' else ('Black' if sl == 'black' else 'Other')
        if feature == 'institution_region':
            mapping = {'south': 'South', 'north east': 'North East', 'northeast': 'North East', 'north-east': 'North East', 'west': 'West', 'midwest': 'Midwest', 'mid west': 'Midwest'}
            return mapping.get(sl, s)
        if feature == 'payer_status':
            if sl == 'medicare':
                return 'Medicare'
            if sl in {'commercial/private', 'commercial', 'private', 'commercial private'}:
                return 'Commercial/Private'
            if sl in {'medicaid/public/government', 'medicaid', 'public', 'government', 'public/government'}:
                return 'Medicaid/Public/Government'
            return 'Other'
        if feature == 'PatTobaccoUse':
            return 'Never' if sl == 'never' else ('Former' if sl == 'former' else ('Current' if sl == 'current' else 'Unknown/Not reported/Multiple'))
        return s

    def _parts(self, X):
        cont = pd.DataFrame(index=X.index)
        disc = pd.DataFrame(index=X.index)
        nom = pd.DataFrame(index=X.index)
        for c in self.continuous_features:
            cont[c] = pd.to_numeric(X[c].map(clean_scalar), errors='coerce')
        for c in self.binary_features:
            disc[c] = X[c].map(lambda z: self._parse_binary(z, c)).astype(float)
        for c in self.ordinal_features:
            disc[c] = X[c].map(lambda z: self._parse_ordinal(z, c)).astype(float)
        for c in self.nominal_features:
            nom[c] = X[c].map(lambda z: self._nominal(c, z)).astype('object')
        return cont, disc, nom

    def fit(self, X):
        cont, disc, nom = self._parts(X)
        self.continuous_imputer = SimpleImputer(strategy='median')
        self.discrete_imputer = SimpleImputer(strategy='most_frequent')
        self.nominal_imputer = SimpleImputer(strategy='constant', fill_value='Missing')
        cont_imp = self.continuous_imputer.fit_transform(cont)
        if self.scale_continuous:
            self.scaler = StandardScaler()
            self.scaler.fit(cont_imp)
        self.discrete_imputer.fit(disc)
        nom_imp = pd.DataFrame(self.nominal_imputer.fit_transform(nom), columns=self.nominal_features)
        cats = []
        for c in self.nominal_features:
            preferred = list(self.preferred_nominal_levels.get(c, []))
            observed = nom_imp[c].astype(str).unique().tolist()
            cats.append(preferred + sorted([x for x in observed if x not in preferred]))
        try:
            self.onehot = OneHotEncoder(categories=cats, handle_unknown='ignore', sparse_output=False)
        except TypeError:
            self.onehot = OneHotEncoder(categories=cats, handle_unknown='ignore', sparse=False)
        self.onehot.fit(nom_imp.astype(str))
        self.output_feature_names_ = self.continuous_features + self.binary_features + self.ordinal_features + self.onehot.get_feature_names_out(self.nominal_features).tolist()
        return self

    def transform(self, X):
        cont, disc, nom = self._parts(X)
        cont_imp = self.continuous_imputer.transform(cont)
        if self.scale_continuous and self.scaler is not None:
            cont_imp = self.scaler.transform(cont_imp)
        disc_imp = self.discrete_imputer.transform(disc)
        nom_imp = pd.DataFrame(self.nominal_imputer.transform(nom), columns=self.nominal_features)
        nom_oh = self.onehot.transform(nom_imp.astype(str))
        return np.concatenate([cont_imp, disc_imp, nom_oh], axis=1).astype(float)

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

    @property
    def output_feature_names(self):
        return self.output_feature_names_

def raw_model_scores(model, Xp, n_iter=None):
    if CLASSIFIER_KEY == 'HistGradientBoosting' and n_iter is not None:
        selected = None
        for i, pred in enumerate(model.staged_predict_proba(Xp), start=1):
            selected = pred
            if i >= int(n_iter):
                break
        if selected is not None:
            return selected[:, 1]
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(Xp)[:, 1]
    if hasattr(model, 'decision_function'):
        return model.decision_function(Xp)
    raise AttributeError('Model provides neither predict_proba nor decision_function.')

def predict(pre, model, X, features, n_iter=None):
    return raw_model_scores(model, pre.transform(X[features]), n_iter=n_iter)

def patient_split(df, target_col, seed):
    group_df = df.groupby(GROUP_COL, dropna=False)[target_col].max().reset_index()
    y_group = group_df[target_col].astype(int).to_numpy()
    groups = group_df[GROUP_COL].to_numpy()
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=seed)
    train_cal_idx, test_idx = next(sss1.split(groups, y_group))
    groups_train_cal = groups[train_cal_idx]
    y_train_cal = y_group[train_cal_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=CALIBRATION_FRACTION_OF_REMAINING, random_state=seed + 1)
    train_idx_rel, cal_idx_rel = next(sss2.split(groups_train_cal, y_train_cal))
    train_groups = set(groups_train_cal[train_idx_rel])
    cal_groups = set(groups_train_cal[cal_idx_rel])
    test_groups = set(groups[test_idx])
    assert train_groups.isdisjoint(cal_groups) and train_groups.isdisjoint(test_groups) and cal_groups.isdisjoint(test_groups)
    return df[GROUP_COL].isin(train_groups).to_numpy(), df[GROUP_COL].isin(cal_groups).to_numpy(), df[GROUP_COL].isin(test_groups).to_numpy()

def cv_splits(y, groups, seed, n_folds=N_CV_FOLDS):
    group_df = pd.DataFrame({'group': groups, 'y': y}).groupby('group')['y'].max().reset_index()
    n_folds = min(n_folds, int(np.sum(group_df.y == 1)), int(np.sum(group_df.y == 0)))
    if n_folds < 2:
        raise ValueError('Not enough positive and negative patient groups for group-aware CV.')
    cv = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    return [(tr, va) for tr, va in cv.split(np.zeros(len(y)), y, groups)]

def feature_types(features):
    cont = [f for f in features if f in CONTINUOUS_FEATURES_ALL or (f not in BINARY_FEATURES and f not in ORDINAL_FEATURES and f not in NOMINAL_FEATURES)]
    binary = [f for f in features if f in BINARY_FEATURES]
    ordinal = [f for f in features if f in ORDINAL_FEATURES]
    nominal = [f for f in features if f in NOMINAL_FEATURES]
    return cont, binary, ordinal, nominal

def fit_pipeline(X, y, features, params, seed, eval_set=None, early=False, n_estimators_override=None):
    cont, binary, ordinal, nominal = feature_types(features)
    pre = ModelPreprocessor(cont, binary, ordinal, nominal, PREFERRED_NOMINAL_LEVELS, scale_continuous=SCALE_CONTINUOUS)
    Xp = pre.fit_transform(X[features])
    eval_transformed = None
    if eval_set is not None:
        X_val, y_val = eval_set
        eval_transformed = (pre.transform(X_val[features]), y_val)
    model, best_iter = fit_estimator(Xp, y, params, seed, eval_set=eval_transformed, use_early_stopping=early, n_estimators_override=n_estimators_override)
    return pre, model, best_iter

def tune(X_train, y_train, groups_train, features, folds, model_key, seed):
    candidate_rows = []
    fold_rows = []
    sampler = list(ParameterSampler(SEARCH_SPACE, n_iter=N_RANDOM_COMBINATIONS, random_state=seed))
    for cid, raw_params in enumerate(sampler, start=1):
        params = sanitize_params(raw_params)
        fold_train_aps, fold_train_aucs, fold_aps, fold_aucs, best_iters = [], [], [], [], []
        t0 = time.time()
        for fid, (tr_idx, va_idx) in enumerate(folds, start=1):
            X_tr = X_train.iloc[tr_idx].reset_index(drop=True)
            y_tr = y_train[tr_idx]
            X_va = X_train.iloc[va_idx].reset_index(drop=True)
            y_va = y_train[va_idx]
            pre, model, best_iter = fit_pipeline(X_tr, y_tr, features, params, seed + cid * 1000 + fid, eval_set=(X_va, y_va), early=USE_EARLY_STOPPING_IN_CV)
            p_tr = predict(pre, model, X_tr, features, n_iter=best_iter)
            p_va = predict(pre, model, X_va, features, n_iter=best_iter)
            tr_ap, tr_auc = safe_average_precision(y_tr, p_tr), safe_roc_auc(y_tr, p_tr)
            va_ap, va_auc = safe_average_precision(y_va, p_va), safe_roc_auc(y_va, p_va)
            fold_train_aps.append(tr_ap); fold_train_aucs.append(tr_auc); fold_aps.append(va_ap); fold_aucs.append(va_auc)
            if best_iter is not None and best_iter > 0:
                best_iters.append(best_iter)
            fold_rows.append({
                'model_key': model_key, 'candidate_id': cid, 'fold': fid,
                'fold_train_AP': tr_ap, 'fold_train_ROC_AUC': tr_auc,
                'fold_validation_AP': va_ap, 'fold_validation_ROC_AUC': va_auc,
                'fold_train_minus_validation_AP_gap': tr_ap - va_ap if np.isfinite(tr_ap) and np.isfinite(va_ap) else np.nan,
                'fold_train_minus_validation_ROC_AUC_gap': tr_auc - va_auc if np.isfinite(tr_auc) and np.isfinite(va_auc) else np.nan,
                'fold_best_iteration': best_iter,
                'positive_weight_used': actual_positive_weight(y_tr, params['positive_weight_multiplier']),
                **params,
            })
        locked = int(np.median(best_iters)) if best_iters and USE_EARLY_STOPPING_IN_CV else (int(params[ITERATION_PARAM]) if ITERATION_PARAM and ITERATION_PARAM in params else np.nan)
        row = {
            'model_key': model_key, 'candidate_id': cid, 'cv_folds': len(folds),
            'cv_train_AP_mean': float(np.nanmean(fold_train_aps)),
            'cv_train_ROC_AUC_mean': float(np.nanmean(fold_train_aucs)),
            'cv_AP_mean': float(np.nanmean(fold_aps)),
            'cv_AP_SD': float(np.nanstd(fold_aps, ddof=1)) if len(fold_aps) > 1 else np.nan,
            'cv_ROC_AUC_mean': float(np.nanmean(fold_aucs)),
            'cv_ROC_AUC_SD': float(np.nanstd(fold_aucs, ddof=1)) if len(fold_aucs) > 1 else np.nan,
            'mean_cv_best_iteration': float(np.mean(best_iters)) if best_iters else np.nan,
            'locked_n_estimators_from_cv': locked,
            'elapsed_seconds': float(time.time() - t0),
            **params,
        }
        candidate_rows.append(row)
        print(f"{model_key} | candidate {cid:03d}/{len(sampler)} | CV AP={row['cv_AP_mean']:.5f} | AUC={row['cv_ROC_AUC_mean']:.5f}")
    return pd.DataFrame(candidate_rows).sort_values('cv_AP_mean', ascending=False).reset_index(drop=True), pd.DataFrame(fold_rows)

def fit_final(X_train, y_train, X_cal, y_cal, X_test, features, params, locked_n, seed):
    n_override = int(locked_n) if pd.notna(locked_n) and USE_EARLY_STOPPING_IN_CV and ITERATION_PARAM else None
    pre, model, _ = fit_pipeline(X_train, y_train, features, params, seed, n_estimators_override=n_override)
    p_train_raw = predict(pre, model, X_train, features)
    p_cal_raw = predict(pre, model, X_cal, features)
    p_test_raw = predict(pre, model, X_test, features)
    calibrator = IsotonicRegression(out_of_bounds='clip')
    calibrator.fit(p_cal_raw, y_cal)
    p_train = calibrator.predict(p_train_raw)
    p_cal = calibrator.predict(p_cal_raw)
    p_test = calibrator.predict(p_test_raw)
    threshold, cal_metrics = select_threshold(y_cal, p_cal)
    return {
        'pre': pre, 'model': model, 'calibrator': calibrator,
        'p_train_raw': p_train_raw, 'p_cal_raw': p_cal_raw, 'p_test_raw': p_test_raw,
        'p_train': p_train, 'p_cal': p_cal, 'p_test': p_test,
        'threshold': threshold, 'cal_metrics': cal_metrics,
    }

def patient_boot_ci(y, p, groups, metric, threshold=None, seed=1):
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({'idx': np.arange(len(y)), 'group': groups, 'y': np.asarray(y).astype(int)})
    group_y = d.groupby('group').y.max()
    pos = group_y[group_y == 1].index.to_numpy()
    neg = group_y[group_y == 0].index.to_numpy()
    if len(pos) == 0 or len(neg) == 0:
        return np.nan, np.nan
    by_group = {g: d.loc[d.group.eq(g), 'idx'].to_numpy() for g in group_y.index}
    vals = []
    for _ in range(N_BOOTSTRAPS):
        sample_groups = np.concatenate([rng.choice(pos, len(pos), replace=True), rng.choice(neg, len(neg), replace=True)])
        sample_idx = np.concatenate([by_group[g] for g in sample_groups])
        yy = np.asarray(y)[sample_idx]
        pp = np.asarray(p)[sample_idx]
        if len(np.unique(yy)) < 2:
            continue
        if metric == 'AP':
            vals.append(average_precision_score(yy, pp))
        elif metric == 'ROC_AUC':
            vals.append(roc_auc_score(yy, pp))
        elif metric == 'F1':
            vals.append(f1_score(yy, pp >= threshold, zero_division=0))
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))) if vals else (np.nan, np.nan)

def paired_delta_boot(y, p0, p1, groups, metric, seed=1):
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({'idx': np.arange(len(y)), 'group': groups, 'y': np.asarray(y).astype(int)})
    group_y = d.groupby('group').y.max()
    pos = group_y[group_y == 1].index.to_numpy()
    neg = group_y[group_y == 0].index.to_numpy()
    by_group = {g: d.loc[d.group.eq(g), 'idx'].to_numpy() for g in group_y.index}
    observed = (safe_average_precision(y, p1) - safe_average_precision(y, p0)) if metric == 'AP' else (safe_roc_auc(y, p1) - safe_roc_auc(y, p0))
    vals = []
    for _ in range(N_BOOTSTRAPS):
        sample_groups = np.concatenate([rng.choice(pos, len(pos), replace=True), rng.choice(neg, len(neg), replace=True)])
        sample_idx = np.concatenate([by_group[g] for g in sample_groups])
        yy = np.asarray(y)[sample_idx]
        if len(np.unique(yy)) < 2:
            continue
        if metric == 'AP':
            vals.append(average_precision_score(yy, np.asarray(p1)[sample_idx]) - average_precision_score(yy, np.asarray(p0)[sample_idx]))
        else:
            vals.append(roc_auc_score(yy, np.asarray(p1)[sample_idx]) - roc_auc_score(yy, np.asarray(p0)[sample_idx]))
    if not vals:
        return observed, np.nan, np.nan, np.nan
    vals = np.asarray(vals)
    ci_low, ci_high = float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))
    p_value = float(2 * min(np.mean(vals <= 0), np.mean(vals >= 0)))
    return float(observed), ci_low, ci_high, min(p_value, 1.0)

def save_learning_curve(X_train, y_train, groups_train, features, params, locked_n, seed, out_path, title):
    group_df = pd.DataFrame({'group': groups_train, 'y': y_train}).groupby('group').y.max().reset_index()
    rng = np.random.default_rng(seed)
    rows = []
    for frac in LEARNING_CURVE_TRAIN_FRACTIONS:
        for rep in range(LEARNING_CURVE_CV_FOLDS):
            selected_groups = []
            for cls in [0, 1]:
                cls_groups = group_df.loc[group_df.y.eq(cls), 'group'].to_numpy()
                k = max(1, int(math.ceil(len(cls_groups) * frac)))
                selected_groups.extend(rng.choice(cls_groups, k, replace=False).tolist())
            mask = np.isin(groups_train, selected_groups)
            X_sub = X_train.loc[mask].reset_index(drop=True)
            y_sub = y_train[mask]
            g_sub = groups_train[mask]
            try:
                folds = cv_splits(y_sub, g_sub, seed + rep, n_folds=3)
            except Exception:
                continue
            for tr, va in folds:
                n_override = int(locked_n) if pd.notna(locked_n) and USE_EARLY_STOPPING_IN_CV and ITERATION_PARAM else None
                pre, model, _ = fit_pipeline(X_sub.iloc[tr].reset_index(drop=True), y_sub[tr], features, params, seed + rep, n_estimators_override=n_override)
                p_tr = predict(pre, model, X_sub.iloc[tr].reset_index(drop=True), features)
                p_va = predict(pre, model, X_sub.iloc[va].reset_index(drop=True), features)
                rows.append({'train_fraction': frac, 'train_AP': safe_average_precision(y_sub[tr], p_tr), 'validation_AP': safe_average_precision(y_sub[va], p_va)})
    lc = pd.DataFrame(rows)
    lc.to_csv(out_path.replace('.png', '.csv'), index=False)
    if not lc.empty:
        s = lc.groupby('train_fraction').agg(train_AP_mean=('train_AP', 'mean'), validation_AP_mean=('validation_AP', 'mean')).reset_index()
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.plot(s.train_fraction, s.train_AP_mean, marker='o', label='Training AP')
        ax.plot(s.train_fraction, s.validation_AP_mean, marker='o', label='Validation AP')
        ax.set_xlabel('Fraction of training patient groups')
        ax.set_ylabel('Average precision')
        ax.set_title(title)
        ax.legend()
        fig.tight_layout()
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close(fig)

def style_excel(writer):
    try:
        from openpyxl.styles import Font, PatternFill, Alignment
        for sheet in writer.sheets:
            ws = writer.sheets[sheet]
            ws.freeze_panes = 'A2'
            ws.auto_filter.ref = ws.dimensions
            for cell in ws[1]:
                cell.font = Font(bold=True)
                cell.fill = PatternFill(start_color='D9EAF7', end_color='D9EAF7', fill_type='solid')
                cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            for col in ws.columns:
                max_len = max([len(str(c.value)) for c in col if c.value is not None] + [12])
                ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 12), 70)
    except Exception:
        pass


def make_model(params, seed, n_estimators_override=None):
    p = sanitize_params(params)
    if n_estimators_override is not None:
        p['max_iter'] = int(max(MIN_FINAL_N_ESTIMATORS, n_estimators_override))
    p.pop('positive_weight_multiplier', None)
    return HistGradientBoostingClassifier(
        loss='log_loss',
        random_state=seed,
        early_stopping=False,
        **p,
    )

def fit_estimator(X_train, y_train, params, seed, eval_set=None, use_early_stopping=False, n_estimators_override=None):
    weights = make_weights(y_train, float(params['positive_weight_multiplier']))
    model = make_model(params, seed, n_estimators_override=n_estimators_override)
    model.fit(X_train, y_train, sample_weight=weights)
    best_iter = None
    if eval_set is not None and use_early_stopping:
        X_val, y_val = eval_set
        staged_scores = []
        for p_val in model.staged_predict_proba(X_val):
            staged_scores.append(safe_average_precision(y_val, p_val[:, 1]))
        if staged_scores:
            best_iter = int(np.nanargmax(staged_scores)) + 1
    return model, best_iter


def add_dynamic_odi_features(df):
    out = df.copy()
    required = ['preop_ODI', 'postop_ODI', DAYS_BETWEEN_PROM_COL]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError('Cannot derive dynamic ODI features because required columns are missing: ' + ', '.join(missing))
    preop = pd.to_numeric(out['preop_ODI'].map(clean_scalar), errors='coerce')
    postop = pd.to_numeric(out['postop_ODI'].map(clean_scalar), errors='coerce')
    days_between = pd.to_numeric(out[DAYS_BETWEEN_PROM_COL].map(clean_scalar), errors='coerce')
    odi_change = postop - preop
    valid_rate = preop.notna() & postop.notna() & days_between.gt(0)
    change_rate = pd.Series(np.nan, index=out.index, dtype='float')
    change_rate.loc[valid_rate] = (odi_change.loc[valid_rate] / days_between.loc[valid_rate]).astype(float)
    improvement = preop - postop
    rel_fraction = improvement / preop.replace(0, np.nan)
    valid_relative = preop.notna() & postop.notna() & preop.gt(0)
    zero_baseline_with_postop = preop.eq(0) & postop.notna()
    relative_mcid = pd.Series(np.nan, index=out.index, dtype='float')
    relative_mcid.loc[valid_relative] = (rel_fraction.loc[valid_relative] >= RELATIVE_ODI_MCID_CUTOFF).astype(float)
    relative_mcid.loc[zero_baseline_with_postop] = 0.0
    out['ODI_change'] = odi_change
    out[PROM_CHANGE_RATE_COL] = change_rate
    out[RELATIVE_ODI_MCID_COL] = relative_mcid
    audit = pd.DataFrame([
        {'item': 'PROM change rate definition', 'value': '(postop_ODI - preop_ODI) / days_between_PROMs'},
        {'item': 'Relative ODI MCID definition', 'value': f'(preop_ODI - postop_ODI) / preop_ODI >= {RELATIVE_ODI_MCID_CUTOFF}; preop_ODI=0 coded as 0 when postop_ODI is available'},
        {'item': 'Rows', 'value': int(len(out))},
        {'item': 'Rows with valid preop/postop ODI and positive days between PROMs', 'value': int(valid_rate.sum())},
        {'item': 'Rows with preop ODI = 0 and postop ODI available coded as Relative ODI MCID = 0', 'value': int(zero_baseline_with_postop.sum())},
    ])
    return out, audit

def run_model(model_type, work, features, folds, seed, all_candidates, all_folds):
    key = model_type
    out_dir = os.path.join(OUTPUT_DIR, model_type)
    plot_dir = os.path.join(PLOT_DIR, model_type)
    artifact_dir = os.path.join(ARTIFACT_DIR, model_type)
    for d in [out_dir, plot_dir, artifact_dir]:
        os.makedirs(d, exist_ok=True)
    train = work[work.split.eq('train')].reset_index(drop=True)
    cal = work[work.split.eq('calibration')].reset_index(drop=True)
    test = work[work.split.eq('test')].reset_index(drop=True)
    y_train = train[TARGET_COL].astype(int).to_numpy()
    y_cal = cal[TARGET_COL].astype(int).to_numpy()
    y_test = test[TARGET_COL].astype(int).to_numpy()
    candidates, fold_metrics = tune(train[features], y_train, train[GROUP_COL].to_numpy(), features, folds, key, seed)
    best_params = sanitize_params({k: candidates.loc[0, k] for k in SEARCH_SPACE.keys()})
    locked_n = candidates.loc[0, 'locked_n_estimators_from_cv']
    final = fit_final(train[features], y_train, cal[features], y_cal, test[features], y_test, features, best_params, locked_n, seed + 991)
    threshold = final['threshold']
    summary = {
        'model_type': model_type, 'classifier': CLASSIFIER_DISPLAY_NAME,
        'n_features_original': len(features), 'best_candidate_id': int(candidates.loc[0, 'candidate_id']),
        'CV_AP_mean': float(candidates.loc[0, 'cv_AP_mean']), 'CV_ROC_AUC_mean': float(candidates.loc[0, 'cv_ROC_AUC_mean']),
        'locked_n_estimators_from_cv': locked_n,
        **eval_preds(y_train, final['p_train'], threshold, prefix='Train_'),
        **eval_preds(y_cal, final['p_cal'], threshold, prefix='Calibration_'),
        **eval_preds(y_test, final['p_test'], threshold, prefix='Test_'),
    }
    summary['Train_minus_CV_AP_gap'] = summary['Train_AP'] - summary['CV_AP_mean'] if pd.notna(summary['CV_AP_mean']) else np.nan
    summary['CV_minus_Test_AP_gap'] = summary['CV_AP_mean'] - summary['Test_AP'] if pd.notna(summary['CV_AP_mean']) else np.nan
    summary['Test_AP_CI_lower'], summary['Test_AP_CI_upper'] = patient_boot_ci(y_test, final['p_test'], test[GROUP_COL].to_numpy(), 'AP', seed=seed + 10)
    summary['Test_ROC_AUC_CI_lower'], summary['Test_ROC_AUC_CI_upper'] = patient_boot_ci(y_test, final['p_test'], test[GROUP_COL].to_numpy(), 'ROC_AUC', seed=seed + 20)
    pd.DataFrame([summary]).to_csv(os.path.join(out_dir, f'performance_{{key}}.csv'), index=False)
    pd.DataFrame({'row_index_in_split': np.arange(len(test)), GROUP_COL: test[GROUP_COL].to_numpy(), 'y_true': y_test, 'p_raw': final['p_test_raw'], 'p_calibrated': final['p_test']}).to_csv(os.path.join(out_dir, f'test_predictions_{{key}}.csv'), index=False)
    candidates.to_csv(os.path.join(out_dir, f'cv_candidates_{{key}}.csv'), index=False)
    fold_metrics.to_csv(os.path.join(out_dir, f'cv_fold_metrics_{{key}}.csv'), index=False)
    joblib.dump(final, os.path.join(artifact_dir, f'final_model_{{key}}.joblib'))
    save_learning_curve(train[features], y_train, train[GROUP_COL].to_numpy(), features, best_params, locked_n, seed + 313, os.path.join(plot_dir, f'learning_curve_{{key}}.png'), key)
    all_candidates.append(candidates)
    all_folds.append(fold_metrics)
    return {'summary': summary, 'preds': pd.read_csv(os.path.join(out_dir, f'test_predictions_{{key}}.csv'))}

def split_audit(df):
    rows = []
    for split in ['train', 'calibration', 'test']:
        sub = df[df.split.eq(split)]
        rows.append({'split': split, 'n_rows': len(sub), 'n_patients': sub[GROUP_COL].nunique(), 'events': int(sub[TARGET_COL].sum()), 'prevalence': float(sub[TARGET_COL].mean())})
    return pd.DataFrame(rows)

def methods_table():
    return pd.DataFrame([
        {'item': 'Paired design', 'rationale': f'Baseline-only and dynamic PROM-expanded {{CLASSIFIER_DISPLAY_NAME}} models are tuned separately under the same protocol.'},
        {'item': 'Paired split/folds', 'rationale': 'Paired models use identical patient-level train/calibration/test splits and identical group-aware CV folds.'},
        {'item': 'Tuning metric', 'rationale': 'Average precision is the primary model-selection metric because delayed reoperation is a rare-event outcome.'},
        {'item': 'Class imbalance', 'rationale': 'Positive-class sample weighting is tuned through a multiplier applied to the natural negative-to-positive ratio.'},
        {'item': 'Calibration and threshold', 'rationale': 'Isotonic calibration and maximum-F1 threshold selection are performed only on the calibration split.'},
        {'item': 'Held-out test set', 'rationale': 'The test set is isolated until the model-development pipeline is locked.'},
    ])

def main():
    t0 = time.time()
    df = pd.read_csv(INPUT_CSV)
    df, dynamic_audit = add_dynamic_odi_features(df)
    required = sorted(set(DYNAMIC_PROM_FEATURES + [TARGET_COL, GROUP_COL, DAYS_BETWEEN_PROM_COL]))
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise ValueError('Missing required columns:\n' + '\n'.join(f' - {{c}}' for c in missing_cols))
    preop = pd.to_numeric(df['preop_ODI'].map(clean_scalar), errors='coerce')
    postop = pd.to_numeric(df['postop_ODI'].map(clean_scalar), errors='coerce')
    days = pd.to_numeric(df[DAYS_BETWEEN_PROM_COL].map(clean_scalar), errors='coerce')
    eligible = preop.notna() & postop.notna() & days.gt(0)
    work = df.loc[eligible, DYNAMIC_PROM_FEATURES + [TARGET_COL, GROUP_COL]].copy()
    work[TARGET_COL] = work[TARGET_COL].map(to_binary_target)
    work = work.dropna(subset=[TARGET_COL]).copy()
    work[TARGET_COL] = work[TARGET_COL].astype(int)
    train_mask, cal_mask, test_mask = patient_split(work, TARGET_COL, RANDOM_STATE)
    work['split'] = np.where(train_mask, 'train', np.where(cal_mask, 'calibration', 'test'))
    train = work[work.split.eq('train')].reset_index(drop=True)
    folds = cv_splits(train[TARGET_COL].astype(int).to_numpy(), train[GROUP_COL].to_numpy(), RANDOM_STATE + 10)
    all_candidates, all_folds = [], []
    baseline = run_model('baseline_only', work, BASELINE_FEATURES, folds, RANDOM_STATE + 100, all_candidates, all_folds)
    expanded = run_model('dynamic_PROM_expanded', work, DYNAMIC_PROM_FEATURES, folds, RANDOM_STATE + 200, all_candidates, all_folds)
    merged = baseline['preds'].merge(expanded['preds'], on=['row_index_in_split', GROUP_COL, 'y_true'], suffixes=('_baseline', '_expanded'), validate='one_to_one')
    y = merged.y_true.astype(int).to_numpy()
    p0 = merged.p_calibrated_baseline.astype(float).to_numpy()
    p1 = merged.p_calibrated_expanded.astype(float).to_numpy()
    groups = merged[GROUP_COL].to_numpy()
    dap = paired_delta_boot(y, p0, p1, groups, 'AP', RANDOM_STATE + 1000)
    dauc = paired_delta_boot(y, p0, p1, groups, 'ROC_AUC', RANDOM_STATE + 2000)
    comparison = pd.DataFrame([{ 'baseline_Test_AP': baseline['summary']['Test_AP'], 'expanded_Test_AP': expanded['summary']['Test_AP'], 'Delta_AP_expanded_minus_baseline': dap[0], 'Delta_AP_CI_lower': dap[1], 'Delta_AP_CI_upper': dap[2], 'Delta_AP_bootstrap_p_value': dap[3], 'baseline_Test_ROC_AUC': baseline['summary']['Test_ROC_AUC'], 'expanded_Test_ROC_AUC': expanded['summary']['Test_ROC_AUC'], 'Delta_ROC_AUC_expanded_minus_baseline': dauc[0], 'Delta_ROC_AUC_CI_lower': dauc[1], 'Delta_ROC_AUC_CI_upper': dauc[2], 'Delta_ROC_AUC_bootstrap_p_value': dauc[3]}])
    summary = pd.DataFrame([baseline['summary'], expanded['summary']])
    candidates = pd.concat(all_candidates, ignore_index=True)
    folds_df = pd.concat(all_folds, ignore_index=True)
    missingness = pd.DataFrame([{'column': c, 'n_missing_or_blank': int(work[c].isna().sum()), 'percent_missing_or_blank': 100 * float(work[c].isna().sum()) / len(work)} for c in DYNAMIC_PROM_FEATURES if c in work.columns])
    xlsx = os.path.join(OUTPUT_DIR, f'Step2_DynamicPROM_{{CLASSIFIER_KEY}}_summary.xlsx')
    with pd.ExcelWriter(xlsx, engine='openpyxl') as writer:
        methods_table().to_excel(writer, 'methods_rationale', index=False)
        dynamic_audit.to_excel(writer, 'dynamic_feature_audit', index=False)
        summary.to_excel(writer, 'model_performance', index=False)
        comparison.to_excel(writer, 'paired_PROM_comparison', index=False)
        candidates.to_excel(writer, 'cv_candidates_all_models', index=False)
        folds_df.to_excel(writer, 'cv_fold_metrics_all', index=False)
        split_audit(work).to_excel(writer, 'split_audit', index=False)
        missingness.to_excel(writer, 'missingness_audit', index=False)
        style_excel(writer)
    summary.to_csv(os.path.join(OUTPUT_DIR, 'model_performance.csv'), index=False)
    comparison.to_csv(os.path.join(OUTPUT_DIR, 'paired_PROM_comparison.csv'), index=False)
    candidates.to_csv(os.path.join(OUTPUT_DIR, 'cv_candidates_all_models.csv'), index=False)
    folds_df.to_csv(os.path.join(OUTPUT_DIR, 'cv_fold_metrics_all.csv'), index=False)
    work[[GROUP_COL, 'split']].drop_duplicates().to_csv(os.path.join(OUTPUT_DIR, 'split_assignment.csv'), index=False)
    manifest = {'classifier': CLASSIFIER_DISPLAY_NAME, 'output_dir': OUTPUT_DIR, 'design': f'paired baseline-only and dynamic PROM-expanded {{CLASSIFIER_DISPLAY_NAME}} models', 'model_selection': 'highest mean group-aware CV average precision inside training split', 'calibration': 'isotonic', 'threshold_strategy': 'maximum F1 on calibration split', 'runtime_minutes': float((time.time() - t0) / 60), 'summary_xlsx': xlsx}
    json.dump(json_native(manifest), open(os.path.join(OUTPUT_DIR, 'run_manifest.json'), 'w'), indent=2, sort_keys=True)
    zip_path = os.path.join(OUTPUT_DIR, f'Step2_DynamicPROM_{{CLASSIFIER_KEY}}_outputs.zip')
    tmp = zip_path + '.tmp'
    for p in [zip_path, tmp]:
        if os.path.exists(p):
            os.remove(p)
    with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=ZIP_COMPRESSION_LEVEL) as z:
        for root, _, files in os.walk(OUTPUT_DIR):
            for name in files:
                full = os.path.join(root, name)
                if full in {zip_path, tmp}:
                    continue
                z.write(full, os.path.relpath(full, OUTPUT_DIR))
    os.replace(tmp, zip_path)
    print('=' * 100)
    print(f'Step 2 {{CLASSIFIER_DISPLAY_NAME}} analysis completed')
    print('Summary Excel:', xlsx)
    print('ZIP archive:', zip_path)
    print('=' * 100)
    if CREATE_COLAB_DOWNLOAD_LINK:
        try:
            from IPython.display import HTML, display
            display(HTML(f'<p><b>Step 2 {{CLASSIFIER_DISPLAY_NAME}} output archive is ready.</b></p><p><a href="/files{{zip_path}}" download>Click here to download the ZIP archive</a></p><p>Path: <code>{{zip_path}}</code></p>'))
        except Exception:
            pass
    if AUTO_DOWNLOAD_ZIP:
        try:
            from google.colab import files
            files.download(zip_path)
        except Exception:
            pass

if __name__ == '__main__':
    main()


#**Step 2: XGBoost**

In [ ]:
"""
Step 2 dynamic postoperative PROM analysis with XGBoost
=======================================================

Input
-----
/content/Step 2_ODI_Cohort.csv

Target
------
final_reop_step2
    1 = reoperation from postoperative day 91 through day 365
    0 = no reoperation from postoperative day 91 through day 365

Design
------
This script runs paired baseline-only and dynamic PROM-expanded XGBoost
models for delayed lumbar reoperation prediction. The baseline-only model
uses the same 35 baseline variables as Step 1. The dynamic PROM-expanded
model includes the same baseline variables plus preoperative PROM,
postoperative PROM, PROM change, PROM change rate, relative MCID status,
and timing of postoperative PROM assessment. Paired models use identical
patient-level train/calibration/test splits and identical group-aware
cross-validation fold construction. Hyperparameter tuning is performed
exclusively within the training split using cross-validated average
precision as the primary selection metric. Probability calibration and
threshold selection are performed only on the calibration split. The
held-out test set is reserved until the model-development pipeline is
locked.
"""

# -*- coding: utf-8 -*-

import os
import re
import sys
import json
import math
import time
import zipfile
import platform
import subprocess
import warnings
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    import joblib
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'joblib'])
    import joblib

try:
    import openpyxl  # noqa
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl'])
    import openpyxl  # noqa

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedGroupKFold, ParameterSampler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_recall_curve,
    roc_curve,
    confusion_matrix,
    f1_score,
)
from sklearn.isotonic import IsotonicRegression
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')


def _ensure_xgboost():
    try:
        from xgboost import XGBClassifier
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost'])
        from xgboost import XGBClassifier
    return XGBClassifier


BASE_DIR = '/content'
INPUT_CSV = os.path.join(BASE_DIR, 'Step 2_ODI_Cohort.csv')
CLASSIFIER_KEY = 'XGBoost'
CLASSIFIER_DISPLAY_NAME = 'XGBoost'
SCALE_CONTINUOUS = False
OUTPUT_DIR = os.path.join(BASE_DIR, 'Step2_DynamicPROM_XGBoost_publication')
PLOT_DIR = os.path.join(OUTPUT_DIR, 'plots')
ARTIFACT_DIR = os.path.join(OUTPUT_DIR, 'model_artifacts')
for d in [OUTPUT_DIR, PLOT_DIR, ARTIFACT_DIR]:
    os.makedirs(d, exist_ok=True)

TARGET_COL = 'final_reop_step2'
GROUP_COL = 'PersonKey'
RANDOM_STATE = 20260524
TEST_FRACTION = 0.20
CALIBRATION_FRACTION_OF_REMAINING = 0.20
N_CV_FOLDS = 5
N_RANDOM_COMBINATIONS = 300
USE_EARLY_STOPPING_IN_CV = True
EARLY_STOPPING_ROUNDS = 100
MIN_FINAL_N_ESTIMATORS = 50
THRESHOLD_STRATEGY = 'max_f1'
N_BOOTSTRAPS = 2000
ECE_N_BINS = 10
N_JOBS = -1
LEARNING_CURVE_TRAIN_FRACTIONS = (0.20, 0.40, 0.60, 0.80, 1.00)
LEARNING_CURVE_CV_FOLDS = 3
ZIP_COMPRESSION_LEVEL = 1
CREATE_COLAB_DOWNLOAD_LINK = True
AUTO_DOWNLOAD_ZIP = False
RELATIVE_ODI_MCID_CUTOFF = 0.30
PROM_CHANGE_RATE_COL = 'ODI_change_rate'
RELATIVE_ODI_MCID_COL = 'ODI_relative_MCID_binary'
DAYS_BETWEEN_PROM_COL = 'days_between_PROMs'

BASELINE_FEATURES = ['finaldx_degenerative', 'finaldx_radicular', 'finaldx_stenosis', 'finaldx_deformity_instability', 'finaldx_other_diagnosis', 'age', 'sex', 'race', 'ethnicity', 'cancer_status', 'chronic_pulmonary_disease', 'congestive_heart_failure', 'connective_tissue_rheumatic_disease', 'diabetes_status', 'myocardial_infarction', 'renal_disease', 'institution_type', 'institution_size', 'institution_region', 'asa', 'bmi', 'payer_status', 'alif_llif', 'corpectomy', 'discectomy', 'foraminotomy', 'instrumentation', 'laminectomy_posterior_decompression', 'pelvic_fixation', 'plf', 'tlif_plif', 'other_lumbar_procedure', 'number_operated_levels', 'operative_region_extent', 'PatTobaccoUse']
assert len(BASELINE_FEATURES) == 35
STEP2_PROM_FEATURES = ['preop_ODI', 'postop_ODI', 'ODI_change', PROM_CHANGE_RATE_COL, RELATIVE_ODI_MCID_COL, 'postop_ODI_day']
DYNAMIC_PROM_FEATURES = BASELINE_FEATURES + STEP2_PROM_FEATURES
CONTINUOUS_BASELINE_FEATURES = ['age', 'bmi']
CONTINUOUS_FEATURES_ALL = CONTINUOUS_BASELINE_FEATURES + ['preop_ODI', 'postop_ODI', 'ODI_change', PROM_CHANGE_RATE_COL, 'postop_ODI_day']
BINARY_FEATURES = ['finaldx_degenerative', 'finaldx_radicular', 'finaldx_stenosis', 'finaldx_deformity_instability', 'finaldx_other_diagnosis', 'sex', 'ethnicity', 'cancer_status', 'chronic_pulmonary_disease', 'congestive_heart_failure', 'connective_tissue_rheumatic_disease', 'myocardial_infarction', 'renal_disease', 'institution_type', 'alif_llif', 'corpectomy', 'discectomy', 'foraminotomy', 'instrumentation', 'laminectomy_posterior_decompression', 'pelvic_fixation', 'plf', 'tlif_plif', 'other_lumbar_procedure', 'operative_region_extent', RELATIVE_ODI_MCID_COL]
ORDINAL_FEATURES = ['diabetes_status', 'institution_size', 'asa', 'number_operated_levels']
NOMINAL_FEATURES = ['race', 'institution_region', 'payer_status', 'PatTobaccoUse']

SEARCH_SPACE = {
    'n_estimators': [300, 500, 800, 1200, 1600, 2000],
    'learning_rate': [0.005, 0.01, 0.02, 0.03, 0.05, 0.08],
    'max_depth': [2, 3, 4, 5, 6, 8],
    'min_child_weight': [1, 3, 5, 10, 20],
    'subsample': [0.60, 0.75, 0.90, 1.00],
    'colsample_bytree': [0.60, 0.75, 0.90, 1.00],
    'reg_alpha': [0.0, 0.001, 0.01, 0.10, 0.50, 1.00, 2.00],
    'reg_lambda': [0.0, 0.001, 0.01, 0.10, 0.50, 1.00, 2.00, 5.00],
    'gamma': [0.0, 0.001, 0.01, 0.05, 0.10],
    'positive_weight_multiplier': [0.25, 0.50, 0.75, 1.00, 1.50, 2.00, 3.00, 4.00, 6.00, 8.00],
}
ITERATION_PARAM = 'n_estimators'
INT_PARAMS = {'n_estimators', 'max_depth', 'min_child_weight'}
FLOAT_PARAMS = {'learning_rate', 'subsample', 'colsample_bytree', 'reg_alpha', 'reg_lambda', 'gamma', 'positive_weight_multiplier'}

PARAMETER_CANDIDATES = list(ParameterSampler(SEARCH_SPACE, n_iter=N_RANDOM_COMBINATIONS, random_state=RANDOM_STATE))

MISSING_STRINGS = {'', ' ', 'na', 'n/a', 'nan', 'none', 'null', '.', 'missing', '<na>'}
BINARY_MAPS = {
    'sex': {'female': 0, 'f': 0, 'male': 1, 'm': 1},
    'ethnicity': {'non-hispanic': 0, 'non hispanic': 0, 'hispanic': 1},
    'cancer_status': {'no cancer': 0, 'no': 0, 'none': 0, 'any cancer': 1, 'yes': 1, 'cancer': 1},
    'institution_type': {'hospital': 0, 'non-hospital': 1, 'non hospital': 1, 'nonhospital': 1},
    'operative_region_extent': {'lumbar only': 0, 'extended_region_involvement': 1, 'extended region involvement': 1, 'extended': 1},
}
ORDINAL_MAPS = {
    'diabetes_status': {'no': 0, 'none': 0, '0': 0, 'without comp': 1, 'without complication': 1, 'without complications': 1, '1': 1, 'with comp': 2, 'with complication': 2, 'with complications': 2, '2': 2},
    'institution_size': {'between 1-99 beds': 0, '1-99': 0, 'between 100-399 beds': 1, '100-399': 1, '>= 400 beds': 2, '>=400 beds': 2, '>=400': 2, '>= 400': 2},
    'asa': {'1': 1, 'i': 1, '2': 2, 'ii': 2, '3': 3, 'iii': 3, '4': 4, 'iv': 4, '>=4': 4, '>= 4': 4, '5': 4, 'v': 4},
    'number_operated_levels': {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '>=4': 4, '>= 4': 4, '5': 4, '6': 4, '7': 4, '8': 4, '9': 4, '10': 4},
}
PREFERRED_NOMINAL_LEVELS = {
    'race': ['White', 'Black', 'Other'],
    'institution_region': ['South', 'North East', 'West', 'Midwest'],
    'payer_status': ['Medicare', 'Commercial/Private', 'Other', 'Medicaid/Public/Government'],
    'PatTobaccoUse': ['Unknown/Not reported/Multiple', 'Never', 'Former', 'Current'],
}
FEATURE_LABELS = {
    'age': 'Age', 'bmi': 'BMI', 'sex': 'Male sex', 'race': 'Race', 'ethnicity': 'Hispanic ethnicity',
    'cancer_status': 'Any cancer', 'diabetes_status': 'Diabetes status', 'institution_type': 'Non-hospital institution',
    'institution_size': 'Institution size', 'institution_region': 'Institution region', 'asa': 'ASA physical status',
    'payer_status': 'Primary payer', 'PatTobaccoUse': 'Tobacco use',
    'finaldx_degenerative': 'Degenerative diagnosis', 'finaldx_radicular': 'Radiculopathy diagnosis',
    'finaldx_stenosis': 'Spinal stenosis diagnosis', 'finaldx_deformity_instability': 'Deformity or instability diagnosis',
    'finaldx_other_diagnosis': 'Other lumbar diagnosis', 'chronic_pulmonary_disease': 'Chronic pulmonary disease',
    'congestive_heart_failure': 'Congestive heart failure', 'connective_tissue_rheumatic_disease': 'Connective tissue/rheumatic disease',
    'myocardial_infarction': 'Myocardial infarction', 'renal_disease': 'Renal disease',
    'alif_llif': 'Anterior/lateral lumbar interbody fusion', 'corpectomy': 'Corpectomy', 'discectomy': 'Discectomy',
    'foraminotomy': 'Foraminotomy', 'instrumentation': 'Instrumentation',
    'laminectomy_posterior_decompression': 'Posterior decompression or laminectomy', 'pelvic_fixation': 'Pelvic fixation',
    'plf': 'Posterolateral fusion', 'tlif_plif': 'Posterior/transforaminal lumbar interbody fusion',
    'other_lumbar_procedure': 'Other lumbar procedure', 'number_operated_levels': 'Number of operated levels',
    'operative_region_extent': 'Operative region extent',
}
FEATURE_LABELS.update({'preop_ODI': 'Preoperative ODI', 'postop_ODI': 'Postoperative ODI', 'ODI_change': 'Change in ODI', PROM_CHANGE_RATE_COL: 'ODI change rate', RELATIVE_ODI_MCID_COL: 'Relative ODI MCID', 'postop_ODI_day': 'Timing of postoperative ODI assessment'})

def clean_scalar(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().replace('≥', '>=')
        return np.nan if s.lower() in MISSING_STRINGS else s
    return x

def norm_text(x):
    x = clean_scalar(x)
    if pd.isna(x):
        return None
    return str(x).strip().replace('≥', '>=').lower()

def to_binary_target(x):
    sx = norm_text(x)
    if sx is None:
        return np.nan
    if sx in {'1', '1.0', 'yes', 'y', 'true', 't'}:
        return 1.0
    if sx in {'0', '0.0', 'no', 'n', 'false', 'f'}:
        return 0.0
    try:
        v = float(sx)
        return float(v) if v in (0.0, 1.0) else np.nan
    except Exception:
        return np.nan

def safe_filename(x):
    return re.sub(r'_+', '_', re.sub(r'[^A-Za-z0-9_.-]+', '_', str(x))).strip('_')[:180] or 'file'

def json_native(obj):
    if isinstance(obj, dict):
        return {str(k): json_native(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_native(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj

def sanitize_params(params):
    out = {}
    for k, v in params.items():
        if k in INT_PARAMS:
            out[k] = int(round(float(v)))
        elif k in FLOAT_PARAMS:
            out[k] = float(v)
        else:
            out[k] = v
    return out

def safe_average_precision(y, p):
    return np.nan if len(np.unique(y)) < 2 else float(average_precision_score(y, p))

def safe_roc_auc(y, p):
    return np.nan if len(np.unique(y)) < 2 else float(roc_auc_score(y, p))

def expected_calibration_error(y, p, n_bins=10):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    out = 0.0
    if len(y) == 0:
        return np.nan
    for i in range(n_bins):
        mask = (p >= bins[i]) & ((p <= bins[i + 1]) if i == n_bins - 1 else (p < bins[i + 1]))
        if np.any(mask):
            out += (np.sum(mask) / len(y)) * abs(float(np.mean(y[mask])) - float(np.mean(p[mask])))
    return float(out)

def classification_metrics(y, p, threshold):
    y = np.asarray(y).astype(int)
    pred = (np.asarray(p) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        'threshold': float(threshold),
        'F1': float(f1_score(y, pred, zero_division=0)),
        'Sensitivity': tp / (tp + fn) if tp + fn > 0 else np.nan,
        'Specificity': tn / (tn + fp) if tn + fp > 0 else np.nan,
        'PPV': tp / (tp + fp) if tp + fp > 0 else np.nan,
        'NPV': tn / (tn + fn) if tn + fn > 0 else np.nan,
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
        'Predicted_positive_rate': float(np.mean(pred)),
    }

def top5_metrics(y, p):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    n = len(y)
    k = max(1, int(math.ceil(n * 0.05)))
    idx = np.argsort(-p)[:k]
    prev = float(np.mean(y)) if n else np.nan
    rate = float(np.mean(y[idx])) if n else np.nan
    return {
        'Top_5pct_n': int(k),
        'Top_5pct_event_rate': rate,
        'Top_5pct_lift': rate / prev if prev > 0 else np.nan,
        'Top_5pct_captured_events': float(np.sum(y[idx]) / np.sum(y)) if np.sum(y) > 0 else np.nan,
    }

def select_threshold(y, p):
    precision, recall, thresholds = precision_recall_curve(y, p)
    if len(thresholds) == 0:
        threshold = 0.5
    else:
        precision = precision[:-1]
        recall = recall[:-1]
        f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
        threshold = float(thresholds[int(np.nanargmax(f1))])
    return threshold, classification_metrics(y, p, threshold)

def eval_preds(y, p, threshold=None, prefix=''):
    out = {
        f'{prefix}AP': safe_average_precision(y, p),
        f'{prefix}ROC_AUC': safe_roc_auc(y, p),
        f'{prefix}Brier_score': float(brier_score_loss(y, p)),
        f'{prefix}ECE': expected_calibration_error(y, p, ECE_N_BINS),
        f'{prefix}N': int(len(y)),
        f'{prefix}Events': int(np.sum(y)),
        f'{prefix}Prevalence': float(np.mean(y)),
    }
    if threshold is not None:
        out.update({f'{prefix}{k}': v for k, v in classification_metrics(y, p, threshold).items()})
        out.update({f'{prefix}{k}': v for k, v in top5_metrics(y, p).items()})
    return out

def make_weights(y, multiplier):
    y = np.asarray(y).astype(int)
    n_pos = int(np.sum(y == 1))
    n_neg = int(np.sum(y == 0))
    if n_pos == 0:
        raise ValueError('No positive events in training fold.')
    pos_weight = (n_neg / max(n_pos, 1)) * float(multiplier)
    return np.where(y == 1, pos_weight, 1.0).astype(float)

def actual_positive_weight(y, multiplier):
    y = np.asarray(y).astype(int)
    return float((np.sum(y == 0) / max(np.sum(y == 1), 1)) * multiplier)

@dataclass
class ModelPreprocessor:
    continuous_features: List[str]
    binary_features: List[str]
    ordinal_features: List[str]
    nominal_features: List[str]
    preferred_nominal_levels: Dict[str, List[str]]
    scale_continuous: bool = False

    def __post_init__(self):
        self.continuous_imputer = None
        self.discrete_imputer = None
        self.nominal_imputer = None
        self.scaler = None
        self.onehot = None
        self.output_feature_names_ = []

    def _parse_binary(self, x, feature):
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in BINARY_MAPS and sx in BINARY_MAPS[feature]:
            return float(BINARY_MAPS[feature][sx])
        if sx in {'1', '1.0', 'yes', 'y', 'true', 't', 'present', 'positive'}:
            return 1.0
        if sx in {'0', '0.0', 'no', 'n', 'false', 'f', 'absent', 'negative'}:
            return 0.0
        try:
            v = float(sx)
            return float(v) if v in (0.0, 1.0) else np.nan
        except Exception:
            return np.nan

    def _parse_ordinal(self, x, feature):
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in ORDINAL_MAPS and sx in ORDINAL_MAPS[feature]:
            return float(ORDINAL_MAPS[feature][sx])
        try:
            v = float(sx)
            if feature == 'asa':
                return float(min(max(int(round(v)), 1), 4))
            if feature == 'number_operated_levels':
                return float(min(max(int(round(v)), 0), 4))
            if feature == 'diabetes_status':
                return float(min(max(int(round(v)), 0), 2))
            if feature == 'institution_size':
                return float(min(max(int(round(v)), 0), 2))
            return float(v)
        except Exception:
            return np.nan

    def _nominal(self, feature, x):
        x = clean_scalar(x)
        if pd.isna(x):
            return np.nan
        s = str(x).strip()
        sl = s.lower()
        if feature == 'race':
            return 'White' if sl == 'white' else ('Black' if sl == 'black' else 'Other')
        if feature == 'institution_region':
            mapping = {'south': 'South', 'north east': 'North East', 'northeast': 'North East', 'north-east': 'North East', 'west': 'West', 'midwest': 'Midwest', 'mid west': 'Midwest'}
            return mapping.get(sl, s)
        if feature == 'payer_status':
            if sl == 'medicare':
                return 'Medicare'
            if sl in {'commercial/private', 'commercial', 'private', 'commercial private'}:
                return 'Commercial/Private'
            if sl in {'medicaid/public/government', 'medicaid', 'public', 'government', 'public/government'}:
                return 'Medicaid/Public/Government'
            return 'Other'
        if feature == 'PatTobaccoUse':
            return 'Never' if sl == 'never' else ('Former' if sl == 'former' else ('Current' if sl == 'current' else 'Unknown/Not reported/Multiple'))
        return s

    def _parts(self, X):
        cont = pd.DataFrame(index=X.index)
        disc = pd.DataFrame(index=X.index)
        nom = pd.DataFrame(index=X.index)
        for c in self.continuous_features:
            cont[c] = pd.to_numeric(X[c].map(clean_scalar), errors='coerce')
        for c in self.binary_features:
            disc[c] = X[c].map(lambda z: self._parse_binary(z, c)).astype(float)
        for c in self.ordinal_features:
            disc[c] = X[c].map(lambda z: self._parse_ordinal(z, c)).astype(float)
        for c in self.nominal_features:
            nom[c] = X[c].map(lambda z: self._nominal(c, z)).astype('object')
        return cont, disc, nom

    def fit(self, X):
        cont, disc, nom = self._parts(X)
        self.continuous_imputer = SimpleImputer(strategy='median')
        self.discrete_imputer = SimpleImputer(strategy='most_frequent')
        self.nominal_imputer = SimpleImputer(strategy='constant', fill_value='Missing')
        cont_imp = self.continuous_imputer.fit_transform(cont)
        if self.scale_continuous:
            self.scaler = StandardScaler()
            self.scaler.fit(cont_imp)
        self.discrete_imputer.fit(disc)
        nom_imp = pd.DataFrame(self.nominal_imputer.fit_transform(nom), columns=self.nominal_features)
        cats = []
        for c in self.nominal_features:
            preferred = list(self.preferred_nominal_levels.get(c, []))
            observed = nom_imp[c].astype(str).unique().tolist()
            cats.append(preferred + sorted([x for x in observed if x not in preferred]))
        try:
            self.onehot = OneHotEncoder(categories=cats, handle_unknown='ignore', sparse_output=False)
        except TypeError:
            self.onehot = OneHotEncoder(categories=cats, handle_unknown='ignore', sparse=False)
        self.onehot.fit(nom_imp.astype(str))
        self.output_feature_names_ = self.continuous_features + self.binary_features + self.ordinal_features + self.onehot.get_feature_names_out(self.nominal_features).tolist()
        return self

    def transform(self, X):
        cont, disc, nom = self._parts(X)
        cont_imp = self.continuous_imputer.transform(cont)
        if self.scale_continuous and self.scaler is not None:
            cont_imp = self.scaler.transform(cont_imp)
        disc_imp = self.discrete_imputer.transform(disc)
        nom_imp = pd.DataFrame(self.nominal_imputer.transform(nom), columns=self.nominal_features)
        nom_oh = self.onehot.transform(nom_imp.astype(str))
        return np.concatenate([cont_imp, disc_imp, nom_oh], axis=1).astype(float)

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

    @property
    def output_feature_names(self):
        return self.output_feature_names_

def raw_model_scores(model, Xp, n_iter=None):
    if CLASSIFIER_KEY == 'HistGradientBoosting' and n_iter is not None:
        selected = None
        for i, pred in enumerate(model.staged_predict_proba(Xp), start=1):
            selected = pred
            if i >= int(n_iter):
                break
        if selected is not None:
            return selected[:, 1]
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(Xp)[:, 1]
    if hasattr(model, 'decision_function'):
        return model.decision_function(Xp)
    raise AttributeError('Model provides neither predict_proba nor decision_function.')

def predict(pre, model, X, features, n_iter=None):
    return raw_model_scores(model, pre.transform(X[features]), n_iter=n_iter)

def patient_split(df, target_col, seed):
    group_df = df.groupby(GROUP_COL, dropna=False)[target_col].max().reset_index()
    y_group = group_df[target_col].astype(int).to_numpy()
    groups = group_df[GROUP_COL].to_numpy()
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=seed)
    train_cal_idx, test_idx = next(sss1.split(groups, y_group))
    groups_train_cal = groups[train_cal_idx]
    y_train_cal = y_group[train_cal_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=CALIBRATION_FRACTION_OF_REMAINING, random_state=seed + 1)
    train_idx_rel, cal_idx_rel = next(sss2.split(groups_train_cal, y_train_cal))
    train_groups = set(groups_train_cal[train_idx_rel])
    cal_groups = set(groups_train_cal[cal_idx_rel])
    test_groups = set(groups[test_idx])
    assert train_groups.isdisjoint(cal_groups) and train_groups.isdisjoint(test_groups) and cal_groups.isdisjoint(test_groups)
    return df[GROUP_COL].isin(train_groups).to_numpy(), df[GROUP_COL].isin(cal_groups).to_numpy(), df[GROUP_COL].isin(test_groups).to_numpy()

def cv_splits(y, groups, seed, n_folds=N_CV_FOLDS):
    group_df = pd.DataFrame({'group': groups, 'y': y}).groupby('group')['y'].max().reset_index()
    n_folds = min(n_folds, int(np.sum(group_df.y == 1)), int(np.sum(group_df.y == 0)))
    if n_folds < 2:
        raise ValueError('Not enough positive and negative patient groups for group-aware CV.')
    cv = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    return [(tr, va) for tr, va in cv.split(np.zeros(len(y)), y, groups)]

def feature_types(features):
    cont = [f for f in features if f in CONTINUOUS_FEATURES_ALL or (f not in BINARY_FEATURES and f not in ORDINAL_FEATURES and f not in NOMINAL_FEATURES)]
    binary = [f for f in features if f in BINARY_FEATURES]
    ordinal = [f for f in features if f in ORDINAL_FEATURES]
    nominal = [f for f in features if f in NOMINAL_FEATURES]
    return cont, binary, ordinal, nominal

def fit_pipeline(X, y, features, params, seed, eval_set=None, early=False, n_estimators_override=None):
    cont, binary, ordinal, nominal = feature_types(features)
    pre = ModelPreprocessor(cont, binary, ordinal, nominal, PREFERRED_NOMINAL_LEVELS, scale_continuous=SCALE_CONTINUOUS)
    Xp = pre.fit_transform(X[features])
    eval_transformed = None
    if eval_set is not None:
        X_val, y_val = eval_set
        eval_transformed = (pre.transform(X_val[features]), y_val)
    model, best_iter = fit_estimator(Xp, y, params, seed, eval_set=eval_transformed, use_early_stopping=early, n_estimators_override=n_estimators_override)
    return pre, model, best_iter

def tune(X_train, y_train, groups_train, features, folds, model_key, seed):
    candidate_rows = []
    fold_rows = []
    sampler = list(ParameterSampler(SEARCH_SPACE, n_iter=N_RANDOM_COMBINATIONS, random_state=seed))
    for cid, raw_params in enumerate(sampler, start=1):
        params = sanitize_params(raw_params)
        fold_train_aps, fold_train_aucs, fold_aps, fold_aucs, best_iters = [], [], [], [], []
        t0 = time.time()
        for fid, (tr_idx, va_idx) in enumerate(folds, start=1):
            X_tr = X_train.iloc[tr_idx].reset_index(drop=True)
            y_tr = y_train[tr_idx]
            X_va = X_train.iloc[va_idx].reset_index(drop=True)
            y_va = y_train[va_idx]
            pre, model, best_iter = fit_pipeline(X_tr, y_tr, features, params, seed + cid * 1000 + fid, eval_set=(X_va, y_va), early=USE_EARLY_STOPPING_IN_CV)
            p_tr = predict(pre, model, X_tr, features, n_iter=best_iter)
            p_va = predict(pre, model, X_va, features, n_iter=best_iter)
            tr_ap, tr_auc = safe_average_precision(y_tr, p_tr), safe_roc_auc(y_tr, p_tr)
            va_ap, va_auc = safe_average_precision(y_va, p_va), safe_roc_auc(y_va, p_va)
            fold_train_aps.append(tr_ap); fold_train_aucs.append(tr_auc); fold_aps.append(va_ap); fold_aucs.append(va_auc)
            if best_iter is not None and best_iter > 0:
                best_iters.append(best_iter)
            fold_rows.append({
                'model_key': model_key, 'candidate_id': cid, 'fold': fid,
                'fold_train_AP': tr_ap, 'fold_train_ROC_AUC': tr_auc,
                'fold_validation_AP': va_ap, 'fold_validation_ROC_AUC': va_auc,
                'fold_train_minus_validation_AP_gap': tr_ap - va_ap if np.isfinite(tr_ap) and np.isfinite(va_ap) else np.nan,
                'fold_train_minus_validation_ROC_AUC_gap': tr_auc - va_auc if np.isfinite(tr_auc) and np.isfinite(va_auc) else np.nan,
                'fold_best_iteration': best_iter,
                'positive_weight_used': actual_positive_weight(y_tr, params['positive_weight_multiplier']),
                **params,
            })
        locked = int(np.median(best_iters)) if best_iters and USE_EARLY_STOPPING_IN_CV else (int(params[ITERATION_PARAM]) if ITERATION_PARAM and ITERATION_PARAM in params else np.nan)
        row = {
            'model_key': model_key, 'candidate_id': cid, 'cv_folds': len(folds),
            'cv_train_AP_mean': float(np.nanmean(fold_train_aps)),
            'cv_train_ROC_AUC_mean': float(np.nanmean(fold_train_aucs)),
            'cv_AP_mean': float(np.nanmean(fold_aps)),
            'cv_AP_SD': float(np.nanstd(fold_aps, ddof=1)) if len(fold_aps) > 1 else np.nan,
            'cv_ROC_AUC_mean': float(np.nanmean(fold_aucs)),
            'cv_ROC_AUC_SD': float(np.nanstd(fold_aucs, ddof=1)) if len(fold_aucs) > 1 else np.nan,
            'mean_cv_best_iteration': float(np.mean(best_iters)) if best_iters else np.nan,
            'locked_n_estimators_from_cv': locked,
            'elapsed_seconds': float(time.time() - t0),
            **params,
        }
        candidate_rows.append(row)
        print(f"{model_key} | candidate {cid:03d}/{len(sampler)} | CV AP={row['cv_AP_mean']:.5f} | AUC={row['cv_ROC_AUC_mean']:.5f}")
    return pd.DataFrame(candidate_rows).sort_values('cv_AP_mean', ascending=False).reset_index(drop=True), pd.DataFrame(fold_rows)

def fit_final(X_train, y_train, X_cal, y_cal, X_test, features, params, locked_n, seed):
    n_override = int(locked_n) if pd.notna(locked_n) and USE_EARLY_STOPPING_IN_CV and ITERATION_PARAM else None
    pre, model, _ = fit_pipeline(X_train, y_train, features, params, seed, n_estimators_override=n_override)
    p_train_raw = predict(pre, model, X_train, features)
    p_cal_raw = predict(pre, model, X_cal, features)
    p_test_raw = predict(pre, model, X_test, features)
    calibrator = IsotonicRegression(out_of_bounds='clip')
    calibrator.fit(p_cal_raw, y_cal)
    p_train = calibrator.predict(p_train_raw)
    p_cal = calibrator.predict(p_cal_raw)
    p_test = calibrator.predict(p_test_raw)
    threshold, cal_metrics = select_threshold(y_cal, p_cal)
    return {
        'pre': pre, 'model': model, 'calibrator': calibrator,
        'p_train_raw': p_train_raw, 'p_cal_raw': p_cal_raw, 'p_test_raw': p_test_raw,
        'p_train': p_train, 'p_cal': p_cal, 'p_test': p_test,
        'threshold': threshold, 'cal_metrics': cal_metrics,
    }

def patient_boot_ci(y, p, groups, metric, threshold=None, seed=1):
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({'idx': np.arange(len(y)), 'group': groups, 'y': np.asarray(y).astype(int)})
    group_y = d.groupby('group').y.max()
    pos = group_y[group_y == 1].index.to_numpy()
    neg = group_y[group_y == 0].index.to_numpy()
    if len(pos) == 0 or len(neg) == 0:
        return np.nan, np.nan
    by_group = {g: d.loc[d.group.eq(g), 'idx'].to_numpy() for g in group_y.index}
    vals = []
    for _ in range(N_BOOTSTRAPS):
        sample_groups = np.concatenate([rng.choice(pos, len(pos), replace=True), rng.choice(neg, len(neg), replace=True)])
        sample_idx = np.concatenate([by_group[g] for g in sample_groups])
        yy = np.asarray(y)[sample_idx]
        pp = np.asarray(p)[sample_idx]
        if len(np.unique(yy)) < 2:
            continue
        if metric == 'AP':
            vals.append(average_precision_score(yy, pp))
        elif metric == 'ROC_AUC':
            vals.append(roc_auc_score(yy, pp))
        elif metric == 'F1':
            vals.append(f1_score(yy, pp >= threshold, zero_division=0))
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))) if vals else (np.nan, np.nan)

def paired_delta_boot(y, p0, p1, groups, metric, seed=1):
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({'idx': np.arange(len(y)), 'group': groups, 'y': np.asarray(y).astype(int)})
    group_y = d.groupby('group').y.max()
    pos = group_y[group_y == 1].index.to_numpy()
    neg = group_y[group_y == 0].index.to_numpy()
    by_group = {g: d.loc[d.group.eq(g), 'idx'].to_numpy() for g in group_y.index}
    observed = (safe_average_precision(y, p1) - safe_average_precision(y, p0)) if metric == 'AP' else (safe_roc_auc(y, p1) - safe_roc_auc(y, p0))
    vals = []
    for _ in range(N_BOOTSTRAPS):
        sample_groups = np.concatenate([rng.choice(pos, len(pos), replace=True), rng.choice(neg, len(neg), replace=True)])
        sample_idx = np.concatenate([by_group[g] for g in sample_groups])
        yy = np.asarray(y)[sample_idx]
        if len(np.unique(yy)) < 2:
            continue
        if metric == 'AP':
            vals.append(average_precision_score(yy, np.asarray(p1)[sample_idx]) - average_precision_score(yy, np.asarray(p0)[sample_idx]))
        else:
            vals.append(roc_auc_score(yy, np.asarray(p1)[sample_idx]) - roc_auc_score(yy, np.asarray(p0)[sample_idx]))
    if not vals:
        return observed, np.nan, np.nan, np.nan
    vals = np.asarray(vals)
    ci_low, ci_high = float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))
    p_value = float(2 * min(np.mean(vals <= 0), np.mean(vals >= 0)))
    return float(observed), ci_low, ci_high, min(p_value, 1.0)

def save_learning_curve(X_train, y_train, groups_train, features, params, locked_n, seed, out_path, title):
    group_df = pd.DataFrame({'group': groups_train, 'y': y_train}).groupby('group').y.max().reset_index()
    rng = np.random.default_rng(seed)
    rows = []
    for frac in LEARNING_CURVE_TRAIN_FRACTIONS:
        for rep in range(LEARNING_CURVE_CV_FOLDS):
            selected_groups = []
            for cls in [0, 1]:
                cls_groups = group_df.loc[group_df.y.eq(cls), 'group'].to_numpy()
                k = max(1, int(math.ceil(len(cls_groups) * frac)))
                selected_groups.extend(rng.choice(cls_groups, k, replace=False).tolist())
            mask = np.isin(groups_train, selected_groups)
            X_sub = X_train.loc[mask].reset_index(drop=True)
            y_sub = y_train[mask]
            g_sub = groups_train[mask]
            try:
                folds = cv_splits(y_sub, g_sub, seed + rep, n_folds=3)
            except Exception:
                continue
            for tr, va in folds:
                n_override = int(locked_n) if pd.notna(locked_n) and USE_EARLY_STOPPING_IN_CV and ITERATION_PARAM else None
                pre, model, _ = fit_pipeline(X_sub.iloc[tr].reset_index(drop=True), y_sub[tr], features, params, seed + rep, n_estimators_override=n_override)
                p_tr = predict(pre, model, X_sub.iloc[tr].reset_index(drop=True), features)
                p_va = predict(pre, model, X_sub.iloc[va].reset_index(drop=True), features)
                rows.append({'train_fraction': frac, 'train_AP': safe_average_precision(y_sub[tr], p_tr), 'validation_AP': safe_average_precision(y_sub[va], p_va)})
    lc = pd.DataFrame(rows)
    lc.to_csv(out_path.replace('.png', '.csv'), index=False)
    if not lc.empty:
        s = lc.groupby('train_fraction').agg(train_AP_mean=('train_AP', 'mean'), validation_AP_mean=('validation_AP', 'mean')).reset_index()
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.plot(s.train_fraction, s.train_AP_mean, marker='o', label='Training AP')
        ax.plot(s.train_fraction, s.validation_AP_mean, marker='o', label='Validation AP')
        ax.set_xlabel('Fraction of training patient groups')
        ax.set_ylabel('Average precision')
        ax.set_title(title)
        ax.legend()
        fig.tight_layout()
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close(fig)

def style_excel(writer):
    try:
        from openpyxl.styles import Font, PatternFill, Alignment
        for sheet in writer.sheets:
            ws = writer.sheets[sheet]
            ws.freeze_panes = 'A2'
            ws.auto_filter.ref = ws.dimensions
            for cell in ws[1]:
                cell.font = Font(bold=True)
                cell.fill = PatternFill(start_color='D9EAF7', end_color='D9EAF7', fill_type='solid')
                cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            for col in ws.columns:
                max_len = max([len(str(c.value)) for c in col if c.value is not None] + [12])
                ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 12), 70)
    except Exception:
        pass


def make_model(params, seed, n_estimators_override=None):
    XGBClassifier = _ensure_xgboost()
    p = sanitize_params(params)
    if n_estimators_override is not None:
        p['n_estimators'] = int(max(MIN_FINAL_N_ESTIMATORS, n_estimators_override))
    p.pop('positive_weight_multiplier', None)
    return XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr',
        random_state=seed,
        n_jobs=N_JOBS,
        tree_method='hist',
        verbosity=0,
        **p,
    )

def fit_estimator(X_train, y_train, params, seed, eval_set=None, use_early_stopping=False, n_estimators_override=None):
    weights = make_weights(y_train, float(params['positive_weight_multiplier']))
    model = make_model(params, seed, n_estimators_override=n_estimators_override)
    best_iter = None
    if eval_set is not None and use_early_stopping:
        X_val, y_val = eval_set
        try:
            model.fit(X_train, y_train, sample_weight=weights, eval_set=[(X_val, y_val)], verbose=False, early_stopping_rounds=EARLY_STOPPING_ROUNDS)
        except TypeError:
            try:
                model.set_params(early_stopping_rounds=EARLY_STOPPING_ROUNDS)
                model.fit(X_train, y_train, sample_weight=weights, eval_set=[(X_val, y_val)], verbose=False)
            except TypeError:
                model.fit(X_train, y_train, sample_weight=weights)
        bi = getattr(model, 'best_iteration', None)
        if bi is not None:
            best_iter = int(bi) + 1
    else:
        model.fit(X_train, y_train, sample_weight=weights)
    return model, best_iter


def add_dynamic_odi_features(df):
    out = df.copy()
    required = ['preop_ODI', 'postop_ODI', DAYS_BETWEEN_PROM_COL]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError('Cannot derive dynamic ODI features because required columns are missing: ' + ', '.join(missing))
    preop = pd.to_numeric(out['preop_ODI'].map(clean_scalar), errors='coerce')
    postop = pd.to_numeric(out['postop_ODI'].map(clean_scalar), errors='coerce')
    days_between = pd.to_numeric(out[DAYS_BETWEEN_PROM_COL].map(clean_scalar), errors='coerce')
    odi_change = postop - preop
    valid_rate = preop.notna() & postop.notna() & days_between.gt(0)
    change_rate = pd.Series(np.nan, index=out.index, dtype='float')
    change_rate.loc[valid_rate] = (odi_change.loc[valid_rate] / days_between.loc[valid_rate]).astype(float)
    improvement = preop - postop
    rel_fraction = improvement / preop.replace(0, np.nan)
    valid_relative = preop.notna() & postop.notna() & preop.gt(0)
    zero_baseline_with_postop = preop.eq(0) & postop.notna()
    relative_mcid = pd.Series(np.nan, index=out.index, dtype='float')
    relative_mcid.loc[valid_relative] = (rel_fraction.loc[valid_relative] >= RELATIVE_ODI_MCID_CUTOFF).astype(float)
    relative_mcid.loc[zero_baseline_with_postop] = 0.0
    out['ODI_change'] = odi_change
    out[PROM_CHANGE_RATE_COL] = change_rate
    out[RELATIVE_ODI_MCID_COL] = relative_mcid
    audit = pd.DataFrame([
        {'item': 'PROM change rate definition', 'value': '(postop_ODI - preop_ODI) / days_between_PROMs'},
        {'item': 'Relative ODI MCID definition', 'value': f'(preop_ODI - postop_ODI) / preop_ODI >= {RELATIVE_ODI_MCID_CUTOFF}; preop_ODI=0 coded as 0 when postop_ODI is available'},
        {'item': 'Rows', 'value': int(len(out))},
        {'item': 'Rows with valid preop/postop ODI and positive days between PROMs', 'value': int(valid_rate.sum())},
        {'item': 'Rows with preop ODI = 0 and postop ODI available coded as Relative ODI MCID = 0', 'value': int(zero_baseline_with_postop.sum())},
    ])
    return out, audit

def run_model(model_type, work, features, folds, seed, all_candidates, all_folds):
    key = model_type
    out_dir = os.path.join(OUTPUT_DIR, model_type)
    plot_dir = os.path.join(PLOT_DIR, model_type)
    artifact_dir = os.path.join(ARTIFACT_DIR, model_type)
    for d in [out_dir, plot_dir, artifact_dir]:
        os.makedirs(d, exist_ok=True)
    train = work[work.split.eq('train')].reset_index(drop=True)
    cal = work[work.split.eq('calibration')].reset_index(drop=True)
    test = work[work.split.eq('test')].reset_index(drop=True)
    y_train = train[TARGET_COL].astype(int).to_numpy()
    y_cal = cal[TARGET_COL].astype(int).to_numpy()
    y_test = test[TARGET_COL].astype(int).to_numpy()
    candidates, fold_metrics = tune(train[features], y_train, train[GROUP_COL].to_numpy(), features, folds, key, seed)
    best_params = sanitize_params({k: candidates.loc[0, k] for k in SEARCH_SPACE.keys()})
    locked_n = candidates.loc[0, 'locked_n_estimators_from_cv']
    final = fit_final(train[features], y_train, cal[features], y_cal, test[features], y_test, features, best_params, locked_n, seed + 991)
    threshold = final['threshold']
    summary = {
        'model_type': model_type, 'classifier': CLASSIFIER_DISPLAY_NAME,
        'n_features_original': len(features), 'best_candidate_id': int(candidates.loc[0, 'candidate_id']),
        'CV_AP_mean': float(candidates.loc[0, 'cv_AP_mean']), 'CV_ROC_AUC_mean': float(candidates.loc[0, 'cv_ROC_AUC_mean']),
        'locked_n_estimators_from_cv': locked_n,
        **eval_preds(y_train, final['p_train'], threshold, prefix='Train_'),
        **eval_preds(y_cal, final['p_cal'], threshold, prefix='Calibration_'),
        **eval_preds(y_test, final['p_test'], threshold, prefix='Test_'),
    }
    summary['Train_minus_CV_AP_gap'] = summary['Train_AP'] - summary['CV_AP_mean'] if pd.notna(summary['CV_AP_mean']) else np.nan
    summary['CV_minus_Test_AP_gap'] = summary['CV_AP_mean'] - summary['Test_AP'] if pd.notna(summary['CV_AP_mean']) else np.nan
    summary['Test_AP_CI_lower'], summary['Test_AP_CI_upper'] = patient_boot_ci(y_test, final['p_test'], test[GROUP_COL].to_numpy(), 'AP', seed=seed + 10)
    summary['Test_ROC_AUC_CI_lower'], summary['Test_ROC_AUC_CI_upper'] = patient_boot_ci(y_test, final['p_test'], test[GROUP_COL].to_numpy(), 'ROC_AUC', seed=seed + 20)
    pd.DataFrame([summary]).to_csv(os.path.join(out_dir, f'performance_{{key}}.csv'), index=False)
    pd.DataFrame({'row_index_in_split': np.arange(len(test)), GROUP_COL: test[GROUP_COL].to_numpy(), 'y_true': y_test, 'p_raw': final['p_test_raw'], 'p_calibrated': final['p_test']}).to_csv(os.path.join(out_dir, f'test_predictions_{{key}}.csv'), index=False)
    candidates.to_csv(os.path.join(out_dir, f'cv_candidates_{{key}}.csv'), index=False)
    fold_metrics.to_csv(os.path.join(out_dir, f'cv_fold_metrics_{{key}}.csv'), index=False)
    joblib.dump(final, os.path.join(artifact_dir, f'final_model_{{key}}.joblib'))
    save_learning_curve(train[features], y_train, train[GROUP_COL].to_numpy(), features, best_params, locked_n, seed + 313, os.path.join(plot_dir, f'learning_curve_{{key}}.png'), key)
    all_candidates.append(candidates)
    all_folds.append(fold_metrics)
    return {'summary': summary, 'preds': pd.read_csv(os.path.join(out_dir, f'test_predictions_{{key}}.csv'))}

def split_audit(df):
    rows = []
    for split in ['train', 'calibration', 'test']:
        sub = df[df.split.eq(split)]
        rows.append({'split': split, 'n_rows': len(sub), 'n_patients': sub[GROUP_COL].nunique(), 'events': int(sub[TARGET_COL].sum()), 'prevalence': float(sub[TARGET_COL].mean())})
    return pd.DataFrame(rows)

def methods_table():
    return pd.DataFrame([
        {'item': 'Paired design', 'rationale': f'Baseline-only and dynamic PROM-expanded {{CLASSIFIER_DISPLAY_NAME}} models are tuned separately under the same protocol.'},
        {'item': 'Paired split/folds', 'rationale': 'Paired models use identical patient-level train/calibration/test splits and identical group-aware CV folds.'},
        {'item': 'Tuning metric', 'rationale': 'Average precision is the primary model-selection metric because delayed reoperation is a rare-event outcome.'},
        {'item': 'Class imbalance', 'rationale': 'Positive-class sample weighting is tuned through a multiplier applied to the natural negative-to-positive ratio.'},
        {'item': 'Calibration and threshold', 'rationale': 'Isotonic calibration and maximum-F1 threshold selection are performed only on the calibration split.'},
        {'item': 'Held-out test set', 'rationale': 'The test set is isolated until the model-development pipeline is locked.'},
    ])

def main():
    t0 = time.time()
    df = pd.read_csv(INPUT_CSV)
    df, dynamic_audit = add_dynamic_odi_features(df)
    required = sorted(set(DYNAMIC_PROM_FEATURES + [TARGET_COL, GROUP_COL, DAYS_BETWEEN_PROM_COL]))
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise ValueError('Missing required columns:\n' + '\n'.join(f' - {{c}}' for c in missing_cols))
    preop = pd.to_numeric(df['preop_ODI'].map(clean_scalar), errors='coerce')
    postop = pd.to_numeric(df['postop_ODI'].map(clean_scalar), errors='coerce')
    days = pd.to_numeric(df[DAYS_BETWEEN_PROM_COL].map(clean_scalar), errors='coerce')
    eligible = preop.notna() & postop.notna() & days.gt(0)
    work = df.loc[eligible, DYNAMIC_PROM_FEATURES + [TARGET_COL, GROUP_COL]].copy()
    work[TARGET_COL] = work[TARGET_COL].map(to_binary_target)
    work = work.dropna(subset=[TARGET_COL]).copy()
    work[TARGET_COL] = work[TARGET_COL].astype(int)
    train_mask, cal_mask, test_mask = patient_split(work, TARGET_COL, RANDOM_STATE)
    work['split'] = np.where(train_mask, 'train', np.where(cal_mask, 'calibration', 'test'))
    train = work[work.split.eq('train')].reset_index(drop=True)
    folds = cv_splits(train[TARGET_COL].astype(int).to_numpy(), train[GROUP_COL].to_numpy(), RANDOM_STATE + 10)
    all_candidates, all_folds = [], []
    baseline = run_model('baseline_only', work, BASELINE_FEATURES, folds, RANDOM_STATE + 100, all_candidates, all_folds)
    expanded = run_model('dynamic_PROM_expanded', work, DYNAMIC_PROM_FEATURES, folds, RANDOM_STATE + 200, all_candidates, all_folds)
    merged = baseline['preds'].merge(expanded['preds'], on=['row_index_in_split', GROUP_COL, 'y_true'], suffixes=('_baseline', '_expanded'), validate='one_to_one')
    y = merged.y_true.astype(int).to_numpy()
    p0 = merged.p_calibrated_baseline.astype(float).to_numpy()
    p1 = merged.p_calibrated_expanded.astype(float).to_numpy()
    groups = merged[GROUP_COL].to_numpy()
    dap = paired_delta_boot(y, p0, p1, groups, 'AP', RANDOM_STATE + 1000)
    dauc = paired_delta_boot(y, p0, p1, groups, 'ROC_AUC', RANDOM_STATE + 2000)
    comparison = pd.DataFrame([{ 'baseline_Test_AP': baseline['summary']['Test_AP'], 'expanded_Test_AP': expanded['summary']['Test_AP'], 'Delta_AP_expanded_minus_baseline': dap[0], 'Delta_AP_CI_lower': dap[1], 'Delta_AP_CI_upper': dap[2], 'Delta_AP_bootstrap_p_value': dap[3], 'baseline_Test_ROC_AUC': baseline['summary']['Test_ROC_AUC'], 'expanded_Test_ROC_AUC': expanded['summary']['Test_ROC_AUC'], 'Delta_ROC_AUC_expanded_minus_baseline': dauc[0], 'Delta_ROC_AUC_CI_lower': dauc[1], 'Delta_ROC_AUC_CI_upper': dauc[2], 'Delta_ROC_AUC_bootstrap_p_value': dauc[3]}])
    summary = pd.DataFrame([baseline['summary'], expanded['summary']])
    candidates = pd.concat(all_candidates, ignore_index=True)
    folds_df = pd.concat(all_folds, ignore_index=True)
    missingness = pd.DataFrame([{'column': c, 'n_missing_or_blank': int(work[c].isna().sum()), 'percent_missing_or_blank': 100 * float(work[c].isna().sum()) / len(work)} for c in DYNAMIC_PROM_FEATURES if c in work.columns])
    xlsx = os.path.join(OUTPUT_DIR, f'Step2_DynamicPROM_{{CLASSIFIER_KEY}}_summary.xlsx')
    with pd.ExcelWriter(xlsx, engine='openpyxl') as writer:
        methods_table().to_excel(writer, 'methods_rationale', index=False)
        dynamic_audit.to_excel(writer, 'dynamic_feature_audit', index=False)
        summary.to_excel(writer, 'model_performance', index=False)
        comparison.to_excel(writer, 'paired_PROM_comparison', index=False)
        candidates.to_excel(writer, 'cv_candidates_all_models', index=False)
        folds_df.to_excel(writer, 'cv_fold_metrics_all', index=False)
        split_audit(work).to_excel(writer, 'split_audit', index=False)
        missingness.to_excel(writer, 'missingness_audit', index=False)
        style_excel(writer)
    summary.to_csv(os.path.join(OUTPUT_DIR, 'model_performance.csv'), index=False)
    comparison.to_csv(os.path.join(OUTPUT_DIR, 'paired_PROM_comparison.csv'), index=False)
    candidates.to_csv(os.path.join(OUTPUT_DIR, 'cv_candidates_all_models.csv'), index=False)
    folds_df.to_csv(os.path.join(OUTPUT_DIR, 'cv_fold_metrics_all.csv'), index=False)
    work[[GROUP_COL, 'split']].drop_duplicates().to_csv(os.path.join(OUTPUT_DIR, 'split_assignment.csv'), index=False)
    manifest = {'classifier': CLASSIFIER_DISPLAY_NAME, 'output_dir': OUTPUT_DIR, 'design': f'paired baseline-only and dynamic PROM-expanded {{CLASSIFIER_DISPLAY_NAME}} models', 'model_selection': 'highest mean group-aware CV average precision inside training split', 'calibration': 'isotonic', 'threshold_strategy': 'maximum F1 on calibration split', 'runtime_minutes': float((time.time() - t0) / 60), 'summary_xlsx': xlsx}
    json.dump(json_native(manifest), open(os.path.join(OUTPUT_DIR, 'run_manifest.json'), 'w'), indent=2, sort_keys=True)
    zip_path = os.path.join(OUTPUT_DIR, f'Step2_DynamicPROM_{{CLASSIFIER_KEY}}_outputs.zip')
    tmp = zip_path + '.tmp'
    for p in [zip_path, tmp]:
        if os.path.exists(p):
            os.remove(p)
    with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=ZIP_COMPRESSION_LEVEL) as z:
        for root, _, files in os.walk(OUTPUT_DIR):
            for name in files:
                full = os.path.join(root, name)
                if full in {zip_path, tmp}:
                    continue
                z.write(full, os.path.relpath(full, OUTPUT_DIR))
    os.replace(tmp, zip_path)
    print('=' * 100)
    print(f'Step 2 {{CLASSIFIER_DISPLAY_NAME}} analysis completed')
    print('Summary Excel:', xlsx)
    print('ZIP archive:', zip_path)
    print('=' * 100)
    if CREATE_COLAB_DOWNLOAD_LINK:
        try:
            from IPython.display import HTML, display
            display(HTML(f'<p><b>Step 2 {{CLASSIFIER_DISPLAY_NAME}} output archive is ready.</b></p><p><a href="/files{{zip_path}}" download>Click here to download the ZIP archive</a></p><p>Path: <code>{{zip_path}}</code></p>'))
        except Exception:
            pass
    if AUTO_DOWNLOAD_ZIP:
        try:
            from google.colab import files
            files.download(zip_path)
        except Exception:
            pass

if __name__ == '__main__':
    main()


#**Step 2: AdaBoost**

In [ ]:
"""
Step 2 dynamic postoperative PROM analysis with AdaBoost
========================================================

Input
-----
/content/Step 2_ODI_Cohort.csv

Target
------
final_reop_step2
    1 = reoperation from postoperative day 91 through day 365
    0 = no reoperation from postoperative day 91 through day 365

Design
------
This script runs paired baseline-only and dynamic PROM-expanded AdaBoost
models for delayed lumbar reoperation prediction. The baseline-only model
uses the same 35 baseline variables as Step 1. The dynamic PROM-expanded
model includes the same baseline variables plus preoperative PROM,
postoperative PROM, PROM change, PROM change rate, relative MCID status,
and timing of postoperative PROM assessment. Paired models use identical
patient-level train/calibration/test splits and identical group-aware
cross-validation fold construction. Hyperparameter tuning is performed
exclusively within the training split using cross-validated average
precision as the primary selection metric. Probability calibration and
threshold selection are performed only on the calibration split. The
held-out test set is reserved until the model-development pipeline is
locked.
"""

# -*- coding: utf-8 -*-

import os
import re
import sys
import json
import math
import time
import zipfile
import platform
import subprocess
import warnings
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    import joblib
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'joblib'])
    import joblib

try:
    import openpyxl  # noqa
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl'])
    import openpyxl  # noqa

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedGroupKFold, ParameterSampler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_recall_curve,
    roc_curve,
    confusion_matrix,
    f1_score,
)
from sklearn.isotonic import IsotonicRegression
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')


from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier


BASE_DIR = '/content'
INPUT_CSV = os.path.join(BASE_DIR, 'Step 2_ODI_Cohort.csv')
CLASSIFIER_KEY = 'AdaBoost'
CLASSIFIER_DISPLAY_NAME = 'AdaBoost'
SCALE_CONTINUOUS = False
OUTPUT_DIR = os.path.join(BASE_DIR, 'Step2_DynamicPROM_AdaBoost_publication')
PLOT_DIR = os.path.join(OUTPUT_DIR, 'plots')
ARTIFACT_DIR = os.path.join(OUTPUT_DIR, 'model_artifacts')
for d in [OUTPUT_DIR, PLOT_DIR, ARTIFACT_DIR]:
    os.makedirs(d, exist_ok=True)

TARGET_COL = 'final_reop_step2'
GROUP_COL = 'PersonKey'
RANDOM_STATE = 20260524
TEST_FRACTION = 0.20
CALIBRATION_FRACTION_OF_REMAINING = 0.20
N_CV_FOLDS = 5
N_RANDOM_COMBINATIONS = 300
USE_EARLY_STOPPING_IN_CV = False
EARLY_STOPPING_ROUNDS = 100
MIN_FINAL_N_ESTIMATORS = 50
THRESHOLD_STRATEGY = 'max_f1'
N_BOOTSTRAPS = 2000
ECE_N_BINS = 10
N_JOBS = -1
LEARNING_CURVE_TRAIN_FRACTIONS = (0.20, 0.40, 0.60, 0.80, 1.00)
LEARNING_CURVE_CV_FOLDS = 3
ZIP_COMPRESSION_LEVEL = 1
CREATE_COLAB_DOWNLOAD_LINK = True
AUTO_DOWNLOAD_ZIP = False
RELATIVE_ODI_MCID_CUTOFF = 0.30
PROM_CHANGE_RATE_COL = 'ODI_change_rate'
RELATIVE_ODI_MCID_COL = 'ODI_relative_MCID_binary'
DAYS_BETWEEN_PROM_COL = 'days_between_PROMs'

BASELINE_FEATURES = ['finaldx_degenerative', 'finaldx_radicular', 'finaldx_stenosis', 'finaldx_deformity_instability', 'finaldx_other_diagnosis', 'age', 'sex', 'race', 'ethnicity', 'cancer_status', 'chronic_pulmonary_disease', 'congestive_heart_failure', 'connective_tissue_rheumatic_disease', 'diabetes_status', 'myocardial_infarction', 'renal_disease', 'institution_type', 'institution_size', 'institution_region', 'asa', 'bmi', 'payer_status', 'alif_llif', 'corpectomy', 'discectomy', 'foraminotomy', 'instrumentation', 'laminectomy_posterior_decompression', 'pelvic_fixation', 'plf', 'tlif_plif', 'other_lumbar_procedure', 'number_operated_levels', 'operative_region_extent', 'PatTobaccoUse']
assert len(BASELINE_FEATURES) == 35
STEP2_PROM_FEATURES = ['preop_ODI', 'postop_ODI', 'ODI_change', PROM_CHANGE_RATE_COL, RELATIVE_ODI_MCID_COL, 'postop_ODI_day']
DYNAMIC_PROM_FEATURES = BASELINE_FEATURES + STEP2_PROM_FEATURES
CONTINUOUS_BASELINE_FEATURES = ['age', 'bmi']
CONTINUOUS_FEATURES_ALL = CONTINUOUS_BASELINE_FEATURES + ['preop_ODI', 'postop_ODI', 'ODI_change', PROM_CHANGE_RATE_COL, 'postop_ODI_day']
BINARY_FEATURES = ['finaldx_degenerative', 'finaldx_radicular', 'finaldx_stenosis', 'finaldx_deformity_instability', 'finaldx_other_diagnosis', 'sex', 'ethnicity', 'cancer_status', 'chronic_pulmonary_disease', 'congestive_heart_failure', 'connective_tissue_rheumatic_disease', 'myocardial_infarction', 'renal_disease', 'institution_type', 'alif_llif', 'corpectomy', 'discectomy', 'foraminotomy', 'instrumentation', 'laminectomy_posterior_decompression', 'pelvic_fixation', 'plf', 'tlif_plif', 'other_lumbar_procedure', 'operative_region_extent', RELATIVE_ODI_MCID_COL]
ORDINAL_FEATURES = ['diabetes_status', 'institution_size', 'asa', 'number_operated_levels']
NOMINAL_FEATURES = ['race', 'institution_region', 'payer_status', 'PatTobaccoUse']

SEARCH_SPACE = {
    'n_estimators': [100, 200, 400, 700, 1000],
    'learning_rate': [0.005, 0.01, 0.03, 0.05, 0.10, 0.30, 0.50, 1.00],
    'base_max_depth': [1, 2, 3],
    'base_min_samples_leaf': [1, 2, 5, 10, 20],
    'positive_weight_multiplier': [0.25, 0.50, 0.75, 1.00, 1.50, 2.00, 3.00, 4.00, 6.00, 8.00],
}
ITERATION_PARAM = 'n_estimators'
INT_PARAMS = {'n_estimators', 'base_max_depth', 'base_min_samples_leaf'}
FLOAT_PARAMS = {'learning_rate', 'positive_weight_multiplier'}

PARAMETER_CANDIDATES = list(ParameterSampler(SEARCH_SPACE, n_iter=N_RANDOM_COMBINATIONS, random_state=RANDOM_STATE))

MISSING_STRINGS = {'', ' ', 'na', 'n/a', 'nan', 'none', 'null', '.', 'missing', '<na>'}
BINARY_MAPS = {
    'sex': {'female': 0, 'f': 0, 'male': 1, 'm': 1},
    'ethnicity': {'non-hispanic': 0, 'non hispanic': 0, 'hispanic': 1},
    'cancer_status': {'no cancer': 0, 'no': 0, 'none': 0, 'any cancer': 1, 'yes': 1, 'cancer': 1},
    'institution_type': {'hospital': 0, 'non-hospital': 1, 'non hospital': 1, 'nonhospital': 1},
    'operative_region_extent': {'lumbar only': 0, 'extended_region_involvement': 1, 'extended region involvement': 1, 'extended': 1},
}
ORDINAL_MAPS = {
    'diabetes_status': {'no': 0, 'none': 0, '0': 0, 'without comp': 1, 'without complication': 1, 'without complications': 1, '1': 1, 'with comp': 2, 'with complication': 2, 'with complications': 2, '2': 2},
    'institution_size': {'between 1-99 beds': 0, '1-99': 0, 'between 100-399 beds': 1, '100-399': 1, '>= 400 beds': 2, '>=400 beds': 2, '>=400': 2, '>= 400': 2},
    'asa': {'1': 1, 'i': 1, '2': 2, 'ii': 2, '3': 3, 'iii': 3, '4': 4, 'iv': 4, '>=4': 4, '>= 4': 4, '5': 4, 'v': 4},
    'number_operated_levels': {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '>=4': 4, '>= 4': 4, '5': 4, '6': 4, '7': 4, '8': 4, '9': 4, '10': 4},
}
PREFERRED_NOMINAL_LEVELS = {
    'race': ['White', 'Black', 'Other'],
    'institution_region': ['South', 'North East', 'West', 'Midwest'],
    'payer_status': ['Medicare', 'Commercial/Private', 'Other', 'Medicaid/Public/Government'],
    'PatTobaccoUse': ['Unknown/Not reported/Multiple', 'Never', 'Former', 'Current'],
}
FEATURE_LABELS = {
    'age': 'Age', 'bmi': 'BMI', 'sex': 'Male sex', 'race': 'Race', 'ethnicity': 'Hispanic ethnicity',
    'cancer_status': 'Any cancer', 'diabetes_status': 'Diabetes status', 'institution_type': 'Non-hospital institution',
    'institution_size': 'Institution size', 'institution_region': 'Institution region', 'asa': 'ASA physical status',
    'payer_status': 'Primary payer', 'PatTobaccoUse': 'Tobacco use',
    'finaldx_degenerative': 'Degenerative diagnosis', 'finaldx_radicular': 'Radiculopathy diagnosis',
    'finaldx_stenosis': 'Spinal stenosis diagnosis', 'finaldx_deformity_instability': 'Deformity or instability diagnosis',
    'finaldx_other_diagnosis': 'Other lumbar diagnosis', 'chronic_pulmonary_disease': 'Chronic pulmonary disease',
    'congestive_heart_failure': 'Congestive heart failure', 'connective_tissue_rheumatic_disease': 'Connective tissue/rheumatic disease',
    'myocardial_infarction': 'Myocardial infarction', 'renal_disease': 'Renal disease',
    'alif_llif': 'Anterior/lateral lumbar interbody fusion', 'corpectomy': 'Corpectomy', 'discectomy': 'Discectomy',
    'foraminotomy': 'Foraminotomy', 'instrumentation': 'Instrumentation',
    'laminectomy_posterior_decompression': 'Posterior decompression or laminectomy', 'pelvic_fixation': 'Pelvic fixation',
    'plf': 'Posterolateral fusion', 'tlif_plif': 'Posterior/transforaminal lumbar interbody fusion',
    'other_lumbar_procedure': 'Other lumbar procedure', 'number_operated_levels': 'Number of operated levels',
    'operative_region_extent': 'Operative region extent',
}
FEATURE_LABELS.update({'preop_ODI': 'Preoperative ODI', 'postop_ODI': 'Postoperative ODI', 'ODI_change': 'Change in ODI', PROM_CHANGE_RATE_COL: 'ODI change rate', RELATIVE_ODI_MCID_COL: 'Relative ODI MCID', 'postop_ODI_day': 'Timing of postoperative ODI assessment'})

def clean_scalar(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().replace('≥', '>=')
        return np.nan if s.lower() in MISSING_STRINGS else s
    return x

def norm_text(x):
    x = clean_scalar(x)
    if pd.isna(x):
        return None
    return str(x).strip().replace('≥', '>=').lower()

def to_binary_target(x):
    sx = norm_text(x)
    if sx is None:
        return np.nan
    if sx in {'1', '1.0', 'yes', 'y', 'true', 't'}:
        return 1.0
    if sx in {'0', '0.0', 'no', 'n', 'false', 'f'}:
        return 0.0
    try:
        v = float(sx)
        return float(v) if v in (0.0, 1.0) else np.nan
    except Exception:
        return np.nan

def safe_filename(x):
    return re.sub(r'_+', '_', re.sub(r'[^A-Za-z0-9_.-]+', '_', str(x))).strip('_')[:180] or 'file'

def json_native(obj):
    if isinstance(obj, dict):
        return {str(k): json_native(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_native(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj

def sanitize_params(params):
    out = {}
    for k, v in params.items():
        if k in INT_PARAMS:
            out[k] = int(round(float(v)))
        elif k in FLOAT_PARAMS:
            out[k] = float(v)
        else:
            out[k] = v
    return out

def safe_average_precision(y, p):
    return np.nan if len(np.unique(y)) < 2 else float(average_precision_score(y, p))

def safe_roc_auc(y, p):
    return np.nan if len(np.unique(y)) < 2 else float(roc_auc_score(y, p))

def expected_calibration_error(y, p, n_bins=10):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    out = 0.0
    if len(y) == 0:
        return np.nan
    for i in range(n_bins):
        mask = (p >= bins[i]) & ((p <= bins[i + 1]) if i == n_bins - 1 else (p < bins[i + 1]))
        if np.any(mask):
            out += (np.sum(mask) / len(y)) * abs(float(np.mean(y[mask])) - float(np.mean(p[mask])))
    return float(out)

def classification_metrics(y, p, threshold):
    y = np.asarray(y).astype(int)
    pred = (np.asarray(p) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        'threshold': float(threshold),
        'F1': float(f1_score(y, pred, zero_division=0)),
        'Sensitivity': tp / (tp + fn) if tp + fn > 0 else np.nan,
        'Specificity': tn / (tn + fp) if tn + fp > 0 else np.nan,
        'PPV': tp / (tp + fp) if tp + fp > 0 else np.nan,
        'NPV': tn / (tn + fn) if tn + fn > 0 else np.nan,
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
        'Predicted_positive_rate': float(np.mean(pred)),
    }

def top5_metrics(y, p):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    n = len(y)
    k = max(1, int(math.ceil(n * 0.05)))
    idx = np.argsort(-p)[:k]
    prev = float(np.mean(y)) if n else np.nan
    rate = float(np.mean(y[idx])) if n else np.nan
    return {
        'Top_5pct_n': int(k),
        'Top_5pct_event_rate': rate,
        'Top_5pct_lift': rate / prev if prev > 0 else np.nan,
        'Top_5pct_captured_events': float(np.sum(y[idx]) / np.sum(y)) if np.sum(y) > 0 else np.nan,
    }

def select_threshold(y, p):
    precision, recall, thresholds = precision_recall_curve(y, p)
    if len(thresholds) == 0:
        threshold = 0.5
    else:
        precision = precision[:-1]
        recall = recall[:-1]
        f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
        threshold = float(thresholds[int(np.nanargmax(f1))])
    return threshold, classification_metrics(y, p, threshold)

def eval_preds(y, p, threshold=None, prefix=''):
    out = {
        f'{prefix}AP': safe_average_precision(y, p),
        f'{prefix}ROC_AUC': safe_roc_auc(y, p),
        f'{prefix}Brier_score': float(brier_score_loss(y, p)),
        f'{prefix}ECE': expected_calibration_error(y, p, ECE_N_BINS),
        f'{prefix}N': int(len(y)),
        f'{prefix}Events': int(np.sum(y)),
        f'{prefix}Prevalence': float(np.mean(y)),
    }
    if threshold is not None:
        out.update({f'{prefix}{k}': v for k, v in classification_metrics(y, p, threshold).items()})
        out.update({f'{prefix}{k}': v for k, v in top5_metrics(y, p).items()})
    return out

def make_weights(y, multiplier):
    y = np.asarray(y).astype(int)
    n_pos = int(np.sum(y == 1))
    n_neg = int(np.sum(y == 0))
    if n_pos == 0:
        raise ValueError('No positive events in training fold.')
    pos_weight = (n_neg / max(n_pos, 1)) * float(multiplier)
    return np.where(y == 1, pos_weight, 1.0).astype(float)

def actual_positive_weight(y, multiplier):
    y = np.asarray(y).astype(int)
    return float((np.sum(y == 0) / max(np.sum(y == 1), 1)) * multiplier)

@dataclass
class ModelPreprocessor:
    continuous_features: List[str]
    binary_features: List[str]
    ordinal_features: List[str]
    nominal_features: List[str]
    preferred_nominal_levels: Dict[str, List[str]]
    scale_continuous: bool = False

    def __post_init__(self):
        self.continuous_imputer = None
        self.discrete_imputer = None
        self.nominal_imputer = None
        self.scaler = None
        self.onehot = None
        self.output_feature_names_ = []

    def _parse_binary(self, x, feature):
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in BINARY_MAPS and sx in BINARY_MAPS[feature]:
            return float(BINARY_MAPS[feature][sx])
        if sx in {'1', '1.0', 'yes', 'y', 'true', 't', 'present', 'positive'}:
            return 1.0
        if sx in {'0', '0.0', 'no', 'n', 'false', 'f', 'absent', 'negative'}:
            return 0.0
        try:
            v = float(sx)
            return float(v) if v in (0.0, 1.0) else np.nan
        except Exception:
            return np.nan

    def _parse_ordinal(self, x, feature):
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in ORDINAL_MAPS and sx in ORDINAL_MAPS[feature]:
            return float(ORDINAL_MAPS[feature][sx])
        try:
            v = float(sx)
            if feature == 'asa':
                return float(min(max(int(round(v)), 1), 4))
            if feature == 'number_operated_levels':
                return float(min(max(int(round(v)), 0), 4))
            if feature == 'diabetes_status':
                return float(min(max(int(round(v)), 0), 2))
            if feature == 'institution_size':
                return float(min(max(int(round(v)), 0), 2))
            return float(v)
        except Exception:
            return np.nan

    def _nominal(self, feature, x):
        x = clean_scalar(x)
        if pd.isna(x):
            return np.nan
        s = str(x).strip()
        sl = s.lower()
        if feature == 'race':
            return 'White' if sl == 'white' else ('Black' if sl == 'black' else 'Other')
        if feature == 'institution_region':
            mapping = {'south': 'South', 'north east': 'North East', 'northeast': 'North East', 'north-east': 'North East', 'west': 'West', 'midwest': 'Midwest', 'mid west': 'Midwest'}
            return mapping.get(sl, s)
        if feature == 'payer_status':
            if sl == 'medicare':
                return 'Medicare'
            if sl in {'commercial/private', 'commercial', 'private', 'commercial private'}:
                return 'Commercial/Private'
            if sl in {'medicaid/public/government', 'medicaid', 'public', 'government', 'public/government'}:
                return 'Medicaid/Public/Government'
            return 'Other'
        if feature == 'PatTobaccoUse':
            return 'Never' if sl == 'never' else ('Former' if sl == 'former' else ('Current' if sl == 'current' else 'Unknown/Not reported/Multiple'))
        return s

    def _parts(self, X):
        cont = pd.DataFrame(index=X.index)
        disc = pd.DataFrame(index=X.index)
        nom = pd.DataFrame(index=X.index)
        for c in self.continuous_features:
            cont[c] = pd.to_numeric(X[c].map(clean_scalar), errors='coerce')
        for c in self.binary_features:
            disc[c] = X[c].map(lambda z: self._parse_binary(z, c)).astype(float)
        for c in self.ordinal_features:
            disc[c] = X[c].map(lambda z: self._parse_ordinal(z, c)).astype(float)
        for c in self.nominal_features:
            nom[c] = X[c].map(lambda z: self._nominal(c, z)).astype('object')
        return cont, disc, nom

    def fit(self, X):
        cont, disc, nom = self._parts(X)
        self.continuous_imputer = SimpleImputer(strategy='median')
        self.discrete_imputer = SimpleImputer(strategy='most_frequent')
        self.nominal_imputer = SimpleImputer(strategy='constant', fill_value='Missing')
        cont_imp = self.continuous_imputer.fit_transform(cont)
        if self.scale_continuous:
            self.scaler = StandardScaler()
            self.scaler.fit(cont_imp)
        self.discrete_imputer.fit(disc)
        nom_imp = pd.DataFrame(self.nominal_imputer.fit_transform(nom), columns=self.nominal_features)
        cats = []
        for c in self.nominal_features:
            preferred = list(self.preferred_nominal_levels.get(c, []))
            observed = nom_imp[c].astype(str).unique().tolist()
            cats.append(preferred + sorted([x for x in observed if x not in preferred]))
        try:
            self.onehot = OneHotEncoder(categories=cats, handle_unknown='ignore', sparse_output=False)
        except TypeError:
            self.onehot = OneHotEncoder(categories=cats, handle_unknown='ignore', sparse=False)
        self.onehot.fit(nom_imp.astype(str))
        self.output_feature_names_ = self.continuous_features + self.binary_features + self.ordinal_features + self.onehot.get_feature_names_out(self.nominal_features).tolist()
        return self

    def transform(self, X):
        cont, disc, nom = self._parts(X)
        cont_imp = self.continuous_imputer.transform(cont)
        if self.scale_continuous and self.scaler is not None:
            cont_imp = self.scaler.transform(cont_imp)
        disc_imp = self.discrete_imputer.transform(disc)
        nom_imp = pd.DataFrame(self.nominal_imputer.transform(nom), columns=self.nominal_features)
        nom_oh = self.onehot.transform(nom_imp.astype(str))
        return np.concatenate([cont_imp, disc_imp, nom_oh], axis=1).astype(float)

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

    @property
    def output_feature_names(self):
        return self.output_feature_names_

def raw_model_scores(model, Xp, n_iter=None):
    if CLASSIFIER_KEY == 'HistGradientBoosting' and n_iter is not None:
        selected = None
        for i, pred in enumerate(model.staged_predict_proba(Xp), start=1):
            selected = pred
            if i >= int(n_iter):
                break
        if selected is not None:
            return selected[:, 1]
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(Xp)[:, 1]
    if hasattr(model, 'decision_function'):
        return model.decision_function(Xp)
    raise AttributeError('Model provides neither predict_proba nor decision_function.')

def predict(pre, model, X, features, n_iter=None):
    return raw_model_scores(model, pre.transform(X[features]), n_iter=n_iter)

def patient_split(df, target_col, seed):
    group_df = df.groupby(GROUP_COL, dropna=False)[target_col].max().reset_index()
    y_group = group_df[target_col].astype(int).to_numpy()
    groups = group_df[GROUP_COL].to_numpy()
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=seed)
    train_cal_idx, test_idx = next(sss1.split(groups, y_group))
    groups_train_cal = groups[train_cal_idx]
    y_train_cal = y_group[train_cal_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=CALIBRATION_FRACTION_OF_REMAINING, random_state=seed + 1)
    train_idx_rel, cal_idx_rel = next(sss2.split(groups_train_cal, y_train_cal))
    train_groups = set(groups_train_cal[train_idx_rel])
    cal_groups = set(groups_train_cal[cal_idx_rel])
    test_groups = set(groups[test_idx])
    assert train_groups.isdisjoint(cal_groups) and train_groups.isdisjoint(test_groups) and cal_groups.isdisjoint(test_groups)
    return df[GROUP_COL].isin(train_groups).to_numpy(), df[GROUP_COL].isin(cal_groups).to_numpy(), df[GROUP_COL].isin(test_groups).to_numpy()

def cv_splits(y, groups, seed, n_folds=N_CV_FOLDS):
    group_df = pd.DataFrame({'group': groups, 'y': y}).groupby('group')['y'].max().reset_index()
    n_folds = min(n_folds, int(np.sum(group_df.y == 1)), int(np.sum(group_df.y == 0)))
    if n_folds < 2:
        raise ValueError('Not enough positive and negative patient groups for group-aware CV.')
    cv = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    return [(tr, va) for tr, va in cv.split(np.zeros(len(y)), y, groups)]

def feature_types(features):
    cont = [f for f in features if f in CONTINUOUS_FEATURES_ALL or (f not in BINARY_FEATURES and f not in ORDINAL_FEATURES and f not in NOMINAL_FEATURES)]
    binary = [f for f in features if f in BINARY_FEATURES]
    ordinal = [f for f in features if f in ORDINAL_FEATURES]
    nominal = [f for f in features if f in NOMINAL_FEATURES]
    return cont, binary, ordinal, nominal

def fit_pipeline(X, y, features, params, seed, eval_set=None, early=False, n_estimators_override=None):
    cont, binary, ordinal, nominal = feature_types(features)
    pre = ModelPreprocessor(cont, binary, ordinal, nominal, PREFERRED_NOMINAL_LEVELS, scale_continuous=SCALE_CONTINUOUS)
    Xp = pre.fit_transform(X[features])
    eval_transformed = None
    if eval_set is not None:
        X_val, y_val = eval_set
        eval_transformed = (pre.transform(X_val[features]), y_val)
    model, best_iter = fit_estimator(Xp, y, params, seed, eval_set=eval_transformed, use_early_stopping=early, n_estimators_override=n_estimators_override)
    return pre, model, best_iter

def tune(X_train, y_train, groups_train, features, folds, model_key, seed):
    candidate_rows = []
    fold_rows = []
    sampler = list(ParameterSampler(SEARCH_SPACE, n_iter=N_RANDOM_COMBINATIONS, random_state=seed))
    for cid, raw_params in enumerate(sampler, start=1):
        params = sanitize_params(raw_params)
        fold_train_aps, fold_train_aucs, fold_aps, fold_aucs, best_iters = [], [], [], [], []
        t0 = time.time()
        for fid, (tr_idx, va_idx) in enumerate(folds, start=1):
            X_tr = X_train.iloc[tr_idx].reset_index(drop=True)
            y_tr = y_train[tr_idx]
            X_va = X_train.iloc[va_idx].reset_index(drop=True)
            y_va = y_train[va_idx]
            pre, model, best_iter = fit_pipeline(X_tr, y_tr, features, params, seed + cid * 1000 + fid, eval_set=(X_va, y_va), early=USE_EARLY_STOPPING_IN_CV)
            p_tr = predict(pre, model, X_tr, features, n_iter=best_iter)
            p_va = predict(pre, model, X_va, features, n_iter=best_iter)
            tr_ap, tr_auc = safe_average_precision(y_tr, p_tr), safe_roc_auc(y_tr, p_tr)
            va_ap, va_auc = safe_average_precision(y_va, p_va), safe_roc_auc(y_va, p_va)
            fold_train_aps.append(tr_ap); fold_train_aucs.append(tr_auc); fold_aps.append(va_ap); fold_aucs.append(va_auc)
            if best_iter is not None and best_iter > 0:
                best_iters.append(best_iter)
            fold_rows.append({
                'model_key': model_key, 'candidate_id': cid, 'fold': fid,
                'fold_train_AP': tr_ap, 'fold_train_ROC_AUC': tr_auc,
                'fold_validation_AP': va_ap, 'fold_validation_ROC_AUC': va_auc,
                'fold_train_minus_validation_AP_gap': tr_ap - va_ap if np.isfinite(tr_ap) and np.isfinite(va_ap) else np.nan,
                'fold_train_minus_validation_ROC_AUC_gap': tr_auc - va_auc if np.isfinite(tr_auc) and np.isfinite(va_auc) else np.nan,
                'fold_best_iteration': best_iter,
                'positive_weight_used': actual_positive_weight(y_tr, params['positive_weight_multiplier']),
                **params,
            })
        locked = int(np.median(best_iters)) if best_iters and USE_EARLY_STOPPING_IN_CV else (int(params[ITERATION_PARAM]) if ITERATION_PARAM and ITERATION_PARAM in params else np.nan)
        row = {
            'model_key': model_key, 'candidate_id': cid, 'cv_folds': len(folds),
            'cv_train_AP_mean': float(np.nanmean(fold_train_aps)),
            'cv_train_ROC_AUC_mean': float(np.nanmean(fold_train_aucs)),
            'cv_AP_mean': float(np.nanmean(fold_aps)),
            'cv_AP_SD': float(np.nanstd(fold_aps, ddof=1)) if len(fold_aps) > 1 else np.nan,
            'cv_ROC_AUC_mean': float(np.nanmean(fold_aucs)),
            'cv_ROC_AUC_SD': float(np.nanstd(fold_aucs, ddof=1)) if len(fold_aucs) > 1 else np.nan,
            'mean_cv_best_iteration': float(np.mean(best_iters)) if best_iters else np.nan,
            'locked_n_estimators_from_cv': locked,
            'elapsed_seconds': float(time.time() - t0),
            **params,
        }
        candidate_rows.append(row)
        print(f"{model_key} | candidate {cid:03d}/{len(sampler)} | CV AP={row['cv_AP_mean']:.5f} | AUC={row['cv_ROC_AUC_mean']:.5f}")
    return pd.DataFrame(candidate_rows).sort_values('cv_AP_mean', ascending=False).reset_index(drop=True), pd.DataFrame(fold_rows)

def fit_final(X_train, y_train, X_cal, y_cal, X_test, features, params, locked_n, seed):
    n_override = int(locked_n) if pd.notna(locked_n) and USE_EARLY_STOPPING_IN_CV and ITERATION_PARAM else None
    pre, model, _ = fit_pipeline(X_train, y_train, features, params, seed, n_estimators_override=n_override)
    p_train_raw = predict(pre, model, X_train, features)
    p_cal_raw = predict(pre, model, X_cal, features)
    p_test_raw = predict(pre, model, X_test, features)
    calibrator = IsotonicRegression(out_of_bounds='clip')
    calibrator.fit(p_cal_raw, y_cal)
    p_train = calibrator.predict(p_train_raw)
    p_cal = calibrator.predict(p_cal_raw)
    p_test = calibrator.predict(p_test_raw)
    threshold, cal_metrics = select_threshold(y_cal, p_cal)
    return {
        'pre': pre, 'model': model, 'calibrator': calibrator,
        'p_train_raw': p_train_raw, 'p_cal_raw': p_cal_raw, 'p_test_raw': p_test_raw,
        'p_train': p_train, 'p_cal': p_cal, 'p_test': p_test,
        'threshold': threshold, 'cal_metrics': cal_metrics,
    }

def patient_boot_ci(y, p, groups, metric, threshold=None, seed=1):
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({'idx': np.arange(len(y)), 'group': groups, 'y': np.asarray(y).astype(int)})
    group_y = d.groupby('group').y.max()
    pos = group_y[group_y == 1].index.to_numpy()
    neg = group_y[group_y == 0].index.to_numpy()
    if len(pos) == 0 or len(neg) == 0:
        return np.nan, np.nan
    by_group = {g: d.loc[d.group.eq(g), 'idx'].to_numpy() for g in group_y.index}
    vals = []
    for _ in range(N_BOOTSTRAPS):
        sample_groups = np.concatenate([rng.choice(pos, len(pos), replace=True), rng.choice(neg, len(neg), replace=True)])
        sample_idx = np.concatenate([by_group[g] for g in sample_groups])
        yy = np.asarray(y)[sample_idx]
        pp = np.asarray(p)[sample_idx]
        if len(np.unique(yy)) < 2:
            continue
        if metric == 'AP':
            vals.append(average_precision_score(yy, pp))
        elif metric == 'ROC_AUC':
            vals.append(roc_auc_score(yy, pp))
        elif metric == 'F1':
            vals.append(f1_score(yy, pp >= threshold, zero_division=0))
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))) if vals else (np.nan, np.nan)

def paired_delta_boot(y, p0, p1, groups, metric, seed=1):
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({'idx': np.arange(len(y)), 'group': groups, 'y': np.asarray(y).astype(int)})
    group_y = d.groupby('group').y.max()
    pos = group_y[group_y == 1].index.to_numpy()
    neg = group_y[group_y == 0].index.to_numpy()
    by_group = {g: d.loc[d.group.eq(g), 'idx'].to_numpy() for g in group_y.index}
    observed = (safe_average_precision(y, p1) - safe_average_precision(y, p0)) if metric == 'AP' else (safe_roc_auc(y, p1) - safe_roc_auc(y, p0))
    vals = []
    for _ in range(N_BOOTSTRAPS):
        sample_groups = np.concatenate([rng.choice(pos, len(pos), replace=True), rng.choice(neg, len(neg), replace=True)])
        sample_idx = np.concatenate([by_group[g] for g in sample_groups])
        yy = np.asarray(y)[sample_idx]
        if len(np.unique(yy)) < 2:
            continue
        if metric == 'AP':
            vals.append(average_precision_score(yy, np.asarray(p1)[sample_idx]) - average_precision_score(yy, np.asarray(p0)[sample_idx]))
        else:
            vals.append(roc_auc_score(yy, np.asarray(p1)[sample_idx]) - roc_auc_score(yy, np.asarray(p0)[sample_idx]))
    if not vals:
        return observed, np.nan, np.nan, np.nan
    vals = np.asarray(vals)
    ci_low, ci_high = float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))
    p_value = float(2 * min(np.mean(vals <= 0), np.mean(vals >= 0)))
    return float(observed), ci_low, ci_high, min(p_value, 1.0)

def save_learning_curve(X_train, y_train, groups_train, features, params, locked_n, seed, out_path, title):
    group_df = pd.DataFrame({'group': groups_train, 'y': y_train}).groupby('group').y.max().reset_index()
    rng = np.random.default_rng(seed)
    rows = []
    for frac in LEARNING_CURVE_TRAIN_FRACTIONS:
        for rep in range(LEARNING_CURVE_CV_FOLDS):
            selected_groups = []
            for cls in [0, 1]:
                cls_groups = group_df.loc[group_df.y.eq(cls), 'group'].to_numpy()
                k = max(1, int(math.ceil(len(cls_groups) * frac)))
                selected_groups.extend(rng.choice(cls_groups, k, replace=False).tolist())
            mask = np.isin(groups_train, selected_groups)
            X_sub = X_train.loc[mask].reset_index(drop=True)
            y_sub = y_train[mask]
            g_sub = groups_train[mask]
            try:
                folds = cv_splits(y_sub, g_sub, seed + rep, n_folds=3)
            except Exception:
                continue
            for tr, va in folds:
                n_override = int(locked_n) if pd.notna(locked_n) and USE_EARLY_STOPPING_IN_CV and ITERATION_PARAM else None
                pre, model, _ = fit_pipeline(X_sub.iloc[tr].reset_index(drop=True), y_sub[tr], features, params, seed + rep, n_estimators_override=n_override)
                p_tr = predict(pre, model, X_sub.iloc[tr].reset_index(drop=True), features)
                p_va = predict(pre, model, X_sub.iloc[va].reset_index(drop=True), features)
                rows.append({'train_fraction': frac, 'train_AP': safe_average_precision(y_sub[tr], p_tr), 'validation_AP': safe_average_precision(y_sub[va], p_va)})
    lc = pd.DataFrame(rows)
    lc.to_csv(out_path.replace('.png', '.csv'), index=False)
    if not lc.empty:
        s = lc.groupby('train_fraction').agg(train_AP_mean=('train_AP', 'mean'), validation_AP_mean=('validation_AP', 'mean')).reset_index()
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.plot(s.train_fraction, s.train_AP_mean, marker='o', label='Training AP')
        ax.plot(s.train_fraction, s.validation_AP_mean, marker='o', label='Validation AP')
        ax.set_xlabel('Fraction of training patient groups')
        ax.set_ylabel('Average precision')
        ax.set_title(title)
        ax.legend()
        fig.tight_layout()
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close(fig)

def style_excel(writer):
    try:
        from openpyxl.styles import Font, PatternFill, Alignment
        for sheet in writer.sheets:
            ws = writer.sheets[sheet]
            ws.freeze_panes = 'A2'
            ws.auto_filter.ref = ws.dimensions
            for cell in ws[1]:
                cell.font = Font(bold=True)
                cell.fill = PatternFill(start_color='D9EAF7', end_color='D9EAF7', fill_type='solid')
                cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            for col in ws.columns:
                max_len = max([len(str(c.value)) for c in col if c.value is not None] + [12])
                ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 12), 70)
    except Exception:
        pass


def make_model(params, seed, n_estimators_override=None):
    p = sanitize_params(params)
    p.pop('positive_weight_multiplier', None)
    base = DecisionTreeClassifier(
        max_depth=int(p.pop('base_max_depth')),
        min_samples_leaf=int(p.pop('base_min_samples_leaf')),
        random_state=seed,
    )
    try:
        return AdaBoostClassifier(estimator=base, random_state=seed, **p)
    except TypeError:
        return AdaBoostClassifier(base_estimator=base, random_state=seed, **p)

def fit_estimator(X_train, y_train, params, seed, eval_set=None, use_early_stopping=False, n_estimators_override=None):
    weights = make_weights(y_train, float(params['positive_weight_multiplier']))
    model = make_model(params, seed)
    model.fit(X_train, y_train, sample_weight=weights)
    return model, None


def add_dynamic_odi_features(df):
    out = df.copy()
    required = ['preop_ODI', 'postop_ODI', DAYS_BETWEEN_PROM_COL]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError('Cannot derive dynamic ODI features because required columns are missing: ' + ', '.join(missing))
    preop = pd.to_numeric(out['preop_ODI'].map(clean_scalar), errors='coerce')
    postop = pd.to_numeric(out['postop_ODI'].map(clean_scalar), errors='coerce')
    days_between = pd.to_numeric(out[DAYS_BETWEEN_PROM_COL].map(clean_scalar), errors='coerce')
    odi_change = postop - preop
    valid_rate = preop.notna() & postop.notna() & days_between.gt(0)
    change_rate = pd.Series(np.nan, index=out.index, dtype='float')
    change_rate.loc[valid_rate] = (odi_change.loc[valid_rate] / days_between.loc[valid_rate]).astype(float)
    improvement = preop - postop
    rel_fraction = improvement / preop.replace(0, np.nan)
    valid_relative = preop.notna() & postop.notna() & preop.gt(0)
    zero_baseline_with_postop = preop.eq(0) & postop.notna()
    relative_mcid = pd.Series(np.nan, index=out.index, dtype='float')
    relative_mcid.loc[valid_relative] = (rel_fraction.loc[valid_relative] >= RELATIVE_ODI_MCID_CUTOFF).astype(float)
    relative_mcid.loc[zero_baseline_with_postop] = 0.0
    out['ODI_change'] = odi_change
    out[PROM_CHANGE_RATE_COL] = change_rate
    out[RELATIVE_ODI_MCID_COL] = relative_mcid
    audit = pd.DataFrame([
        {'item': 'PROM change rate definition', 'value': '(postop_ODI - preop_ODI) / days_between_PROMs'},
        {'item': 'Relative ODI MCID definition', 'value': f'(preop_ODI - postop_ODI) / preop_ODI >= {RELATIVE_ODI_MCID_CUTOFF}; preop_ODI=0 coded as 0 when postop_ODI is available'},
        {'item': 'Rows', 'value': int(len(out))},
        {'item': 'Rows with valid preop/postop ODI and positive days between PROMs', 'value': int(valid_rate.sum())},
        {'item': 'Rows with preop ODI = 0 and postop ODI available coded as Relative ODI MCID = 0', 'value': int(zero_baseline_with_postop.sum())},
    ])
    return out, audit

def run_model(model_type, work, features, folds, seed, all_candidates, all_folds):
    key = model_type
    out_dir = os.path.join(OUTPUT_DIR, model_type)
    plot_dir = os.path.join(PLOT_DIR, model_type)
    artifact_dir = os.path.join(ARTIFACT_DIR, model_type)
    for d in [out_dir, plot_dir, artifact_dir]:
        os.makedirs(d, exist_ok=True)
    train = work[work.split.eq('train')].reset_index(drop=True)
    cal = work[work.split.eq('calibration')].reset_index(drop=True)
    test = work[work.split.eq('test')].reset_index(drop=True)
    y_train = train[TARGET_COL].astype(int).to_numpy()
    y_cal = cal[TARGET_COL].astype(int).to_numpy()
    y_test = test[TARGET_COL].astype(int).to_numpy()
    candidates, fold_metrics = tune(train[features], y_train, train[GROUP_COL].to_numpy(), features, folds, key, seed)
    best_params = sanitize_params({k: candidates.loc[0, k] for k in SEARCH_SPACE.keys()})
    locked_n = candidates.loc[0, 'locked_n_estimators_from_cv']
    final = fit_final(train[features], y_train, cal[features], y_cal, test[features], y_test, features, best_params, locked_n, seed + 991)
    threshold = final['threshold']
    summary = {
        'model_type': model_type, 'classifier': CLASSIFIER_DISPLAY_NAME,
        'n_features_original': len(features), 'best_candidate_id': int(candidates.loc[0, 'candidate_id']),
        'CV_AP_mean': float(candidates.loc[0, 'cv_AP_mean']), 'CV_ROC_AUC_mean': float(candidates.loc[0, 'cv_ROC_AUC_mean']),
        'locked_n_estimators_from_cv': locked_n,
        **eval_preds(y_train, final['p_train'], threshold, prefix='Train_'),
        **eval_preds(y_cal, final['p_cal'], threshold, prefix='Calibration_'),
        **eval_preds(y_test, final['p_test'], threshold, prefix='Test_'),
    }
    summary['Train_minus_CV_AP_gap'] = summary['Train_AP'] - summary['CV_AP_mean'] if pd.notna(summary['CV_AP_mean']) else np.nan
    summary['CV_minus_Test_AP_gap'] = summary['CV_AP_mean'] - summary['Test_AP'] if pd.notna(summary['CV_AP_mean']) else np.nan
    summary['Test_AP_CI_lower'], summary['Test_AP_CI_upper'] = patient_boot_ci(y_test, final['p_test'], test[GROUP_COL].to_numpy(), 'AP', seed=seed + 10)
    summary['Test_ROC_AUC_CI_lower'], summary['Test_ROC_AUC_CI_upper'] = patient_boot_ci(y_test, final['p_test'], test[GROUP_COL].to_numpy(), 'ROC_AUC', seed=seed + 20)
    pd.DataFrame([summary]).to_csv(os.path.join(out_dir, f'performance_{{key}}.csv'), index=False)
    pd.DataFrame({'row_index_in_split': np.arange(len(test)), GROUP_COL: test[GROUP_COL].to_numpy(), 'y_true': y_test, 'p_raw': final['p_test_raw'], 'p_calibrated': final['p_test']}).to_csv(os.path.join(out_dir, f'test_predictions_{{key}}.csv'), index=False)
    candidates.to_csv(os.path.join(out_dir, f'cv_candidates_{{key}}.csv'), index=False)
    fold_metrics.to_csv(os.path.join(out_dir, f'cv_fold_metrics_{{key}}.csv'), index=False)
    joblib.dump(final, os.path.join(artifact_dir, f'final_model_{{key}}.joblib'))
    save_learning_curve(train[features], y_train, train[GROUP_COL].to_numpy(), features, best_params, locked_n, seed + 313, os.path.join(plot_dir, f'learning_curve_{{key}}.png'), key)
    all_candidates.append(candidates)
    all_folds.append(fold_metrics)
    return {'summary': summary, 'preds': pd.read_csv(os.path.join(out_dir, f'test_predictions_{{key}}.csv'))}

def split_audit(df):
    rows = []
    for split in ['train', 'calibration', 'test']:
        sub = df[df.split.eq(split)]
        rows.append({'split': split, 'n_rows': len(sub), 'n_patients': sub[GROUP_COL].nunique(), 'events': int(sub[TARGET_COL].sum()), 'prevalence': float(sub[TARGET_COL].mean())})
    return pd.DataFrame(rows)

def methods_table():
    return pd.DataFrame([
        {'item': 'Paired design', 'rationale': f'Baseline-only and dynamic PROM-expanded {{CLASSIFIER_DISPLAY_NAME}} models are tuned separately under the same protocol.'},
        {'item': 'Paired split/folds', 'rationale': 'Paired models use identical patient-level train/calibration/test splits and identical group-aware CV folds.'},
        {'item': 'Tuning metric', 'rationale': 'Average precision is the primary model-selection metric because delayed reoperation is a rare-event outcome.'},
        {'item': 'Class imbalance', 'rationale': 'Positive-class sample weighting is tuned through a multiplier applied to the natural negative-to-positive ratio.'},
        {'item': 'Calibration and threshold', 'rationale': 'Isotonic calibration and maximum-F1 threshold selection are performed only on the calibration split.'},
        {'item': 'Held-out test set', 'rationale': 'The test set is isolated until the model-development pipeline is locked.'},
    ])

def main():
    t0 = time.time()
    df = pd.read_csv(INPUT_CSV)
    df, dynamic_audit = add_dynamic_odi_features(df)
    required = sorted(set(DYNAMIC_PROM_FEATURES + [TARGET_COL, GROUP_COL, DAYS_BETWEEN_PROM_COL]))
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise ValueError('Missing required columns:\n' + '\n'.join(f' - {{c}}' for c in missing_cols))
    preop = pd.to_numeric(df['preop_ODI'].map(clean_scalar), errors='coerce')
    postop = pd.to_numeric(df['postop_ODI'].map(clean_scalar), errors='coerce')
    days = pd.to_numeric(df[DAYS_BETWEEN_PROM_COL].map(clean_scalar), errors='coerce')
    eligible = preop.notna() & postop.notna() & days.gt(0)
    work = df.loc[eligible, DYNAMIC_PROM_FEATURES + [TARGET_COL, GROUP_COL]].copy()
    work[TARGET_COL] = work[TARGET_COL].map(to_binary_target)
    work = work.dropna(subset=[TARGET_COL]).copy()
    work[TARGET_COL] = work[TARGET_COL].astype(int)
    train_mask, cal_mask, test_mask = patient_split(work, TARGET_COL, RANDOM_STATE)
    work['split'] = np.where(train_mask, 'train', np.where(cal_mask, 'calibration', 'test'))
    train = work[work.split.eq('train')].reset_index(drop=True)
    folds = cv_splits(train[TARGET_COL].astype(int).to_numpy(), train[GROUP_COL].to_numpy(), RANDOM_STATE + 10)
    all_candidates, all_folds = [], []
    baseline = run_model('baseline_only', work, BASELINE_FEATURES, folds, RANDOM_STATE + 100, all_candidates, all_folds)
    expanded = run_model('dynamic_PROM_expanded', work, DYNAMIC_PROM_FEATURES, folds, RANDOM_STATE + 200, all_candidates, all_folds)
    merged = baseline['preds'].merge(expanded['preds'], on=['row_index_in_split', GROUP_COL, 'y_true'], suffixes=('_baseline', '_expanded'), validate='one_to_one')
    y = merged.y_true.astype(int).to_numpy()
    p0 = merged.p_calibrated_baseline.astype(float).to_numpy()
    p1 = merged.p_calibrated_expanded.astype(float).to_numpy()
    groups = merged[GROUP_COL].to_numpy()
    dap = paired_delta_boot(y, p0, p1, groups, 'AP', RANDOM_STATE + 1000)
    dauc = paired_delta_boot(y, p0, p1, groups, 'ROC_AUC', RANDOM_STATE + 2000)
    comparison = pd.DataFrame([{ 'baseline_Test_AP': baseline['summary']['Test_AP'], 'expanded_Test_AP': expanded['summary']['Test_AP'], 'Delta_AP_expanded_minus_baseline': dap[0], 'Delta_AP_CI_lower': dap[1], 'Delta_AP_CI_upper': dap[2], 'Delta_AP_bootstrap_p_value': dap[3], 'baseline_Test_ROC_AUC': baseline['summary']['Test_ROC_AUC'], 'expanded_Test_ROC_AUC': expanded['summary']['Test_ROC_AUC'], 'Delta_ROC_AUC_expanded_minus_baseline': dauc[0], 'Delta_ROC_AUC_CI_lower': dauc[1], 'Delta_ROC_AUC_CI_upper': dauc[2], 'Delta_ROC_AUC_bootstrap_p_value': dauc[3]}])
    summary = pd.DataFrame([baseline['summary'], expanded['summary']])
    candidates = pd.concat(all_candidates, ignore_index=True)
    folds_df = pd.concat(all_folds, ignore_index=True)
    missingness = pd.DataFrame([{'column': c, 'n_missing_or_blank': int(work[c].isna().sum()), 'percent_missing_or_blank': 100 * float(work[c].isna().sum()) / len(work)} for c in DYNAMIC_PROM_FEATURES if c in work.columns])
    xlsx = os.path.join(OUTPUT_DIR, f'Step2_DynamicPROM_{{CLASSIFIER_KEY}}_summary.xlsx')
    with pd.ExcelWriter(xlsx, engine='openpyxl') as writer:
        methods_table().to_excel(writer, 'methods_rationale', index=False)
        dynamic_audit.to_excel(writer, 'dynamic_feature_audit', index=False)
        summary.to_excel(writer, 'model_performance', index=False)
        comparison.to_excel(writer, 'paired_PROM_comparison', index=False)
        candidates.to_excel(writer, 'cv_candidates_all_models', index=False)
        folds_df.to_excel(writer, 'cv_fold_metrics_all', index=False)
        split_audit(work).to_excel(writer, 'split_audit', index=False)
        missingness.to_excel(writer, 'missingness_audit', index=False)
        style_excel(writer)
    summary.to_csv(os.path.join(OUTPUT_DIR, 'model_performance.csv'), index=False)
    comparison.to_csv(os.path.join(OUTPUT_DIR, 'paired_PROM_comparison.csv'), index=False)
    candidates.to_csv(os.path.join(OUTPUT_DIR, 'cv_candidates_all_models.csv'), index=False)
    folds_df.to_csv(os.path.join(OUTPUT_DIR, 'cv_fold_metrics_all.csv'), index=False)
    work[[GROUP_COL, 'split']].drop_duplicates().to_csv(os.path.join(OUTPUT_DIR, 'split_assignment.csv'), index=False)
    manifest = {'classifier': CLASSIFIER_DISPLAY_NAME, 'output_dir': OUTPUT_DIR, 'design': f'paired baseline-only and dynamic PROM-expanded {{CLASSIFIER_DISPLAY_NAME}} models', 'model_selection': 'highest mean group-aware CV average precision inside training split', 'calibration': 'isotonic', 'threshold_strategy': 'maximum F1 on calibration split', 'runtime_minutes': float((time.time() - t0) / 60), 'summary_xlsx': xlsx}
    json.dump(json_native(manifest), open(os.path.join(OUTPUT_DIR, 'run_manifest.json'), 'w'), indent=2, sort_keys=True)
    zip_path = os.path.join(OUTPUT_DIR, f'Step2_DynamicPROM_{{CLASSIFIER_KEY}}_outputs.zip')
    tmp = zip_path + '.tmp'
    for p in [zip_path, tmp]:
        if os.path.exists(p):
            os.remove(p)
    with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=ZIP_COMPRESSION_LEVEL) as z:
        for root, _, files in os.walk(OUTPUT_DIR):
            for name in files:
                full = os.path.join(root, name)
                if full in {zip_path, tmp}:
                    continue
                z.write(full, os.path.relpath(full, OUTPUT_DIR))
    os.replace(tmp, zip_path)
    print('=' * 100)
    print(f'Step 2 {{CLASSIFIER_DISPLAY_NAME}} analysis completed')
    print('Summary Excel:', xlsx)
    print('ZIP archive:', zip_path)
    print('=' * 100)
    if CREATE_COLAB_DOWNLOAD_LINK:
        try:
            from IPython.display import HTML, display
            display(HTML(f'<p><b>Step 2 {{CLASSIFIER_DISPLAY_NAME}} output archive is ready.</b></p><p><a href="/files{{zip_path}}" download>Click here to download the ZIP archive</a></p><p>Path: <code>{{zip_path}}</code></p>'))
        except Exception:
            pass
    if AUTO_DOWNLOAD_ZIP:
        try:
            from google.colab import files
            files.download(zip_path)
        except Exception:
            pass

if __name__ == '__main__':
    main()



#**Step 2: Linear Support Vector**

In [ ]:
"""
Step 2 dynamic postoperative PROM analysis with linear support vector classifier
================================================================================

Input
-----
/content/Step 2_ODI_Cohort.csv

Target
------
final_reop_step2
    1 = reoperation from postoperative day 91 through day 365
    0 = no reoperation from postoperative day 91 through day 365

Design
------
This script runs paired baseline-only and dynamic PROM-expanded linear support vector classifier
models for delayed lumbar reoperation prediction. The baseline-only model
uses the same 35 baseline variables as Step 1. The dynamic PROM-expanded
model includes the same baseline variables plus preoperative PROM,
postoperative PROM, PROM change, PROM change rate, relative MCID status,
and timing of postoperative PROM assessment. Paired models use identical
patient-level train/calibration/test splits and identical group-aware
cross-validation fold construction. Hyperparameter tuning is performed
exclusively within the training split using cross-validated average
precision as the primary selection metric. Probability calibration and
threshold selection are performed only on the calibration split. The
held-out test set is reserved until the model-development pipeline is
locked.
"""

# -*- coding: utf-8 -*-

import os
import re
import sys
import json
import math
import time
import zipfile
import platform
import subprocess
import warnings
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    import joblib
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'joblib'])
    import joblib

try:
    import openpyxl  # noqa
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl'])
    import openpyxl  # noqa

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedGroupKFold, ParameterSampler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_recall_curve,
    roc_curve,
    confusion_matrix,
    f1_score,
)
from sklearn.isotonic import IsotonicRegression
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')


from sklearn.svm import LinearSVC


BASE_DIR = '/content'
INPUT_CSV = os.path.join(BASE_DIR, 'Step 2_ODI_Cohort.csv')
CLASSIFIER_KEY = 'LinearSVC'
CLASSIFIER_DISPLAY_NAME = 'linear support vector classifier'
SCALE_CONTINUOUS = True
OUTPUT_DIR = os.path.join(BASE_DIR, 'Step2_DynamicPROM_LinearSVC_publication')
PLOT_DIR = os.path.join(OUTPUT_DIR, 'plots')
ARTIFACT_DIR = os.path.join(OUTPUT_DIR, 'model_artifacts')
for d in [OUTPUT_DIR, PLOT_DIR, ARTIFACT_DIR]:
    os.makedirs(d, exist_ok=True)

TARGET_COL = 'final_reop_step2'
GROUP_COL = 'PersonKey'
RANDOM_STATE = 20260524
TEST_FRACTION = 0.20
CALIBRATION_FRACTION_OF_REMAINING = 0.20
N_CV_FOLDS = 5
N_RANDOM_COMBINATIONS = 300
USE_EARLY_STOPPING_IN_CV = False
EARLY_STOPPING_ROUNDS = 100
MIN_FINAL_N_ESTIMATORS = 50
THRESHOLD_STRATEGY = 'max_f1'
N_BOOTSTRAPS = 2000
ECE_N_BINS = 10
N_JOBS = -1
LEARNING_CURVE_TRAIN_FRACTIONS = (0.20, 0.40, 0.60, 0.80, 1.00)
LEARNING_CURVE_CV_FOLDS = 3
ZIP_COMPRESSION_LEVEL = 1
CREATE_COLAB_DOWNLOAD_LINK = True
AUTO_DOWNLOAD_ZIP = False
RELATIVE_ODI_MCID_CUTOFF = 0.30
PROM_CHANGE_RATE_COL = 'ODI_change_rate'
RELATIVE_ODI_MCID_COL = 'ODI_relative_MCID_binary'
DAYS_BETWEEN_PROM_COL = 'days_between_PROMs'

BASELINE_FEATURES = ['finaldx_degenerative', 'finaldx_radicular', 'finaldx_stenosis', 'finaldx_deformity_instability', 'finaldx_other_diagnosis', 'age', 'sex', 'race', 'ethnicity', 'cancer_status', 'chronic_pulmonary_disease', 'congestive_heart_failure', 'connective_tissue_rheumatic_disease', 'diabetes_status', 'myocardial_infarction', 'renal_disease', 'institution_type', 'institution_size', 'institution_region', 'asa', 'bmi', 'payer_status', 'alif_llif', 'corpectomy', 'discectomy', 'foraminotomy', 'instrumentation', 'laminectomy_posterior_decompression', 'pelvic_fixation', 'plf', 'tlif_plif', 'other_lumbar_procedure', 'number_operated_levels', 'operative_region_extent', 'PatTobaccoUse']
assert len(BASELINE_FEATURES) == 35
STEP2_PROM_FEATURES = ['preop_ODI', 'postop_ODI', 'ODI_change', PROM_CHANGE_RATE_COL, RELATIVE_ODI_MCID_COL, 'postop_ODI_day']
DYNAMIC_PROM_FEATURES = BASELINE_FEATURES + STEP2_PROM_FEATURES
CONTINUOUS_BASELINE_FEATURES = ['age', 'bmi']
CONTINUOUS_FEATURES_ALL = CONTINUOUS_BASELINE_FEATURES + ['preop_ODI', 'postop_ODI', 'ODI_change', PROM_CHANGE_RATE_COL, 'postop_ODI_day']
BINARY_FEATURES = ['finaldx_degenerative', 'finaldx_radicular', 'finaldx_stenosis', 'finaldx_deformity_instability', 'finaldx_other_diagnosis', 'sex', 'ethnicity', 'cancer_status', 'chronic_pulmonary_disease', 'congestive_heart_failure', 'connective_tissue_rheumatic_disease', 'myocardial_infarction', 'renal_disease', 'institution_type', 'alif_llif', 'corpectomy', 'discectomy', 'foraminotomy', 'instrumentation', 'laminectomy_posterior_decompression', 'pelvic_fixation', 'plf', 'tlif_plif', 'other_lumbar_procedure', 'operative_region_extent', RELATIVE_ODI_MCID_COL]
ORDINAL_FEATURES = ['diabetes_status', 'institution_size', 'asa', 'number_operated_levels']
NOMINAL_FEATURES = ['race', 'institution_region', 'payer_status', 'PatTobaccoUse']

SEARCH_SPACE = {
    'C': [0.001, 0.003, 0.01, 0.03, 0.10, 0.30, 1.00, 3.00, 10.00],
    'tol': [1e-5, 1e-4, 1e-3],
    'positive_weight_multiplier': [0.25, 0.50, 0.75, 1.00, 1.50, 2.00, 3.00, 4.00, 6.00, 8.00],
}
ITERATION_PARAM = None
INT_PARAMS = set()
FLOAT_PARAMS = {'C', 'tol', 'positive_weight_multiplier'}

PARAMETER_CANDIDATES = list(ParameterSampler(SEARCH_SPACE, n_iter=N_RANDOM_COMBINATIONS, random_state=RANDOM_STATE))

MISSING_STRINGS = {'', ' ', 'na', 'n/a', 'nan', 'none', 'null', '.', 'missing', '<na>'}
BINARY_MAPS = {
    'sex': {'female': 0, 'f': 0, 'male': 1, 'm': 1},
    'ethnicity': {'non-hispanic': 0, 'non hispanic': 0, 'hispanic': 1},
    'cancer_status': {'no cancer': 0, 'no': 0, 'none': 0, 'any cancer': 1, 'yes': 1, 'cancer': 1},
    'institution_type': {'hospital': 0, 'non-hospital': 1, 'non hospital': 1, 'nonhospital': 1},
    'operative_region_extent': {'lumbar only': 0, 'extended_region_involvement': 1, 'extended region involvement': 1, 'extended': 1},
}
ORDINAL_MAPS = {
    'diabetes_status': {'no': 0, 'none': 0, '0': 0, 'without comp': 1, 'without complication': 1, 'without complications': 1, '1': 1, 'with comp': 2, 'with complication': 2, 'with complications': 2, '2': 2},
    'institution_size': {'between 1-99 beds': 0, '1-99': 0, 'between 100-399 beds': 1, '100-399': 1, '>= 400 beds': 2, '>=400 beds': 2, '>=400': 2, '>= 400': 2},
    'asa': {'1': 1, 'i': 1, '2': 2, 'ii': 2, '3': 3, 'iii': 3, '4': 4, 'iv': 4, '>=4': 4, '>= 4': 4, '5': 4, 'v': 4},
    'number_operated_levels': {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '>=4': 4, '>= 4': 4, '5': 4, '6': 4, '7': 4, '8': 4, '9': 4, '10': 4},
}
PREFERRED_NOMINAL_LEVELS = {
    'race': ['White', 'Black', 'Other'],
    'institution_region': ['South', 'North East', 'West', 'Midwest'],
    'payer_status': ['Medicare', 'Commercial/Private', 'Other', 'Medicaid/Public/Government'],
    'PatTobaccoUse': ['Unknown/Not reported/Multiple', 'Never', 'Former', 'Current'],
}
FEATURE_LABELS = {
    'age': 'Age', 'bmi': 'BMI', 'sex': 'Male sex', 'race': 'Race', 'ethnicity': 'Hispanic ethnicity',
    'cancer_status': 'Any cancer', 'diabetes_status': 'Diabetes status', 'institution_type': 'Non-hospital institution',
    'institution_size': 'Institution size', 'institution_region': 'Institution region', 'asa': 'ASA physical status',
    'payer_status': 'Primary payer', 'PatTobaccoUse': 'Tobacco use',
    'finaldx_degenerative': 'Degenerative diagnosis', 'finaldx_radicular': 'Radiculopathy diagnosis',
    'finaldx_stenosis': 'Spinal stenosis diagnosis', 'finaldx_deformity_instability': 'Deformity or instability diagnosis',
    'finaldx_other_diagnosis': 'Other lumbar diagnosis', 'chronic_pulmonary_disease': 'Chronic pulmonary disease',
    'congestive_heart_failure': 'Congestive heart failure', 'connective_tissue_rheumatic_disease': 'Connective tissue/rheumatic disease',
    'myocardial_infarction': 'Myocardial infarction', 'renal_disease': 'Renal disease',
    'alif_llif': 'Anterior/lateral lumbar interbody fusion', 'corpectomy': 'Corpectomy', 'discectomy': 'Discectomy',
    'foraminotomy': 'Foraminotomy', 'instrumentation': 'Instrumentation',
    'laminectomy_posterior_decompression': 'Posterior decompression or laminectomy', 'pelvic_fixation': 'Pelvic fixation',
    'plf': 'Posterolateral fusion', 'tlif_plif': 'Posterior/transforaminal lumbar interbody fusion',
    'other_lumbar_procedure': 'Other lumbar procedure', 'number_operated_levels': 'Number of operated levels',
    'operative_region_extent': 'Operative region extent',
}
FEATURE_LABELS.update({'preop_ODI': 'Preoperative ODI', 'postop_ODI': 'Postoperative ODI', 'ODI_change': 'Change in ODI', PROM_CHANGE_RATE_COL: 'ODI change rate', RELATIVE_ODI_MCID_COL: 'Relative ODI MCID', 'postop_ODI_day': 'Timing of postoperative ODI assessment'})

def clean_scalar(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().replace('≥', '>=')
        return np.nan if s.lower() in MISSING_STRINGS else s
    return x

def norm_text(x):
    x = clean_scalar(x)
    if pd.isna(x):
        return None
    return str(x).strip().replace('≥', '>=').lower()

def to_binary_target(x):
    sx = norm_text(x)
    if sx is None:
        return np.nan
    if sx in {'1', '1.0', 'yes', 'y', 'true', 't'}:
        return 1.0
    if sx in {'0', '0.0', 'no', 'n', 'false', 'f'}:
        return 0.0
    try:
        v = float(sx)
        return float(v) if v in (0.0, 1.0) else np.nan
    except Exception:
        return np.nan

def safe_filename(x):
    return re.sub(r'_+', '_', re.sub(r'[^A-Za-z0-9_.-]+', '_', str(x))).strip('_')[:180] or 'file'

def json_native(obj):
    if isinstance(obj, dict):
        return {str(k): json_native(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_native(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj

def sanitize_params(params):
    out = {}
    for k, v in params.items():
        if k in INT_PARAMS:
            out[k] = int(round(float(v)))
        elif k in FLOAT_PARAMS:
            out[k] = float(v)
        else:
            out[k] = v
    return out

def safe_average_precision(y, p):
    return np.nan if len(np.unique(y)) < 2 else float(average_precision_score(y, p))

def safe_roc_auc(y, p):
    return np.nan if len(np.unique(y)) < 2 else float(roc_auc_score(y, p))

def expected_calibration_error(y, p, n_bins=10):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    out = 0.0
    if len(y) == 0:
        return np.nan
    for i in range(n_bins):
        mask = (p >= bins[i]) & ((p <= bins[i + 1]) if i == n_bins - 1 else (p < bins[i + 1]))
        if np.any(mask):
            out += (np.sum(mask) / len(y)) * abs(float(np.mean(y[mask])) - float(np.mean(p[mask])))
    return float(out)

def classification_metrics(y, p, threshold):
    y = np.asarray(y).astype(int)
    pred = (np.asarray(p) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        'threshold': float(threshold),
        'F1': float(f1_score(y, pred, zero_division=0)),
        'Sensitivity': tp / (tp + fn) if tp + fn > 0 else np.nan,
        'Specificity': tn / (tn + fp) if tn + fp > 0 else np.nan,
        'PPV': tp / (tp + fp) if tp + fp > 0 else np.nan,
        'NPV': tn / (tn + fn) if tn + fn > 0 else np.nan,
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
        'Predicted_positive_rate': float(np.mean(pred)),
    }

def top5_metrics(y, p):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    n = len(y)
    k = max(1, int(math.ceil(n * 0.05)))
    idx = np.argsort(-p)[:k]
    prev = float(np.mean(y)) if n else np.nan
    rate = float(np.mean(y[idx])) if n else np.nan
    return {
        'Top_5pct_n': int(k),
        'Top_5pct_event_rate': rate,
        'Top_5pct_lift': rate / prev if prev > 0 else np.nan,
        'Top_5pct_captured_events': float(np.sum(y[idx]) / np.sum(y)) if np.sum(y) > 0 else np.nan,
    }

def select_threshold(y, p):
    precision, recall, thresholds = precision_recall_curve(y, p)
    if len(thresholds) == 0:
        threshold = 0.5
    else:
        precision = precision[:-1]
        recall = recall[:-1]
        f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
        threshold = float(thresholds[int(np.nanargmax(f1))])
    return threshold, classification_metrics(y, p, threshold)

def eval_preds(y, p, threshold=None, prefix=''):
    out = {
        f'{prefix}AP': safe_average_precision(y, p),
        f'{prefix}ROC_AUC': safe_roc_auc(y, p),
        f'{prefix}Brier_score': float(brier_score_loss(y, p)),
        f'{prefix}ECE': expected_calibration_error(y, p, ECE_N_BINS),
        f'{prefix}N': int(len(y)),
        f'{prefix}Events': int(np.sum(y)),
        f'{prefix}Prevalence': float(np.mean(y)),
    }
    if threshold is not None:
        out.update({f'{prefix}{k}': v for k, v in classification_metrics(y, p, threshold).items()})
        out.update({f'{prefix}{k}': v for k, v in top5_metrics(y, p).items()})
    return out

def make_weights(y, multiplier):
    y = np.asarray(y).astype(int)
    n_pos = int(np.sum(y == 1))
    n_neg = int(np.sum(y == 0))
    if n_pos == 0:
        raise ValueError('No positive events in training fold.')
    pos_weight = (n_neg / max(n_pos, 1)) * float(multiplier)
    return np.where(y == 1, pos_weight, 1.0).astype(float)

def actual_positive_weight(y, multiplier):
    y = np.asarray(y).astype(int)
    return float((np.sum(y == 0) / max(np.sum(y == 1), 1)) * multiplier)

@dataclass
class ModelPreprocessor:
    continuous_features: List[str]
    binary_features: List[str]
    ordinal_features: List[str]
    nominal_features: List[str]
    preferred_nominal_levels: Dict[str, List[str]]
    scale_continuous: bool = False

    def __post_init__(self):
        self.continuous_imputer = None
        self.discrete_imputer = None
        self.nominal_imputer = None
        self.scaler = None
        self.onehot = None
        self.output_feature_names_ = []

    def _parse_binary(self, x, feature):
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in BINARY_MAPS and sx in BINARY_MAPS[feature]:
            return float(BINARY_MAPS[feature][sx])
        if sx in {'1', '1.0', 'yes', 'y', 'true', 't', 'present', 'positive'}:
            return 1.0
        if sx in {'0', '0.0', 'no', 'n', 'false', 'f', 'absent', 'negative'}:
            return 0.0
        try:
            v = float(sx)
            return float(v) if v in (0.0, 1.0) else np.nan
        except Exception:
            return np.nan

    def _parse_ordinal(self, x, feature):
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in ORDINAL_MAPS and sx in ORDINAL_MAPS[feature]:
            return float(ORDINAL_MAPS[feature][sx])
        try:
            v = float(sx)
            if feature == 'asa':
                return float(min(max(int(round(v)), 1), 4))
            if feature == 'number_operated_levels':
                return float(min(max(int(round(v)), 0), 4))
            if feature == 'diabetes_status':
                return float(min(max(int(round(v)), 0), 2))
            if feature == 'institution_size':
                return float(min(max(int(round(v)), 0), 2))
            return float(v)
        except Exception:
            return np.nan

    def _nominal(self, feature, x):
        x = clean_scalar(x)
        if pd.isna(x):
            return np.nan
        s = str(x).strip()
        sl = s.lower()
        if feature == 'race':
            return 'White' if sl == 'white' else ('Black' if sl == 'black' else 'Other')
        if feature == 'institution_region':
            mapping = {'south': 'South', 'north east': 'North East', 'northeast': 'North East', 'north-east': 'North East', 'west': 'West', 'midwest': 'Midwest', 'mid west': 'Midwest'}
            return mapping.get(sl, s)
        if feature == 'payer_status':
            if sl == 'medicare':
                return 'Medicare'
            if sl in {'commercial/private', 'commercial', 'private', 'commercial private'}:
                return 'Commercial/Private'
            if sl in {'medicaid/public/government', 'medicaid', 'public', 'government', 'public/government'}:
                return 'Medicaid/Public/Government'
            return 'Other'
        if feature == 'PatTobaccoUse':
            return 'Never' if sl == 'never' else ('Former' if sl == 'former' else ('Current' if sl == 'current' else 'Unknown/Not reported/Multiple'))
        return s

    def _parts(self, X):
        cont = pd.DataFrame(index=X.index)
        disc = pd.DataFrame(index=X.index)
        nom = pd.DataFrame(index=X.index)
        for c in self.continuous_features:
            cont[c] = pd.to_numeric(X[c].map(clean_scalar), errors='coerce')
        for c in self.binary_features:
            disc[c] = X[c].map(lambda z: self._parse_binary(z, c)).astype(float)
        for c in self.ordinal_features:
            disc[c] = X[c].map(lambda z: self._parse_ordinal(z, c)).astype(float)
        for c in self.nominal_features:
            nom[c] = X[c].map(lambda z: self._nominal(c, z)).astype('object')
        return cont, disc, nom

    def fit(self, X):
        cont, disc, nom = self._parts(X)
        self.continuous_imputer = SimpleImputer(strategy='median')
        self.discrete_imputer = SimpleImputer(strategy='most_frequent')
        self.nominal_imputer = SimpleImputer(strategy='constant', fill_value='Missing')
        cont_imp = self.continuous_imputer.fit_transform(cont)
        if self.scale_continuous:
            self.scaler = StandardScaler()
            self.scaler.fit(cont_imp)
        self.discrete_imputer.fit(disc)
        nom_imp = pd.DataFrame(self.nominal_imputer.fit_transform(nom), columns=self.nominal_features)
        cats = []
        for c in self.nominal_features:
            preferred = list(self.preferred_nominal_levels.get(c, []))
            observed = nom_imp[c].astype(str).unique().tolist()
            cats.append(preferred + sorted([x for x in observed if x not in preferred]))
        try:
            self.onehot = OneHotEncoder(categories=cats, handle_unknown='ignore', sparse_output=False)
        except TypeError:
            self.onehot = OneHotEncoder(categories=cats, handle_unknown='ignore', sparse=False)
        self.onehot.fit(nom_imp.astype(str))
        self.output_feature_names_ = self.continuous_features + self.binary_features + self.ordinal_features + self.onehot.get_feature_names_out(self.nominal_features).tolist()
        return self

    def transform(self, X):
        cont, disc, nom = self._parts(X)
        cont_imp = self.continuous_imputer.transform(cont)
        if self.scale_continuous and self.scaler is not None:
            cont_imp = self.scaler.transform(cont_imp)
        disc_imp = self.discrete_imputer.transform(disc)
        nom_imp = pd.DataFrame(self.nominal_imputer.transform(nom), columns=self.nominal_features)
        nom_oh = self.onehot.transform(nom_imp.astype(str))
        return np.concatenate([cont_imp, disc_imp, nom_oh], axis=1).astype(float)

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

    @property
    def output_feature_names(self):
        return self.output_feature_names_

def raw_model_scores(model, Xp, n_iter=None):
    if CLASSIFIER_KEY == 'HistGradientBoosting' and n_iter is not None:
        selected = None
        for i, pred in enumerate(model.staged_predict_proba(Xp), start=1):
            selected = pred
            if i >= int(n_iter):
                break
        if selected is not None:
            return selected[:, 1]
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(Xp)[:, 1]
    if hasattr(model, 'decision_function'):
        return model.decision_function(Xp)
    raise AttributeError('Model provides neither predict_proba nor decision_function.')

def predict(pre, model, X, features, n_iter=None):
    return raw_model_scores(model, pre.transform(X[features]), n_iter=n_iter)

def patient_split(df, target_col, seed):
    group_df = df.groupby(GROUP_COL, dropna=False)[target_col].max().reset_index()
    y_group = group_df[target_col].astype(int).to_numpy()
    groups = group_df[GROUP_COL].to_numpy()
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=seed)
    train_cal_idx, test_idx = next(sss1.split(groups, y_group))
    groups_train_cal = groups[train_cal_idx]
    y_train_cal = y_group[train_cal_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=CALIBRATION_FRACTION_OF_REMAINING, random_state=seed + 1)
    train_idx_rel, cal_idx_rel = next(sss2.split(groups_train_cal, y_train_cal))
    train_groups = set(groups_train_cal[train_idx_rel])
    cal_groups = set(groups_train_cal[cal_idx_rel])
    test_groups = set(groups[test_idx])
    assert train_groups.isdisjoint(cal_groups) and train_groups.isdisjoint(test_groups) and cal_groups.isdisjoint(test_groups)
    return df[GROUP_COL].isin(train_groups).to_numpy(), df[GROUP_COL].isin(cal_groups).to_numpy(), df[GROUP_COL].isin(test_groups).to_numpy()

def cv_splits(y, groups, seed, n_folds=N_CV_FOLDS):
    group_df = pd.DataFrame({'group': groups, 'y': y}).groupby('group')['y'].max().reset_index()
    n_folds = min(n_folds, int(np.sum(group_df.y == 1)), int(np.sum(group_df.y == 0)))
    if n_folds < 2:
        raise ValueError('Not enough positive and negative patient groups for group-aware CV.')
    cv = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    return [(tr, va) for tr, va in cv.split(np.zeros(len(y)), y, groups)]

def feature_types(features):
    cont = [f for f in features if f in CONTINUOUS_FEATURES_ALL or (f not in BINARY_FEATURES and f not in ORDINAL_FEATURES and f not in NOMINAL_FEATURES)]
    binary = [f for f in features if f in BINARY_FEATURES]
    ordinal = [f for f in features if f in ORDINAL_FEATURES]
    nominal = [f for f in features if f in NOMINAL_FEATURES]
    return cont, binary, ordinal, nominal

def fit_pipeline(X, y, features, params, seed, eval_set=None, early=False, n_estimators_override=None):
    cont, binary, ordinal, nominal = feature_types(features)
    pre = ModelPreprocessor(cont, binary, ordinal, nominal, PREFERRED_NOMINAL_LEVELS, scale_continuous=SCALE_CONTINUOUS)
    Xp = pre.fit_transform(X[features])
    eval_transformed = None
    if eval_set is not None:
        X_val, y_val = eval_set
        eval_transformed = (pre.transform(X_val[features]), y_val)
    model, best_iter = fit_estimator(Xp, y, params, seed, eval_set=eval_transformed, use_early_stopping=early, n_estimators_override=n_estimators_override)
    return pre, model, best_iter

def tune(X_train, y_train, groups_train, features, folds, model_key, seed):
    candidate_rows = []
    fold_rows = []
    sampler = list(ParameterSampler(SEARCH_SPACE, n_iter=N_RANDOM_COMBINATIONS, random_state=seed))
    for cid, raw_params in enumerate(sampler, start=1):
        params = sanitize_params(raw_params)
        fold_train_aps, fold_train_aucs, fold_aps, fold_aucs, best_iters = [], [], [], [], []
        t0 = time.time()
        for fid, (tr_idx, va_idx) in enumerate(folds, start=1):
            X_tr = X_train.iloc[tr_idx].reset_index(drop=True)
            y_tr = y_train[tr_idx]
            X_va = X_train.iloc[va_idx].reset_index(drop=True)
            y_va = y_train[va_idx]
            pre, model, best_iter = fit_pipeline(X_tr, y_tr, features, params, seed + cid * 1000 + fid, eval_set=(X_va, y_va), early=USE_EARLY_STOPPING_IN_CV)
            p_tr = predict(pre, model, X_tr, features, n_iter=best_iter)
            p_va = predict(pre, model, X_va, features, n_iter=best_iter)
            tr_ap, tr_auc = safe_average_precision(y_tr, p_tr), safe_roc_auc(y_tr, p_tr)
            va_ap, va_auc = safe_average_precision(y_va, p_va), safe_roc_auc(y_va, p_va)
            fold_train_aps.append(tr_ap); fold_train_aucs.append(tr_auc); fold_aps.append(va_ap); fold_aucs.append(va_auc)
            if best_iter is not None and best_iter > 0:
                best_iters.append(best_iter)
            fold_rows.append({
                'model_key': model_key, 'candidate_id': cid, 'fold': fid,
                'fold_train_AP': tr_ap, 'fold_train_ROC_AUC': tr_auc,
                'fold_validation_AP': va_ap, 'fold_validation_ROC_AUC': va_auc,
                'fold_train_minus_validation_AP_gap': tr_ap - va_ap if np.isfinite(tr_ap) and np.isfinite(va_ap) else np.nan,
                'fold_train_minus_validation_ROC_AUC_gap': tr_auc - va_auc if np.isfinite(tr_auc) and np.isfinite(va_auc) else np.nan,
                'fold_best_iteration': best_iter,
                'positive_weight_used': actual_positive_weight(y_tr, params['positive_weight_multiplier']),
                **params,
            })
        locked = int(np.median(best_iters)) if best_iters and USE_EARLY_STOPPING_IN_CV else (int(params[ITERATION_PARAM]) if ITERATION_PARAM and ITERATION_PARAM in params else np.nan)
        row = {
            'model_key': model_key, 'candidate_id': cid, 'cv_folds': len(folds),
            'cv_train_AP_mean': float(np.nanmean(fold_train_aps)),
            'cv_train_ROC_AUC_mean': float(np.nanmean(fold_train_aucs)),
            'cv_AP_mean': float(np.nanmean(fold_aps)),
            'cv_AP_SD': float(np.nanstd(fold_aps, ddof=1)) if len(fold_aps) > 1 else np.nan,
            'cv_ROC_AUC_mean': float(np.nanmean(fold_aucs)),
            'cv_ROC_AUC_SD': float(np.nanstd(fold_aucs, ddof=1)) if len(fold_aucs) > 1 else np.nan,
            'mean_cv_best_iteration': float(np.mean(best_iters)) if best_iters else np.nan,
            'locked_n_estimators_from_cv': locked,
            'elapsed_seconds': float(time.time() - t0),
            **params,
        }
        candidate_rows.append(row)
        print(f"{model_key} | candidate {cid:03d}/{len(sampler)} | CV AP={row['cv_AP_mean']:.5f} | AUC={row['cv_ROC_AUC_mean']:.5f}")
    return pd.DataFrame(candidate_rows).sort_values('cv_AP_mean', ascending=False).reset_index(drop=True), pd.DataFrame(fold_rows)

def fit_final(X_train, y_train, X_cal, y_cal, X_test, features, params, locked_n, seed):
    n_override = int(locked_n) if pd.notna(locked_n) and USE_EARLY_STOPPING_IN_CV and ITERATION_PARAM else None
    pre, model, _ = fit_pipeline(X_train, y_train, features, params, seed, n_estimators_override=n_override)
    p_train_raw = predict(pre, model, X_train, features)
    p_cal_raw = predict(pre, model, X_cal, features)
    p_test_raw = predict(pre, model, X_test, features)
    calibrator = IsotonicRegression(out_of_bounds='clip')
    calibrator.fit(p_cal_raw, y_cal)
    p_train = calibrator.predict(p_train_raw)
    p_cal = calibrator.predict(p_cal_raw)
    p_test = calibrator.predict(p_test_raw)
    threshold, cal_metrics = select_threshold(y_cal, p_cal)
    return {
        'pre': pre, 'model': model, 'calibrator': calibrator,
        'p_train_raw': p_train_raw, 'p_cal_raw': p_cal_raw, 'p_test_raw': p_test_raw,
        'p_train': p_train, 'p_cal': p_cal, 'p_test': p_test,
        'threshold': threshold, 'cal_metrics': cal_metrics,
    }

def patient_boot_ci(y, p, groups, metric, threshold=None, seed=1):
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({'idx': np.arange(len(y)), 'group': groups, 'y': np.asarray(y).astype(int)})
    group_y = d.groupby('group').y.max()
    pos = group_y[group_y == 1].index.to_numpy()
    neg = group_y[group_y == 0].index.to_numpy()
    if len(pos) == 0 or len(neg) == 0:
        return np.nan, np.nan
    by_group = {g: d.loc[d.group.eq(g), 'idx'].to_numpy() for g in group_y.index}
    vals = []
    for _ in range(N_BOOTSTRAPS):
        sample_groups = np.concatenate([rng.choice(pos, len(pos), replace=True), rng.choice(neg, len(neg), replace=True)])
        sample_idx = np.concatenate([by_group[g] for g in sample_groups])
        yy = np.asarray(y)[sample_idx]
        pp = np.asarray(p)[sample_idx]
        if len(np.unique(yy)) < 2:
            continue
        if metric == 'AP':
            vals.append(average_precision_score(yy, pp))
        elif metric == 'ROC_AUC':
            vals.append(roc_auc_score(yy, pp))
        elif metric == 'F1':
            vals.append(f1_score(yy, pp >= threshold, zero_division=0))
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))) if vals else (np.nan, np.nan)

def paired_delta_boot(y, p0, p1, groups, metric, seed=1):
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({'idx': np.arange(len(y)), 'group': groups, 'y': np.asarray(y).astype(int)})
    group_y = d.groupby('group').y.max()
    pos = group_y[group_y == 1].index.to_numpy()
    neg = group_y[group_y == 0].index.to_numpy()
    by_group = {g: d.loc[d.group.eq(g), 'idx'].to_numpy() for g in group_y.index}
    observed = (safe_average_precision(y, p1) - safe_average_precision(y, p0)) if metric == 'AP' else (safe_roc_auc(y, p1) - safe_roc_auc(y, p0))
    vals = []
    for _ in range(N_BOOTSTRAPS):
        sample_groups = np.concatenate([rng.choice(pos, len(pos), replace=True), rng.choice(neg, len(neg), replace=True)])
        sample_idx = np.concatenate([by_group[g] for g in sample_groups])
        yy = np.asarray(y)[sample_idx]
        if len(np.unique(yy)) < 2:
            continue
        if metric == 'AP':
            vals.append(average_precision_score(yy, np.asarray(p1)[sample_idx]) - average_precision_score(yy, np.asarray(p0)[sample_idx]))
        else:
            vals.append(roc_auc_score(yy, np.asarray(p1)[sample_idx]) - roc_auc_score(yy, np.asarray(p0)[sample_idx]))
    if not vals:
        return observed, np.nan, np.nan, np.nan
    vals = np.asarray(vals)
    ci_low, ci_high = float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))
    p_value = float(2 * min(np.mean(vals <= 0), np.mean(vals >= 0)))
    return float(observed), ci_low, ci_high, min(p_value, 1.0)

def save_learning_curve(X_train, y_train, groups_train, features, params, locked_n, seed, out_path, title):
    group_df = pd.DataFrame({'group': groups_train, 'y': y_train}).groupby('group').y.max().reset_index()
    rng = np.random.default_rng(seed)
    rows = []
    for frac in LEARNING_CURVE_TRAIN_FRACTIONS:
        for rep in range(LEARNING_CURVE_CV_FOLDS):
            selected_groups = []
            for cls in [0, 1]:
                cls_groups = group_df.loc[group_df.y.eq(cls), 'group'].to_numpy()
                k = max(1, int(math.ceil(len(cls_groups) * frac)))
                selected_groups.extend(rng.choice(cls_groups, k, replace=False).tolist())
            mask = np.isin(groups_train, selected_groups)
            X_sub = X_train.loc[mask].reset_index(drop=True)
            y_sub = y_train[mask]
            g_sub = groups_train[mask]
            try:
                folds = cv_splits(y_sub, g_sub, seed + rep, n_folds=3)
            except Exception:
                continue
            for tr, va in folds:
                n_override = int(locked_n) if pd.notna(locked_n) and USE_EARLY_STOPPING_IN_CV and ITERATION_PARAM else None
                pre, model, _ = fit_pipeline(X_sub.iloc[tr].reset_index(drop=True), y_sub[tr], features, params, seed + rep, n_estimators_override=n_override)
                p_tr = predict(pre, model, X_sub.iloc[tr].reset_index(drop=True), features)
                p_va = predict(pre, model, X_sub.iloc[va].reset_index(drop=True), features)
                rows.append({'train_fraction': frac, 'train_AP': safe_average_precision(y_sub[tr], p_tr), 'validation_AP': safe_average_precision(y_sub[va], p_va)})
    lc = pd.DataFrame(rows)
    lc.to_csv(out_path.replace('.png', '.csv'), index=False)
    if not lc.empty:
        s = lc.groupby('train_fraction').agg(train_AP_mean=('train_AP', 'mean'), validation_AP_mean=('validation_AP', 'mean')).reset_index()
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.plot(s.train_fraction, s.train_AP_mean, marker='o', label='Training AP')
        ax.plot(s.train_fraction, s.validation_AP_mean, marker='o', label='Validation AP')
        ax.set_xlabel('Fraction of training patient groups')
        ax.set_ylabel('Average precision')
        ax.set_title(title)
        ax.legend()
        fig.tight_layout()
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close(fig)

def style_excel(writer):
    try:
        from openpyxl.styles import Font, PatternFill, Alignment
        for sheet in writer.sheets:
            ws = writer.sheets[sheet]
            ws.freeze_panes = 'A2'
            ws.auto_filter.ref = ws.dimensions
            for cell in ws[1]:
                cell.font = Font(bold=True)
                cell.fill = PatternFill(start_color='D9EAF7', end_color='D9EAF7', fill_type='solid')
                cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            for col in ws.columns:
                max_len = max([len(str(c.value)) for c in col if c.value is not None] + [12])
                ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 12), 70)
    except Exception:
        pass


def make_model(params, seed, n_estimators_override=None):
    p = sanitize_params(params)
    p.pop('positive_weight_multiplier', None)
    try:
        return LinearSVC(random_state=seed, dual='auto', max_iter=20000, **p)
    except TypeError:
        return LinearSVC(random_state=seed, dual=False, max_iter=20000, **p)

def fit_estimator(X_train, y_train, params, seed, eval_set=None, use_early_stopping=False, n_estimators_override=None):
    weights = make_weights(y_train, float(params['positive_weight_multiplier']))
    model = make_model(params, seed)
    model.fit(X_train, y_train, sample_weight=weights)
    return model, None


def add_dynamic_odi_features(df):
    out = df.copy()
    required = ['preop_ODI', 'postop_ODI', DAYS_BETWEEN_PROM_COL]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError('Cannot derive dynamic ODI features because required columns are missing: ' + ', '.join(missing))
    preop = pd.to_numeric(out['preop_ODI'].map(clean_scalar), errors='coerce')
    postop = pd.to_numeric(out['postop_ODI'].map(clean_scalar), errors='coerce')
    days_between = pd.to_numeric(out[DAYS_BETWEEN_PROM_COL].map(clean_scalar), errors='coerce')
    odi_change = postop - preop
    valid_rate = preop.notna() & postop.notna() & days_between.gt(0)
    change_rate = pd.Series(np.nan, index=out.index, dtype='float')
    change_rate.loc[valid_rate] = (odi_change.loc[valid_rate] / days_between.loc[valid_rate]).astype(float)
    improvement = preop - postop
    rel_fraction = improvement / preop.replace(0, np.nan)
    valid_relative = preop.notna() & postop.notna() & preop.gt(0)
    zero_baseline_with_postop = preop.eq(0) & postop.notna()
    relative_mcid = pd.Series(np.nan, index=out.index, dtype='float')
    relative_mcid.loc[valid_relative] = (rel_fraction.loc[valid_relative] >= RELATIVE_ODI_MCID_CUTOFF).astype(float)
    relative_mcid.loc[zero_baseline_with_postop] = 0.0
    out['ODI_change'] = odi_change
    out[PROM_CHANGE_RATE_COL] = change_rate
    out[RELATIVE_ODI_MCID_COL] = relative_mcid
    audit = pd.DataFrame([
        {'item': 'PROM change rate definition', 'value': '(postop_ODI - preop_ODI) / days_between_PROMs'},
        {'item': 'Relative ODI MCID definition', 'value': f'(preop_ODI - postop_ODI) / preop_ODI >= {RELATIVE_ODI_MCID_CUTOFF}; preop_ODI=0 coded as 0 when postop_ODI is available'},
        {'item': 'Rows', 'value': int(len(out))},
        {'item': 'Rows with valid preop/postop ODI and positive days between PROMs', 'value': int(valid_rate.sum())},
        {'item': 'Rows with preop ODI = 0 and postop ODI available coded as Relative ODI MCID = 0', 'value': int(zero_baseline_with_postop.sum())},
    ])
    return out, audit

def run_model(model_type, work, features, folds, seed, all_candidates, all_folds):
    key = model_type
    out_dir = os.path.join(OUTPUT_DIR, model_type)
    plot_dir = os.path.join(PLOT_DIR, model_type)
    artifact_dir = os.path.join(ARTIFACT_DIR, model_type)
    for d in [out_dir, plot_dir, artifact_dir]:
        os.makedirs(d, exist_ok=True)
    train = work[work.split.eq('train')].reset_index(drop=True)
    cal = work[work.split.eq('calibration')].reset_index(drop=True)
    test = work[work.split.eq('test')].reset_index(drop=True)
    y_train = train[TARGET_COL].astype(int).to_numpy()
    y_cal = cal[TARGET_COL].astype(int).to_numpy()
    y_test = test[TARGET_COL].astype(int).to_numpy()
    candidates, fold_metrics = tune(train[features], y_train, train[GROUP_COL].to_numpy(), features, folds, key, seed)
    best_params = sanitize_params({k: candidates.loc[0, k] for k in SEARCH_SPACE.keys()})
    locked_n = candidates.loc[0, 'locked_n_estimators_from_cv']
    final = fit_final(train[features], y_train, cal[features], y_cal, test[features], y_test, features, best_params, locked_n, seed + 991)
    threshold = final['threshold']
    summary = {
        'model_type': model_type, 'classifier': CLASSIFIER_DISPLAY_NAME,
        'n_features_original': len(features), 'best_candidate_id': int(candidates.loc[0, 'candidate_id']),
        'CV_AP_mean': float(candidates.loc[0, 'cv_AP_mean']), 'CV_ROC_AUC_mean': float(candidates.loc[0, 'cv_ROC_AUC_mean']),
        'locked_n_estimators_from_cv': locked_n,
        **eval_preds(y_train, final['p_train'], threshold, prefix='Train_'),
        **eval_preds(y_cal, final['p_cal'], threshold, prefix='Calibration_'),
        **eval_preds(y_test, final['p_test'], threshold, prefix='Test_'),
    }
    summary['Train_minus_CV_AP_gap'] = summary['Train_AP'] - summary['CV_AP_mean'] if pd.notna(summary['CV_AP_mean']) else np.nan
    summary['CV_minus_Test_AP_gap'] = summary['CV_AP_mean'] - summary['Test_AP'] if pd.notna(summary['CV_AP_mean']) else np.nan
    summary['Test_AP_CI_lower'], summary['Test_AP_CI_upper'] = patient_boot_ci(y_test, final['p_test'], test[GROUP_COL].to_numpy(), 'AP', seed=seed + 10)
    summary['Test_ROC_AUC_CI_lower'], summary['Test_ROC_AUC_CI_upper'] = patient_boot_ci(y_test, final['p_test'], test[GROUP_COL].to_numpy(), 'ROC_AUC', seed=seed + 20)
    pd.DataFrame([summary]).to_csv(os.path.join(out_dir, f'performance_{{key}}.csv'), index=False)
    pd.DataFrame({'row_index_in_split': np.arange(len(test)), GROUP_COL: test[GROUP_COL].to_numpy(), 'y_true': y_test, 'p_raw': final['p_test_raw'], 'p_calibrated': final['p_test']}).to_csv(os.path.join(out_dir, f'test_predictions_{{key}}.csv'), index=False)
    candidates.to_csv(os.path.join(out_dir, f'cv_candidates_{{key}}.csv'), index=False)
    fold_metrics.to_csv(os.path.join(out_dir, f'cv_fold_metrics_{{key}}.csv'), index=False)
    joblib.dump(final, os.path.join(artifact_dir, f'final_model_{{key}}.joblib'))
    save_learning_curve(train[features], y_train, train[GROUP_COL].to_numpy(), features, best_params, locked_n, seed + 313, os.path.join(plot_dir, f'learning_curve_{{key}}.png'), key)
    all_candidates.append(candidates)
    all_folds.append(fold_metrics)
    return {'summary': summary, 'preds': pd.read_csv(os.path.join(out_dir, f'test_predictions_{{key}}.csv'))}

def split_audit(df):
    rows = []
    for split in ['train', 'calibration', 'test']:
        sub = df[df.split.eq(split)]
        rows.append({'split': split, 'n_rows': len(sub), 'n_patients': sub[GROUP_COL].nunique(), 'events': int(sub[TARGET_COL].sum()), 'prevalence': float(sub[TARGET_COL].mean())})
    return pd.DataFrame(rows)

def methods_table():
    return pd.DataFrame([
        {'item': 'Paired design', 'rationale': f'Baseline-only and dynamic PROM-expanded {{CLASSIFIER_DISPLAY_NAME}} models are tuned separately under the same protocol.'},
        {'item': 'Paired split/folds', 'rationale': 'Paired models use identical patient-level train/calibration/test splits and identical group-aware CV folds.'},
        {'item': 'Tuning metric', 'rationale': 'Average precision is the primary model-selection metric because delayed reoperation is a rare-event outcome.'},
        {'item': 'Class imbalance', 'rationale': 'Positive-class sample weighting is tuned through a multiplier applied to the natural negative-to-positive ratio.'},
        {'item': 'Calibration and threshold', 'rationale': 'Isotonic calibration and maximum-F1 threshold selection are performed only on the calibration split.'},
        {'item': 'Held-out test set', 'rationale': 'The test set is isolated until the model-development pipeline is locked.'},
    ])

def main():
    t0 = time.time()
    df = pd.read_csv(INPUT_CSV)
    df, dynamic_audit = add_dynamic_odi_features(df)
    required = sorted(set(DYNAMIC_PROM_FEATURES + [TARGET_COL, GROUP_COL, DAYS_BETWEEN_PROM_COL]))
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise ValueError('Missing required columns:\n' + '\n'.join(f' - {{c}}' for c in missing_cols))
    preop = pd.to_numeric(df['preop_ODI'].map(clean_scalar), errors='coerce')
    postop = pd.to_numeric(df['postop_ODI'].map(clean_scalar), errors='coerce')
    days = pd.to_numeric(df[DAYS_BETWEEN_PROM_COL].map(clean_scalar), errors='coerce')
    eligible = preop.notna() & postop.notna() & days.gt(0)
    work = df.loc[eligible, DYNAMIC_PROM_FEATURES + [TARGET_COL, GROUP_COL]].copy()
    work[TARGET_COL] = work[TARGET_COL].map(to_binary_target)
    work = work.dropna(subset=[TARGET_COL]).copy()
    work[TARGET_COL] = work[TARGET_COL].astype(int)
    train_mask, cal_mask, test_mask = patient_split(work, TARGET_COL, RANDOM_STATE)
    work['split'] = np.where(train_mask, 'train', np.where(cal_mask, 'calibration', 'test'))
    train = work[work.split.eq('train')].reset_index(drop=True)
    folds = cv_splits(train[TARGET_COL].astype(int).to_numpy(), train[GROUP_COL].to_numpy(), RANDOM_STATE + 10)
    all_candidates, all_folds = [], []
    baseline = run_model('baseline_only', work, BASELINE_FEATURES, folds, RANDOM_STATE + 100, all_candidates, all_folds)
    expanded = run_model('dynamic_PROM_expanded', work, DYNAMIC_PROM_FEATURES, folds, RANDOM_STATE + 200, all_candidates, all_folds)
    merged = baseline['preds'].merge(expanded['preds'], on=['row_index_in_split', GROUP_COL, 'y_true'], suffixes=('_baseline', '_expanded'), validate='one_to_one')
    y = merged.y_true.astype(int).to_numpy()
    p0 = merged.p_calibrated_baseline.astype(float).to_numpy()
    p1 = merged.p_calibrated_expanded.astype(float).to_numpy()
    groups = merged[GROUP_COL].to_numpy()
    dap = paired_delta_boot(y, p0, p1, groups, 'AP', RANDOM_STATE + 1000)
    dauc = paired_delta_boot(y, p0, p1, groups, 'ROC_AUC', RANDOM_STATE + 2000)
    comparison = pd.DataFrame([{ 'baseline_Test_AP': baseline['summary']['Test_AP'], 'expanded_Test_AP': expanded['summary']['Test_AP'], 'Delta_AP_expanded_minus_baseline': dap[0], 'Delta_AP_CI_lower': dap[1], 'Delta_AP_CI_upper': dap[2], 'Delta_AP_bootstrap_p_value': dap[3], 'baseline_Test_ROC_AUC': baseline['summary']['Test_ROC_AUC'], 'expanded_Test_ROC_AUC': expanded['summary']['Test_ROC_AUC'], 'Delta_ROC_AUC_expanded_minus_baseline': dauc[0], 'Delta_ROC_AUC_CI_lower': dauc[1], 'Delta_ROC_AUC_CI_upper': dauc[2], 'Delta_ROC_AUC_bootstrap_p_value': dauc[3]}])
    summary = pd.DataFrame([baseline['summary'], expanded['summary']])
    candidates = pd.concat(all_candidates, ignore_index=True)
    folds_df = pd.concat(all_folds, ignore_index=True)
    missingness = pd.DataFrame([{'column': c, 'n_missing_or_blank': int(work[c].isna().sum()), 'percent_missing_or_blank': 100 * float(work[c].isna().sum()) / len(work)} for c in DYNAMIC_PROM_FEATURES if c in work.columns])
    xlsx = os.path.join(OUTPUT_DIR, f'Step2_DynamicPROM_{{CLASSIFIER_KEY}}_summary.xlsx')
    with pd.ExcelWriter(xlsx, engine='openpyxl') as writer:
        methods_table().to_excel(writer, 'methods_rationale', index=False)
        dynamic_audit.to_excel(writer, 'dynamic_feature_audit', index=False)
        summary.to_excel(writer, 'model_performance', index=False)
        comparison.to_excel(writer, 'paired_PROM_comparison', index=False)
        candidates.to_excel(writer, 'cv_candidates_all_models', index=False)
        folds_df.to_excel(writer, 'cv_fold_metrics_all', index=False)
        split_audit(work).to_excel(writer, 'split_audit', index=False)
        missingness.to_excel(writer, 'missingness_audit', index=False)
        style_excel(writer)
    summary.to_csv(os.path.join(OUTPUT_DIR, 'model_performance.csv'), index=False)
    comparison.to_csv(os.path.join(OUTPUT_DIR, 'paired_PROM_comparison.csv'), index=False)
    candidates.to_csv(os.path.join(OUTPUT_DIR, 'cv_candidates_all_models.csv'), index=False)
    folds_df.to_csv(os.path.join(OUTPUT_DIR, 'cv_fold_metrics_all.csv'), index=False)
    work[[GROUP_COL, 'split']].drop_duplicates().to_csv(os.path.join(OUTPUT_DIR, 'split_assignment.csv'), index=False)
    manifest = {'classifier': CLASSIFIER_DISPLAY_NAME, 'output_dir': OUTPUT_DIR, 'design': f'paired baseline-only and dynamic PROM-expanded {{CLASSIFIER_DISPLAY_NAME}} models', 'model_selection': 'highest mean group-aware CV average precision inside training split', 'calibration': 'isotonic', 'threshold_strategy': 'maximum F1 on calibration split', 'runtime_minutes': float((time.time() - t0) / 60), 'summary_xlsx': xlsx}
    json.dump(json_native(manifest), open(os.path.join(OUTPUT_DIR, 'run_manifest.json'), 'w'), indent=2, sort_keys=True)
    zip_path = os.path.join(OUTPUT_DIR, f'Step2_DynamicPROM_{{CLASSIFIER_KEY}}_outputs.zip')
    tmp = zip_path + '.tmp'
    for p in [zip_path, tmp]:
        if os.path.exists(p):
            os.remove(p)
    with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=ZIP_COMPRESSION_LEVEL) as z:
        for root, _, files in os.walk(OUTPUT_DIR):
            for name in files:
                full = os.path.join(root, name)
                if full in {zip_path, tmp}:
                    continue
                z.write(full, os.path.relpath(full, OUTPUT_DIR))
    os.replace(tmp, zip_path)
    print('=' * 100)
    print(f'Step 2 {{CLASSIFIER_DISPLAY_NAME}} analysis completed')
    print('Summary Excel:', xlsx)
    print('ZIP archive:', zip_path)
    print('=' * 100)
    if CREATE_COLAB_DOWNLOAD_LINK:
        try:
            from IPython.display import HTML, display
            display(HTML(f'<p><b>Step 2 {{CLASSIFIER_DISPLAY_NAME}} output archive is ready.</b></p><p><a href="/files{{zip_path}}" download>Click here to download the ZIP archive</a></p><p>Path: <code>{{zip_path}}</code></p>'))
        except Exception:
            pass
    if AUTO_DOWNLOAD_ZIP:
        try:
            from google.colab import files
            files.download(zip_path)
        except Exception:
            pass

if __name__ == '__main__':
    main()


#**Step 2: Logistic Regression**

In [ ]:
"""
Step 2 dynamic postoperative PROM analysis with logistic regression
===================================================================

Input
-----
/content/Step 2_ODI_Cohort.csv

Target
------
final_reop_step2
    1 = reoperation from postoperative day 91 through day 365
    0 = no reoperation from postoperative day 91 through day 365

Design
------
This script runs paired baseline-only and dynamic PROM-expanded logistic regression
models for delayed lumbar reoperation prediction. The baseline-only model
uses the same 35 baseline variables as Step 1. The dynamic PROM-expanded
model includes the same baseline variables plus preoperative PROM,
postoperative PROM, PROM change, PROM change rate, relative MCID status,
and timing of postoperative PROM assessment. Paired models use identical
patient-level train/calibration/test splits and identical group-aware
cross-validation fold construction. Hyperparameter tuning is performed
exclusively within the training split using cross-validated average
precision as the primary selection metric. Probability calibration and
threshold selection are performed only on the calibration split. The
held-out test set is reserved until the model-development pipeline is
locked.
"""

# -*- coding: utf-8 -*-

import os
import re
import sys
import json
import math
import time
import zipfile
import platform
import subprocess
import warnings
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    import joblib
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'joblib'])
    import joblib

try:
    import openpyxl  # noqa
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl'])
    import openpyxl  # noqa

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedGroupKFold, ParameterSampler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_recall_curve,
    roc_curve,
    confusion_matrix,
    f1_score,
)
from sklearn.isotonic import IsotonicRegression
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')


from sklearn.linear_model import LogisticRegression


BASE_DIR = '/content'
INPUT_CSV = os.path.join(BASE_DIR, 'Step 2_ODI_Cohort.csv')
CLASSIFIER_KEY = 'LogisticRegression'
CLASSIFIER_DISPLAY_NAME = 'logistic regression'
SCALE_CONTINUOUS = True
OUTPUT_DIR = os.path.join(BASE_DIR, 'Step2_DynamicPROM_LogisticRegression_publication')
PLOT_DIR = os.path.join(OUTPUT_DIR, 'plots')
ARTIFACT_DIR = os.path.join(OUTPUT_DIR, 'model_artifacts')
for d in [OUTPUT_DIR, PLOT_DIR, ARTIFACT_DIR]:
    os.makedirs(d, exist_ok=True)

TARGET_COL = 'final_reop_step2'
GROUP_COL = 'PersonKey'
RANDOM_STATE = 20260524
TEST_FRACTION = 0.20
CALIBRATION_FRACTION_OF_REMAINING = 0.20
N_CV_FOLDS = 5
N_RANDOM_COMBINATIONS = 300
USE_EARLY_STOPPING_IN_CV = False
EARLY_STOPPING_ROUNDS = 100
MIN_FINAL_N_ESTIMATORS = 50
THRESHOLD_STRATEGY = 'max_f1'
N_BOOTSTRAPS = 2000
ECE_N_BINS = 10
N_JOBS = -1
LEARNING_CURVE_TRAIN_FRACTIONS = (0.20, 0.40, 0.60, 0.80, 1.00)
LEARNING_CURVE_CV_FOLDS = 3
ZIP_COMPRESSION_LEVEL = 1
CREATE_COLAB_DOWNLOAD_LINK = True
AUTO_DOWNLOAD_ZIP = False
RELATIVE_ODI_MCID_CUTOFF = 0.30
PROM_CHANGE_RATE_COL = 'ODI_change_rate'
RELATIVE_ODI_MCID_COL = 'ODI_relative_MCID_binary'
DAYS_BETWEEN_PROM_COL = 'days_between_PROMs'

BASELINE_FEATURES = ['finaldx_degenerative', 'finaldx_radicular', 'finaldx_stenosis', 'finaldx_deformity_instability', 'finaldx_other_diagnosis', 'age', 'sex', 'race', 'ethnicity', 'cancer_status', 'chronic_pulmonary_disease', 'congestive_heart_failure', 'connective_tissue_rheumatic_disease', 'diabetes_status', 'myocardial_infarction', 'renal_disease', 'institution_type', 'institution_size', 'institution_region', 'asa', 'bmi', 'payer_status', 'alif_llif', 'corpectomy', 'discectomy', 'foraminotomy', 'instrumentation', 'laminectomy_posterior_decompression', 'pelvic_fixation', 'plf', 'tlif_plif', 'other_lumbar_procedure', 'number_operated_levels', 'operative_region_extent', 'PatTobaccoUse']
assert len(BASELINE_FEATURES) == 35
STEP2_PROM_FEATURES = ['preop_ODI', 'postop_ODI', 'ODI_change', PROM_CHANGE_RATE_COL, RELATIVE_ODI_MCID_COL, 'postop_ODI_day']
DYNAMIC_PROM_FEATURES = BASELINE_FEATURES + STEP2_PROM_FEATURES
CONTINUOUS_BASELINE_FEATURES = ['age', 'bmi']
CONTINUOUS_FEATURES_ALL = CONTINUOUS_BASELINE_FEATURES + ['preop_ODI', 'postop_ODI', 'ODI_change', PROM_CHANGE_RATE_COL, 'postop_ODI_day']
BINARY_FEATURES = ['finaldx_degenerative', 'finaldx_radicular', 'finaldx_stenosis', 'finaldx_deformity_instability', 'finaldx_other_diagnosis', 'sex', 'ethnicity', 'cancer_status', 'chronic_pulmonary_disease', 'congestive_heart_failure', 'connective_tissue_rheumatic_disease', 'myocardial_infarction', 'renal_disease', 'institution_type', 'alif_llif', 'corpectomy', 'discectomy', 'foraminotomy', 'instrumentation', 'laminectomy_posterior_decompression', 'pelvic_fixation', 'plf', 'tlif_plif', 'other_lumbar_procedure', 'operative_region_extent', RELATIVE_ODI_MCID_COL]
ORDINAL_FEATURES = ['diabetes_status', 'institution_size', 'asa', 'number_operated_levels']
NOMINAL_FEATURES = ['race', 'institution_region', 'payer_status', 'PatTobaccoUse']

SEARCH_SPACE = {
    'C': [0.001, 0.003, 0.01, 0.03, 0.10, 0.30, 1.00, 3.00, 10.00],
    'tol': [1e-5, 1e-4, 1e-3],
    'positive_weight_multiplier': [0.25, 0.50, 0.75, 1.00, 1.50, 2.00, 3.00, 4.00, 6.00, 8.00],
}
ITERATION_PARAM = None
INT_PARAMS = set()
FLOAT_PARAMS = {'C', 'tol', 'positive_weight_multiplier'}

PARAMETER_CANDIDATES = list(ParameterSampler(SEARCH_SPACE, n_iter=N_RANDOM_COMBINATIONS, random_state=RANDOM_STATE))

MISSING_STRINGS = {'', ' ', 'na', 'n/a', 'nan', 'none', 'null', '.', 'missing', '<na>'}
BINARY_MAPS = {
    'sex': {'female': 0, 'f': 0, 'male': 1, 'm': 1},
    'ethnicity': {'non-hispanic': 0, 'non hispanic': 0, 'hispanic': 1},
    'cancer_status': {'no cancer': 0, 'no': 0, 'none': 0, 'any cancer': 1, 'yes': 1, 'cancer': 1},
    'institution_type': {'hospital': 0, 'non-hospital': 1, 'non hospital': 1, 'nonhospital': 1},
    'operative_region_extent': {'lumbar only': 0, 'extended_region_involvement': 1, 'extended region involvement': 1, 'extended': 1},
}
ORDINAL_MAPS = {
    'diabetes_status': {'no': 0, 'none': 0, '0': 0, 'without comp': 1, 'without complication': 1, 'without complications': 1, '1': 1, 'with comp': 2, 'with complication': 2, 'with complications': 2, '2': 2},
    'institution_size': {'between 1-99 beds': 0, '1-99': 0, 'between 100-399 beds': 1, '100-399': 1, '>= 400 beds': 2, '>=400 beds': 2, '>=400': 2, '>= 400': 2},
    'asa': {'1': 1, 'i': 1, '2': 2, 'ii': 2, '3': 3, 'iii': 3, '4': 4, 'iv': 4, '>=4': 4, '>= 4': 4, '5': 4, 'v': 4},
    'number_operated_levels': {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '>=4': 4, '>= 4': 4, '5': 4, '6': 4, '7': 4, '8': 4, '9': 4, '10': 4},
}
PREFERRED_NOMINAL_LEVELS = {
    'race': ['White', 'Black', 'Other'],
    'institution_region': ['South', 'North East', 'West', 'Midwest'],
    'payer_status': ['Medicare', 'Commercial/Private', 'Other', 'Medicaid/Public/Government'],
    'PatTobaccoUse': ['Unknown/Not reported/Multiple', 'Never', 'Former', 'Current'],
}
FEATURE_LABELS = {
    'age': 'Age', 'bmi': 'BMI', 'sex': 'Male sex', 'race': 'Race', 'ethnicity': 'Hispanic ethnicity',
    'cancer_status': 'Any cancer', 'diabetes_status': 'Diabetes status', 'institution_type': 'Non-hospital institution',
    'institution_size': 'Institution size', 'institution_region': 'Institution region', 'asa': 'ASA physical status',
    'payer_status': 'Primary payer', 'PatTobaccoUse': 'Tobacco use',
    'finaldx_degenerative': 'Degenerative diagnosis', 'finaldx_radicular': 'Radiculopathy diagnosis',
    'finaldx_stenosis': 'Spinal stenosis diagnosis', 'finaldx_deformity_instability': 'Deformity or instability diagnosis',
    'finaldx_other_diagnosis': 'Other lumbar diagnosis', 'chronic_pulmonary_disease': 'Chronic pulmonary disease',
    'congestive_heart_failure': 'Congestive heart failure', 'connective_tissue_rheumatic_disease': 'Connective tissue/rheumatic disease',
    'myocardial_infarction': 'Myocardial infarction', 'renal_disease': 'Renal disease',
    'alif_llif': 'Anterior/lateral lumbar interbody fusion', 'corpectomy': 'Corpectomy', 'discectomy': 'Discectomy',
    'foraminotomy': 'Foraminotomy', 'instrumentation': 'Instrumentation',
    'laminectomy_posterior_decompression': 'Posterior decompression or laminectomy', 'pelvic_fixation': 'Pelvic fixation',
    'plf': 'Posterolateral fusion', 'tlif_plif': 'Posterior/transforaminal lumbar interbody fusion',
    'other_lumbar_procedure': 'Other lumbar procedure', 'number_operated_levels': 'Number of operated levels',
    'operative_region_extent': 'Operative region extent',
}
FEATURE_LABELS.update({'preop_ODI': 'Preoperative ODI', 'postop_ODI': 'Postoperative ODI', 'ODI_change': 'Change in ODI', PROM_CHANGE_RATE_COL: 'ODI change rate', RELATIVE_ODI_MCID_COL: 'Relative ODI MCID', 'postop_ODI_day': 'Timing of postoperative ODI assessment'})

def clean_scalar(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().replace('≥', '>=')
        return np.nan if s.lower() in MISSING_STRINGS else s
    return x

def norm_text(x):
    x = clean_scalar(x)
    if pd.isna(x):
        return None
    return str(x).strip().replace('≥', '>=').lower()

def to_binary_target(x):
    sx = norm_text(x)
    if sx is None:
        return np.nan
    if sx in {'1', '1.0', 'yes', 'y', 'true', 't'}:
        return 1.0
    if sx in {'0', '0.0', 'no', 'n', 'false', 'f'}:
        return 0.0
    try:
        v = float(sx)
        return float(v) if v in (0.0, 1.0) else np.nan
    except Exception:
        return np.nan

def safe_filename(x):
    return re.sub(r'_+', '_', re.sub(r'[^A-Za-z0-9_.-]+', '_', str(x))).strip('_')[:180] or 'file'

def json_native(obj):
    if isinstance(obj, dict):
        return {str(k): json_native(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_native(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj

def sanitize_params(params):
    out = {}
    for k, v in params.items():
        if k in INT_PARAMS:
            out[k] = int(round(float(v)))
        elif k in FLOAT_PARAMS:
            out[k] = float(v)
        else:
            out[k] = v
    return out

def safe_average_precision(y, p):
    return np.nan if len(np.unique(y)) < 2 else float(average_precision_score(y, p))

def safe_roc_auc(y, p):
    return np.nan if len(np.unique(y)) < 2 else float(roc_auc_score(y, p))

def expected_calibration_error(y, p, n_bins=10):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    out = 0.0
    if len(y) == 0:
        return np.nan
    for i in range(n_bins):
        mask = (p >= bins[i]) & ((p <= bins[i + 1]) if i == n_bins - 1 else (p < bins[i + 1]))
        if np.any(mask):
            out += (np.sum(mask) / len(y)) * abs(float(np.mean(y[mask])) - float(np.mean(p[mask])))
    return float(out)

def classification_metrics(y, p, threshold):
    y = np.asarray(y).astype(int)
    pred = (np.asarray(p) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        'threshold': float(threshold),
        'F1': float(f1_score(y, pred, zero_division=0)),
        'Sensitivity': tp / (tp + fn) if tp + fn > 0 else np.nan,
        'Specificity': tn / (tn + fp) if tn + fp > 0 else np.nan,
        'PPV': tp / (tp + fp) if tp + fp > 0 else np.nan,
        'NPV': tn / (tn + fn) if tn + fn > 0 else np.nan,
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
        'Predicted_positive_rate': float(np.mean(pred)),
    }

def top5_metrics(y, p):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    n = len(y)
    k = max(1, int(math.ceil(n * 0.05)))
    idx = np.argsort(-p)[:k]
    prev = float(np.mean(y)) if n else np.nan
    rate = float(np.mean(y[idx])) if n else np.nan
    return {
        'Top_5pct_n': int(k),
        'Top_5pct_event_rate': rate,
        'Top_5pct_lift': rate / prev if prev > 0 else np.nan,
        'Top_5pct_captured_events': float(np.sum(y[idx]) / np.sum(y)) if np.sum(y) > 0 else np.nan,
    }

def select_threshold(y, p):
    precision, recall, thresholds = precision_recall_curve(y, p)
    if len(thresholds) == 0:
        threshold = 0.5
    else:
        precision = precision[:-1]
        recall = recall[:-1]
        f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
        threshold = float(thresholds[int(np.nanargmax(f1))])
    return threshold, classification_metrics(y, p, threshold)

def eval_preds(y, p, threshold=None, prefix=''):
    out = {
        f'{prefix}AP': safe_average_precision(y, p),
        f'{prefix}ROC_AUC': safe_roc_auc(y, p),
        f'{prefix}Brier_score': float(brier_score_loss(y, p)),
        f'{prefix}ECE': expected_calibration_error(y, p, ECE_N_BINS),
        f'{prefix}N': int(len(y)),
        f'{prefix}Events': int(np.sum(y)),
        f'{prefix}Prevalence': float(np.mean(y)),
    }
    if threshold is not None:
        out.update({f'{prefix}{k}': v for k, v in classification_metrics(y, p, threshold).items()})
        out.update({f'{prefix}{k}': v for k, v in top5_metrics(y, p).items()})
    return out

def make_weights(y, multiplier):
    y = np.asarray(y).astype(int)
    n_pos = int(np.sum(y == 1))
    n_neg = int(np.sum(y == 0))
    if n_pos == 0:
        raise ValueError('No positive events in training fold.')
    pos_weight = (n_neg / max(n_pos, 1)) * float(multiplier)
    return np.where(y == 1, pos_weight, 1.0).astype(float)

def actual_positive_weight(y, multiplier):
    y = np.asarray(y).astype(int)
    return float((np.sum(y == 0) / max(np.sum(y == 1), 1)) * multiplier)

@dataclass
class ModelPreprocessor:
    continuous_features: List[str]
    binary_features: List[str]
    ordinal_features: List[str]
    nominal_features: List[str]
    preferred_nominal_levels: Dict[str, List[str]]
    scale_continuous: bool = False

    def __post_init__(self):
        self.continuous_imputer = None
        self.discrete_imputer = None
        self.nominal_imputer = None
        self.scaler = None
        self.onehot = None
        self.output_feature_names_ = []

    def _parse_binary(self, x, feature):
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in BINARY_MAPS and sx in BINARY_MAPS[feature]:
            return float(BINARY_MAPS[feature][sx])
        if sx in {'1', '1.0', 'yes', 'y', 'true', 't', 'present', 'positive'}:
            return 1.0
        if sx in {'0', '0.0', 'no', 'n', 'false', 'f', 'absent', 'negative'}:
            return 0.0
        try:
            v = float(sx)
            return float(v) if v in (0.0, 1.0) else np.nan
        except Exception:
            return np.nan

    def _parse_ordinal(self, x, feature):
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in ORDINAL_MAPS and sx in ORDINAL_MAPS[feature]:
            return float(ORDINAL_MAPS[feature][sx])
        try:
            v = float(sx)
            if feature == 'asa':
                return float(min(max(int(round(v)), 1), 4))
            if feature == 'number_operated_levels':
                return float(min(max(int(round(v)), 0), 4))
            if feature == 'diabetes_status':
                return float(min(max(int(round(v)), 0), 2))
            if feature == 'institution_size':
                return float(min(max(int(round(v)), 0), 2))
            return float(v)
        except Exception:
            return np.nan

    def _nominal(self, feature, x):
        x = clean_scalar(x)
        if pd.isna(x):
            return np.nan
        s = str(x).strip()
        sl = s.lower()
        if feature == 'race':
            return 'White' if sl == 'white' else ('Black' if sl == 'black' else 'Other')
        if feature == 'institution_region':
            mapping = {'south': 'South', 'north east': 'North East', 'northeast': 'North East', 'north-east': 'North East', 'west': 'West', 'midwest': 'Midwest', 'mid west': 'Midwest'}
            return mapping.get(sl, s)
        if feature == 'payer_status':
            if sl == 'medicare':
                return 'Medicare'
            if sl in {'commercial/private', 'commercial', 'private', 'commercial private'}:
                return 'Commercial/Private'
            if sl in {'medicaid/public/government', 'medicaid', 'public', 'government', 'public/government'}:
                return 'Medicaid/Public/Government'
            return 'Other'
        if feature == 'PatTobaccoUse':
            return 'Never' if sl == 'never' else ('Former' if sl == 'former' else ('Current' if sl == 'current' else 'Unknown/Not reported/Multiple'))
        return s

    def _parts(self, X):
        cont = pd.DataFrame(index=X.index)
        disc = pd.DataFrame(index=X.index)
        nom = pd.DataFrame(index=X.index)
        for c in self.continuous_features:
            cont[c] = pd.to_numeric(X[c].map(clean_scalar), errors='coerce')
        for c in self.binary_features:
            disc[c] = X[c].map(lambda z: self._parse_binary(z, c)).astype(float)
        for c in self.ordinal_features:
            disc[c] = X[c].map(lambda z: self._parse_ordinal(z, c)).astype(float)
        for c in self.nominal_features:
            nom[c] = X[c].map(lambda z: self._nominal(c, z)).astype('object')
        return cont, disc, nom

    def fit(self, X):
        cont, disc, nom = self._parts(X)
        self.continuous_imputer = SimpleImputer(strategy='median')
        self.discrete_imputer = SimpleImputer(strategy='most_frequent')
        self.nominal_imputer = SimpleImputer(strategy='constant', fill_value='Missing')
        cont_imp = self.continuous_imputer.fit_transform(cont)
        if self.scale_continuous:
            self.scaler = StandardScaler()
            self.scaler.fit(cont_imp)
        self.discrete_imputer.fit(disc)
        nom_imp = pd.DataFrame(self.nominal_imputer.fit_transform(nom), columns=self.nominal_features)
        cats = []
        for c in self.nominal_features:
            preferred = list(self.preferred_nominal_levels.get(c, []))
            observed = nom_imp[c].astype(str).unique().tolist()
            cats.append(preferred + sorted([x for x in observed if x not in preferred]))
        try:
            self.onehot = OneHotEncoder(categories=cats, handle_unknown='ignore', sparse_output=False)
        except TypeError:
            self.onehot = OneHotEncoder(categories=cats, handle_unknown='ignore', sparse=False)
        self.onehot.fit(nom_imp.astype(str))
        self.output_feature_names_ = self.continuous_features + self.binary_features + self.ordinal_features + self.onehot.get_feature_names_out(self.nominal_features).tolist()
        return self

    def transform(self, X):
        cont, disc, nom = self._parts(X)
        cont_imp = self.continuous_imputer.transform(cont)
        if self.scale_continuous and self.scaler is not None:
            cont_imp = self.scaler.transform(cont_imp)
        disc_imp = self.discrete_imputer.transform(disc)
        nom_imp = pd.DataFrame(self.nominal_imputer.transform(nom), columns=self.nominal_features)
        nom_oh = self.onehot.transform(nom_imp.astype(str))
        return np.concatenate([cont_imp, disc_imp, nom_oh], axis=1).astype(float)

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

    @property
    def output_feature_names(self):
        return self.output_feature_names_

def raw_model_scores(model, Xp, n_iter=None):
    if CLASSIFIER_KEY == 'HistGradientBoosting' and n_iter is not None:
        selected = None
        for i, pred in enumerate(model.staged_predict_proba(Xp), start=1):
            selected = pred
            if i >= int(n_iter):
                break
        if selected is not None:
            return selected[:, 1]
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(Xp)[:, 1]
    if hasattr(model, 'decision_function'):
        return model.decision_function(Xp)
    raise AttributeError('Model provides neither predict_proba nor decision_function.')

def predict(pre, model, X, features, n_iter=None):
    return raw_model_scores(model, pre.transform(X[features]), n_iter=n_iter)

def patient_split(df, target_col, seed):
    group_df = df.groupby(GROUP_COL, dropna=False)[target_col].max().reset_index()
    y_group = group_df[target_col].astype(int).to_numpy()
    groups = group_df[GROUP_COL].to_numpy()
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=seed)
    train_cal_idx, test_idx = next(sss1.split(groups, y_group))
    groups_train_cal = groups[train_cal_idx]
    y_train_cal = y_group[train_cal_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=CALIBRATION_FRACTION_OF_REMAINING, random_state=seed + 1)
    train_idx_rel, cal_idx_rel = next(sss2.split(groups_train_cal, y_train_cal))
    train_groups = set(groups_train_cal[train_idx_rel])
    cal_groups = set(groups_train_cal[cal_idx_rel])
    test_groups = set(groups[test_idx])
    assert train_groups.isdisjoint(cal_groups) and train_groups.isdisjoint(test_groups) and cal_groups.isdisjoint(test_groups)
    return df[GROUP_COL].isin(train_groups).to_numpy(), df[GROUP_COL].isin(cal_groups).to_numpy(), df[GROUP_COL].isin(test_groups).to_numpy()

def cv_splits(y, groups, seed, n_folds=N_CV_FOLDS):
    group_df = pd.DataFrame({'group': groups, 'y': y}).groupby('group')['y'].max().reset_index()
    n_folds = min(n_folds, int(np.sum(group_df.y == 1)), int(np.sum(group_df.y == 0)))
    if n_folds < 2:
        raise ValueError('Not enough positive and negative patient groups for group-aware CV.')
    cv = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    return [(tr, va) for tr, va in cv.split(np.zeros(len(y)), y, groups)]

def feature_types(features):
    cont = [f for f in features if f in CONTINUOUS_FEATURES_ALL or (f not in BINARY_FEATURES and f not in ORDINAL_FEATURES and f not in NOMINAL_FEATURES)]
    binary = [f for f in features if f in BINARY_FEATURES]
    ordinal = [f for f in features if f in ORDINAL_FEATURES]
    nominal = [f for f in features if f in NOMINAL_FEATURES]
    return cont, binary, ordinal, nominal

def fit_pipeline(X, y, features, params, seed, eval_set=None, early=False, n_estimators_override=None):
    cont, binary, ordinal, nominal = feature_types(features)
    pre = ModelPreprocessor(cont, binary, ordinal, nominal, PREFERRED_NOMINAL_LEVELS, scale_continuous=SCALE_CONTINUOUS)
    Xp = pre.fit_transform(X[features])
    eval_transformed = None
    if eval_set is not None:
        X_val, y_val = eval_set
        eval_transformed = (pre.transform(X_val[features]), y_val)
    model, best_iter = fit_estimator(Xp, y, params, seed, eval_set=eval_transformed, use_early_stopping=early, n_estimators_override=n_estimators_override)
    return pre, model, best_iter

def tune(X_train, y_train, groups_train, features, folds, model_key, seed):
    candidate_rows = []
    fold_rows = []
    sampler = list(ParameterSampler(SEARCH_SPACE, n_iter=N_RANDOM_COMBINATIONS, random_state=seed))
    for cid, raw_params in enumerate(sampler, start=1):
        params = sanitize_params(raw_params)
        fold_train_aps, fold_train_aucs, fold_aps, fold_aucs, best_iters = [], [], [], [], []
        t0 = time.time()
        for fid, (tr_idx, va_idx) in enumerate(folds, start=1):
            X_tr = X_train.iloc[tr_idx].reset_index(drop=True)
            y_tr = y_train[tr_idx]
            X_va = X_train.iloc[va_idx].reset_index(drop=True)
            y_va = y_train[va_idx]
            pre, model, best_iter = fit_pipeline(X_tr, y_tr, features, params, seed + cid * 1000 + fid, eval_set=(X_va, y_va), early=USE_EARLY_STOPPING_IN_CV)
            p_tr = predict(pre, model, X_tr, features, n_iter=best_iter)
            p_va = predict(pre, model, X_va, features, n_iter=best_iter)
            tr_ap, tr_auc = safe_average_precision(y_tr, p_tr), safe_roc_auc(y_tr, p_tr)
            va_ap, va_auc = safe_average_precision(y_va, p_va), safe_roc_auc(y_va, p_va)
            fold_train_aps.append(tr_ap); fold_train_aucs.append(tr_auc); fold_aps.append(va_ap); fold_aucs.append(va_auc)
            if best_iter is not None and best_iter > 0:
                best_iters.append(best_iter)
            fold_rows.append({
                'model_key': model_key, 'candidate_id': cid, 'fold': fid,
                'fold_train_AP': tr_ap, 'fold_train_ROC_AUC': tr_auc,
                'fold_validation_AP': va_ap, 'fold_validation_ROC_AUC': va_auc,
                'fold_train_minus_validation_AP_gap': tr_ap - va_ap if np.isfinite(tr_ap) and np.isfinite(va_ap) else np.nan,
                'fold_train_minus_validation_ROC_AUC_gap': tr_auc - va_auc if np.isfinite(tr_auc) and np.isfinite(va_auc) else np.nan,
                'fold_best_iteration': best_iter,
                'positive_weight_used': actual_positive_weight(y_tr, params['positive_weight_multiplier']),
                **params,
            })
        locked = int(np.median(best_iters)) if best_iters and USE_EARLY_STOPPING_IN_CV else (int(params[ITERATION_PARAM]) if ITERATION_PARAM and ITERATION_PARAM in params else np.nan)
        row = {
            'model_key': model_key, 'candidate_id': cid, 'cv_folds': len(folds),
            'cv_train_AP_mean': float(np.nanmean(fold_train_aps)),
            'cv_train_ROC_AUC_mean': float(np.nanmean(fold_train_aucs)),
            'cv_AP_mean': float(np.nanmean(fold_aps)),
            'cv_AP_SD': float(np.nanstd(fold_aps, ddof=1)) if len(fold_aps) > 1 else np.nan,
            'cv_ROC_AUC_mean': float(np.nanmean(fold_aucs)),
            'cv_ROC_AUC_SD': float(np.nanstd(fold_aucs, ddof=1)) if len(fold_aucs) > 1 else np.nan,
            'mean_cv_best_iteration': float(np.mean(best_iters)) if best_iters else np.nan,
            'locked_n_estimators_from_cv': locked,
            'elapsed_seconds': float(time.time() - t0),
            **params,
        }
        candidate_rows.append(row)
        print(f"{model_key} | candidate {cid:03d}/{len(sampler)} | CV AP={row['cv_AP_mean']:.5f} | AUC={row['cv_ROC_AUC_mean']:.5f}")
    return pd.DataFrame(candidate_rows).sort_values('cv_AP_mean', ascending=False).reset_index(drop=True), pd.DataFrame(fold_rows)

def fit_final(X_train, y_train, X_cal, y_cal, X_test, features, params, locked_n, seed):
    n_override = int(locked_n) if pd.notna(locked_n) and USE_EARLY_STOPPING_IN_CV and ITERATION_PARAM else None
    pre, model, _ = fit_pipeline(X_train, y_train, features, params, seed, n_estimators_override=n_override)
    p_train_raw = predict(pre, model, X_train, features)
    p_cal_raw = predict(pre, model, X_cal, features)
    p_test_raw = predict(pre, model, X_test, features)
    calibrator = IsotonicRegression(out_of_bounds='clip')
    calibrator.fit(p_cal_raw, y_cal)
    p_train = calibrator.predict(p_train_raw)
    p_cal = calibrator.predict(p_cal_raw)
    p_test = calibrator.predict(p_test_raw)
    threshold, cal_metrics = select_threshold(y_cal, p_cal)
    return {
        'pre': pre, 'model': model, 'calibrator': calibrator,
        'p_train_raw': p_train_raw, 'p_cal_raw': p_cal_raw, 'p_test_raw': p_test_raw,
        'p_train': p_train, 'p_cal': p_cal, 'p_test': p_test,
        'threshold': threshold, 'cal_metrics': cal_metrics,
    }

def patient_boot_ci(y, p, groups, metric, threshold=None, seed=1):
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({'idx': np.arange(len(y)), 'group': groups, 'y': np.asarray(y).astype(int)})
    group_y = d.groupby('group').y.max()
    pos = group_y[group_y == 1].index.to_numpy()
    neg = group_y[group_y == 0].index.to_numpy()
    if len(pos) == 0 or len(neg) == 0:
        return np.nan, np.nan
    by_group = {g: d.loc[d.group.eq(g), 'idx'].to_numpy() for g in group_y.index}
    vals = []
    for _ in range(N_BOOTSTRAPS):
        sample_groups = np.concatenate([rng.choice(pos, len(pos), replace=True), rng.choice(neg, len(neg), replace=True)])
        sample_idx = np.concatenate([by_group[g] for g in sample_groups])
        yy = np.asarray(y)[sample_idx]
        pp = np.asarray(p)[sample_idx]
        if len(np.unique(yy)) < 2:
            continue
        if metric == 'AP':
            vals.append(average_precision_score(yy, pp))
        elif metric == 'ROC_AUC':
            vals.append(roc_auc_score(yy, pp))
        elif metric == 'F1':
            vals.append(f1_score(yy, pp >= threshold, zero_division=0))
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))) if vals else (np.nan, np.nan)

def paired_delta_boot(y, p0, p1, groups, metric, seed=1):
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({'idx': np.arange(len(y)), 'group': groups, 'y': np.asarray(y).astype(int)})
    group_y = d.groupby('group').y.max()
    pos = group_y[group_y == 1].index.to_numpy()
    neg = group_y[group_y == 0].index.to_numpy()
    by_group = {g: d.loc[d.group.eq(g), 'idx'].to_numpy() for g in group_y.index}
    observed = (safe_average_precision(y, p1) - safe_average_precision(y, p0)) if metric == 'AP' else (safe_roc_auc(y, p1) - safe_roc_auc(y, p0))
    vals = []
    for _ in range(N_BOOTSTRAPS):
        sample_groups = np.concatenate([rng.choice(pos, len(pos), replace=True), rng.choice(neg, len(neg), replace=True)])
        sample_idx = np.concatenate([by_group[g] for g in sample_groups])
        yy = np.asarray(y)[sample_idx]
        if len(np.unique(yy)) < 2:
            continue
        if metric == 'AP':
            vals.append(average_precision_score(yy, np.asarray(p1)[sample_idx]) - average_precision_score(yy, np.asarray(p0)[sample_idx]))
        else:
            vals.append(roc_auc_score(yy, np.asarray(p1)[sample_idx]) - roc_auc_score(yy, np.asarray(p0)[sample_idx]))
    if not vals:
        return observed, np.nan, np.nan, np.nan
    vals = np.asarray(vals)
    ci_low, ci_high = float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))
    p_value = float(2 * min(np.mean(vals <= 0), np.mean(vals >= 0)))
    return float(observed), ci_low, ci_high, min(p_value, 1.0)

def save_learning_curve(X_train, y_train, groups_train, features, params, locked_n, seed, out_path, title):
    group_df = pd.DataFrame({'group': groups_train, 'y': y_train}).groupby('group').y.max().reset_index()
    rng = np.random.default_rng(seed)
    rows = []
    for frac in LEARNING_CURVE_TRAIN_FRACTIONS:
        for rep in range(LEARNING_CURVE_CV_FOLDS):
            selected_groups = []
            for cls in [0, 1]:
                cls_groups = group_df.loc[group_df.y.eq(cls), 'group'].to_numpy()
                k = max(1, int(math.ceil(len(cls_groups) * frac)))
                selected_groups.extend(rng.choice(cls_groups, k, replace=False).tolist())
            mask = np.isin(groups_train, selected_groups)
            X_sub = X_train.loc[mask].reset_index(drop=True)
            y_sub = y_train[mask]
            g_sub = groups_train[mask]
            try:
                folds = cv_splits(y_sub, g_sub, seed + rep, n_folds=3)
            except Exception:
                continue
            for tr, va in folds:
                n_override = int(locked_n) if pd.notna(locked_n) and USE_EARLY_STOPPING_IN_CV and ITERATION_PARAM else None
                pre, model, _ = fit_pipeline(X_sub.iloc[tr].reset_index(drop=True), y_sub[tr], features, params, seed + rep, n_estimators_override=n_override)
                p_tr = predict(pre, model, X_sub.iloc[tr].reset_index(drop=True), features)
                p_va = predict(pre, model, X_sub.iloc[va].reset_index(drop=True), features)
                rows.append({'train_fraction': frac, 'train_AP': safe_average_precision(y_sub[tr], p_tr), 'validation_AP': safe_average_precision(y_sub[va], p_va)})
    lc = pd.DataFrame(rows)
    lc.to_csv(out_path.replace('.png', '.csv'), index=False)
    if not lc.empty:
        s = lc.groupby('train_fraction').agg(train_AP_mean=('train_AP', 'mean'), validation_AP_mean=('validation_AP', 'mean')).reset_index()
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.plot(s.train_fraction, s.train_AP_mean, marker='o', label='Training AP')
        ax.plot(s.train_fraction, s.validation_AP_mean, marker='o', label='Validation AP')
        ax.set_xlabel('Fraction of training patient groups')
        ax.set_ylabel('Average precision')
        ax.set_title(title)
        ax.legend()
        fig.tight_layout()
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close(fig)

def style_excel(writer):
    try:
        from openpyxl.styles import Font, PatternFill, Alignment
        for sheet in writer.sheets:
            ws = writer.sheets[sheet]
            ws.freeze_panes = 'A2'
            ws.auto_filter.ref = ws.dimensions
            for cell in ws[1]:
                cell.font = Font(bold=True)
                cell.fill = PatternFill(start_color='D9EAF7', end_color='D9EAF7', fill_type='solid')
                cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            for col in ws.columns:
                max_len = max([len(str(c.value)) for c in col if c.value is not None] + [12])
                ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 12), 70)
    except Exception:
        pass


def make_model(params, seed, n_estimators_override=None):
    p = sanitize_params(params)
    p.pop('positive_weight_multiplier', None)
    return LogisticRegression(
        penalty='l2',
        solver='lbfgs',
        max_iter=5000,
        random_state=seed,
        **p,
    )

def fit_estimator(X_train, y_train, params, seed, eval_set=None, use_early_stopping=False, n_estimators_override=None):
    weights = make_weights(y_train, float(params['positive_weight_multiplier']))
    model = make_model(params, seed)
    model.fit(X_train, y_train, sample_weight=weights)
    return model, None


def add_dynamic_odi_features(df):
    out = df.copy()
    required = ['preop_ODI', 'postop_ODI', DAYS_BETWEEN_PROM_COL]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError('Cannot derive dynamic ODI features because required columns are missing: ' + ', '.join(missing))
    preop = pd.to_numeric(out['preop_ODI'].map(clean_scalar), errors='coerce')
    postop = pd.to_numeric(out['postop_ODI'].map(clean_scalar), errors='coerce')
    days_between = pd.to_numeric(out[DAYS_BETWEEN_PROM_COL].map(clean_scalar), errors='coerce')
    odi_change = postop - preop
    valid_rate = preop.notna() & postop.notna() & days_between.gt(0)
    change_rate = pd.Series(np.nan, index=out.index, dtype='float')
    change_rate.loc[valid_rate] = (odi_change.loc[valid_rate] / days_between.loc[valid_rate]).astype(float)
    improvement = preop - postop
    rel_fraction = improvement / preop.replace(0, np.nan)
    valid_relative = preop.notna() & postop.notna() & preop.gt(0)
    zero_baseline_with_postop = preop.eq(0) & postop.notna()
    relative_mcid = pd.Series(np.nan, index=out.index, dtype='float')
    relative_mcid.loc[valid_relative] = (rel_fraction.loc[valid_relative] >= RELATIVE_ODI_MCID_CUTOFF).astype(float)
    relative_mcid.loc[zero_baseline_with_postop] = 0.0
    out['ODI_change'] = odi_change
    out[PROM_CHANGE_RATE_COL] = change_rate
    out[RELATIVE_ODI_MCID_COL] = relative_mcid
    audit = pd.DataFrame([
        {'item': 'PROM change rate definition', 'value': '(postop_ODI - preop_ODI) / days_between_PROMs'},
        {'item': 'Relative ODI MCID definition', 'value': f'(preop_ODI - postop_ODI) / preop_ODI >= {RELATIVE_ODI_MCID_CUTOFF}; preop_ODI=0 coded as 0 when postop_ODI is available'},
        {'item': 'Rows', 'value': int(len(out))},
        {'item': 'Rows with valid preop/postop ODI and positive days between PROMs', 'value': int(valid_rate.sum())},
        {'item': 'Rows with preop ODI = 0 and postop ODI available coded as Relative ODI MCID = 0', 'value': int(zero_baseline_with_postop.sum())},
    ])
    return out, audit

def run_model(model_type, work, features, folds, seed, all_candidates, all_folds):
    key = model_type
    out_dir = os.path.join(OUTPUT_DIR, model_type)
    plot_dir = os.path.join(PLOT_DIR, model_type)
    artifact_dir = os.path.join(ARTIFACT_DIR, model_type)
    for d in [out_dir, plot_dir, artifact_dir]:
        os.makedirs(d, exist_ok=True)
    train = work[work.split.eq('train')].reset_index(drop=True)
    cal = work[work.split.eq('calibration')].reset_index(drop=True)
    test = work[work.split.eq('test')].reset_index(drop=True)
    y_train = train[TARGET_COL].astype(int).to_numpy()
    y_cal = cal[TARGET_COL].astype(int).to_numpy()
    y_test = test[TARGET_COL].astype(int).to_numpy()
    candidates, fold_metrics = tune(train[features], y_train, train[GROUP_COL].to_numpy(), features, folds, key, seed)
    best_params = sanitize_params({k: candidates.loc[0, k] for k in SEARCH_SPACE.keys()})
    locked_n = candidates.loc[0, 'locked_n_estimators_from_cv']
    final = fit_final(train[features], y_train, cal[features], y_cal, test[features], y_test, features, best_params, locked_n, seed + 991)
    threshold = final['threshold']
    summary = {
        'model_type': model_type, 'classifier': CLASSIFIER_DISPLAY_NAME,
        'n_features_original': len(features), 'best_candidate_id': int(candidates.loc[0, 'candidate_id']),
        'CV_AP_mean': float(candidates.loc[0, 'cv_AP_mean']), 'CV_ROC_AUC_mean': float(candidates.loc[0, 'cv_ROC_AUC_mean']),
        'locked_n_estimators_from_cv': locked_n,
        **eval_preds(y_train, final['p_train'], threshold, prefix='Train_'),
        **eval_preds(y_cal, final['p_cal'], threshold, prefix='Calibration_'),
        **eval_preds(y_test, final['p_test'], threshold, prefix='Test_'),
    }
    summary['Train_minus_CV_AP_gap'] = summary['Train_AP'] - summary['CV_AP_mean'] if pd.notna(summary['CV_AP_mean']) else np.nan
    summary['CV_minus_Test_AP_gap'] = summary['CV_AP_mean'] - summary['Test_AP'] if pd.notna(summary['CV_AP_mean']) else np.nan
    summary['Test_AP_CI_lower'], summary['Test_AP_CI_upper'] = patient_boot_ci(y_test, final['p_test'], test[GROUP_COL].to_numpy(), 'AP', seed=seed + 10)
    summary['Test_ROC_AUC_CI_lower'], summary['Test_ROC_AUC_CI_upper'] = patient_boot_ci(y_test, final['p_test'], test[GROUP_COL].to_numpy(), 'ROC_AUC', seed=seed + 20)
    pd.DataFrame([summary]).to_csv(os.path.join(out_dir, f'performance_{{key}}.csv'), index=False)
    pd.DataFrame({'row_index_in_split': np.arange(len(test)), GROUP_COL: test[GROUP_COL].to_numpy(), 'y_true': y_test, 'p_raw': final['p_test_raw'], 'p_calibrated': final['p_test']}).to_csv(os.path.join(out_dir, f'test_predictions_{{key}}.csv'), index=False)
    candidates.to_csv(os.path.join(out_dir, f'cv_candidates_{{key}}.csv'), index=False)
    fold_metrics.to_csv(os.path.join(out_dir, f'cv_fold_metrics_{{key}}.csv'), index=False)
    joblib.dump(final, os.path.join(artifact_dir, f'final_model_{{key}}.joblib'))
    save_learning_curve(train[features], y_train, train[GROUP_COL].to_numpy(), features, best_params, locked_n, seed + 313, os.path.join(plot_dir, f'learning_curve_{{key}}.png'), key)
    all_candidates.append(candidates)
    all_folds.append(fold_metrics)
    return {'summary': summary, 'preds': pd.read_csv(os.path.join(out_dir, f'test_predictions_{{key}}.csv'))}

def split_audit(df):
    rows = []
    for split in ['train', 'calibration', 'test']:
        sub = df[df.split.eq(split)]
        rows.append({'split': split, 'n_rows': len(sub), 'n_patients': sub[GROUP_COL].nunique(), 'events': int(sub[TARGET_COL].sum()), 'prevalence': float(sub[TARGET_COL].mean())})
    return pd.DataFrame(rows)

def methods_table():
    return pd.DataFrame([
        {'item': 'Paired design', 'rationale': f'Baseline-only and dynamic PROM-expanded {{CLASSIFIER_DISPLAY_NAME}} models are tuned separately under the same protocol.'},
        {'item': 'Paired split/folds', 'rationale': 'Paired models use identical patient-level train/calibration/test splits and identical group-aware CV folds.'},
        {'item': 'Tuning metric', 'rationale': 'Average precision is the primary model-selection metric because delayed reoperation is a rare-event outcome.'},
        {'item': 'Class imbalance', 'rationale': 'Positive-class sample weighting is tuned through a multiplier applied to the natural negative-to-positive ratio.'},
        {'item': 'Calibration and threshold', 'rationale': 'Isotonic calibration and maximum-F1 threshold selection are performed only on the calibration split.'},
        {'item': 'Held-out test set', 'rationale': 'The test set is isolated until the model-development pipeline is locked.'},
    ])

def main():
    t0 = time.time()
    df = pd.read_csv(INPUT_CSV)
    df, dynamic_audit = add_dynamic_odi_features(df)
    required = sorted(set(DYNAMIC_PROM_FEATURES + [TARGET_COL, GROUP_COL, DAYS_BETWEEN_PROM_COL]))
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise ValueError('Missing required columns:\n' + '\n'.join(f' - {{c}}' for c in missing_cols))
    preop = pd.to_numeric(df['preop_ODI'].map(clean_scalar), errors='coerce')
    postop = pd.to_numeric(df['postop_ODI'].map(clean_scalar), errors='coerce')
    days = pd.to_numeric(df[DAYS_BETWEEN_PROM_COL].map(clean_scalar), errors='coerce')
    eligible = preop.notna() & postop.notna() & days.gt(0)
    work = df.loc[eligible, DYNAMIC_PROM_FEATURES + [TARGET_COL, GROUP_COL]].copy()
    work[TARGET_COL] = work[TARGET_COL].map(to_binary_target)
    work = work.dropna(subset=[TARGET_COL]).copy()
    work[TARGET_COL] = work[TARGET_COL].astype(int)
    train_mask, cal_mask, test_mask = patient_split(work, TARGET_COL, RANDOM_STATE)
    work['split'] = np.where(train_mask, 'train', np.where(cal_mask, 'calibration', 'test'))
    train = work[work.split.eq('train')].reset_index(drop=True)
    folds = cv_splits(train[TARGET_COL].astype(int).to_numpy(), train[GROUP_COL].to_numpy(), RANDOM_STATE + 10)
    all_candidates, all_folds = [], []
    baseline = run_model('baseline_only', work, BASELINE_FEATURES, folds, RANDOM_STATE + 100, all_candidates, all_folds)
    expanded = run_model('dynamic_PROM_expanded', work, DYNAMIC_PROM_FEATURES, folds, RANDOM_STATE + 200, all_candidates, all_folds)
    merged = baseline['preds'].merge(expanded['preds'], on=['row_index_in_split', GROUP_COL, 'y_true'], suffixes=('_baseline', '_expanded'), validate='one_to_one')
    y = merged.y_true.astype(int).to_numpy()
    p0 = merged.p_calibrated_baseline.astype(float).to_numpy()
    p1 = merged.p_calibrated_expanded.astype(float).to_numpy()
    groups = merged[GROUP_COL].to_numpy()
    dap = paired_delta_boot(y, p0, p1, groups, 'AP', RANDOM_STATE + 1000)
    dauc = paired_delta_boot(y, p0, p1, groups, 'ROC_AUC', RANDOM_STATE + 2000)
    comparison = pd.DataFrame([{ 'baseline_Test_AP': baseline['summary']['Test_AP'], 'expanded_Test_AP': expanded['summary']['Test_AP'], 'Delta_AP_expanded_minus_baseline': dap[0], 'Delta_AP_CI_lower': dap[1], 'Delta_AP_CI_upper': dap[2], 'Delta_AP_bootstrap_p_value': dap[3], 'baseline_Test_ROC_AUC': baseline['summary']['Test_ROC_AUC'], 'expanded_Test_ROC_AUC': expanded['summary']['Test_ROC_AUC'], 'Delta_ROC_AUC_expanded_minus_baseline': dauc[0], 'Delta_ROC_AUC_CI_lower': dauc[1], 'Delta_ROC_AUC_CI_upper': dauc[2], 'Delta_ROC_AUC_bootstrap_p_value': dauc[3]}])
    summary = pd.DataFrame([baseline['summary'], expanded['summary']])
    candidates = pd.concat(all_candidates, ignore_index=True)
    folds_df = pd.concat(all_folds, ignore_index=True)
    missingness = pd.DataFrame([{'column': c, 'n_missing_or_blank': int(work[c].isna().sum()), 'percent_missing_or_blank': 100 * float(work[c].isna().sum()) / len(work)} for c in DYNAMIC_PROM_FEATURES if c in work.columns])
    xlsx = os.path.join(OUTPUT_DIR, f'Step2_DynamicPROM_{{CLASSIFIER_KEY}}_summary.xlsx')
    with pd.ExcelWriter(xlsx, engine='openpyxl') as writer:
        methods_table().to_excel(writer, 'methods_rationale', index=False)
        dynamic_audit.to_excel(writer, 'dynamic_feature_audit', index=False)
        summary.to_excel(writer, 'model_performance', index=False)
        comparison.to_excel(writer, 'paired_PROM_comparison', index=False)
        candidates.to_excel(writer, 'cv_candidates_all_models', index=False)
        folds_df.to_excel(writer, 'cv_fold_metrics_all', index=False)
        split_audit(work).to_excel(writer, 'split_audit', index=False)
        missingness.to_excel(writer, 'missingness_audit', index=False)
        style_excel(writer)
    summary.to_csv(os.path.join(OUTPUT_DIR, 'model_performance.csv'), index=False)
    comparison.to_csv(os.path.join(OUTPUT_DIR, 'paired_PROM_comparison.csv'), index=False)
    candidates.to_csv(os.path.join(OUTPUT_DIR, 'cv_candidates_all_models.csv'), index=False)
    folds_df.to_csv(os.path.join(OUTPUT_DIR, 'cv_fold_metrics_all.csv'), index=False)
    work[[GROUP_COL, 'split']].drop_duplicates().to_csv(os.path.join(OUTPUT_DIR, 'split_assignment.csv'), index=False)
    manifest = {'classifier': CLASSIFIER_DISPLAY_NAME, 'output_dir': OUTPUT_DIR, 'design': f'paired baseline-only and dynamic PROM-expanded {{CLASSIFIER_DISPLAY_NAME}} models', 'model_selection': 'highest mean group-aware CV average precision inside training split', 'calibration': 'isotonic', 'threshold_strategy': 'maximum F1 on calibration split', 'runtime_minutes': float((time.time() - t0) / 60), 'summary_xlsx': xlsx}
    json.dump(json_native(manifest), open(os.path.join(OUTPUT_DIR, 'run_manifest.json'), 'w'), indent=2, sort_keys=True)
    zip_path = os.path.join(OUTPUT_DIR, f'Step2_DynamicPROM_{{CLASSIFIER_KEY}}_outputs.zip')
    tmp = zip_path + '.tmp'
    for p in [zip_path, tmp]:
        if os.path.exists(p):
            os.remove(p)
    with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=ZIP_COMPRESSION_LEVEL) as z:
        for root, _, files in os.walk(OUTPUT_DIR):
            for name in files:
                full = os.path.join(root, name)
                if full in {zip_path, tmp}:
                    continue
                z.write(full, os.path.relpath(full, OUTPUT_DIR))
    os.replace(tmp, zip_path)
    print('=' * 100)
    print(f'Step 2 {{CLASSIFIER_DISPLAY_NAME}} analysis completed')
    print('Summary Excel:', xlsx)
    print('ZIP archive:', zip_path)
    print('=' * 100)
    if CREATE_COLAB_DOWNLOAD_LINK:
        try:
            from IPython.display import HTML, display
            display(HTML(f'<p><b>Step 2 {{CLASSIFIER_DISPLAY_NAME}} output archive is ready.</b></p><p><a href="/files{{zip_path}}" download>Click here to download the ZIP archive</a></p><p>Path: <code>{{zip_path}}</code></p>'))
        except Exception:
            pass
    if AUTO_DOWNLOAD_ZIP:
        try:
            from google.colab import files
            files.download(zip_path)
        except Exception:
            pass

if __name__ == '__main__':
    main()


#**Step 2: Final Tuned Model**

In [ ]:
# -*- coding: utf-8 -*-
"""

Step 2 LightGBM final locked PROM SHAP suite
============================================

This script reports only the final locked PROM-expanded LightGBM model for each Step 2 dynamic PROM-expanded cohort. It does not tune, refit, compare, or report alternative
settings.

"""

# ============================================================
# 0) Imports
# ============================================================

import os
import re
import sys
import json
import zipfile
import shutil
import subprocess
import warnings
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    import lightgbm as lgb  # noqa: F401
    from lightgbm import LGBMClassifier
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
    import lightgbm as lgb  # noqa: F401
    from lightgbm import LGBMClassifier

try:
    import shap
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
    import shap

try:
    import joblib
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "joblib"])
    import joblib

try:
    import openpyxl  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl"])
    import openpyxl  # noqa: F401

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import average_precision_score, roc_auc_score
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ============================================================
# 1) Configuration
# ============================================================

BASE_DIR = "/content"
FALLBACK_DIR = "/mnt/data"
INPUT_CSV = os.path.join(BASE_DIR, "Step 2_ODI_Cohort.csv")
FALLBACK_INPUT_CSV = os.path.join(FALLBACK_DIR, "Step 2_ODI_Cohort.csv")

SOURCE_RUN_BASENAME = "Step2_DynamicPROM_LightGBM_Nature_Paired_FinalConfigs_Final"
SOURCE_DIR = os.path.join(BASE_DIR, SOURCE_RUN_BASENAME)
SOURCE_ZIP = os.path.join(BASE_DIR, SOURCE_RUN_BASENAME + ".zip")
FALLBACK_SOURCE_ZIP = os.path.join(FALLBACK_DIR, SOURCE_RUN_BASENAME + ".zip")

OUTPUT_DIR = os.path.join(BASE_DIR, "Step2_LightGBM_Final_DynamicPROM_SHAP_Suite_NaturalThreshold_Top25_AllTestHeatmap_Final")
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
for d in [OUTPUT_DIR, PLOT_DIR, TABLE_DIR]:
    os.makedirs(d, exist_ok=True)

TARGET_COL = "final_reop_step2"
GROUP_COL = "PersonKey"
RANDOM_STATE = 20260530

RELATIVE_PROM_MCID_CUTOFF = 0.30
LEGACY_ODI_MCID_COL = "ODI_MCID_binary"
PROM_CHANGE_RATE_COL = "ODI_change_rate"
RELATIVE_ODI_MCID_COL = "ODI_relative_MCID_binary"
DAYS_BETWEEN_PROM_COL = "days_between_PROMs"

SHAP_MAX_EXPLAIN_ROWS = None  # use all held-out test patients
SHAP_INTERACTION_MAX_ROWS = None  # use all held-out test patients for interaction plots
SHAP_BEESWARM_MAX_DISPLAY = 25
SHAP_BAR_MAX_DISPLAY = 25
SHAP_HEATMAP_MAX_DISPLAY_FEATURES = 25
SHAP_HEATMAP_MAX_PATIENTS = None  # show all held-out test patients on the heatmap x-axis
SHAP_THRESHOLD_TOP_N_NUMERIC = 20
SHAP_THRESHOLD_MAX_BINS = 35
ZIP_COMPRESSION_LEVEL = 1
AUTO_DOWNLOAD_ZIP = False
CREATE_COLAB_DOWNLOAD_LINK = True

# Beeswarm, bar chart, and heatmap are limited to the top 25 SHAP-final_indexed features.
# Heatmap columns include all held-out test patients, sorted by calibrated predicted risk.
# Matched to standard SHAP beeswarm low/high palette.
SHAP_BEESWARM_BLUE = "#008BFB"
SHAP_BEESWARM_PINK = "#FF0051"
SHAP_COLOR_CMAP = LinearSegmentedColormap.from_list("shap_blue_pink", [SHAP_BEESWARM_BLUE, SHAP_BEESWARM_PINK])
SHAP_DIVERGING_CMAP = LinearSegmentedColormap.from_list("shap_blue_white_pink", [SHAP_BEESWARM_BLUE, "#FFFFFF", SHAP_BEESWARM_PINK])

# ============================================================
# 2) Feature definitions — must match the original Step 2 script
# ============================================================

BASE_FEATURES = [
    "finaldx_degenerative", "finaldx_radicular", "finaldx_stenosis",
    "finaldx_deformity_instability", "finaldx_other_diagnosis",
    "age", "sex", "race", "ethnicity", "cancer_status",
    "chronic_pulmonary_disease", "congestive_heart_failure",
    "connective_tissue_rheumatic_disease", "diabetes_status",
    "myocardial_infarction", "renal_disease", "institution_type",
    "institution_size", "institution_region", "asa", "bmi", "payer_status",
    "alif_llif", "corpectomy", "discectomy", "foraminotomy",
    "instrumentation", "laminectomy_posterior_decompression",
    "pelvic_fixation", "plf", "tlif_plif", "other_lumbar_procedure",
    "number_operated_levels", "operative_region_extent", "PatTobaccoUse",
]

STEP2_DYNAMIC_PROM_FEATURES = [
    "preop_ODI",
    "postop_ODI",
    "ODI_change",
    PROM_CHANGE_RATE_COL,
    RELATIVE_ODI_MCID_COL,
    "postop_ODI_day",
]
FEATURES = BASE_FEATURES + STEP2_DYNAMIC_PROM_FEATURES

CONTINUOUS_FEATURES = [
    "age", "bmi", "preop_ODI", "postop_ODI", "ODI_change", PROM_CHANGE_RATE_COL, "postop_ODI_day",
]
BINARY_FEATURES = [
    "finaldx_degenerative", "finaldx_radicular", "finaldx_stenosis",
    "finaldx_deformity_instability", "finaldx_other_diagnosis", "sex",
    "ethnicity", "cancer_status", "chronic_pulmonary_disease",
    "congestive_heart_failure", "connective_tissue_rheumatic_disease",
    "myocardial_infarction", "renal_disease", "institution_type",
    "alif_llif", "corpectomy", "discectomy", "foraminotomy", "instrumentation",
    "laminectomy_posterior_decompression", "pelvic_fixation", "plf", "tlif_plif",
    "other_lumbar_procedure", "operative_region_extent", RELATIVE_ODI_MCID_COL,
]
ORDINAL_FEATURES = ["diabetes_status", "institution_size", "asa", "number_operated_levels"]
NOMINAL_FEATURES = ["race", "institution_region", "payer_status", "PatTobaccoUse"]

MISSING_STRINGS = {"", " ", "na", "n/a", "nan", "none", "null", ".", "missing", "<na>"}
BINARY_MAPS = {
    "sex": {"female": 0, "f": 0, "male": 1, "m": 1},
    "ethnicity": {"non-hispanic": 0, "non hispanic": 0, "hispanic": 1},
    "cancer_status": {"no cancer": 0, "no": 0, "none": 0, "any cancer": 1, "yes": 1, "cancer": 1},
    "institution_type": {"hospital": 0, "non-hospital": 1, "non hospital": 1, "nonhospital": 1},
    "operative_region_extent": {"lumbar only": 0, "extended_region_involvement": 1, "extended region involvement": 1, "extended": 1},
}
ORDINAL_MAPS = {
    "diabetes_status": {"no": 0, "none": 0, "0": 0, "without comp": 1, "without complication": 1, "without complications": 1, "1": 1, "with comp": 2, "with complication": 2, "with complications": 2, "2": 2},
    "institution_size": {"between 1-99 beds": 0, "1-99": 0, "between 100-399 beds": 1, "100-399": 1, ">= 400 beds": 2, ">=400 beds": 2, ">=400": 2, ">= 400": 2},
    "asa": {"1": 1, "i": 1, "2": 2, "ii": 2, "3": 3, "iii": 3, "4": 4, "iv": 4, ">=4": 4, ">= 4": 4, "5": 4, "v": 4},
    "number_operated_levels": {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4, ">=4": 4, ">= 4": 4, "5": 4, "6": 4, "7": 4, "8": 4, "9": 4, "10": 4},
}
PREFERRED_NOMINAL_LEVELS = {
    "race": ["White", "Black", "Other"],
    "institution_region": ["South", "North East", "West", "Midwest"],
    "payer_status": ["Medicare", "Commercial/Private", "Other", "Medicaid/Public/Government"],
    "PatTobaccoUse": ["Unknown/Not reported/Multiple", "Never", "Former", "Current"],
}
FEATURE_LABELS = {
    "finaldx_degenerative": "Degenerative diagnosis",
    "finaldx_radicular": "Radiculopathy diagnosis",
    "finaldx_stenosis": "Spinal stenosis diagnosis",
    "finaldx_deformity_instability": "Deformity or instability diagnosis",
    "finaldx_other_diagnosis": "Other lumbar diagnosis",
    "age": "Age",
    "sex": "Sex",
    "race": "Race",
    "ethnicity": "Ethnicity",
    "cancer_status": "Cancer status",
    "chronic_pulmonary_disease": "Chronic pulmonary disease",
    "congestive_heart_failure": "Congestive heart failure",
    "connective_tissue_rheumatic_disease": "Connective tissue/rheumatic disease",
    "diabetes_status": "Diabetes status",
    "myocardial_infarction": "Myocardial infarction",
    "renal_disease": "Renal disease",
    "institution_type": "Institution type",
    "institution_size": "Institution size",
    "institution_region": "Institution region",
    "asa": "ASA physical status",
    "bmi": "BMI",
    "payer_status": "Primary payer",
    "alif_llif": "Anterior/lateral lumbar interbody fusion",
    "corpectomy": "Corpectomy",
    "discectomy": "Discectomy",
    "foraminotomy": "Foraminotomy",
    "instrumentation": "Instrumentation",
    "laminectomy_posterior_decompression": "Posterior decompression or laminectomy",
    "pelvic_fixation": "Pelvic fixation",
    "plf": "Posterolateral fusion",
    "tlif_plif": "Posterior/transforaminal lumbar interbody fusion",
    "other_lumbar_procedure": "Other lumbar procedure",
    "number_operated_levels": "Number of operated levels",
    "operative_region_extent": "Operative region extent",
    "PatTobaccoUse": "Tobacco use",
    "preop_ODI": "Preoperative ODI",
    "postop_ODI": "Postoperative ODI",
    "ODI_change": "Change in ODI",
    PROM_CHANGE_RATE_COL: "ODI change rate",
    RELATIVE_ODI_MCID_COL: "Relative ODI MCID",
    "postop_ODI_day": "Timing of postoperative ODI assessment",
}

# ============================================================
# 3) Preprocessor class needed for joblib artifact loading
# ============================================================

def clean_scalar(x: Any) -> Any:
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().replace("≥", ">=")
        return np.nan if s.lower() in MISSING_STRINGS else s
    return x


def norm_text(x: Any) -> Optional[str]:
    x = clean_scalar(x)
    if pd.isna(x):
        return None
    return str(x).strip().replace("≥", ">=").lower()


def pretty_feature_name(feature: str) -> str:
    return FEATURE_LABELS.get(feature, feature.replace("_", " "))


def safe_filename(x: str) -> str:
    x = str(x).replace("≥", "ge").replace("≤", "le")
    x = re.sub(r"[^A-Za-z0-9_.-]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x[:180] if x else "file"


def json_native(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(k): json_native(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_native(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj


@dataclass
class Step2Preprocessor:
    continuous_features: List[str]
    binary_features: List[str]
    ordinal_features: List[str]
    nominal_features: List[str]
    preferred_nominal_levels: Dict[str, List[str]]

    def __post_init__(self):
        self.continuous_imputer: Optional[SimpleImputer] = None
        self.discrete_imputer: Optional[SimpleImputer] = None
        self.nominal_imputer: Optional[SimpleImputer] = None
        self.onehot: Optional[OneHotEncoder] = None
        self.numeric_feature_names_: List[str] = []
        self.output_feature_names_: List[str] = []

    def _parse_binary(self, x: Any, feature: str) -> float:
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in BINARY_MAPS and sx in BINARY_MAPS[feature]:
            return float(BINARY_MAPS[feature][sx])
        if sx in {"1", "1.0", "yes", "y", "true", "t", "present", "positive"}:
            return 1.0
        if sx in {"0", "0.0", "no", "n", "false", "f", "absent", "negative"}:
            return 0.0
        try:
            v = float(sx)
            return float(v) if v in (0.0, 1.0) else np.nan
        except Exception:
            return np.nan

    def _parse_ordinal(self, x: Any, feature: str) -> float:
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in ORDINAL_MAPS and sx in ORDINAL_MAPS[feature]:
            return float(ORDINAL_MAPS[feature][sx])
        try:
            v = float(sx)
            if feature == "asa":
                return float(min(max(int(round(v)), 1), 4))
            if feature == "number_operated_levels":
                return float(min(max(int(round(v)), 0), 4))
            if feature == "diabetes_status":
                return float(min(max(int(round(v)), 0), 2))
            if feature == "institution_size":
                return float(min(max(int(round(v)), 0), 2))
            return float(v)
        except Exception:
            return np.nan

    def _canonical_nominal(self, feature: str, x: Any) -> Any:
        x = clean_scalar(x)
        if pd.isna(x):
            return np.nan
        s = str(x).strip()
        sl = s.lower()
        if feature == "race":
            if sl == "white":
                return "White"
            if sl == "black":
                return "Black"
            return "Other"
        if feature == "institution_region":
            mapping = {"south": "South", "north east": "North East", "northeast": "North East", "north-east": "North East", "west": "West", "midwest": "Midwest", "mid west": "Midwest"}
            return mapping.get(sl, s)
        if feature == "payer_status":
            if sl == "medicare":
                return "Medicare"
            if sl in {"commercial/private", "commercial", "private", "commercial private"}:
                return "Commercial/Private"
            if sl in {"medicaid/public/government", "medicaid", "public", "government", "public/government"}:
                return "Medicaid/Public/Government"
            return "Other"
        if feature == "PatTobaccoUse":
            if sl == "never":
                return "Never"
            if sl == "former":
                return "Former"
            if sl == "current":
                return "Current"
            return "Unknown/Not reported/Multiple"
        return s

    def _make_parts(self, X: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        cont = pd.DataFrame(index=X.index)
        for c in self.continuous_features:
            cont[c] = pd.to_numeric(X[c].map(clean_scalar), errors="coerce")
        discrete = pd.DataFrame(index=X.index)
        for c in self.binary_features:
            discrete[c] = X[c].map(lambda z: self._parse_binary(z, c)).astype(float)
        for c in self.ordinal_features:
            discrete[c] = X[c].map(lambda z: self._parse_ordinal(z, c)).astype(float)
        nominal = pd.DataFrame(index=X.index)
        for c in self.nominal_features:
            nominal[c] = X[c].map(lambda z: self._canonical_nominal(c, z)).astype("object")
        return cont, discrete, nominal

    def fit(self, X: pd.DataFrame):
        cont, discrete, nominal = self._make_parts(X)
        self.continuous_imputer = SimpleImputer(strategy="median")
        self.discrete_imputer = SimpleImputer(strategy="most_frequent")
        self.nominal_imputer = SimpleImputer(strategy="constant", fill_value="Missing")
        self.continuous_imputer.fit(cont)
        self.discrete_imputer.fit(discrete)
        nominal_imp = pd.DataFrame(self.nominal_imputer.fit_transform(nominal), columns=self.nominal_features)
        categories = []
        for c in self.nominal_features:
            preferred = list(self.preferred_nominal_levels.get(c, []))
            observed = nominal_imp[c].astype(str).unique().tolist()
            categories.append(preferred + sorted([x for x in observed if x not in preferred]))
        try:
            self.onehot = OneHotEncoder(categories=categories, handle_unknown="ignore", sparse_output=False)
        except TypeError:
            self.onehot = OneHotEncoder(categories=categories, handle_unknown="ignore", sparse=False)
        self.onehot.fit(nominal_imp.astype(str))
        self.numeric_feature_names_ = self.continuous_features + self.binary_features + self.ordinal_features
        self.output_feature_names_ = list(self.numeric_feature_names_) + self.onehot.get_feature_names_out(self.nominal_features).tolist()
        return self

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        cont, discrete, nominal = self._make_parts(X)
        cont_imp = self.continuous_imputer.transform(cont)
        discrete_imp = self.discrete_imputer.transform(discrete)
        nominal_imp = pd.DataFrame(self.nominal_imputer.transform(nominal), columns=self.nominal_features)
        nominal_oh = self.onehot.transform(nominal_imp.astype(str))
        return np.concatenate([cont_imp, discrete_imp, nominal_oh], axis=1).astype(float)

    @property
    def output_feature_names(self) -> List[str]:
        return self.output_feature_names_

# ============================================================
# 4) Data and source-run helpers
# ============================================================

SOURCE_ARCHIVE_NAME = "Step2_DynamicPROM_LightGBM_TopConfig_Final.zip"
SOURCE_FOLDER_NAME = "Step2_DynamicPROM_LightGBM_TopConfig_Final"


def _source_has_required_structure(path: str) -> bool:
    if not os.path.isdir(path):
        return False
    return os.path.isdir(os.path.join(path, "dynamic_PROM_expanded"))


def _archive_has_required_structure(zip_path: str) -> bool:
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            names = [n.replace(chr(92), "/") for n in zf.namelist()]
    except Exception:
        return False
    has_model = any(
        "dynamic_PROM_expanded/" in n
        and n.endswith("_lightgbm_pipeline_artifact_Final.joblib")
        for n in names
    )
    has_split = any(n.endswith(".xlsx") for n in names)
    return bool(has_model and has_split)


def _find_source_dir_inside(search_root: str) -> Optional[str]:
    if _source_has_required_structure(search_root):
        return search_root
    for root, dirs, files in os.walk(search_root):
        depth = os.path.relpath(root, search_root).count(os.sep)
        if depth > 5:
            dirs[:] = []
            continue
        if _source_has_required_structure(root):
            return root
    return None


def ensure_source_dir() -> str:
    folder_candidates = [
        os.path.join(BASE_DIR, SOURCE_FOLDER_NAME),
        os.path.join(FALLBACK_DIR, SOURCE_FOLDER_NAME),
    ]
    for folder in folder_candidates:
        if _source_has_required_structure(folder):
            return folder

    archive_candidates = [
        os.path.join(BASE_DIR, SOURCE_ARCHIVE_NAME),
        os.path.join(FALLBACK_DIR, SOURCE_ARCHIVE_NAME),
    ]
    archive_path = None
    for candidate in archive_candidates:
        if os.path.exists(candidate) and _archive_has_required_structure(candidate):
            archive_path = candidate
            break

    if archive_path is None:
        raise FileNotFoundError(
            f"Required source archive was not found or did not contain the final locked Step 2 model artifact. "
            f"Expected archive name: {SOURCE_ARCHIVE_NAME}"
        )

    extract_root = os.path.join(BASE_DIR, "_step2_final_source")
    if os.path.isdir(extract_root):
        shutil.rmtree(extract_root)
    os.makedirs(extract_root, exist_ok=True)

    print(f"Extracting source archive: {archive_path}")
    with zipfile.ZipFile(archive_path, "r") as z:
        z.extractall(extract_root)

    detected = _find_source_dir_inside(extract_root)
    if detected is None:
        raise FileNotFoundError("The extracted source archive did not contain the expected final Step 2 model structure.")
    return detected


def ensure_input_csv() -> str:
    if os.path.exists(INPUT_CSV):
        return INPUT_CSV
    if os.path.exists(FALLBACK_INPUT_CSV):
        shutil.copy2(FALLBACK_INPUT_CSV, INPUT_CSV)
        return INPUT_CSV
    raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")


def to_binary_target(x: Any) -> float:
    sx = norm_text(x)
    if sx is None:
        return np.nan
    if sx in {"1", "1.0", "yes", "y", "true", "t"}:
        return 1.0
    if sx in {"0", "0.0", "no", "n", "false", "f"}:
        return 0.0
    try:
        v = float(sx)
        return float(v) if v in (0.0, 1.0) else np.nan
    except Exception:
        return np.nan


def derive_dynamic_prom_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    required = ["preop_ODI", "postop_ODI", DAYS_BETWEEN_PROM_COL, "postop_ODI_day", TARGET_COL, GROUP_COL]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"Missing required Step 2 columns: {missing}")
    out["preop_ODI"] = pd.to_numeric(out["preop_ODI"], errors="coerce")
    out["postop_ODI"] = pd.to_numeric(out["postop_ODI"], errors="coerce")
    out[DAYS_BETWEEN_PROM_COL] = pd.to_numeric(out[DAYS_BETWEEN_PROM_COL], errors="coerce")
    out["ODI_change"] = out["postop_ODI"] - out["preop_ODI"]
    out[PROM_CHANGE_RATE_COL] = out["ODI_change"] / out[DAYS_BETWEEN_PROM_COL]
    valid_rel = out["preop_ODI"].notna() & out["postop_ODI"].notna() & (out["preop_ODI"] > 0)
    zero_baseline_rel = out["preop_ODI"].notna() & out["postop_ODI"].notna() & out["preop_ODI"].eq(0)
    out[RELATIVE_ODI_MCID_COL] = np.nan
    out.loc[valid_rel, RELATIVE_ODI_MCID_COL] = (
        (out.loc[valid_rel, "preop_ODI"] - out.loc[valid_rel, "postop_ODI"])
        / out.loc[valid_rel, "preop_ODI"]
        >= RELATIVE_PROM_MCID_CUTOFF
    ).astype(int)
    out.loc[zero_baseline_rel, RELATIVE_ODI_MCID_COL] = 0.0
    out[TARGET_COL] = out[TARGET_COL].map(to_binary_target)
    eligibility = (
        out["preop_ODI"].notna() &
        out["postop_ODI"].notna() &
        out[DAYS_BETWEEN_PROM_COL].notna() &
        (out[DAYS_BETWEEN_PROM_COL] > 0) &
        out[TARGET_COL].isin([0.0, 1.0])
    )
    out = out.loc[eligibility].copy()
    out[TARGET_COL] = out[TARGET_COL].astype(int)
    return out


def load_split_assignment(source_dir: str) -> pd.DataFrame:
    workbooks = []
    for root, _, files in os.walk(source_dir):
        for fn in files:
            if fn.endswith(".xlsx"):
                workbooks.append(os.path.join(root, fn))

    for xlsx in workbooks:
        try:
            xl = pd.ExcelFile(xlsx)
            if "split_assignment" not in xl.sheet_names:
                continue
            split = pd.read_excel(xlsx, sheet_name="split_assignment")
            split.columns = [str(c).strip() for c in split.columns]
            if GROUP_COL in split.columns and "split" in split.columns:
                return split[[GROUP_COL, "split"]].drop_duplicates()
        except Exception:
            continue

    raise FileNotFoundError("Could not find a workbook containing a valid split_assignment sheet.")


def get_test_dataframe(source_dir: str) -> pd.DataFrame:
    csv_path = ensure_input_csv()
    raw = pd.read_csv(csv_path, low_memory=False)
    raw.columns = [str(c).strip() for c in raw.columns]
    work = derive_dynamic_prom_features(raw)
    split = load_split_assignment(source_dir)
    work = work.merge(split, on=GROUP_COL, how="left")
    if work["split"].isna().any():
        n_missing = int(work["split"].isna().sum())
        raise RuntimeError(f"{n_missing} rows could not be matched to split_assignment by PersonKey.")
    test = work[work["split"].eq("test")].reset_index(drop=True)
    print(f"Held-out test set reconstructed: rows={len(test):,}, events={int(test[TARGET_COL].sum()):,}, prevalence={test[TARGET_COL].mean():.4%}")
    return test


def find_final_artifact(source_dir: str) -> str:
    root = os.path.join(source_dir, "dynamic_PROM_expanded")
    if not os.path.isdir(root):
        raise FileNotFoundError(f"Dynamic PROM artifact root not found: {root}")

    selected = []
    for folder_name in os.listdir(root):
        folder_path = os.path.join(root, folder_name)
        if not os.path.isdir(folder_path):
            continue
        if not folder_name.startswith("top01_"):
            continue
        art_dir = os.path.join(folder_path, "model_artifacts")
        if not os.path.isdir(art_dir):
            continue
        artifacts = [
            os.path.join(art_dir, fn)
            for fn in os.listdir(art_dir)
            if fn.endswith("_lightgbm_pipeline_artifact_Final.joblib")
        ]
        selected.extend(artifacts)

    if len(selected) != 1:
        raise RuntimeError(
            "Could not identify exactly one final locked LightGBM artifact in the source archive."
        )

    return selected[0]

# ============================================================
# 5) SHAP helpers
# ============================================================

def raw_feature_numeric_values(X_raw: pd.DataFrame, feature: str) -> pd.Series:
    if feature in CONTINUOUS_FEATURES:
        s = pd.to_numeric(X_raw[feature].map(clean_scalar), errors="coerce")
    elif feature in BINARY_FEATURES:
        tmp_pre = Step2Preprocessor([], [], [], [], {})
        s = X_raw[feature].map(lambda z: tmp_pre._parse_binary(z, feature)).astype(float)
    elif feature in ORDINAL_FEATURES:
        tmp_pre = Step2Preprocessor([], [], [], [], {})
        s = X_raw[feature].map(lambda z: tmp_pre._parse_ordinal(z, feature)).astype(float)
    else:
        vals = X_raw[feature].map(lambda z: str(clean_scalar(z)) if not pd.isna(clean_scalar(z)) else "Missing")
        cats = {v: i for i, v in enumerate(sorted(vals.dropna().unique()))}
        s = vals.map(cats).astype(float)
    if s.isna().any():
        med = s.median(skipna=True)
        s = s.fillna(0.0 if pd.isna(med) else med)
    return s.astype(float)


def encoded_to_group_mapping(pre: Step2Preprocessor, features: List[str]) -> Dict[str, List[int]]:
    encoded_names = list(pre.output_feature_names)
    mapping = {}
    for feature in features:
        idx = []
        if feature in encoded_names:
            idx.append(encoded_names.index(feature))
        if feature in NOMINAL_FEATURES:
            prefix = feature + "_"
            idx.extend([i for i, name in enumerate(encoded_names) if name.startswith(prefix)])
        if idx:
            mapping[feature] = sorted(set(idx))
    return mapping


def stratified_explain_indices(y: np.ndarray, max_rows: Optional[int], seed: int) -> np.ndarray:
    """
    Return indices for SHAP explanation.

    max_rows=None means all held-out test patients are explained. This ensures
    that the SHAP heatmap x-axis represents every test patient rather than a
    downsampled subset.
    """
    n = len(y)
    if max_rows is None or n <= int(max_rows):
        return np.arange(n)

    max_rows = int(max_rows)
    rng = np.random.default_rng(seed)
    ev = np.where(np.asarray(y).astype(int) == 1)[0]
    ne = np.where(np.asarray(y).astype(int) == 0)[0]
    ev_keep = min(len(ev), max_rows // 2)
    if len(ev) > ev_keep:
        ev = rng.choice(ev, ev_keep, replace=False)
    remaining = max_rows - len(ev)
    if len(ne) > remaining:
        ne = rng.choice(ne, remaining, replace=False)
    return np.sort(np.concatenate([ev, ne]))


def compute_grouped_shap(pre, model, X_raw: pd.DataFrame, y_true: np.ndarray, p_calibrated: np.ndarray, features: List[str], seed: int):
    idx = stratified_explain_indices(y_true, SHAP_MAX_EXPLAIN_ROWS, seed)
    X_explain_raw = X_raw.iloc[idx].reset_index(drop=True)
    y_explain = np.asarray(y_true)[idx]
    p_explain = np.asarray(p_calibrated)[idx]
    X_explain_enc = pre.transform(X_explain_raw[features])
    encoded_names = list(pre.output_feature_names)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_explain_enc)
    shap_values = shap_values[-1] if isinstance(shap_values, list) else shap_values
    shap_values = np.asarray(shap_values)
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, -1]
    mapping = encoded_to_group_mapping(pre, features)
    grouped_values = {}
    grouped_data = {}
    mapping_names = {}
    for raw_feature, ids in mapping.items():
        display = pretty_feature_name(raw_feature)
        grouped_values[display] = shap_values[:, ids].sum(axis=1)
        grouped_data[display] = raw_feature_numeric_values(X_explain_raw, raw_feature).values
        mapping_names[raw_feature] = [encoded_names[i] for i in ids]
    shap_df = pd.DataFrame(grouped_values)
    data_df = pd.DataFrame(grouped_data)
    total_abs = float(np.abs(shap_df.values).mean(axis=0).sum())
    imp = []
    for raw_feature, ids in mapping.items():
        display = pretty_feature_name(raw_feature)
        mean_abs = float(np.abs(shap_df[display]).mean())
        imp.append({
            "raw_feature": raw_feature,
            "display_feature": display,
            "mean_abs_SHAP": mean_abs,
            "percent_of_grouped_SHAP": 100.0 * mean_abs / total_abs if total_abs > 0 else np.nan,
            "n_encoded_columns_grouped": len(ids),
            "feature_type": "Continuous" if raw_feature in CONTINUOUS_FEATURES else "Binary" if raw_feature in BINARY_FEATURES else "Ordinal" if raw_feature in ORDINAL_FEATURES else "Nominal" if raw_feature in NOMINAL_FEATURES else "Unknown",
        })
    imp_df = pd.DataFrame(imp).sort_values("mean_abs_SHAP", ascending=False).reset_index(drop=True)
    shap_df.insert(0, "__row_id__", idx)
    shap_df.insert(1, "__y_true__", y_explain)
    shap_df.insert(2, "__p_calibrated__", p_explain)
    return shap_df, data_df, imp_df, mapping_names, explainer, X_explain_raw, X_explain_enc, encoded_names


def save_beeswarm(shap_df, data_df, imp_df, path, title):
    ordered = imp_df["display_feature"].tolist()[: min(SHAP_BEESWARM_MAX_DISPLAY, len(imp_df))]
    plt.figure(figsize=(11.0, max(8.8, 0.32 * len(ordered) + 2)))
    try:
        shap.summary_plot(shap_df[ordered].values, features=data_df[ordered], feature_names=ordered, max_display=len(ordered), show=False, plot_size=None, color=SHAP_COLOR_CMAP)
    except TypeError:
        shap.summary_plot(shap_df[ordered].values, features=data_df[ordered], feature_names=ordered, max_display=len(ordered), show=False, plot_size=None)
    plt.title(title, fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()


def save_bar(imp_df, path, title):
    d = imp_df.head(SHAP_BAR_MAX_DISPLAY).copy().iloc[::-1]
    fig, ax = plt.subplots(figsize=(10.5, max(7, 0.35 * len(d))))
    bars = ax.barh(d["display_feature"], d["mean_abs_SHAP"], color=SHAP_BEESWARM_BLUE)
    mx = float(d["mean_abs_SHAP"].max()) if len(d) else 1.0
    ax.set_xlim(0, mx * 1.35 if mx > 0 else 1.0)
    ax.set_xlabel("Mean absolute SHAP value")
    ax.set_title(title, fontsize=15, fontweight="bold")
    for b, v, pct in zip(bars, d["mean_abs_SHAP"], d["percent_of_grouped_SHAP"]):
        ax.text(b.get_width() + mx * 0.015, b.get_y() + b.get_height() / 2, f"{v:.3f} ({pct:.1f}%)", va="center", fontsize=9)
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def save_heatmap(shap_df, imp_df, path, title):
    features = imp_df["display_feature"].tolist()[: min(SHAP_HEATMAP_MAX_DISPLAY_FEATURES, len(imp_df))]
    tmp = shap_df[["__p_calibrated__"] + features].copy().sort_values("__p_calibrated__", ascending=False).reset_index(drop=True)

    mat = tmp[features].T.values
    fx = tmp["__p_calibrated__"].to_numpy(dtype=float)
    vmax = max(float(np.nanquantile(np.abs(mat), 0.99)), 1e-6)

    fig_height = max(8.2, 0.34 * len(features) + 2.8)
    fig = plt.figure(figsize=(14, fig_height))
    gs = fig.add_gridspec(
        nrows=2,
        ncols=1,
        height_ratios=[1.0, max(5.0, 0.34 * len(features) + 1.6)],
        hspace=0.05,
    )
    ax_fx = fig.add_subplot(gs[0])
    ax = fig.add_subplot(gs[1], sharex=ax_fx)

    x_positions = np.arange(len(tmp))
    ax_fx.plot(x_positions, fx, color="black", linewidth=1.5)
    ax_fx.set_ylabel("f(x)", rotation=0, labelpad=20, fontweight="bold")
    ax_fx.set_xlim(-0.5, len(tmp) - 0.5)
    ax_fx.tick_params(axis="x", bottom=False, labelbottom=False)
    ax_fx.spines["right"].set_visible(False)
    ax_fx.spines["top"].set_visible(False)
    ax_fx.spines["bottom"].set_visible(False)
    ax_fx.grid(axis="y", alpha=0.20, linewidth=0.6)
    ax_fx.set_title(title + " (held-out test set)", fontsize=14, fontweight="bold", pad=8)

    im = ax.imshow(mat, aspect="auto", cmap=SHAP_DIVERGING_CMAP, vmin=-vmax, vmax=vmax, interpolation="nearest")
    ax.set_xlabel("Test-set patients sorted by calibrated predicted risk")
    ax.set_ylabel("Features")
    ax.set_yticks(np.arange(len(features)))
    ax.set_yticklabels(features)
    ticks = np.linspace(0, max(len(tmp) - 1, 1), min(12, len(tmp))).astype(int)
    ax.set_xticks(ticks)
    ax.set_xticklabels((ticks + 1).astype(int))

    cbar = fig.colorbar(im, ax=[ax_fx, ax], pad=0.02)
    cbar.set_label("SHAP value")

    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)

# ============================================================
# 6) Threshold and interaction plots
# ============================================================

def build_binned_curve(x: np.ndarray, shap_y: np.ndarray, pred: np.ndarray, max_bins: int = SHAP_THRESHOLD_MAX_BINS) -> pd.DataFrame:
    x = np.asarray(x, dtype=float)
    shap_y = np.asarray(shap_y, dtype=float)
    pred = np.asarray(pred, dtype=float)

    valid = np.isfinite(x) & np.isfinite(shap_y) & np.isfinite(pred)
    d = pd.DataFrame({"x": x[valid], "shap": shap_y[valid], "pred": pred[valid]})

    if d.empty:
        return pd.DataFrame(columns=["x", "shap_mean", "pred_mean", "n"])

    unique_x = np.sort(d["x"].unique())
    if len(unique_x) <= min(max_bins, 20):
        curve = (
            d.groupby("x", as_index=False)
            .agg(shap_mean=("shap", "mean"), pred_mean=("pred", "mean"), n=("x", "size"))
            .sort_values("x")
            .reset_index(drop=True)
        )
    else:
        q = min(max_bins, max(8, int(np.sqrt(len(d)))))
        d["bin"] = pd.qcut(d["x"].rank(method="first"), q=q, duplicates="drop")
        curve = (
            d.groupby("bin", observed=True)
            .agg(x=("x", "median"), shap_mean=("shap", "mean"), pred_mean=("pred", "mean"), n=("x", "size"))
            .reset_index(drop=True)
            .sort_values("x")
            .reset_index(drop=True)
        )

    return curve


def _smoothed_shap_curve(curve: pd.DataFrame, window: int = 3) -> np.ndarray:
    ys = curve["shap_mean"].astype(float).to_numpy()
    if len(ys) <= 2:
        return ys
    return pd.Series(ys).rolling(window=window, center=True, min_periods=1).mean().to_numpy()


def find_all_zero_crossings(curve: pd.DataFrame) -> List[float]:
    if curve.empty:
        return []

    xs = curve["x"].astype(float).to_numpy()
    ys = _smoothed_shap_curve(curve, window=3)

    thresholds = []
    for i in range(len(xs) - 1):
        y1, y2 = ys[i], ys[i + 1]
        if not (np.isfinite(y1) and np.isfinite(y2)):
            continue
        if y1 == 0:
            thresholds.append(float(xs[i]))
        elif y1 * y2 < 0:
            denom = abs(y1) + abs(y2)
            frac = abs(y1) / denom if denom > 0 else 0.5
            thresholds.append(float(xs[i] + frac * (xs[i + 1] - xs[i])))

    if len(thresholds) <= 1:
        return thresholds

    x_range = float(np.nanmax(xs) - np.nanmin(xs)) if len(xs) else 1.0
    tolerance = max(1e-8, 0.005 * x_range)
    out = []
    for value in sorted(thresholds):
        if not out or abs(value - out[-1]) > tolerance:
            out.append(value)
    return out


def save_threshold_plot(x, shap_y, pred, display, raw_feature, path, model_label):
    x = np.asarray(x, dtype=float)
    shap_y = np.asarray(shap_y, dtype=float)
    pred = np.asarray(pred, dtype=float)

    valid = np.isfinite(x) & np.isfinite(shap_y) & np.isfinite(pred)
    x_valid = x[valid]
    shap_valid = shap_y[valid]
    pred_valid = pred[valid]

    curve = build_binned_curve(x_valid, shap_valid, pred_valid)
    thresholds = find_all_zero_crossings(curve)

    fig, ax1 = plt.subplots(figsize=(8.8, 5.5))
    ax1.scatter(x_valid, shap_valid, s=14, alpha=0.12, color="gray", edgecolors="none")

    if not curve.empty:
        xs = curve["x"].to_numpy(dtype=float)
        ys_plot = _smoothed_shap_curve(curve, window=3)

        ax1.plot(xs, ys_plot, color="black", linewidth=2.0)
        ax1.fill_between(xs, 0, ys_plot, where=ys_plot >= 0, color=SHAP_BEESWARM_PINK, alpha=0.42, interpolate=True)
        ax1.fill_between(xs, 0, ys_plot, where=ys_plot < 0, color=SHAP_BEESWARM_BLUE, alpha=0.42, interpolate=True)

    ax1.axhline(0, color="black", linewidth=0.8, alpha=0.7)

    ymin, _ = ax1.get_ylim()
    for threshold in thresholds:
        ax1.axvline(threshold, color=SHAP_BEESWARM_BLUE, linestyle=":", linewidth=1.8)
        ax1.text(threshold, ymin, f"{threshold:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

    ax1.set_xlabel(display, fontweight="bold")
    ax1.set_ylabel(f"SHAP value for {display}")
    ax1.set_title(f"{model_label}: SHAP threshold analysis — {display}", fontsize=13, fontweight="bold")

    ax2 = ax1.twinx()
    if not curve.empty:
        ax2.plot(curve["x"], curve["pred_mean"] * 100, color=SHAP_BEESWARM_PINK, linestyle="--", linewidth=1.8, alpha=0.95)
    ax2.set_ylabel("Predicted risk of reoperation (%)")

    if thresholds:
        threshold_text = "\n".join([f"Threshold {i + 1} = {v:.2f}" for i, v in enumerate(thresholds)])
        method = "all_zero_crossings"
    else:
        threshold_text = "No zero crossing detected"
        method = "no_zero_crossing"

    ax1.text(
        0.02, 0.97,
        f"{threshold_text}\nMethod = {method}",
        transform=ax1.transAxes,
        ha="left",
        va="top",
        fontsize=8.5,
        bbox=dict(boxstyle="round,pad=0.30", facecolor="white", edgecolor="gray", alpha=0.90),
    )

    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    return {
        "raw_feature": raw_feature,
        "display_feature": display,
        "n_thresholds": len(thresholds),
        "thresholds": "; ".join([f"{v:.6g}" for v in thresholds]),
        "threshold_method": method,
        "plot_path": path,
    }


def compute_interaction_vector(explainer, Xenc: np.ndarray, ids_a: List[int], ids_b: List[int]) -> np.ndarray:
    ints = explainer.shap_interaction_values(Xenc)
    ints = ints[-1] if isinstance(ints, list) else ints
    ints = np.asarray(ints)
    if ints.ndim == 4:
        ints = ints[:, :, :, -1]
    out = np.zeros(Xenc.shape[0], dtype=float)
    for i in ids_a:
        for j in ids_b:
            out += ints[:, i, j]
    return out


def save_interaction_plot(prom_x, other_x, int_values, prom_name, other_name, path, title):
    fig, ax = plt.subplots(figsize=(7.6, 5.6))
    sc = ax.scatter(prom_x, int_values, c=other_x, cmap=SHAP_COLOR_CMAP, s=24, alpha=0.72, edgecolors="none")
    ax.axhline(0, color="black", linewidth=0.8, alpha=0.7)
    ax.set_xlabel(prom_name, fontweight="bold")
    ax.set_ylabel(f"SHAP interaction value\n({prom_name} × {other_name})")
    ax.set_title(title, fontsize=13, fontweight="bold")
    cbar = fig.colorbar(sc, ax=ax, pad=0.02)
    cbar.set_label(other_name)
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)

# ============================================================
# 7) Main model processing
# ============================================================

def process_final_artifact(artifact_path: str, test: pd.DataFrame) -> Dict[str, Any]:
    art = joblib.load(artifact_path)
    features = list(art["features"])
    pre = art["preprocessor"]
    model = art["model"]
    calibrator = art["calibrator"]
    model_label = art.get("model_label", "Dynamic PROM-expanded")

    X_test = test[features].copy()
    y_test = test[TARGET_COL].astype(int).to_numpy()
    Xenc_test = pre.transform(X_test)
    p_raw = model.predict_proba(Xenc_test)[:, 1]
    p_cal = np.clip(calibrator.predict(p_raw), 0, 1)

    out_dir = OUTPUT_DIR
    plot_dir = os.path.join(out_dir, "plots")
    table_dir = os.path.join(out_dir, "tables")
    for d in [out_dir, plot_dir, table_dir]:
        os.makedirs(d, exist_ok=True)

    title_base = "Step 2 LightGBM dynamic PROM-expanded final model"
    shap_df, data_df, imp_df, mapping_names, explainer, X_explain_raw, Xenc_explain, encoded_names = compute_grouped_shap(
        pre, model, X_test, y_test, p_cal, features, seed=RANDOM_STATE
    )

    shap_df.to_csv(os.path.join(table_dir, "grouped_SHAP_values_Final.csv"), index=False)
    data_df.to_csv(os.path.join(table_dir, "grouped_SHAP_feature_values_Final.csv"), index=False)
    imp_df.to_csv(os.path.join(table_dir, "grouped_SHAP_importance_Final.csv"), index=False)
    with open(os.path.join(table_dir, "grouped_SHAP_feature_mapping_Final.json"), "w") as f:
        json.dump(json_native(mapping_names), f, indent=2, sort_keys=True)

    bees = os.path.join(plot_dir, "SHAP_beeswarm_GROUPED_Final.png")
    bar = os.path.join(plot_dir, "SHAP_bar_GROUPED_Final.png")
    heatmap = os.path.join(plot_dir, "SHAP_heatmap_fx_GROUPED_Final.png")
    save_beeswarm(shap_df, data_df, imp_df, bees, title_base + ": grouped SHAP beeswarm")
    save_bar(imp_df, bar, title_base + ": grouped SHAP importance")
    save_heatmap(shap_df, imp_df, heatmap, title_base + ": grouped SHAP heatmap")

    # Threshold analyses for top-final_indexed numeric/binary/ordinal features.
    row_ids = shap_df["__row_id__"].astype(int).to_numpy()
    X_thr = X_test.iloc[row_ids].reset_index(drop=True)
    p_thr = p_cal[row_ids]
    threshold_rows = []
    numeric_like = set(CONTINUOUS_FEATURES + BINARY_FEATURES + ORDINAL_FEATURES)
    candidates = imp_df[imp_df["raw_feature"].isin(numeric_like)].head(SHAP_THRESHOLD_TOP_N_NUMERIC)
    for _, r in candidates.iterrows():
        raw_feature = r["raw_feature"]
        display = r["display_feature"]
        x = raw_feature_numeric_values(X_thr, raw_feature).to_numpy(dtype=float)
        shap_y = shap_df[display].to_numpy(dtype=float)
        path = os.path.join(plot_dir, f"SHAP_threshold_{safe_filename(display)}_Final.png")
        threshold_rows.append(save_threshold_plot(x, shap_y, p_thr, display, raw_feature, path, model_label))
    threshold_df = pd.DataFrame(threshold_rows)
    threshold_df.to_csv(os.path.join(table_dir, "SHAP_threshold_summary_Final.csv"), index=False)

    # Exact SHAP interaction plots: preoperative PROM × Age and preoperative PROM × BMI.
    interaction_rows = []
    encoded_names = list(pre.output_feature_names)
    prom_ids = [encoded_names.index("preop_ODI")] if "preop_ODI" in encoded_names else []
    age_ids = [encoded_names.index("age")] if "age" in encoded_names else []
    bmi_ids = [encoded_names.index("bmi")] if "bmi" in encoded_names else []
    if not prom_ids:
        raise RuntimeError("preop_ODI was not found in encoded feature names.")

    idx_int = stratified_explain_indices(y_test, SHAP_INTERACTION_MAX_ROWS, RANDOM_STATE + 9000)
    X_int = X_test.iloc[idx_int].reset_index(drop=True)
    Xenc_int = pre.transform(X_int)
    prom_x = raw_feature_numeric_values(X_int, "preop_ODI").to_numpy(dtype=float)

    for other_feature, other_ids, other_label in [("age", age_ids, "Age"), ("bmi", bmi_ids, "BMI")]:
        if not other_ids:
            continue
        other_x = raw_feature_numeric_values(X_int, other_feature).to_numpy(dtype=float)
        try:
            int_vals = compute_interaction_vector(explainer, Xenc_int, prom_ids, other_ids)
            pth = os.path.join(plot_dir, f"SHAP_interaction_Preoperative_ODI_x_{safe_filename(other_label)}_Final.png")
            save_interaction_plot(
                prom_x=prom_x,
                other_x=other_x,
                int_values=int_vals,
                prom_name="Preoperative ODI",
                other_name=other_label,
                path=pth,
                title=f"{title_base}: Preoperative ODI × {other_label}",
            )
            interaction_rows.append({
                "interaction_pair": f"Preoperative ODI x {other_label}",
                "n_interaction_rows": int(len(int_vals)),
                "mean_abs_interaction": float(np.mean(np.abs(int_vals))),
                "plot_path": pth,
                "status": "success",
            })
        except Exception as e:
            interaction_rows.append({
                "interaction_pair": f"Preoperative ODI x {other_label}",
                "n_interaction_rows": 0,
                "mean_abs_interaction": np.nan,
                "plot_path": "",
                "status": f"failed: {repr(e)}",
            })
    interaction_df = pd.DataFrame(interaction_rows)
    interaction_df.to_csv(os.path.join(table_dir, "SHAP_interaction_summary_Final.csv"), index=False)

    with open(os.path.join(table_dir, "locked_final_artifact_metadata_Final.json"), "w") as f:
        json.dump(json_native({k: v for k, v in art.items() if k not in {"preprocessor", "model", "calibrator"}}), f, indent=2, sort_keys=True)

    return {
        "artifact_path": artifact_path,
        "test_AP": float(average_precision_score(y_test, p_cal)),
        "test_ROC_AUC": float(roc_auc_score(y_test, p_cal)),
        "beeswarm_plot": bees,
        "bar_plot": bar,
        "heatmap_plot": heatmap,
        "n_threshold_plots": int(len(threshold_df)),
        "n_interaction_plots": int((interaction_df["status"] == "success").sum()) if not interaction_df.empty else 0,
    }


# ============================================================
# 8) Main
# ============================================================

def main():
    print("=" * 100)
    print("Step 2 LightGBM final locked dynamic PROM SHAP suite")
    print("=" * 100)

    source_dir = ensure_source_dir()
    print("Source run:", source_dir)

    test = get_test_dataframe(source_dir)
    artifact_path = find_final_artifact(source_dir)
    print("Final locked artifact:", artifact_path)

    manifest = pd.DataFrame([process_final_artifact(artifact_path, test)])
    manifest.to_csv(os.path.join(TABLE_DIR, "SHAP_suite_manifest_Final.csv"), index=False)

    table_dir = os.path.join(OUTPUT_DIR, "tables")
    threshold_path = os.path.join(table_dir, "SHAP_threshold_summary_Final.csv")
    interaction_path = os.path.join(table_dir, "SHAP_interaction_summary_Final.csv")
    importance_path = os.path.join(table_dir, "grouped_SHAP_importance_Final.csv")

    summary_xlsx = os.path.join(OUTPUT_DIR, "Step2_DynamicPROM_LightGBM_Final_SHAP_HeatmapFx_summary_Final.xlsx")
    with pd.ExcelWriter(summary_xlsx, engine="openpyxl") as writer:
        manifest.to_excel(writer, sheet_name="manifest", index=False)
        if os.path.exists(threshold_path):
            pd.read_csv(threshold_path).to_excel(writer, sheet_name="threshold_summary", index=False)
        if os.path.exists(interaction_path):
            pd.read_csv(interaction_path).to_excel(writer, sheet_name="interaction_summary", index=False)
        if os.path.exists(importance_path):
            pd.read_csv(importance_path).to_excel(writer, sheet_name="SHAP_importance", index=False)

    zip_out = os.path.join(BASE_DIR, "Step2_DynamicPROM_LightGBM_Final_SHAP_HeatmapFx_Final.zip")
    tmp_zip = zip_out + ".tmp"
    for p in [zip_out, tmp_zip]:
        if os.path.exists(p):
            os.remove(p)
    with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=ZIP_COMPRESSION_LEVEL) as z:
        for root, _, files in os.walk(OUTPUT_DIR):
            for fn in files:
                full = os.path.join(root, fn)
                rel = os.path.relpath(full, os.path.dirname(OUTPUT_DIR))
                z.write(full, rel)
    os.replace(tmp_zip, zip_out)

    print("\nDONE")
    print("Output folder:", OUTPUT_DIR)
    print("Summary Excel:", summary_xlsx)
    print("ZIP:", zip_out)

    if CREATE_COLAB_DOWNLOAD_LINK:
        try:
            from IPython.display import HTML, display
            display(HTML(f'<p><b>Step 2 SHAP suite output archive is ready.</b></p><p><a href="/files{zip_out}" download>Click here to download the ZIP archive</a></p><p>Path: <code>{zip_out}</code></p>'))
        except Exception as e:
            print("Download link display skipped:", repr(e))

    if AUTO_DOWNLOAD_ZIP:
        try:
            from google.colab import files
            files.download(zip_out)
        except Exception as e:
            print("Automatic download skipped:", repr(e))


if __name__ == "__main__":
    main()

#**Sensitivity Analysis Across Different Diagnoses**

In [ ]:
# -*- coding: utf-8 -*-
"""
Step 2 diagnosis-stratified sensitivity analysis with locked LightGBM models
============================================================================

This script evaluates diagnosis-stratified sensitivity analyses for the locked
Step 2 LightGBM models. It uses the final baseline-only and dynamic
PROM-expanded model artifacts, applies the locked calibration objects, and does
not refit, retune, recalibrate, or re-optimize thresholds.

Outputs are limited to subgroup-specific ΔAP and SHAP beeswarm plots for the
locked dynamic PROM-expanded model. For subgroup-specific SHAP visualizations
only, the subgroup-defining feature is omitted from the corresponding beeswarm
plot to avoid tautological display; the locked model and ΔAP calculations are
unchanged.
"""

# ============================================================
# Imports
# ============================================================

import os
import re
import sys
import json
import time
import zipfile
import shutil
import warnings
import subprocess
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    import lightgbm as lgb  # noqa: F401
    from lightgbm import LGBMClassifier  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
    import lightgbm as lgb  # noqa: F401
    from lightgbm import LGBMClassifier  # noqa: F401

try:
    import shap
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
    import shap

try:
    import joblib
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "joblib"])
    import joblib

try:
    import openpyxl  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl"])
    import openpyxl  # noqa: F401

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import average_precision_score
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# ============================================================
# Configuration
# ============================================================

BASE_DIR = "/content"
FALLBACK_DIR = "/mnt/data"

INPUT_CSV = os.path.join(BASE_DIR, "Step 2_ODI_Cohort.csv")
FALLBACK_INPUT_CSV = os.path.join(FALLBACK_DIR, "Step 2_ODI_Cohort.csv")

SOURCE_ARCHIVE_NAME = "Step2_DynamicPROM_LightGBM_TopConfig_Final.zip"
SOURCE_FOLDER_NAME = "Step2_DynamicPROM_LightGBM_TopConfig_Final"

OUTPUT_DIR = os.path.join(BASE_DIR, "Step2_ODI_Diagnosis_Sensitivity_DeltaAP_SHAP")
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
SHAP_DIR = os.path.join(OUTPUT_DIR, "shap")
AUDIT_DIR = os.path.join(OUTPUT_DIR, "audit")
for d in [OUTPUT_DIR, TABLE_DIR, PLOT_DIR, SHAP_DIR, AUDIT_DIR]:
    os.makedirs(d, exist_ok=True)

TARGET_COL = "final_reop_step2"
GROUP_COL = "PersonKey"
RANDOM_STATE = 20260530
N_BOOTSTRAPS = 2000

RELATIVE_PROM_MCID_CUTOFF = 0.30
PROM_CHANGE_RATE_COL = "ODI_change_rate"
RELATIVE_ODI_MCID_COL = "ODI_relative_MCID_binary"
DAYS_BETWEEN_PROM_COL = "days_between_PROMs"

SHAP_BEESWARM_MAX_DISPLAY = 25
ZIP_COMPRESSION_LEVEL = 1
AUTO_DOWNLOAD_ZIP = False
CREATE_COLAB_DOWNLOAD_LINK = True

SHAP_BEESWARM_BLUE = "#008BFB"
SHAP_BEESWARM_PINK = "#FF0051"
SHAP_COLOR_CMAP = LinearSegmentedColormap.from_list(
    "shap_blue_pink", [SHAP_BEESWARM_BLUE, SHAP_BEESWARM_PINK]
)

DIAGNOSIS_SUBGROUPS = [
    {"column": "finaldx_degenerative", "label": "Degenerative diagnosis"},
    {"column": "finaldx_radicular", "label": "Radiculopathy diagnosis"},
    {"column": "finaldx_stenosis", "label": "Spinal stenosis diagnosis"},
    {"column": "finaldx_deformity_instability", "label": "Deformity or instability diagnosis"},
    {"column": "finaldx_other_diagnosis", "label": "Other lumbar diagnosis"},
]

BASE_FEATURES = [
    "finaldx_degenerative", "finaldx_radicular", "finaldx_stenosis",
    "finaldx_deformity_instability", "finaldx_other_diagnosis",
    "age", "sex", "race", "ethnicity", "cancer_status",
    "chronic_pulmonary_disease", "congestive_heart_failure",
    "connective_tissue_rheumatic_disease", "diabetes_status",
    "myocardial_infarction", "renal_disease", "institution_type",
    "institution_size", "institution_region", "asa", "bmi", "payer_status",
    "alif_llif", "corpectomy", "discectomy", "foraminotomy",
    "instrumentation", "laminectomy_posterior_decompression",
    "pelvic_fixation", "plf", "tlif_plif", "other_lumbar_procedure",
    "number_operated_levels", "operative_region_extent", "PatTobaccoUse",
]

STEP2_DYNAMIC_PROM_FEATURES = [
    "preop_ODI", "postop_ODI", "ODI_change", PROM_CHANGE_RATE_COL,
    RELATIVE_ODI_MCID_COL, "postop_ODI_day",
]
FEATURES = BASE_FEATURES + STEP2_DYNAMIC_PROM_FEATURES

CONTINUOUS_FEATURES = [
    "age", "bmi", "preop_ODI", "postop_ODI", "ODI_change",
    PROM_CHANGE_RATE_COL, "postop_ODI_day",
]
BINARY_FEATURES = [
    "finaldx_degenerative", "finaldx_radicular", "finaldx_stenosis",
    "finaldx_deformity_instability", "finaldx_other_diagnosis", "sex",
    "ethnicity", "cancer_status", "chronic_pulmonary_disease",
    "congestive_heart_failure", "connective_tissue_rheumatic_disease",
    "myocardial_infarction", "renal_disease", "institution_type",
    "alif_llif", "corpectomy", "discectomy", "foraminotomy", "instrumentation",
    "laminectomy_posterior_decompression", "pelvic_fixation", "plf", "tlif_plif",
    "other_lumbar_procedure", "operative_region_extent", RELATIVE_ODI_MCID_COL,
]
ORDINAL_FEATURES = ["diabetes_status", "institution_size", "asa", "number_operated_levels"]
NOMINAL_FEATURES = ["race", "institution_region", "payer_status", "PatTobaccoUse"]

MISSING_STRINGS = {"", " ", "na", "n/a", "nan", "none", "null", ".", "missing", "<na>"}
BINARY_MAPS = {
    "sex": {"female": 0, "f": 0, "male": 1, "m": 1},
    "ethnicity": {"non-hispanic": 0, "non hispanic": 0, "hispanic": 1},
    "cancer_status": {"no cancer": 0, "no": 0, "none": 0, "any cancer": 1, "yes": 1, "cancer": 1},
    "institution_type": {"hospital": 0, "non-hospital": 1, "non hospital": 1, "nonhospital": 1},
    "operative_region_extent": {"lumbar only": 0, "extended_region_involvement": 1, "extended region involvement": 1, "extended": 1},
}
ORDINAL_MAPS = {
    "diabetes_status": {"no": 0, "none": 0, "0": 0, "without comp": 1, "without complication": 1, "without complications": 1, "1": 1, "with comp": 2, "with complication": 2, "with complications": 2, "2": 2},
    "institution_size": {"between 1-99 beds": 0, "1-99": 0, "between 100-399 beds": 1, "100-399": 1, ">= 400 beds": 2, ">=400 beds": 2, ">=400": 2, ">= 400": 2},
    "asa": {"1": 1, "i": 1, "2": 2, "ii": 2, "3": 3, "iii": 3, "4": 4, "iv": 4, ">=4": 4, ">= 4": 4, "5": 4, "v": 4},
    "number_operated_levels": {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4, ">=4": 4, ">= 4": 4, "5": 4, "6": 4, "7": 4, "8": 4, "9": 4, "10": 4},
}
PREFERRED_NOMINAL_LEVELS = {
    "race": ["White", "Black", "Other"],
    "institution_region": ["South", "North East", "West", "Midwest"],
    "payer_status": ["Medicare", "Commercial/Private", "Other", "Medicaid/Public/Government"],
    "PatTobaccoUse": ["Unknown/Not reported/Multiple", "Never", "Former", "Current"],
}
FEATURE_LABELS = {
    "finaldx_degenerative": "Degenerative diagnosis",
    "finaldx_radicular": "Radiculopathy diagnosis",
    "finaldx_stenosis": "Spinal stenosis diagnosis",
    "finaldx_deformity_instability": "Deformity or instability diagnosis",
    "finaldx_other_diagnosis": "Other lumbar diagnosis",
    "age": "Age",
    "sex": "Sex",
    "race": "Race",
    "ethnicity": "Ethnicity",
    "cancer_status": "Cancer status",
    "chronic_pulmonary_disease": "Chronic pulmonary disease",
    "congestive_heart_failure": "Congestive heart failure",
    "connective_tissue_rheumatic_disease": "Connective tissue/rheumatic disease",
    "diabetes_status": "Diabetes status",
    "myocardial_infarction": "Myocardial infarction",
    "renal_disease": "Renal disease",
    "institution_type": "Institution type",
    "institution_size": "Institution size",
    "institution_region": "Institution region",
    "asa": "ASA physical status",
    "bmi": "BMI",
    "payer_status": "Primary payer",
    "alif_llif": "Anterior/lateral lumbar interbody fusion",
    "corpectomy": "Corpectomy",
    "discectomy": "Discectomy",
    "foraminotomy": "Foraminotomy",
    "instrumentation": "Instrumentation",
    "laminectomy_posterior_decompression": "Posterior decompression or laminectomy",
    "pelvic_fixation": "Pelvic fixation",
    "plf": "Posterolateral fusion",
    "tlif_plif": "Posterior/transforaminal lumbar interbody fusion",
    "other_lumbar_procedure": "Other lumbar procedure",
    "number_operated_levels": "Number of operated levels",
    "operative_region_extent": "Operative region extent",
    "PatTobaccoUse": "Tobacco use",
    "preop_ODI": "Preoperative ODI",
    "postop_ODI": "Postoperative ODI",
    "ODI_change": "Change in ODI",
    PROM_CHANGE_RATE_COL: "ODI change rate",
    RELATIVE_ODI_MCID_COL: "Relative ODI MCID",
    "postop_ODI_day": "Timing of postoperative ODI assessment",
}


# ============================================================
# Preprocessor class for joblib artifact loading
# ============================================================

def clean_scalar(x: Any) -> Any:
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().replace("≥", ">=")
        return np.nan if s.lower() in MISSING_STRINGS else s
    return x


def norm_text(x: Any) -> Optional[str]:
    x = clean_scalar(x)
    if pd.isna(x):
        return None
    return str(x).strip().replace("≥", ">=").lower()


def to_binary_target(x: Any) -> float:
    sx = norm_text(x)
    if sx is None:
        return np.nan
    if sx in {"1", "1.0", "yes", "y", "true", "t"}:
        return 1.0
    if sx in {"0", "0.0", "no", "n", "false", "f"}:
        return 0.0
    try:
        v = float(sx)
        return float(v) if v in (0.0, 1.0) else np.nan
    except Exception:
        return np.nan


def pretty_feature_name(feature: str) -> str:
    return FEATURE_LABELS.get(feature, feature.replace("_", " "))


def safe_filename(x: str) -> str:
    x = str(x).replace("≥", "ge").replace("≤", "le").replace("<", "lt").replace(">", "gt")
    x = re.sub(r"[^A-Za-z0-9_.-]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x[:180] if x else "file"


def json_native(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(k): json_native(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_native(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj


@dataclass
class Step2Preprocessor:
    continuous_features: List[str]
    binary_features: List[str]
    ordinal_features: List[str]
    nominal_features: List[str]
    preferred_nominal_levels: Dict[str, List[str]]

    def __post_init__(self):
        self.continuous_imputer: Optional[SimpleImputer] = None
        self.discrete_imputer: Optional[SimpleImputer] = None
        self.nominal_imputer: Optional[SimpleImputer] = None
        self.onehot: Optional[OneHotEncoder] = None
        self.numeric_feature_names_: List[str] = []
        self.output_feature_names_: List[str] = []

    def _parse_binary(self, x: Any, feature: str) -> float:
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in BINARY_MAPS and sx in BINARY_MAPS[feature]:
            return float(BINARY_MAPS[feature][sx])
        if sx in {"1", "1.0", "yes", "y", "true", "t", "present", "positive", "performed"}:
            return 1.0
        if sx in {"0", "0.0", "no", "n", "false", "f", "absent", "negative", "not performed"}:
            return 0.0
        try:
            v = float(sx)
            return float(v) if v in (0.0, 1.0) else np.nan
        except Exception:
            return np.nan

    def _parse_ordinal(self, x: Any, feature: str) -> float:
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in ORDINAL_MAPS and sx in ORDINAL_MAPS[feature]:
            return float(ORDINAL_MAPS[feature][sx])
        try:
            v = float(sx)
            if feature == "asa":
                return float(min(max(int(round(v)), 1), 4))
            if feature == "number_operated_levels":
                return float(min(max(int(round(v)), 0), 4))
            if feature == "diabetes_status":
                return float(min(max(int(round(v)), 0), 2))
            if feature == "institution_size":
                return float(min(max(int(round(v)), 0), 2))
            return float(v)
        except Exception:
            return np.nan

    def _canonical_nominal(self, feature: str, x: Any) -> Any:
        x = clean_scalar(x)
        if pd.isna(x):
            return np.nan
        s = str(x).strip()
        sl = s.lower()
        if feature == "race":
            if sl == "white":
                return "White"
            if sl == "black":
                return "Black"
            return "Other"
        if feature == "institution_region":
            mapping = {"south": "South", "north east": "North East", "northeast": "North East", "north-east": "North East", "west": "West", "midwest": "Midwest", "mid west": "Midwest"}
            return mapping.get(sl, s)
        if feature == "payer_status":
            if sl == "medicare":
                return "Medicare"
            if sl in {"commercial/private", "commercial", "private", "commercial private"}:
                return "Commercial/Private"
            if sl in {"medicaid/public/government", "medicaid", "public", "government", "public/government"}:
                return "Medicaid/Public/Government"
            return "Other"
        if feature == "PatTobaccoUse":
            if sl == "never":
                return "Never"
            if sl == "former":
                return "Former"
            if sl == "current":
                return "Current"
            return "Unknown/Not reported/Multiple"
        return s

    def _make_parts(self, X: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        cont = pd.DataFrame(index=X.index)
        for c in self.continuous_features:
            cont[c] = pd.to_numeric(X[c].map(clean_scalar), errors="coerce")
        discrete = pd.DataFrame(index=X.index)
        for c in self.binary_features:
            discrete[c] = X[c].map(lambda z: self._parse_binary(z, c)).astype(float)
        for c in self.ordinal_features:
            discrete[c] = X[c].map(lambda z: self._parse_ordinal(z, c)).astype(float)
        nominal = pd.DataFrame(index=X.index)
        for c in self.nominal_features:
            nominal[c] = X[c].map(lambda z: self._canonical_nominal(c, z)).astype("object")
        return cont, discrete, nominal

    def fit(self, X: pd.DataFrame):
        cont, discrete, nominal = self._make_parts(X)
        self.continuous_imputer = SimpleImputer(strategy="median")
        self.discrete_imputer = SimpleImputer(strategy="most_frequent")
        self.nominal_imputer = SimpleImputer(strategy="constant", fill_value="Missing")
        self.continuous_imputer.fit(cont)
        self.discrete_imputer.fit(discrete)
        nominal_imp = pd.DataFrame(self.nominal_imputer.fit_transform(nominal), columns=self.nominal_features)
        categories = []
        for c in self.nominal_features:
            preferred = list(self.preferred_nominal_levels.get(c, []))
            observed = nominal_imp[c].astype(str).unique().tolist()
            categories.append(preferred + sorted([x for x in observed if x not in preferred]))
        try:
            self.onehot = OneHotEncoder(categories=categories, handle_unknown="ignore", sparse_output=False)
        except TypeError:
            self.onehot = OneHotEncoder(categories=categories, handle_unknown="ignore", sparse=False)
        self.onehot.fit(nominal_imp.astype(str))
        self.numeric_feature_names_ = self.continuous_features + self.binary_features + self.ordinal_features
        self.output_feature_names_ = list(self.numeric_feature_names_) + self.onehot.get_feature_names_out(self.nominal_features).tolist()
        return self

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        cont, discrete, nominal = self._make_parts(X)
        cont_imp = self.continuous_imputer.transform(cont)
        discrete_imp = self.discrete_imputer.transform(discrete)
        nominal_imp = pd.DataFrame(self.nominal_imputer.transform(nominal), columns=self.nominal_features)
        nominal_oh = self.onehot.transform(nominal_imp.astype(str))
        return np.concatenate([cont_imp, discrete_imp, nominal_oh], axis=1).astype(float)

    @property
    def output_feature_names(self) -> List[str]:
        return self.output_feature_names_


# ============================================================
# Source and data helpers
# ============================================================

def ensure_input_csv() -> str:
    if os.path.exists(INPUT_CSV):
        return INPUT_CSV
    if os.path.exists(FALLBACK_INPUT_CSV):
        shutil.copy2(FALLBACK_INPUT_CSV, INPUT_CSV)
        return INPUT_CSV
    raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")


def _source_has_required_structure(path: str) -> bool:
    if not os.path.isdir(path):
        return False
    return any(
        os.path.isdir(os.path.join(path, sub))
        for sub in ["dynamic_PROM_expanded", "baseline_only", "baseline"]
    )


def _archive_has_required_structure(zip_path: str) -> bool:
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            names = [n.replace(chr(92), "/") for n in zf.namelist()]
    except Exception:
        return False
    has_dynamic = any("dynamic_PROM_expanded/" in n and n.endswith("_lightgbm_pipeline_artifact_Final.joblib") for n in names)
    has_baseline = any(("baseline_only/" in n or "baseline/" in n) and n.endswith("_lightgbm_pipeline_artifact_Final.joblib") for n in names)
    has_split = any(n.endswith(".xlsx") for n in names)
    return bool(has_dynamic and has_baseline and has_split)


def _find_source_dir_inside(search_root: str) -> Optional[str]:
    if _source_has_required_structure(search_root):
        return search_root
    for root, dirs, _ in os.walk(search_root):
        depth = os.path.relpath(root, search_root).count(os.sep)
        if depth > 6:
            dirs[:] = []
            continue
        if _source_has_required_structure(root):
            return root
    return None


def ensure_source_dir() -> str:
    folder_candidates = [
        os.path.join(BASE_DIR, SOURCE_FOLDER_NAME),
        os.path.join(FALLBACK_DIR, SOURCE_FOLDER_NAME),
    ]
    for folder in folder_candidates:
        if _source_has_required_structure(folder):
            return folder

    archive_candidates = [
        os.path.join(BASE_DIR, SOURCE_ARCHIVE_NAME),
        os.path.join(FALLBACK_DIR, SOURCE_ARCHIVE_NAME),
    ]
    archive_path = None
    for candidate in archive_candidates:
        if os.path.exists(candidate) and _archive_has_required_structure(candidate):
            archive_path = candidate
            break

    if archive_path is None:
        raise FileNotFoundError(
            "Required Step 2 source archive/folder was not found or did not contain both locked baseline-only and dynamic PROM-expanded model artifacts. "
            f"Expected archive name: {SOURCE_ARCHIVE_NAME}"
        )

    extract_root = os.path.join(BASE_DIR, "_step2_diagnosis_sensitivity_source")
    if os.path.isdir(extract_root):
        shutil.rmtree(extract_root)
    os.makedirs(extract_root, exist_ok=True)
    print(f"Extracting source archive: {archive_path}")
    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(extract_root)

    detected = _find_source_dir_inside(extract_root)
    if detected is None:
        raise FileNotFoundError("The extracted archive did not contain the expected Step 2 model structure.")
    return detected


def find_locked_artifact(source_dir: str, model_kind: str) -> str:
    if model_kind == "dynamic":
        preferred_roots = ["dynamic_PROM_expanded"]
    elif model_kind == "baseline":
        preferred_roots = ["baseline_only", "baseline"]
    else:
        raise ValueError(model_kind)

    selected = []
    for root_name in preferred_roots:
        root = os.path.join(source_dir, root_name)
        if not os.path.isdir(root):
            continue
        for folder_name in os.listdir(root):
            folder_path = os.path.join(root, folder_name)
            if not os.path.isdir(folder_path) or not folder_name.startswith("top01_"):
                continue
            art_dir = os.path.join(folder_path, "model_artifacts")
            if not os.path.isdir(art_dir):
                continue
            for fn in os.listdir(art_dir):
                if fn.endswith("_lightgbm_pipeline_artifact_Final.joblib"):
                    selected.append(os.path.join(art_dir, fn))

    if len(selected) != 1:
        raise RuntimeError(f"Could not identify exactly one locked {model_kind} LightGBM artifact. Found {len(selected)}.")
    return selected[0]


def load_split_assignment(source_dir: str) -> pd.DataFrame:
    workbooks = []
    for root, _, files in os.walk(source_dir):
        for fn in files:
            if fn.endswith(".xlsx"):
                workbooks.append(os.path.join(root, fn))

    for xlsx in workbooks:
        try:
            xl = pd.ExcelFile(xlsx)
            if "split_assignment" not in xl.sheet_names:
                continue
            split = pd.read_excel(xlsx, sheet_name="split_assignment")
            split.columns = [str(c).strip() for c in split.columns]
            if GROUP_COL in split.columns and "split" in split.columns:
                return split[[GROUP_COL, "split"]].drop_duplicates()
        except Exception:
            continue
    raise FileNotFoundError("Could not find a workbook containing a valid split_assignment sheet.")


def derive_dynamic_prom_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    required = ["preop_ODI", "postop_ODI", DAYS_BETWEEN_PROM_COL, "postop_ODI_day", TARGET_COL, GROUP_COL]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"Missing required Step 2 columns: {missing}")

    out["preop_ODI"] = pd.to_numeric(out["preop_ODI"], errors="coerce")
    out["postop_ODI"] = pd.to_numeric(out["postop_ODI"], errors="coerce")
    out[DAYS_BETWEEN_PROM_COL] = pd.to_numeric(out[DAYS_BETWEEN_PROM_COL], errors="coerce")
    out["ODI_change"] = out["postop_ODI"] - out["preop_ODI"]
    out[PROM_CHANGE_RATE_COL] = out["ODI_change"] / out[DAYS_BETWEEN_PROM_COL]

    valid_rel = out["preop_ODI"].notna() & out["postop_ODI"].notna() & (out["preop_ODI"] > 0)
    zero_baseline_rel = out["preop_ODI"].notna() & out["postop_ODI"].notna() & out["preop_ODI"].eq(0)
    out[RELATIVE_ODI_MCID_COL] = np.nan
    out.loc[valid_rel, RELATIVE_ODI_MCID_COL] = (
        (out.loc[valid_rel, "preop_ODI"] - out.loc[valid_rel, "postop_ODI"])
        / out.loc[valid_rel, "preop_ODI"]
        >= RELATIVE_PROM_MCID_CUTOFF
    ).astype(int)
    out.loc[zero_baseline_rel, RELATIVE_ODI_MCID_COL] = 0.0

    out[TARGET_COL] = out[TARGET_COL].map(to_binary_target)
    eligibility = (
        out["preop_ODI"].notna()
        & out["postop_ODI"].notna()
        & out[DAYS_BETWEEN_PROM_COL].notna()
        & (out[DAYS_BETWEEN_PROM_COL] > 0)
        & out[TARGET_COL].isin([0.0, 1.0])
    )
    out = out.loc[eligibility].copy()
    out[TARGET_COL] = out[TARGET_COL].astype(int)
    return out.reset_index(drop=True)


def get_test_dataframe(source_dir: str) -> pd.DataFrame:
    csv_path = ensure_input_csv()
    raw = pd.read_csv(csv_path, low_memory=False)
    raw.columns = [str(c).strip() for c in raw.columns]
    work = derive_dynamic_prom_features(raw)
    split = load_split_assignment(source_dir)
    work = work.merge(split, on=GROUP_COL, how="left")
    if work["split"].isna().any():
        n_missing = int(work["split"].isna().sum())
        raise RuntimeError(f"{n_missing} rows could not be matched to split_assignment by PersonKey.")
    test = work[work["split"].astype(str).eq("test")].reset_index(drop=True)
    if test.empty:
        raise RuntimeError("No held-out test rows were found after applying split_assignment.")
    return test


def predict_locked_artifact(artifact_path: str, test: pd.DataFrame) -> Tuple[Dict[str, Any], np.ndarray]:
    artifact = joblib.load(artifact_path)
    features = list(artifact["features"])
    pre = artifact["preprocessor"]
    model = artifact["model"]
    calibrator = artifact["calibrator"]
    X = test[features].copy()
    Xenc = pre.transform(X)
    p_raw = model.predict_proba(Xenc)[:, 1]
    p_cal = np.clip(calibrator.predict(p_raw), 0, 1)
    return artifact, p_cal


# ============================================================
# Metrics and bootstrap
# ============================================================

def safe_ap(y: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    if len(y) == 0 or len(np.unique(y)) < 2:
        return np.nan
    return float(average_precision_score(y, p))


def paired_delta_ap_bootstrap(y: np.ndarray, p_base: np.ndarray, p_expanded: np.ndarray, n_boot: int, seed: int) -> Dict[str, Any]:
    y = np.asarray(y).astype(int)
    p_base = np.asarray(p_base).astype(float)
    p_expanded = np.asarray(p_expanded).astype(float)
    n = len(y)

    ap_base = safe_ap(y, p_base)
    ap_expanded = safe_ap(y, p_expanded)
    delta = ap_expanded - ap_base if np.isfinite(ap_base) and np.isfinite(ap_expanded) else np.nan

    if n == 0 or len(np.unique(y)) < 2:
        return {
            "AP_baseline": ap_base,
            "AP_dynamic_PROM_expanded": ap_expanded,
            "delta_AP": delta,
            "delta_AP_CI_lower": np.nan,
            "delta_AP_CI_upper": np.nan,
            "delta_AP_p_value": np.nan,
            "bootstrap_valid_iterations": 0,
        }

    rng = np.random.default_rng(seed)
    boot = []
    idx = np.arange(n)
    for _ in range(n_boot):
        b = rng.choice(idx, size=n, replace=True)
        if len(np.unique(y[b])) < 2:
            continue
        b_base = safe_ap(y[b], p_base[b])
        b_exp = safe_ap(y[b], p_expanded[b])
        if np.isfinite(b_base) and np.isfinite(b_exp):
            boot.append(b_exp - b_base)

    boot = np.asarray(boot, dtype=float)
    if len(boot) == 0:
        ci_l = ci_u = p_val = np.nan
    else:
        ci_l, ci_u = np.percentile(boot, [2.5, 97.5])
        p_val = 2.0 * min(np.mean(boot <= 0), np.mean(boot >= 0))
        p_val = min(float(p_val), 1.0)

    return {
        "AP_baseline": ap_base,
        "AP_dynamic_PROM_expanded": ap_expanded,
        "delta_AP": float(delta) if np.isfinite(delta) else np.nan,
        "delta_AP_CI_lower": float(ci_l) if np.isfinite(ci_l) else np.nan,
        "delta_AP_CI_upper": float(ci_u) if np.isfinite(ci_u) else np.nan,
        "delta_AP_p_value": p_val,
        "bootstrap_valid_iterations": int(len(boot)),
    }


def diagnosis_present(series: pd.Series) -> pd.Series:
    tmp = Step2Preprocessor([], [], [], [], {})
    parsed = series.map(lambda z: tmp._parse_binary(z, "diagnosis_indicator")).astype(float)
    fallback = pd.to_numeric(series.map(clean_scalar), errors="coerce")
    parsed = parsed.where(parsed.notna(), fallback)
    return parsed.eq(1.0)


# ============================================================
# SHAP helpers
# ============================================================

def raw_feature_numeric_values(X_raw: pd.DataFrame, feature: str) -> pd.Series:
    if feature in CONTINUOUS_FEATURES:
        s = pd.to_numeric(X_raw[feature].map(clean_scalar), errors="coerce")
    elif feature in BINARY_FEATURES:
        tmp_pre = Step2Preprocessor([], [], [], [], {})
        s = X_raw[feature].map(lambda z: tmp_pre._parse_binary(z, feature)).astype(float)
    elif feature in ORDINAL_FEATURES:
        tmp_pre = Step2Preprocessor([], [], [], [], {})
        s = X_raw[feature].map(lambda z: tmp_pre._parse_ordinal(z, feature)).astype(float)
    else:
        vals = X_raw[feature].map(lambda z: str(clean_scalar(z)) if not pd.isna(clean_scalar(z)) else "Missing")
        cats = {v: i for i, v in enumerate(sorted(vals.dropna().unique()))}
        s = vals.map(cats).astype(float)
    if s.isna().any():
        med = s.median(skipna=True)
        s = s.fillna(0.0 if pd.isna(med) else med)
    return s.astype(float)


def encoded_to_group_mapping(pre: Step2Preprocessor, features: List[str]) -> Dict[str, List[int]]:
    encoded_names = list(pre.output_feature_names)
    mapping: Dict[str, List[int]] = {}
    for feature in features:
        idx = []
        if feature in encoded_names:
            idx.append(encoded_names.index(feature))
        if feature in NOMINAL_FEATURES:
            prefix = feature + "_"
            idx.extend([i for i, name in enumerate(encoded_names) if name.startswith(prefix)])
        if idx:
            mapping[feature] = sorted(set(idx))
    return mapping


def compute_grouped_shap(artifact: Dict[str, Any], X_raw: pd.DataFrame, y_true: np.ndarray, p_calibrated: np.ndarray) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    features = list(artifact["features"])
    pre = artifact["preprocessor"]
    model = artifact["model"]

    X_model = X_raw[features].copy().reset_index(drop=True)
    y_true = np.asarray(y_true).astype(int)
    p_calibrated = np.asarray(p_calibrated).astype(float)

    Xenc = pre.transform(X_model)
    encoded_names = list(pre.output_feature_names)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(Xenc)
    shap_values = shap_values[-1] if isinstance(shap_values, list) else shap_values
    shap_values = np.asarray(shap_values)
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, -1]

    mapping = encoded_to_group_mapping(pre, features)
    grouped_values = {}
    grouped_data = {}
    importance_rows = []

    for raw_feature, ids in mapping.items():
        display = pretty_feature_name(raw_feature)
        grouped_values[display] = shap_values[:, ids].sum(axis=1)
        grouped_data[display] = raw_feature_numeric_values(X_model, raw_feature).values

    shap_df = pd.DataFrame(grouped_values)
    data_df = pd.DataFrame(grouped_data)
    total_abs = float(np.abs(shap_df.values).mean(axis=0).sum()) if shap_df.shape[1] else 0.0

    for raw_feature, ids in mapping.items():
        display = pretty_feature_name(raw_feature)
        mean_abs = float(np.abs(shap_df[display]).mean())
        importance_rows.append({
            "raw_feature": raw_feature,
            "display_feature": display,
            "mean_abs_SHAP": mean_abs,
            "percent_of_grouped_SHAP": 100.0 * mean_abs / total_abs if total_abs > 0 else np.nan,
            "n_encoded_columns_grouped": len(ids),
            "encoded_columns": "; ".join([encoded_names[i] for i in ids]),
        })

    imp_df = pd.DataFrame(importance_rows).sort_values("mean_abs_SHAP", ascending=False).reset_index(drop=True)
    shap_df.insert(0, "__row_id__", np.arange(len(shap_df)))
    shap_df.insert(1, "__y_true__", y_true)
    shap_df.insert(2, "__p_calibrated__", p_calibrated)
    return shap_df, data_df, imp_df


def subgroup_importance(shap_sub: pd.DataFrame, imp_all: pd.DataFrame) -> pd.DataFrame:
    feature_rows = []
    features = [f for f in imp_all["display_feature"].tolist() if f in shap_sub.columns]
    total_abs = float(np.abs(shap_sub[features].values).mean(axis=0).sum()) if features else 0.0
    raw_lookup = dict(zip(imp_all["display_feature"], imp_all["raw_feature"]))
    enc_lookup = dict(zip(imp_all["display_feature"], imp_all["encoded_columns"])) if "encoded_columns" in imp_all.columns else {}

    for display in features:
        mean_abs = float(np.abs(shap_sub[display]).mean())
        feature_rows.append({
            "raw_feature": raw_lookup.get(display, display),
            "display_feature": display,
            "mean_abs_SHAP": mean_abs,
            "percent_of_grouped_SHAP": 100.0 * mean_abs / total_abs if total_abs > 0 else np.nan,
            "encoded_columns": enc_lookup.get(display, ""),
        })
    return pd.DataFrame(feature_rows).sort_values("mean_abs_SHAP", ascending=False).reset_index(drop=True)


def omit_stratifying_feature_from_shap_visualization(
    shap_sub: pd.DataFrame,
    data_sub: pd.DataFrame,
    imp_sub: pd.DataFrame,
    subgroup_column: str,
    subgroup_label: str,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, List[str]]:
    """Remove the subgroup-defining feature from SHAP beeswarm display only.

    The locked model, calibrated predictions, ΔAP calculations, and stored full
    SHAP tables are unchanged. This function only prepares plotting-specific
    data so the beeswarm visualization does not include a tautological feature
    that is constant or nearly constant by construction within the subgroup.
    """
    candidates = []
    for value in [
        subgroup_label,
        pretty_feature_name(subgroup_column),
        subgroup_column,
        subgroup_column.replace("_", " "),
    ]:
        if value is not None and str(value).strip() and value not in candidates:
            candidates.append(str(value).strip())

    raw_col = "raw_feature"
    display_col = "display_feature"
    omit_display = set()
    if display_col in imp_sub.columns:
        omit_display.update(imp_sub.loc[imp_sub[display_col].isin(candidates), display_col].astype(str).tolist())
    if raw_col in imp_sub.columns:
        omit_display.update(
            imp_sub.loc[imp_sub[raw_col].astype(str).eq(str(subgroup_column)), display_col].astype(str).tolist()
            if display_col in imp_sub.columns else []
        )
    omit_display.update([c for c in candidates if c in shap_sub.columns or c in data_sub.columns])

    omitted = sorted([c for c in omit_display if c in shap_sub.columns or c in data_sub.columns or (display_col in imp_sub.columns and c in set(imp_sub[display_col].astype(str)))])
    if not omitted:
        return shap_sub, data_sub, imp_sub, []

    shap_plot = shap_sub.drop(columns=[c for c in omitted if c in shap_sub.columns], errors="ignore")
    data_plot = data_sub.drop(columns=[c for c in omitted if c in data_sub.columns], errors="ignore")
    imp_plot = imp_sub.copy()
    if display_col in imp_plot.columns:
        imp_plot = imp_plot.loc[~imp_plot[display_col].astype(str).isin(set(omitted))].copy()
    if raw_col in imp_plot.columns:
        imp_plot = imp_plot.loc[~imp_plot[raw_col].astype(str).eq(str(subgroup_column))].copy()
    imp_plot = imp_plot.reset_index(drop=True)
    return shap_plot, data_plot, imp_plot, omitted


def save_beeswarm(shap_df: pd.DataFrame, data_df: pd.DataFrame, imp_df: pd.DataFrame, path: str, title: str) -> None:
    ordered = [f for f in imp_df["display_feature"].tolist() if f in shap_df.columns and f in data_df.columns]
    ordered = ordered[: min(SHAP_BEESWARM_MAX_DISPLAY, len(ordered))]
    if not ordered:
        return

    plt.figure(figsize=(11.0, max(8.8, 0.32 * len(ordered) + 2)))
    try:
        shap.summary_plot(
            shap_df[ordered].values,
            features=data_df[ordered],
            feature_names=ordered,
            max_display=len(ordered),
            show=False,
            plot_size=None,
            color=SHAP_COLOR_CMAP,
        )
    except TypeError:
        shap.summary_plot(
            shap_df[ordered].values,
            features=data_df[ordered],
            feature_names=ordered,
            max_display=len(ordered),
            show=False,
            plot_size=None,
        )
    plt.title(title, fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()


# ============================================================
# Main analysis
# ============================================================

def make_zip(src_dir: str, zip_path: str) -> None:
    if os.path.exists(zip_path):
        os.remove(zip_path)
    tmp_zip = zip_path + ".tmp"
    if os.path.exists(tmp_zip):
        os.remove(tmp_zip)
    with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=ZIP_COMPRESSION_LEVEL) as zf:
        for root, _, files in os.walk(src_dir):
            for fn in files:
                full = os.path.join(root, fn)
                rel = os.path.relpath(full, os.path.dirname(src_dir))
                zf.write(full, rel)
    os.replace(tmp_zip, zip_path)


def main() -> None:
    print("=" * 100)
    print("Step 2 diagnosis-stratified sensitivity analysis with locked LightGBM models")
    print("=" * 100)

    source_dir = ensure_source_dir()
    print("Source run:", source_dir)

    test = get_test_dataframe(source_dir)
    y_test = test[TARGET_COL].astype(int).to_numpy()
    print(f"Held-out test set: rows={len(test):,}, events={int(y_test.sum()):,}, prevalence={np.mean(y_test):.4%}")

    baseline_artifact_path = find_locked_artifact(source_dir, "baseline")
    dynamic_artifact_path = find_locked_artifact(source_dir, "dynamic")
    print("Locked baseline artifact:", baseline_artifact_path)
    print("Locked dynamic PROM-expanded artifact:", dynamic_artifact_path)

    baseline_artifact, p_baseline = predict_locked_artifact(baseline_artifact_path, test)
    dynamic_artifact, p_dynamic = predict_locked_artifact(dynamic_artifact_path, test)

    predictions = test[[GROUP_COL, TARGET_COL] + [d["column"] for d in DIAGNOSIS_SUBGROUPS if d["column"] in test.columns]].copy()
    predictions["p_baseline_calibrated"] = p_baseline
    predictions["p_dynamic_PROM_expanded_calibrated"] = p_dynamic
    predictions.to_csv(os.path.join(TABLE_DIR, "Step2_ODI_paired_test_predictions_with_diagnosis.csv"), index=False)

    delta_rows = []
    audit_rows = []
    subgroup_masks: Dict[str, np.ndarray] = {}

    for i, subgroup in enumerate(DIAGNOSIS_SUBGROUPS, start=1):
        col = subgroup["column"]
        label = subgroup["label"]
        if col not in test.columns:
            audit_rows.append({"subgroup": label, "column": col, "status": "missing_column"})
            continue

        mask = diagnosis_present(test[col]).to_numpy()
        subgroup_masks[label] = mask
        y_sub = y_test[mask]
        p_base_sub = p_baseline[mask]
        p_dyn_sub = p_dynamic[mask]
        metrics = paired_delta_ap_bootstrap(
            y_sub,
            p_base_sub,
            p_dyn_sub,
            n_boot=N_BOOTSTRAPS,
            seed=RANDOM_STATE + 100 * i,
        )
        row = {
            "subgroup_type": "diagnosis",
            "subgroup_column": col,
            "subgroup_label": label,
            "definition": "diagnosis present",
            "n": int(mask.sum()),
            "events": int(y_sub.sum()) if len(y_sub) else 0,
            "prevalence": float(np.mean(y_sub)) if len(y_sub) else np.nan,
        }
        row.update(metrics)
        delta_rows.append(row)
        audit_rows.append({
            "subgroup": label,
            "column": col,
            "status": "evaluated",
            "n": int(mask.sum()),
            "events": int(y_sub.sum()) if len(y_sub) else 0,
            "non_events": int(len(y_sub) - int(y_sub.sum())) if len(y_sub) else 0,
        })

    delta_df = pd.DataFrame(delta_rows)
    delta_path = os.path.join(TABLE_DIR, "Step2_ODI_diagnosis_delta_AP.csv")
    delta_df.to_csv(delta_path, index=False)

    audit_df = pd.DataFrame(audit_rows)
    audit_df.to_csv(os.path.join(AUDIT_DIR, "Step2_ODI_diagnosis_sensitivity_audit.csv"), index=False)

    print("Computing SHAP values for the locked dynamic PROM-expanded model...")
    shap_df_all, data_df_all, imp_df_all = compute_grouped_shap(dynamic_artifact, test, y_test, p_dynamic)
    shap_df_all.to_csv(os.path.join(SHAP_DIR, "Step2_grouped_SHAP_values_all_test_Final.csv"), index=False)
    data_df_all.to_csv(os.path.join(SHAP_DIR, "Step2_grouped_SHAP_feature_values_all_test_Final.csv"), index=False)
    imp_df_all.to_csv(os.path.join(SHAP_DIR, "Step2_grouped_SHAP_importance_all_test_Final.csv"), index=False)

    shap_manifest_rows = []
    for subgroup in DIAGNOSIS_SUBGROUPS:
        label = subgroup["label"]
        col = subgroup["column"]
        if label not in subgroup_masks:
            continue
        mask = subgroup_masks[label]
        idx = np.where(mask)[0]
        if len(idx) == 0:
            continue

        safe_label = safe_filename(label)
        subgroup_dir = os.path.join(SHAP_DIR, safe_label)
        os.makedirs(subgroup_dir, exist_ok=True)

        shap_sub = shap_df_all.iloc[idx].reset_index(drop=True)
        data_sub = data_df_all.iloc[idx].reset_index(drop=True)
        imp_sub = subgroup_importance(shap_sub, imp_df_all)

        shap_sub.to_csv(os.path.join(subgroup_dir, f"grouped_SHAP_values_{safe_label}.csv"), index=False)
        data_sub.to_csv(os.path.join(subgroup_dir, f"grouped_SHAP_feature_values_{safe_label}.csv"), index=False)
        imp_sub.to_csv(os.path.join(subgroup_dir, f"grouped_SHAP_importance_{safe_label}.csv"), index=False)

        shap_plot, data_plot, imp_plot, omitted_features = omit_stratifying_feature_from_shap_visualization(
            shap_sub=shap_sub,
            data_sub=data_sub,
            imp_sub=imp_sub,
            subgroup_column=col,
            subgroup_label=label,
        )
        imp_plot.to_csv(os.path.join(subgroup_dir, f"grouped_SHAP_importance_{safe_label}_beeswarm_display.csv"), index=False)

        plot_path = os.path.join(PLOT_DIR, f"SHAP_beeswarm_{safe_label}.png")
        save_beeswarm(
            shap_df=shap_plot,
            data_df=data_plot,
            imp_df=imp_plot,
            path=plot_path,
            title=f"Step 2 diagnosis sensitivity: {label}",
        )
        shap_manifest_rows.append({
            "subgroup_type": "diagnosis",
            "subgroup_column": col,
            "subgroup_label": label,
            "n_shap_rows": int(len(shap_sub)),
            "events_in_shap_rows": int(shap_sub["__y_true__"].sum()) if "__y_true__" in shap_sub.columns else np.nan,
            "beeswarm_plot": plot_path,
            "stratifying_feature_omitted_from_beeswarm": "; ".join(omitted_features),
            "top_5_SHAP_features": "; ".join(imp_plot["display_feature"].head(5).tolist()),
        })

    shap_manifest = pd.DataFrame(shap_manifest_rows)
    shap_manifest.to_csv(os.path.join(TABLE_DIR, "Step2_ODI_diagnosis_SHAP_beeswarm_manifest.csv"), index=False)

    run_manifest = {
        "analysis": "Step 2 ODI diagnosis-stratified sensitivity analysis",
        "model": "locked LightGBM",
        "source_dir": source_dir,
        "baseline_artifact_path": baseline_artifact_path,
        "dynamic_artifact_path": dynamic_artifact_path,
        "refit": False,
        "retune": False,
        "recalibrate": False,
        "threshold_reoptimization": False,
        "reported_metrics": ["delta_AP"],
        "reported_plots": ["SHAP beeswarm"],
        "stratifying_feature_omitted_from_SHAP_beeswarm_only": True,
        "model_and_delta_AP_calculations_unchanged_by_SHAP_display_omission": True,
        "bootstrap_iterations_for_delta_AP_CI": N_BOOTSTRAPS,
        "subgroups": [
            {"column": s["column"], "label": s["label"], "definition": "diagnosis present"}
            for s in DIAGNOSIS_SUBGROUPS
        ],
        "test_rows": int(len(test)),
        "test_events": int(y_test.sum()),
        "test_prevalence": float(np.mean(y_test)),
        "timestamp_unix": time.time(),
    }
    with open(os.path.join(OUTPUT_DIR, "run_manifest.json"), "w") as f:
        json.dump(json_native(run_manifest), f, indent=2, sort_keys=True)

    summary_xlsx = os.path.join(OUTPUT_DIR, "Step2_ODI_Diagnosis_Sensitivity_DeltaAP_SHAP_summary.xlsx")
    with pd.ExcelWriter(summary_xlsx, engine="openpyxl") as writer:
        delta_df.to_excel(writer, sheet_name="delta_AP", index=False)
        shap_manifest.to_excel(writer, sheet_name="SHAP_beeswarm_manifest", index=False)
        audit_df.to_excel(writer, sheet_name="audit", index=False)
        imp_df_all.to_excel(writer, sheet_name="SHAP_importance_all_test", index=False)

    zip_path = os.path.join(BASE_DIR, "Step2_ODI_Diagnosis_Sensitivity_DeltaAP_SHAP.zip")
    make_zip(OUTPUT_DIR, zip_path)

    print("\nDONE")
    print("Output folder:", OUTPUT_DIR)
    print("Delta AP table:", delta_path)
    print("Summary Excel:", summary_xlsx)
    print("ZIP:", zip_path)

    if CREATE_COLAB_DOWNLOAD_LINK:
        try:
            from IPython.display import HTML, display
            display(HTML(
                f'<p><b>Step 2 diagnosis sensitivity output archive is ready.</b></p>'
                f'<p><a href="/files{zip_path}" download>Click here to download the ZIP archive</a></p>'
                f'<p>Path: <code>{zip_path}</code></p>'
            ))
        except Exception as e:
            print("Download link display skipped:", repr(e))

    if AUTO_DOWNLOAD_ZIP:
        try:
            from google.colab import files
            files.download(zip_path)
        except Exception as e:
            print("Automatic download skipped:", repr(e))


if __name__ == "__main__":
    main()

#**Sensitivity Analysis Across Different Procedures**

In [ ]:
# -*- coding: utf-8 -*-
"""
Step 2 procedure-stratified sensitivity analysis with locked LightGBM models
============================================================================

This script evaluates procedure-stratified sensitivity analyses for the locked
Step 2 LightGBM models. It uses the final baseline-only and dynamic
PROM-expanded model artifacts, applies the locked calibration objects, and does
not refit, retune, recalibrate, or re-optimize thresholds.

Outputs are limited to subgroup-specific ΔAP and SHAP beeswarm plots for the
locked dynamic PROM-expanded model. For subgroup-specific SHAP visualizations
only, the subgroup-defining feature is omitted from the corresponding beeswarm
plot to avoid tautological display; the locked model and ΔAP calculations are
unchanged.
"""

# ============================================================
# Imports
# ============================================================

import os
import re
import sys
import json
import time
import zipfile
import shutil
import warnings
import subprocess
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    import lightgbm as lgb  # noqa: F401
    from lightgbm import LGBMClassifier  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
    import lightgbm as lgb  # noqa: F401
    from lightgbm import LGBMClassifier  # noqa: F401

try:
    import shap
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
    import shap

try:
    import joblib
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "joblib"])
    import joblib

try:
    import openpyxl  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl"])
    import openpyxl  # noqa: F401

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import average_precision_score
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# ============================================================
# Configuration
# ============================================================

BASE_DIR = "/content"
FALLBACK_DIR = "/mnt/data"

INPUT_CSV = os.path.join(BASE_DIR, "Step 2_ODI_Cohort.csv")
FALLBACK_INPUT_CSV = os.path.join(FALLBACK_DIR, "Step 2_ODI_Cohort.csv")

SOURCE_ARCHIVE_NAME = "Step2_DynamicPROM_LightGBM_TopConfig_Final.zip"
SOURCE_FOLDER_NAME = "Step2_DynamicPROM_LightGBM_TopConfig_Final"

OUTPUT_DIR = os.path.join(BASE_DIR, "Step2_ODI_Procedure_Sensitivity_DeltaAP_SHAP")
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
SHAP_DIR = os.path.join(OUTPUT_DIR, "shap")
AUDIT_DIR = os.path.join(OUTPUT_DIR, "audit")
for d in [OUTPUT_DIR, TABLE_DIR, PLOT_DIR, SHAP_DIR, AUDIT_DIR]:
    os.makedirs(d, exist_ok=True)

TARGET_COL = "final_reop_step2"
GROUP_COL = "PersonKey"
RANDOM_STATE = 20260530
N_BOOTSTRAPS = 2000

RELATIVE_PROM_MCID_CUTOFF = 0.30
PROM_CHANGE_RATE_COL = "ODI_change_rate"
RELATIVE_ODI_MCID_COL = "ODI_relative_MCID_binary"
DAYS_BETWEEN_PROM_COL = "days_between_PROMs"

SHAP_BEESWARM_MAX_DISPLAY = 25
ZIP_COMPRESSION_LEVEL = 1
AUTO_DOWNLOAD_ZIP = True
CREATE_COLAB_DOWNLOAD_LINK = True

SHAP_BEESWARM_BLUE = "#008BFB"
SHAP_BEESWARM_PINK = "#FF0051"
SHAP_COLOR_CMAP = LinearSegmentedColormap.from_list(
    "shap_blue_pink", [SHAP_BEESWARM_BLUE, SHAP_BEESWARM_PINK]
)

PROCEDURE_SUBGROUPS = [
    {"column": "alif_llif", "label": "Anterior/lateral lumbar interbody fusion"},
    {"column": "corpectomy", "label": "Corpectomy"},
    {"column": "discectomy", "label": "Discectomy"},
    {"column": "foraminotomy", "label": "Foraminotomy"},
    {"column": "instrumentation", "label": "Instrumentation"},
    {"column": "laminectomy_posterior_decompression", "label": "Posterior decompression or laminectomy"},
    {"column": "pelvic_fixation", "label": "Pelvic fixation"},
    {"column": "plf", "label": "Posterolateral fusion"},
    {"column": "tlif_plif", "label": "Posterior/transforaminal lumbar interbody fusion"},
    {"column": "other_lumbar_procedure", "label": "Other lumbar procedure"},
]

BASE_FEATURES = [
    "finaldx_degenerative", "finaldx_radicular", "finaldx_stenosis",
    "finaldx_deformity_instability", "finaldx_other_diagnosis",
    "age", "sex", "race", "ethnicity", "cancer_status",
    "chronic_pulmonary_disease", "congestive_heart_failure",
    "connective_tissue_rheumatic_disease", "diabetes_status",
    "myocardial_infarction", "renal_disease", "institution_type",
    "institution_size", "institution_region", "asa", "bmi", "payer_status",
    "alif_llif", "corpectomy", "discectomy", "foraminotomy",
    "instrumentation", "laminectomy_posterior_decompression",
    "pelvic_fixation", "plf", "tlif_plif", "other_lumbar_procedure",
    "number_operated_levels", "operative_region_extent", "PatTobaccoUse",
]

STEP2_DYNAMIC_PROM_FEATURES = [
    "preop_ODI", "postop_ODI", "ODI_change", PROM_CHANGE_RATE_COL,
    RELATIVE_ODI_MCID_COL, "postop_ODI_day",
]
FEATURES = BASE_FEATURES + STEP2_DYNAMIC_PROM_FEATURES

CONTINUOUS_FEATURES = [
    "age", "bmi", "preop_ODI", "postop_ODI", "ODI_change",
    PROM_CHANGE_RATE_COL, "postop_ODI_day",
]
BINARY_FEATURES = [
    "finaldx_degenerative", "finaldx_radicular", "finaldx_stenosis",
    "finaldx_deformity_instability", "finaldx_other_diagnosis", "sex",
    "ethnicity", "cancer_status", "chronic_pulmonary_disease",
    "congestive_heart_failure", "connective_tissue_rheumatic_disease",
    "myocardial_infarction", "renal_disease", "institution_type",
    "alif_llif", "corpectomy", "discectomy", "foraminotomy", "instrumentation",
    "laminectomy_posterior_decompression", "pelvic_fixation", "plf", "tlif_plif",
    "other_lumbar_procedure", "operative_region_extent", RELATIVE_ODI_MCID_COL,
]
ORDINAL_FEATURES = ["diabetes_status", "institution_size", "asa", "number_operated_levels"]
NOMINAL_FEATURES = ["race", "institution_region", "payer_status", "PatTobaccoUse"]

MISSING_STRINGS = {"", " ", "na", "n/a", "nan", "none", "null", ".", "missing", "<na>"}
BINARY_MAPS = {
    "sex": {"female": 0, "f": 0, "male": 1, "m": 1},
    "ethnicity": {"non-hispanic": 0, "non hispanic": 0, "hispanic": 1},
    "cancer_status": {"no cancer": 0, "no": 0, "none": 0, "any cancer": 1, "yes": 1, "cancer": 1},
    "institution_type": {"hospital": 0, "non-hospital": 1, "non hospital": 1, "nonhospital": 1},
    "operative_region_extent": {"lumbar only": 0, "extended_region_involvement": 1, "extended region involvement": 1, "extended": 1},
}
ORDINAL_MAPS = {
    "diabetes_status": {"no": 0, "none": 0, "0": 0, "without comp": 1, "without complication": 1, "without complications": 1, "1": 1, "with comp": 2, "with complication": 2, "with complications": 2, "2": 2},
    "institution_size": {"between 1-99 beds": 0, "1-99": 0, "between 100-399 beds": 1, "100-399": 1, ">= 400 beds": 2, ">=400 beds": 2, ">=400": 2, ">= 400": 2},
    "asa": {"1": 1, "i": 1, "2": 2, "ii": 2, "3": 3, "iii": 3, "4": 4, "iv": 4, ">=4": 4, ">= 4": 4, "5": 4, "v": 4},
    "number_operated_levels": {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4, ">=4": 4, ">= 4": 4, "5": 4, "6": 4, "7": 4, "8": 4, "9": 4, "10": 4},
}
PREFERRED_NOMINAL_LEVELS = {
    "race": ["White", "Black", "Other"],
    "institution_region": ["South", "North East", "West", "Midwest"],
    "payer_status": ["Medicare", "Commercial/Private", "Other", "Medicaid/Public/Government"],
    "PatTobaccoUse": ["Unknown/Not reported/Multiple", "Never", "Former", "Current"],
}
FEATURE_LABELS = {
    "finaldx_degenerative": "Degenerative diagnosis",
    "finaldx_radicular": "Radiculopathy diagnosis",
    "finaldx_stenosis": "Spinal stenosis diagnosis",
    "finaldx_deformity_instability": "Deformity or instability diagnosis",
    "finaldx_other_diagnosis": "Other lumbar diagnosis",
    "age": "Age",
    "sex": "Sex",
    "race": "Race",
    "ethnicity": "Ethnicity",
    "cancer_status": "Cancer status",
    "chronic_pulmonary_disease": "Chronic pulmonary disease",
    "congestive_heart_failure": "Congestive heart failure",
    "connective_tissue_rheumatic_disease": "Connective tissue/rheumatic disease",
    "diabetes_status": "Diabetes status",
    "myocardial_infarction": "Myocardial infarction",
    "renal_disease": "Renal disease",
    "institution_type": "Institution type",
    "institution_size": "Institution size",
    "institution_region": "Institution region",
    "asa": "ASA physical status",
    "bmi": "BMI",
    "payer_status": "Primary payer",
    "alif_llif": "Anterior/lateral lumbar interbody fusion",
    "corpectomy": "Corpectomy",
    "discectomy": "Discectomy",
    "foraminotomy": "Foraminotomy",
    "instrumentation": "Instrumentation",
    "laminectomy_posterior_decompression": "Posterior decompression or laminectomy",
    "pelvic_fixation": "Pelvic fixation",
    "plf": "Posterolateral fusion",
    "tlif_plif": "Posterior/transforaminal lumbar interbody fusion",
    "other_lumbar_procedure": "Other lumbar procedure",
    "number_operated_levels": "Number of operated levels",
    "operative_region_extent": "Operative region extent",
    "PatTobaccoUse": "Tobacco use",
    "preop_ODI": "Preoperative ODI",
    "postop_ODI": "Postoperative ODI",
    "ODI_change": "Change in ODI",
    PROM_CHANGE_RATE_COL: "ODI change rate",
    RELATIVE_ODI_MCID_COL: "Relative ODI MCID",
    "postop_ODI_day": "Timing of postoperative ODI assessment",
}


# ============================================================
# Preprocessor class for joblib artifact loading
# ============================================================

def clean_scalar(x: Any) -> Any:
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().replace("≥", ">=")
        return np.nan if s.lower() in MISSING_STRINGS else s
    return x


def norm_text(x: Any) -> Optional[str]:
    x = clean_scalar(x)
    if pd.isna(x):
        return None
    return str(x).strip().replace("≥", ">=").lower()


def to_binary_target(x: Any) -> float:
    sx = norm_text(x)
    if sx is None:
        return np.nan
    if sx in {"1", "1.0", "yes", "y", "true", "t"}:
        return 1.0
    if sx in {"0", "0.0", "no", "n", "false", "f"}:
        return 0.0
    try:
        v = float(sx)
        return float(v) if v in (0.0, 1.0) else np.nan
    except Exception:
        return np.nan


def pretty_feature_name(feature: str) -> str:
    return FEATURE_LABELS.get(feature, feature.replace("_", " "))


def safe_filename(x: str) -> str:
    x = str(x).replace("≥", "ge").replace("≤", "le").replace("<", "lt").replace(">", "gt")
    x = re.sub(r"[^A-Za-z0-9_.-]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x[:180] if x else "file"


def json_native(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(k): json_native(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_native(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj


@dataclass
class Step2Preprocessor:
    continuous_features: List[str]
    binary_features: List[str]
    ordinal_features: List[str]
    nominal_features: List[str]
    preferred_nominal_levels: Dict[str, List[str]]

    def __post_init__(self):
        self.continuous_imputer: Optional[SimpleImputer] = None
        self.discrete_imputer: Optional[SimpleImputer] = None
        self.nominal_imputer: Optional[SimpleImputer] = None
        self.onehot: Optional[OneHotEncoder] = None
        self.numeric_feature_names_: List[str] = []
        self.output_feature_names_: List[str] = []

    def _parse_binary(self, x: Any, feature: str) -> float:
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in BINARY_MAPS and sx in BINARY_MAPS[feature]:
            return float(BINARY_MAPS[feature][sx])
        if sx in {"1", "1.0", "yes", "y", "true", "t", "present", "positive", "performed"}:
            return 1.0
        if sx in {"0", "0.0", "no", "n", "false", "f", "absent", "negative", "not performed"}:
            return 0.0
        try:
            v = float(sx)
            return float(v) if v in (0.0, 1.0) else np.nan
        except Exception:
            return np.nan

    def _parse_ordinal(self, x: Any, feature: str) -> float:
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in ORDINAL_MAPS and sx in ORDINAL_MAPS[feature]:
            return float(ORDINAL_MAPS[feature][sx])
        try:
            v = float(sx)
            if feature == "asa":
                return float(min(max(int(round(v)), 1), 4))
            if feature == "number_operated_levels":
                return float(min(max(int(round(v)), 0), 4))
            if feature == "diabetes_status":
                return float(min(max(int(round(v)), 0), 2))
            if feature == "institution_size":
                return float(min(max(int(round(v)), 0), 2))
            return float(v)
        except Exception:
            return np.nan

    def _canonical_nominal(self, feature: str, x: Any) -> Any:
        x = clean_scalar(x)
        if pd.isna(x):
            return np.nan
        s = str(x).strip()
        sl = s.lower()
        if feature == "race":
            if sl == "white":
                return "White"
            if sl == "black":
                return "Black"
            return "Other"
        if feature == "institution_region":
            mapping = {"south": "South", "north east": "North East", "northeast": "North East", "north-east": "North East", "west": "West", "midwest": "Midwest", "mid west": "Midwest"}
            return mapping.get(sl, s)
        if feature == "payer_status":
            if sl == "medicare":
                return "Medicare"
            if sl in {"commercial/private", "commercial", "private", "commercial private"}:
                return "Commercial/Private"
            if sl in {"medicaid/public/government", "medicaid", "public", "government", "public/government"}:
                return "Medicaid/Public/Government"
            return "Other"
        if feature == "PatTobaccoUse":
            if sl == "never":
                return "Never"
            if sl == "former":
                return "Former"
            if sl == "current":
                return "Current"
            return "Unknown/Not reported/Multiple"
        return s

    def _make_parts(self, X: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        cont = pd.DataFrame(index=X.index)
        for c in self.continuous_features:
            cont[c] = pd.to_numeric(X[c].map(clean_scalar), errors="coerce")
        discrete = pd.DataFrame(index=X.index)
        for c in self.binary_features:
            discrete[c] = X[c].map(lambda z: self._parse_binary(z, c)).astype(float)
        for c in self.ordinal_features:
            discrete[c] = X[c].map(lambda z: self._parse_ordinal(z, c)).astype(float)
        nominal = pd.DataFrame(index=X.index)
        for c in self.nominal_features:
            nominal[c] = X[c].map(lambda z: self._canonical_nominal(c, z)).astype("object")
        return cont, discrete, nominal

    def fit(self, X: pd.DataFrame):
        cont, discrete, nominal = self._make_parts(X)
        self.continuous_imputer = SimpleImputer(strategy="median")
        self.discrete_imputer = SimpleImputer(strategy="most_frequent")
        self.nominal_imputer = SimpleImputer(strategy="constant", fill_value="Missing")
        self.continuous_imputer.fit(cont)
        self.discrete_imputer.fit(discrete)
        nominal_imp = pd.DataFrame(self.nominal_imputer.fit_transform(nominal), columns=self.nominal_features)
        categories = []
        for c in self.nominal_features:
            preferred = list(self.preferred_nominal_levels.get(c, []))
            observed = nominal_imp[c].astype(str).unique().tolist()
            categories.append(preferred + sorted([x for x in observed if x not in preferred]))
        try:
            self.onehot = OneHotEncoder(categories=categories, handle_unknown="ignore", sparse_output=False)
        except TypeError:
            self.onehot = OneHotEncoder(categories=categories, handle_unknown="ignore", sparse=False)
        self.onehot.fit(nominal_imp.astype(str))
        self.numeric_feature_names_ = self.continuous_features + self.binary_features + self.ordinal_features
        self.output_feature_names_ = list(self.numeric_feature_names_) + self.onehot.get_feature_names_out(self.nominal_features).tolist()
        return self

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        cont, discrete, nominal = self._make_parts(X)
        cont_imp = self.continuous_imputer.transform(cont)
        discrete_imp = self.discrete_imputer.transform(discrete)
        nominal_imp = pd.DataFrame(self.nominal_imputer.transform(nominal), columns=self.nominal_features)
        nominal_oh = self.onehot.transform(nominal_imp.astype(str))
        return np.concatenate([cont_imp, discrete_imp, nominal_oh], axis=1).astype(float)

    @property
    def output_feature_names(self) -> List[str]:
        return self.output_feature_names_


# ============================================================
# Source and data helpers
# ============================================================

def ensure_input_csv() -> str:
    if os.path.exists(INPUT_CSV):
        return INPUT_CSV
    if os.path.exists(FALLBACK_INPUT_CSV):
        shutil.copy2(FALLBACK_INPUT_CSV, INPUT_CSV)
        return INPUT_CSV
    raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")


def _source_has_required_structure(path: str) -> bool:
    if not os.path.isdir(path):
        return False
    return any(
        os.path.isdir(os.path.join(path, sub))
        for sub in ["dynamic_PROM_expanded", "baseline_only", "baseline"]
    )


def _archive_has_required_structure(zip_path: str) -> bool:
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            names = [n.replace(chr(92), "/") for n in zf.namelist()]
    except Exception:
        return False
    has_dynamic = any("dynamic_PROM_expanded/" in n and n.endswith("_lightgbm_pipeline_artifact_Final.joblib") for n in names)
    has_baseline = any(("baseline_only/" in n or "baseline/" in n) and n.endswith("_lightgbm_pipeline_artifact_Final.joblib") for n in names)
    has_split = any(n.endswith(".xlsx") for n in names)
    return bool(has_dynamic and has_baseline and has_split)


def _find_source_dir_inside(search_root: str) -> Optional[str]:
    if _source_has_required_structure(search_root):
        return search_root
    for root, dirs, _ in os.walk(search_root):
        depth = os.path.relpath(root, search_root).count(os.sep)
        if depth > 6:
            dirs[:] = []
            continue
        if _source_has_required_structure(root):
            return root
    return None


def ensure_source_dir() -> str:
    folder_candidates = [
        os.path.join(BASE_DIR, SOURCE_FOLDER_NAME),
        os.path.join(FALLBACK_DIR, SOURCE_FOLDER_NAME),
    ]
    for folder in folder_candidates:
        if _source_has_required_structure(folder):
            return folder

    archive_candidates = [
        os.path.join(BASE_DIR, SOURCE_ARCHIVE_NAME),
        os.path.join(FALLBACK_DIR, SOURCE_ARCHIVE_NAME),
    ]
    archive_path = None
    for candidate in archive_candidates:
        if os.path.exists(candidate) and _archive_has_required_structure(candidate):
            archive_path = candidate
            break

    if archive_path is None:
        raise FileNotFoundError(
            "Required Step 2 source archive/folder was not found or did not contain both locked baseline-only and dynamic PROM-expanded model artifacts. "
            f"Expected archive name: {SOURCE_ARCHIVE_NAME}"
        )

    extract_root = os.path.join(BASE_DIR, "_step2_procedure_sensitivity_source")
    if os.path.isdir(extract_root):
        shutil.rmtree(extract_root)
    os.makedirs(extract_root, exist_ok=True)
    print(f"Extracting source archive: {archive_path}")
    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(extract_root)

    detected = _find_source_dir_inside(extract_root)
    if detected is None:
        raise FileNotFoundError("The extracted archive did not contain the expected Step 2 model structure.")
    return detected


def find_locked_artifact(source_dir: str, model_kind: str) -> str:
    if model_kind == "dynamic":
        preferred_roots = ["dynamic_PROM_expanded"]
    elif model_kind == "baseline":
        preferred_roots = ["baseline_only", "baseline"]
    else:
        raise ValueError(model_kind)

    selected = []
    for root_name in preferred_roots:
        root = os.path.join(source_dir, root_name)
        if not os.path.isdir(root):
            continue
        for folder_name in os.listdir(root):
            folder_path = os.path.join(root, folder_name)
            if not os.path.isdir(folder_path) or not folder_name.startswith("top01_"):
                continue
            art_dir = os.path.join(folder_path, "model_artifacts")
            if not os.path.isdir(art_dir):
                continue
            for fn in os.listdir(art_dir):
                if fn.endswith("_lightgbm_pipeline_artifact_Final.joblib"):
                    selected.append(os.path.join(art_dir, fn))

    if len(selected) != 1:
        raise RuntimeError(f"Could not identify exactly one locked {model_kind} LightGBM artifact. Found {len(selected)}.")
    return selected[0]


def load_split_assignment(source_dir: str) -> pd.DataFrame:
    workbooks = []
    for root, _, files in os.walk(source_dir):
        for fn in files:
            if fn.endswith(".xlsx"):
                workbooks.append(os.path.join(root, fn))

    for xlsx in workbooks:
        try:
            xl = pd.ExcelFile(xlsx)
            if "split_assignment" not in xl.sheet_names:
                continue
            split = pd.read_excel(xlsx, sheet_name="split_assignment")
            split.columns = [str(c).strip() for c in split.columns]
            if GROUP_COL in split.columns and "split" in split.columns:
                return split[[GROUP_COL, "split"]].drop_duplicates()
        except Exception:
            continue
    raise FileNotFoundError("Could not find a workbook containing a valid split_assignment sheet.")


def derive_dynamic_prom_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    required = ["preop_ODI", "postop_ODI", DAYS_BETWEEN_PROM_COL, "postop_ODI_day", TARGET_COL, GROUP_COL]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"Missing required Step 2 columns: {missing}")

    out["preop_ODI"] = pd.to_numeric(out["preop_ODI"], errors="coerce")
    out["postop_ODI"] = pd.to_numeric(out["postop_ODI"], errors="coerce")
    out[DAYS_BETWEEN_PROM_COL] = pd.to_numeric(out[DAYS_BETWEEN_PROM_COL], errors="coerce")
    out["ODI_change"] = out["postop_ODI"] - out["preop_ODI"]
    out[PROM_CHANGE_RATE_COL] = out["ODI_change"] / out[DAYS_BETWEEN_PROM_COL]

    valid_rel = out["preop_ODI"].notna() & out["postop_ODI"].notna() & (out["preop_ODI"] > 0)
    zero_baseline_rel = out["preop_ODI"].notna() & out["postop_ODI"].notna() & out["preop_ODI"].eq(0)
    out[RELATIVE_ODI_MCID_COL] = np.nan
    out.loc[valid_rel, RELATIVE_ODI_MCID_COL] = (
        (out.loc[valid_rel, "preop_ODI"] - out.loc[valid_rel, "postop_ODI"])
        / out.loc[valid_rel, "preop_ODI"]
        >= RELATIVE_PROM_MCID_CUTOFF
    ).astype(int)
    out.loc[zero_baseline_rel, RELATIVE_ODI_MCID_COL] = 0.0

    out[TARGET_COL] = out[TARGET_COL].map(to_binary_target)
    eligibility = (
        out["preop_ODI"].notna()
        & out["postop_ODI"].notna()
        & out[DAYS_BETWEEN_PROM_COL].notna()
        & (out[DAYS_BETWEEN_PROM_COL] > 0)
        & out[TARGET_COL].isin([0.0, 1.0])
    )
    out = out.loc[eligibility].copy()
    out[TARGET_COL] = out[TARGET_COL].astype(int)
    return out.reset_index(drop=True)


def get_test_dataframe(source_dir: str) -> pd.DataFrame:
    csv_path = ensure_input_csv()
    raw = pd.read_csv(csv_path, low_memory=False)
    raw.columns = [str(c).strip() for c in raw.columns]
    work = derive_dynamic_prom_features(raw)
    split = load_split_assignment(source_dir)
    work = work.merge(split, on=GROUP_COL, how="left")
    if work["split"].isna().any():
        n_missing = int(work["split"].isna().sum())
        raise RuntimeError(f"{n_missing} rows could not be matched to split_assignment by PersonKey.")
    test = work[work["split"].astype(str).eq("test")].reset_index(drop=True)
    if test.empty:
        raise RuntimeError("No held-out test rows were found after applying split_assignment.")
    return test


def predict_locked_artifact(artifact_path: str, test: pd.DataFrame) -> Tuple[Dict[str, Any], np.ndarray]:
    artifact = joblib.load(artifact_path)
    features = list(artifact["features"])
    pre = artifact["preprocessor"]
    model = artifact["model"]
    calibrator = artifact["calibrator"]
    X = test[features].copy()
    Xenc = pre.transform(X)
    p_raw = model.predict_proba(Xenc)[:, 1]
    p_cal = np.clip(calibrator.predict(p_raw), 0, 1)
    return artifact, p_cal


# ============================================================
# Metrics and bootstrap
# ============================================================

def safe_ap(y: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    if len(y) == 0 or len(np.unique(y)) < 2:
        return np.nan
    return float(average_precision_score(y, p))


def paired_delta_ap_bootstrap(y: np.ndarray, p_base: np.ndarray, p_expanded: np.ndarray, n_boot: int, seed: int) -> Dict[str, Any]:
    y = np.asarray(y).astype(int)
    p_base = np.asarray(p_base).astype(float)
    p_expanded = np.asarray(p_expanded).astype(float)
    n = len(y)

    ap_base = safe_ap(y, p_base)
    ap_expanded = safe_ap(y, p_expanded)
    delta = ap_expanded - ap_base if np.isfinite(ap_base) and np.isfinite(ap_expanded) else np.nan

    if n == 0 or len(np.unique(y)) < 2:
        return {
            "AP_baseline": ap_base,
            "AP_dynamic_PROM_expanded": ap_expanded,
            "delta_AP": delta,
            "delta_AP_CI_lower": np.nan,
            "delta_AP_CI_upper": np.nan,
            "delta_AP_p_value": np.nan,
            "bootstrap_valid_iterations": 0,
        }

    rng = np.random.default_rng(seed)
    boot = []
    idx = np.arange(n)
    for _ in range(n_boot):
        b = rng.choice(idx, size=n, replace=True)
        if len(np.unique(y[b])) < 2:
            continue
        b_base = safe_ap(y[b], p_base[b])
        b_exp = safe_ap(y[b], p_expanded[b])
        if np.isfinite(b_base) and np.isfinite(b_exp):
            boot.append(b_exp - b_base)

    boot = np.asarray(boot, dtype=float)
    if len(boot) == 0:
        ci_l = ci_u = p_val = np.nan
    else:
        ci_l, ci_u = np.percentile(boot, [2.5, 97.5])
        p_val = 2.0 * min(np.mean(boot <= 0), np.mean(boot >= 0))
        p_val = min(float(p_val), 1.0)

    return {
        "AP_baseline": ap_base,
        "AP_dynamic_PROM_expanded": ap_expanded,
        "delta_AP": float(delta) if np.isfinite(delta) else np.nan,
        "delta_AP_CI_lower": float(ci_l) if np.isfinite(ci_l) else np.nan,
        "delta_AP_CI_upper": float(ci_u) if np.isfinite(ci_u) else np.nan,
        "delta_AP_p_value": p_val,
        "bootstrap_valid_iterations": int(len(boot)),
    }


def procedure_present(series: pd.Series) -> pd.Series:
    tmp = Step2Preprocessor([], [], [], [], {})
    parsed = series.map(lambda z: tmp._parse_binary(z, "procedure_indicator")).astype(float)
    fallback = pd.to_numeric(series.map(clean_scalar), errors="coerce")
    parsed = parsed.where(parsed.notna(), fallback)
    return parsed.eq(1.0)


# ============================================================
# SHAP helpers
# ============================================================

def raw_feature_numeric_values(X_raw: pd.DataFrame, feature: str) -> pd.Series:
    if feature in CONTINUOUS_FEATURES:
        s = pd.to_numeric(X_raw[feature].map(clean_scalar), errors="coerce")
    elif feature in BINARY_FEATURES:
        tmp_pre = Step2Preprocessor([], [], [], [], {})
        s = X_raw[feature].map(lambda z: tmp_pre._parse_binary(z, feature)).astype(float)
    elif feature in ORDINAL_FEATURES:
        tmp_pre = Step2Preprocessor([], [], [], [], {})
        s = X_raw[feature].map(lambda z: tmp_pre._parse_ordinal(z, feature)).astype(float)
    else:
        vals = X_raw[feature].map(lambda z: str(clean_scalar(z)) if not pd.isna(clean_scalar(z)) else "Missing")
        cats = {v: i for i, v in enumerate(sorted(vals.dropna().unique()))}
        s = vals.map(cats).astype(float)
    if s.isna().any():
        med = s.median(skipna=True)
        s = s.fillna(0.0 if pd.isna(med) else med)
    return s.astype(float)


def encoded_to_group_mapping(pre: Step2Preprocessor, features: List[str]) -> Dict[str, List[int]]:
    encoded_names = list(pre.output_feature_names)
    mapping: Dict[str, List[int]] = {}
    for feature in features:
        idx = []
        if feature in encoded_names:
            idx.append(encoded_names.index(feature))
        if feature in NOMINAL_FEATURES:
            prefix = feature + "_"
            idx.extend([i for i, name in enumerate(encoded_names) if name.startswith(prefix)])
        if idx:
            mapping[feature] = sorted(set(idx))
    return mapping


def compute_grouped_shap(artifact: Dict[str, Any], X_raw: pd.DataFrame, y_true: np.ndarray, p_calibrated: np.ndarray) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    features = list(artifact["features"])
    pre = artifact["preprocessor"]
    model = artifact["model"]

    X_model = X_raw[features].copy().reset_index(drop=True)
    y_true = np.asarray(y_true).astype(int)
    p_calibrated = np.asarray(p_calibrated).astype(float)

    Xenc = pre.transform(X_model)
    encoded_names = list(pre.output_feature_names)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(Xenc)
    shap_values = shap_values[-1] if isinstance(shap_values, list) else shap_values
    shap_values = np.asarray(shap_values)
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, -1]

    mapping = encoded_to_group_mapping(pre, features)
    grouped_values = {}
    grouped_data = {}
    importance_rows = []

    for raw_feature, ids in mapping.items():
        display = pretty_feature_name(raw_feature)
        grouped_values[display] = shap_values[:, ids].sum(axis=1)
        grouped_data[display] = raw_feature_numeric_values(X_model, raw_feature).values

    shap_df = pd.DataFrame(grouped_values)
    data_df = pd.DataFrame(grouped_data)
    total_abs = float(np.abs(shap_df.values).mean(axis=0).sum()) if shap_df.shape[1] else 0.0

    for raw_feature, ids in mapping.items():
        display = pretty_feature_name(raw_feature)
        mean_abs = float(np.abs(shap_df[display]).mean())
        importance_rows.append({
            "raw_feature": raw_feature,
            "display_feature": display,
            "mean_abs_SHAP": mean_abs,
            "percent_of_grouped_SHAP": 100.0 * mean_abs / total_abs if total_abs > 0 else np.nan,
            "n_encoded_columns_grouped": len(ids),
            "encoded_columns": "; ".join([encoded_names[i] for i in ids]),
        })

    imp_df = pd.DataFrame(importance_rows).sort_values("mean_abs_SHAP", ascending=False).reset_index(drop=True)
    shap_df.insert(0, "__row_id__", np.arange(len(shap_df)))
    shap_df.insert(1, "__y_true__", y_true)
    shap_df.insert(2, "__p_calibrated__", p_calibrated)
    return shap_df, data_df, imp_df


def subgroup_importance(shap_sub: pd.DataFrame, imp_all: pd.DataFrame) -> pd.DataFrame:
    feature_rows = []
    features = [f for f in imp_all["display_feature"].tolist() if f in shap_sub.columns]
    total_abs = float(np.abs(shap_sub[features].values).mean(axis=0).sum()) if features else 0.0
    raw_lookup = dict(zip(imp_all["display_feature"], imp_all["raw_feature"]))
    enc_lookup = dict(zip(imp_all["display_feature"], imp_all["encoded_columns"])) if "encoded_columns" in imp_all.columns else {}

    for display in features:
        mean_abs = float(np.abs(shap_sub[display]).mean())
        feature_rows.append({
            "raw_feature": raw_lookup.get(display, display),
            "display_feature": display,
            "mean_abs_SHAP": mean_abs,
            "percent_of_grouped_SHAP": 100.0 * mean_abs / total_abs if total_abs > 0 else np.nan,
            "encoded_columns": enc_lookup.get(display, ""),
        })
    return pd.DataFrame(feature_rows).sort_values("mean_abs_SHAP", ascending=False).reset_index(drop=True)


def omit_stratifying_feature_from_shap_visualization(
    shap_sub: pd.DataFrame,
    data_sub: pd.DataFrame,
    imp_sub: pd.DataFrame,
    subgroup_column: str,
    subgroup_label: str,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, List[str]]:
    """Remove the subgroup-defining feature from SHAP beeswarm display only.

    The locked model, calibrated predictions, ΔAP calculations, and stored full
    SHAP tables are unchanged. This function only prepares plotting-specific
    data so the beeswarm visualization does not include a tautological feature
    that is constant or nearly constant by construction within the subgroup.
    """
    candidates = []
    for value in [
        subgroup_label,
        pretty_feature_name(subgroup_column),
        subgroup_column,
        subgroup_column.replace("_", " "),
    ]:
        if value is not None and str(value).strip() and value not in candidates:
            candidates.append(str(value).strip())

    raw_col = "raw_feature"
    display_col = "display_feature"
    omit_display = set()
    if display_col in imp_sub.columns:
        omit_display.update(imp_sub.loc[imp_sub[display_col].isin(candidates), display_col].astype(str).tolist())
    if raw_col in imp_sub.columns:
        omit_display.update(
            imp_sub.loc[imp_sub[raw_col].astype(str).eq(str(subgroup_column)), display_col].astype(str).tolist()
            if display_col in imp_sub.columns else []
        )
    omit_display.update([c for c in candidates if c in shap_sub.columns or c in data_sub.columns])

    omitted = sorted([c for c in omit_display if c in shap_sub.columns or c in data_sub.columns or (display_col in imp_sub.columns and c in set(imp_sub[display_col].astype(str)))])
    if not omitted:
        return shap_sub, data_sub, imp_sub, []

    shap_plot = shap_sub.drop(columns=[c for c in omitted if c in shap_sub.columns], errors="ignore")
    data_plot = data_sub.drop(columns=[c for c in omitted if c in data_sub.columns], errors="ignore")
    imp_plot = imp_sub.copy()
    if display_col in imp_plot.columns:
        imp_plot = imp_plot.loc[~imp_plot[display_col].astype(str).isin(set(omitted))].copy()
    if raw_col in imp_plot.columns:
        imp_plot = imp_plot.loc[~imp_plot[raw_col].astype(str).eq(str(subgroup_column))].copy()
    imp_plot = imp_plot.reset_index(drop=True)
    return shap_plot, data_plot, imp_plot, omitted


def save_beeswarm(shap_df: pd.DataFrame, data_df: pd.DataFrame, imp_df: pd.DataFrame, path: str, title: str) -> None:
    ordered = [f for f in imp_df["display_feature"].tolist() if f in shap_df.columns and f in data_df.columns]
    ordered = ordered[: min(SHAP_BEESWARM_MAX_DISPLAY, len(ordered))]
    if not ordered:
        return

    plt.figure(figsize=(11.0, max(8.8, 0.32 * len(ordered) + 2)))
    try:
        shap.summary_plot(
            shap_df[ordered].values,
            features=data_df[ordered],
            feature_names=ordered,
            max_display=len(ordered),
            show=False,
            plot_size=None,
            color=SHAP_COLOR_CMAP,
        )
    except TypeError:
        shap.summary_plot(
            shap_df[ordered].values,
            features=data_df[ordered],
            feature_names=ordered,
            max_display=len(ordered),
            show=False,
            plot_size=None,
        )
    plt.title(title, fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()


# ============================================================
# Main analysis
# ============================================================

def make_zip(src_dir: str, zip_path: str) -> None:
    if os.path.exists(zip_path):
        os.remove(zip_path)
    tmp_zip = zip_path + ".tmp"
    if os.path.exists(tmp_zip):
        os.remove(tmp_zip)
    with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=ZIP_COMPRESSION_LEVEL) as zf:
        for root, _, files in os.walk(src_dir):
            for fn in files:
                full = os.path.join(root, fn)
                rel = os.path.relpath(full, os.path.dirname(src_dir))
                zf.write(full, rel)
    os.replace(tmp_zip, zip_path)


def main() -> None:
    print("=" * 100)
    print("Step 2 procedure-stratified sensitivity analysis with locked LightGBM models")
    print("=" * 100)

    source_dir = ensure_source_dir()
    print("Source run:", source_dir)

    test = get_test_dataframe(source_dir)
    y_test = test[TARGET_COL].astype(int).to_numpy()
    print(f"Held-out test set: rows={len(test):,}, events={int(y_test.sum()):,}, prevalence={np.mean(y_test):.4%}")

    baseline_artifact_path = find_locked_artifact(source_dir, "baseline")
    dynamic_artifact_path = find_locked_artifact(source_dir, "dynamic")
    print("Locked baseline artifact:", baseline_artifact_path)
    print("Locked dynamic PROM-expanded artifact:", dynamic_artifact_path)

    baseline_artifact, p_baseline = predict_locked_artifact(baseline_artifact_path, test)
    dynamic_artifact, p_dynamic = predict_locked_artifact(dynamic_artifact_path, test)

    predictions = test[[GROUP_COL, TARGET_COL] + [d["column"] for d in PROCEDURE_SUBGROUPS if d["column"] in test.columns]].copy()
    predictions["p_baseline_calibrated"] = p_baseline
    predictions["p_dynamic_PROM_expanded_calibrated"] = p_dynamic
    predictions.to_csv(os.path.join(TABLE_DIR, "Step2_ODI_paired_test_predictions_with_procedure.csv"), index=False)

    delta_rows = []
    audit_rows = []
    subgroup_masks: Dict[str, np.ndarray] = {}

    for i, subgroup in enumerate(PROCEDURE_SUBGROUPS, start=1):
        col = subgroup["column"]
        label = subgroup["label"]
        if col not in test.columns:
            audit_rows.append({"subgroup": label, "column": col, "status": "missing_column"})
            continue

        mask = procedure_present(test[col]).to_numpy()
        subgroup_masks[label] = mask
        y_sub = y_test[mask]
        p_base_sub = p_baseline[mask]
        p_dyn_sub = p_dynamic[mask]
        metrics = paired_delta_ap_bootstrap(
            y_sub,
            p_base_sub,
            p_dyn_sub,
            n_boot=N_BOOTSTRAPS,
            seed=RANDOM_STATE + 100 * i,
        )
        row = {
            "subgroup_type": "procedure",
            "subgroup_column": col,
            "subgroup_label": label,
            "definition": "procedure performed",
            "n": int(mask.sum()),
            "events": int(y_sub.sum()) if len(y_sub) else 0,
            "prevalence": float(np.mean(y_sub)) if len(y_sub) else np.nan,
        }
        row.update(metrics)
        delta_rows.append(row)
        audit_rows.append({
            "subgroup": label,
            "column": col,
            "status": "evaluated",
            "n": int(mask.sum()),
            "events": int(y_sub.sum()) if len(y_sub) else 0,
            "non_events": int(len(y_sub) - int(y_sub.sum())) if len(y_sub) else 0,
        })

    delta_df = pd.DataFrame(delta_rows)
    delta_path = os.path.join(TABLE_DIR, "Step2_ODI_procedure_delta_AP.csv")
    delta_df.to_csv(delta_path, index=False)

    audit_df = pd.DataFrame(audit_rows)
    audit_df.to_csv(os.path.join(AUDIT_DIR, "Step2_ODI_procedure_sensitivity_audit.csv"), index=False)

    print("Computing SHAP values for the locked dynamic PROM-expanded model...")
    shap_df_all, data_df_all, imp_df_all = compute_grouped_shap(dynamic_artifact, test, y_test, p_dynamic)
    shap_df_all.to_csv(os.path.join(SHAP_DIR, "Step2_grouped_SHAP_values_all_test_Final.csv"), index=False)
    data_df_all.to_csv(os.path.join(SHAP_DIR, "Step2_grouped_SHAP_feature_values_all_test_Final.csv"), index=False)
    imp_df_all.to_csv(os.path.join(SHAP_DIR, "Step2_grouped_SHAP_importance_all_test_Final.csv"), index=False)

    shap_manifest_rows = []
    for subgroup in PROCEDURE_SUBGROUPS:
        label = subgroup["label"]
        col = subgroup["column"]
        if label not in subgroup_masks:
            continue
        mask = subgroup_masks[label]
        idx = np.where(mask)[0]
        if len(idx) == 0:
            continue

        safe_label = safe_filename(label)
        subgroup_dir = os.path.join(SHAP_DIR, safe_label)
        os.makedirs(subgroup_dir, exist_ok=True)

        shap_sub = shap_df_all.iloc[idx].reset_index(drop=True)
        data_sub = data_df_all.iloc[idx].reset_index(drop=True)
        imp_sub = subgroup_importance(shap_sub, imp_df_all)

        shap_sub.to_csv(os.path.join(subgroup_dir, f"grouped_SHAP_values_{safe_label}.csv"), index=False)
        data_sub.to_csv(os.path.join(subgroup_dir, f"grouped_SHAP_feature_values_{safe_label}.csv"), index=False)
        imp_sub.to_csv(os.path.join(subgroup_dir, f"grouped_SHAP_importance_{safe_label}.csv"), index=False)

        shap_plot, data_plot, imp_plot, omitted_features = omit_stratifying_feature_from_shap_visualization(
            shap_sub=shap_sub,
            data_sub=data_sub,
            imp_sub=imp_sub,
            subgroup_column=col,
            subgroup_label=label,
        )
        imp_plot.to_csv(os.path.join(subgroup_dir, f"grouped_SHAP_importance_{safe_label}_beeswarm_display.csv"), index=False)

        plot_path = os.path.join(PLOT_DIR, f"SHAP_beeswarm_{safe_label}.png")
        save_beeswarm(
            shap_df=shap_plot,
            data_df=data_plot,
            imp_df=imp_plot,
            path=plot_path,
            title=f"Step 2 procedure sensitivity: {label}",
        )
        shap_manifest_rows.append({
            "subgroup_type": "procedure",
            "subgroup_column": col,
            "subgroup_label": label,
            "n_shap_rows": int(len(shap_sub)),
            "events_in_shap_rows": int(shap_sub["__y_true__"].sum()) if "__y_true__" in shap_sub.columns else np.nan,
            "beeswarm_plot": plot_path,
            "stratifying_feature_omitted_from_beeswarm": "; ".join(omitted_features),
            "top_5_SHAP_features": "; ".join(imp_plot["display_feature"].head(5).tolist()),
        })

    shap_manifest = pd.DataFrame(shap_manifest_rows)
    shap_manifest.to_csv(os.path.join(TABLE_DIR, "Step2_ODI_procedure_SHAP_beeswarm_manifest.csv"), index=False)

    run_manifest = {
        "analysis": "Step 2 ODI procedure-stratified sensitivity analysis",
        "model": "locked LightGBM",
        "source_dir": source_dir,
        "baseline_artifact_path": baseline_artifact_path,
        "dynamic_artifact_path": dynamic_artifact_path,
        "refit": False,
        "retune": False,
        "recalibrate": False,
        "threshold_reoptimization": False,
        "reported_metrics": ["delta_AP"],
        "reported_plots": ["SHAP beeswarm"],
        "stratifying_feature_omitted_from_SHAP_beeswarm_only": True,
        "model_and_delta_AP_calculations_unchanged_by_SHAP_display_omission": True,
        "bootstrap_iterations_for_delta_AP_CI": N_BOOTSTRAPS,
        "subgroups": [
            {"column": s["column"], "label": s["label"], "definition": "procedure performed"}
            for s in PROCEDURE_SUBGROUPS
        ],
        "test_rows": int(len(test)),
        "test_events": int(y_test.sum()),
        "test_prevalence": float(np.mean(y_test)),
        "timestamp_unix": time.time(),
    }
    with open(os.path.join(OUTPUT_DIR, "run_manifest.json"), "w") as f:
        json.dump(json_native(run_manifest), f, indent=2, sort_keys=True)

    summary_xlsx = os.path.join(OUTPUT_DIR, "Step2_ODI_Procedure_Sensitivity_DeltaAP_SHAP_summary.xlsx")
    with pd.ExcelWriter(summary_xlsx, engine="openpyxl") as writer:
        delta_df.to_excel(writer, sheet_name="delta_AP", index=False)
        shap_manifest.to_excel(writer, sheet_name="SHAP_beeswarm_manifest", index=False)
        audit_df.to_excel(writer, sheet_name="audit", index=False)
        imp_df_all.to_excel(writer, sheet_name="SHAP_importance_all_test", index=False)

    zip_path = os.path.join(BASE_DIR, "Step2_ODI_Procedure_Sensitivity_DeltaAP_SHAP.zip")
    make_zip(OUTPUT_DIR, zip_path)

    print("\nDONE")
    print("Output folder:", OUTPUT_DIR)
    print("Delta AP table:", delta_path)
    print("Summary Excel:", summary_xlsx)
    print("ZIP:", zip_path)

    if CREATE_COLAB_DOWNLOAD_LINK:
        try:
            from IPython.display import HTML, display
            display(HTML(
                f'<p><b>Step 2 procedure sensitivity output archive is ready.</b></p>'
                f'<p><a href="/files{zip_path}" download>Click here to download the ZIP archive</a></p>'
                f'<p>Path: <code>{zip_path}</code></p>'
            ))
        except Exception as e:
            print("Download link display skipped:", repr(e))

    if AUTO_DOWNLOAD_ZIP:
        try:
            from google.colab import files
            files.download(zip_path)
        except Exception as e:
            print("Automatic download skipped:", repr(e))


if __name__ == "__main__":
    main()

#**Hospital-stratified Internal Validation**

In [ ]:
# -*- coding: utf-8 -*-
"""
Step 2 hospital-level internal validation with locked LightGBM models
====================================================================

This script evaluates hospital-level internal validation for the locked Step 2
LightGBM models. It uses the final baseline-only and dynamic PROM-expanded
model artifacts, applies the locked calibration objects, and does not refit,
retune, recalibrate, or re-optimize thresholds.

Outputs are limited to hospital-stratum-specific ΔAP and SHAP beeswarm plots
for the locked dynamic PROM-expanded model.
"""

# ============================================================
# Imports
# ============================================================

import os
import re
import sys
import json
import time
import zipfile
import shutil
import warnings
import subprocess
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    import lightgbm as lgb  # noqa: F401
    from lightgbm import LGBMClassifier  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
    import lightgbm as lgb  # noqa: F401
    from lightgbm import LGBMClassifier  # noqa: F401

try:
    import shap
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
    import shap

try:
    import joblib
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "joblib"])
    import joblib

try:
    import openpyxl  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl"])
    import openpyxl  # noqa: F401

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import average_precision_score
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# ============================================================
# Configuration
# ============================================================

BASE_DIR = "/content"
FALLBACK_DIR = "/mnt/data"

INPUT_CSV = os.path.join(BASE_DIR, "Step 2_ODI_Cohort.csv")
FALLBACK_INPUT_CSV = os.path.join(FALLBACK_DIR, "Step 2_ODI_Cohort.csv")

SOURCE_ARCHIVE_NAME = "Step2_DynamicPROM_LightGBM_TopConfig_Final.zip"
SOURCE_FOLDER_NAME = "Step2_DynamicPROM_LightGBM_TopConfig_Final"

OUTPUT_DIR = os.path.join(BASE_DIR, "Step2_ODI_Hospital_InternalValidation_DeltaAP_SHAP")
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
SHAP_DIR = os.path.join(OUTPUT_DIR, "shap")
AUDIT_DIR = os.path.join(OUTPUT_DIR, "audit")
for d in [OUTPUT_DIR, TABLE_DIR, PLOT_DIR, SHAP_DIR, AUDIT_DIR]:
    os.makedirs(d, exist_ok=True)

TARGET_COL = "final_reop_step2"
GROUP_COL = "PersonKey"
RANDOM_STATE = 20260530
N_BOOTSTRAPS = 2000

RELATIVE_PROM_MCID_CUTOFF = 0.30
PROM_CHANGE_RATE_COL = "ODI_change_rate"
RELATIVE_ODI_MCID_COL = "ODI_relative_MCID_binary"
DAYS_BETWEEN_PROM_COL = "days_between_PROMs"

SHAP_BEESWARM_MAX_DISPLAY = 25
ZIP_COMPRESSION_LEVEL = 1
AUTO_DOWNLOAD_ZIP = True
CREATE_COLAB_DOWNLOAD_LINK = True

SHAP_BEESWARM_BLUE = "#008BFB"
SHAP_BEESWARM_PINK = "#FF0051"
SHAP_COLOR_CMAP = LinearSegmentedColormap.from_list(
    "shap_blue_pink", [SHAP_BEESWARM_BLUE, SHAP_BEESWARM_PINK]
)

HOSPITAL_ID_CANDIDATES = [
    "InstitutionNPI1",
    "InstitutionName",
    "InstitutionKey",
    "HospitalID",
    "SiteID",
    "FacilityID",
]
HOSPITAL_LABEL_CANDIDATES = ["InstitutionName", "InstitutionNPI1", "InstitutionState"]

# Step 2 has fewer delayed reoperation events than Step 1. These criteria keep
# individual hospital estimates restricted to strata with minimum information;
# all remaining hospitals are pooled into a lower-volume stratum.
MIN_HOSPITAL_TEST_ROWS = 50
MIN_HOSPITAL_EVENTS = 3
LOWER_VOLUME_STRATUM_LABEL = "Lower-volume hospital stratum"

BASE_FEATURES = [
    "finaldx_degenerative", "finaldx_radicular", "finaldx_stenosis",
    "finaldx_deformity_instability", "finaldx_other_diagnosis",
    "age", "sex", "race", "ethnicity", "cancer_status",
    "chronic_pulmonary_disease", "congestive_heart_failure",
    "connective_tissue_rheumatic_disease", "diabetes_status",
    "myocardial_infarction", "renal_disease", "institution_type",
    "institution_size", "institution_region", "asa", "bmi", "payer_status",
    "alif_llif", "corpectomy", "discectomy", "foraminotomy",
    "instrumentation", "laminectomy_posterior_decompression",
    "pelvic_fixation", "plf", "tlif_plif", "other_lumbar_procedure",
    "number_operated_levels", "operative_region_extent", "PatTobaccoUse",
]

STEP2_DYNAMIC_PROM_FEATURES = [
    "preop_ODI", "postop_ODI", "ODI_change", PROM_CHANGE_RATE_COL,
    RELATIVE_ODI_MCID_COL, "postop_ODI_day",
]
FEATURES = BASE_FEATURES + STEP2_DYNAMIC_PROM_FEATURES

CONTINUOUS_FEATURES = [
    "age", "bmi", "preop_ODI", "postop_ODI", "ODI_change",
    PROM_CHANGE_RATE_COL, "postop_ODI_day",
]
BINARY_FEATURES = [
    "finaldx_degenerative", "finaldx_radicular", "finaldx_stenosis",
    "finaldx_deformity_instability", "finaldx_other_diagnosis", "sex",
    "ethnicity", "cancer_status", "chronic_pulmonary_disease",
    "congestive_heart_failure", "connective_tissue_rheumatic_disease",
    "myocardial_infarction", "renal_disease", "institution_type",
    "alif_llif", "corpectomy", "discectomy", "foraminotomy", "instrumentation",
    "laminectomy_posterior_decompression", "pelvic_fixation", "plf", "tlif_plif",
    "other_lumbar_procedure", "operative_region_extent", RELATIVE_ODI_MCID_COL,
]
ORDINAL_FEATURES = ["diabetes_status", "institution_size", "asa", "number_operated_levels"]
NOMINAL_FEATURES = ["race", "institution_region", "payer_status", "PatTobaccoUse"]

MISSING_STRINGS = {"", " ", "na", "n/a", "nan", "none", "null", ".", "missing", "<na>"}
BINARY_MAPS = {
    "sex": {"female": 0, "f": 0, "male": 1, "m": 1},
    "ethnicity": {"non-hispanic": 0, "non hispanic": 0, "hispanic": 1},
    "cancer_status": {"no cancer": 0, "no": 0, "none": 0, "any cancer": 1, "yes": 1, "cancer": 1},
    "institution_type": {"hospital": 0, "non-hospital": 1, "non hospital": 1, "nonhospital": 1},
    "operative_region_extent": {"lumbar only": 0, "extended_region_involvement": 1, "extended region involvement": 1, "extended": 1},
}
ORDINAL_MAPS = {
    "diabetes_status": {"no": 0, "none": 0, "0": 0, "without comp": 1, "without complication": 1, "without complications": 1, "1": 1, "with comp": 2, "with complication": 2, "with complications": 2, "2": 2},
    "institution_size": {"between 1-99 beds": 0, "1-99": 0, "between 100-399 beds": 1, "100-399": 1, ">= 400 beds": 2, ">=400 beds": 2, ">=400": 2, ">= 400": 2},
    "asa": {"1": 1, "i": 1, "2": 2, "ii": 2, "3": 3, "iii": 3, "4": 4, "iv": 4, ">=4": 4, ">= 4": 4, "5": 4, "v": 4},
    "number_operated_levels": {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4, ">=4": 4, ">= 4": 4, "5": 4, "6": 4, "7": 4, "8": 4, "9": 4, "10": 4},
}
PREFERRED_NOMINAL_LEVELS = {
    "race": ["White", "Black", "Other"],
    "institution_region": ["South", "North East", "West", "Midwest"],
    "payer_status": ["Medicare", "Commercial/Private", "Other", "Medicaid/Public/Government"],
    "PatTobaccoUse": ["Unknown/Not reported/Multiple", "Never", "Former", "Current"],
}
FEATURE_LABELS = {
    "finaldx_degenerative": "Degenerative diagnosis",
    "finaldx_radicular": "Radiculopathy diagnosis",
    "finaldx_stenosis": "Spinal stenosis diagnosis",
    "finaldx_deformity_instability": "Deformity or instability diagnosis",
    "finaldx_other_diagnosis": "Other lumbar diagnosis",
    "age": "Age",
    "sex": "Sex",
    "race": "Race",
    "ethnicity": "Ethnicity",
    "cancer_status": "Cancer status",
    "chronic_pulmonary_disease": "Chronic pulmonary disease",
    "congestive_heart_failure": "Congestive heart failure",
    "connective_tissue_rheumatic_disease": "Connective tissue/rheumatic disease",
    "diabetes_status": "Diabetes status",
    "myocardial_infarction": "Myocardial infarction",
    "renal_disease": "Renal disease",
    "institution_type": "Institution type",
    "institution_size": "Institution size",
    "institution_region": "Institution region",
    "asa": "ASA physical status",
    "bmi": "BMI",
    "payer_status": "Primary payer",
    "alif_llif": "Anterior/lateral lumbar interbody fusion",
    "corpectomy": "Corpectomy",
    "discectomy": "Discectomy",
    "foraminotomy": "Foraminotomy",
    "instrumentation": "Instrumentation",
    "laminectomy_posterior_decompression": "Posterior decompression or laminectomy",
    "pelvic_fixation": "Pelvic fixation",
    "plf": "Posterolateral fusion",
    "tlif_plif": "Posterior/transforaminal lumbar interbody fusion",
    "other_lumbar_procedure": "Other lumbar procedure",
    "number_operated_levels": "Number of operated levels",
    "operative_region_extent": "Operative region extent",
    "PatTobaccoUse": "Tobacco use",
    "preop_ODI": "Preoperative ODI",
    "postop_ODI": "Postoperative ODI",
    "ODI_change": "Change in ODI",
    PROM_CHANGE_RATE_COL: "ODI change rate",
    RELATIVE_ODI_MCID_COL: "Relative ODI MCID",
    "postop_ODI_day": "Timing of postoperative ODI assessment",
}


# ============================================================
# Preprocessor class for joblib artifact loading
# ============================================================

def clean_scalar(x: Any) -> Any:
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().replace("≥", ">=")
        return np.nan if s.lower() in MISSING_STRINGS else s
    return x


def norm_text(x: Any) -> Optional[str]:
    x = clean_scalar(x)
    if pd.isna(x):
        return None
    return str(x).strip().replace("≥", ">=").lower()


def to_binary_target(x: Any) -> float:
    sx = norm_text(x)
    if sx is None:
        return np.nan
    if sx in {"1", "1.0", "yes", "y", "true", "t"}:
        return 1.0
    if sx in {"0", "0.0", "no", "n", "false", "f"}:
        return 0.0
    try:
        v = float(sx)
        return float(v) if v in (0.0, 1.0) else np.nan
    except Exception:
        return np.nan


def pretty_feature_name(feature: str) -> str:
    return FEATURE_LABELS.get(feature, feature.replace("_", " "))


def safe_filename(x: str) -> str:
    x = str(x).replace("≥", "ge").replace("≤", "le").replace("<", "lt").replace(">", "gt")
    x = re.sub(r"[^A-Za-z0-9_.-]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x[:180] if x else "file"


def json_native(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(k): json_native(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_native(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj


@dataclass
class Step2Preprocessor:
    continuous_features: List[str]
    binary_features: List[str]
    ordinal_features: List[str]
    nominal_features: List[str]
    preferred_nominal_levels: Dict[str, List[str]]

    def __post_init__(self):
        self.continuous_imputer: Optional[SimpleImputer] = None
        self.discrete_imputer: Optional[SimpleImputer] = None
        self.nominal_imputer: Optional[SimpleImputer] = None
        self.onehot: Optional[OneHotEncoder] = None
        self.numeric_feature_names_: List[str] = []
        self.output_feature_names_: List[str] = []

    def _parse_binary(self, x: Any, feature: str) -> float:
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in BINARY_MAPS and sx in BINARY_MAPS[feature]:
            return float(BINARY_MAPS[feature][sx])
        if sx in {"1", "1.0", "yes", "y", "true", "t", "present", "positive", "performed"}:
            return 1.0
        if sx in {"0", "0.0", "no", "n", "false", "f", "absent", "negative", "not performed"}:
            return 0.0
        try:
            v = float(sx)
            return float(v) if v in (0.0, 1.0) else np.nan
        except Exception:
            return np.nan

    def _parse_ordinal(self, x: Any, feature: str) -> float:
        sx = norm_text(x)
        if sx is None:
            return np.nan
        if feature in ORDINAL_MAPS and sx in ORDINAL_MAPS[feature]:
            return float(ORDINAL_MAPS[feature][sx])
        try:
            v = float(sx)
            if feature == "asa":
                return float(min(max(int(round(v)), 1), 4))
            if feature == "number_operated_levels":
                return float(min(max(int(round(v)), 0), 4))
            if feature == "diabetes_status":
                return float(min(max(int(round(v)), 0), 2))
            if feature == "institution_size":
                return float(min(max(int(round(v)), 0), 2))
            return float(v)
        except Exception:
            return np.nan

    def _canonical_nominal(self, feature: str, x: Any) -> Any:
        x = clean_scalar(x)
        if pd.isna(x):
            return np.nan
        s = str(x).strip()
        sl = s.lower()
        if feature == "race":
            if sl == "white":
                return "White"
            if sl == "black":
                return "Black"
            return "Other"
        if feature == "institution_region":
            mapping = {"south": "South", "north east": "North East", "northeast": "North East", "north-east": "North East", "west": "West", "midwest": "Midwest", "mid west": "Midwest"}
            return mapping.get(sl, s)
        if feature == "payer_status":
            if sl == "medicare":
                return "Medicare"
            if sl in {"commercial/private", "commercial", "private", "commercial private"}:
                return "Commercial/Private"
            if sl in {"medicaid/public/government", "medicaid", "public", "government", "public/government"}:
                return "Medicaid/Public/Government"
            return "Other"
        if feature == "PatTobaccoUse":
            if sl == "never":
                return "Never"
            if sl == "former":
                return "Former"
            if sl == "current":
                return "Current"
            return "Unknown/Not reported/Multiple"
        return s

    def _make_parts(self, X: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        cont = pd.DataFrame(index=X.index)
        for c in self.continuous_features:
            cont[c] = pd.to_numeric(X[c].map(clean_scalar), errors="coerce")
        discrete = pd.DataFrame(index=X.index)
        for c in self.binary_features:
            discrete[c] = X[c].map(lambda z: self._parse_binary(z, c)).astype(float)
        for c in self.ordinal_features:
            discrete[c] = X[c].map(lambda z: self._parse_ordinal(z, c)).astype(float)
        nominal = pd.DataFrame(index=X.index)
        for c in self.nominal_features:
            nominal[c] = X[c].map(lambda z: self._canonical_nominal(c, z)).astype("object")
        return cont, discrete, nominal

    def fit(self, X: pd.DataFrame):
        cont, discrete, nominal = self._make_parts(X)
        self.continuous_imputer = SimpleImputer(strategy="median")
        self.discrete_imputer = SimpleImputer(strategy="most_frequent")
        self.nominal_imputer = SimpleImputer(strategy="constant", fill_value="Missing")
        self.continuous_imputer.fit(cont)
        self.discrete_imputer.fit(discrete)
        nominal_imp = pd.DataFrame(self.nominal_imputer.fit_transform(nominal), columns=self.nominal_features)
        categories = []
        for c in self.nominal_features:
            preferred = list(self.preferred_nominal_levels.get(c, []))
            observed = nominal_imp[c].astype(str).unique().tolist()
            categories.append(preferred + sorted([x for x in observed if x not in preferred]))
        try:
            self.onehot = OneHotEncoder(categories=categories, handle_unknown="ignore", sparse_output=False)
        except TypeError:
            self.onehot = OneHotEncoder(categories=categories, handle_unknown="ignore", sparse=False)
        self.onehot.fit(nominal_imp.astype(str))
        self.numeric_feature_names_ = self.continuous_features + self.binary_features + self.ordinal_features
        self.output_feature_names_ = list(self.numeric_feature_names_) + self.onehot.get_feature_names_out(self.nominal_features).tolist()
        return self

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        cont, discrete, nominal = self._make_parts(X)
        cont_imp = self.continuous_imputer.transform(cont)
        discrete_imp = self.discrete_imputer.transform(discrete)
        nominal_imp = pd.DataFrame(self.nominal_imputer.transform(nominal), columns=self.nominal_features)
        nominal_oh = self.onehot.transform(nominal_imp.astype(str))
        return np.concatenate([cont_imp, discrete_imp, nominal_oh], axis=1).astype(float)

    @property
    def output_feature_names(self) -> List[str]:
        return self.output_feature_names_


# ============================================================
# Source and data helpers
# ============================================================

def ensure_input_csv() -> str:
    if os.path.exists(INPUT_CSV):
        return INPUT_CSV
    if os.path.exists(FALLBACK_INPUT_CSV):
        shutil.copy2(FALLBACK_INPUT_CSV, INPUT_CSV)
        return INPUT_CSV
    raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")


def _source_has_required_structure(path: str) -> bool:
    if not os.path.isdir(path):
        return False
    return any(
        os.path.isdir(os.path.join(path, sub))
        for sub in ["dynamic_PROM_expanded", "baseline_only", "baseline"]
    )


def _archive_has_required_structure(zip_path: str) -> bool:
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            names = [n.replace(chr(92), "/") for n in zf.namelist()]
    except Exception:
        return False
    has_dynamic = any("dynamic_PROM_expanded/" in n and n.endswith("_lightgbm_pipeline_artifact_Final.joblib") for n in names)
    has_baseline = any(("baseline_only/" in n or "baseline/" in n) and n.endswith("_lightgbm_pipeline_artifact_Final.joblib") for n in names)
    has_split = any(n.endswith(".xlsx") for n in names)
    return bool(has_dynamic and has_baseline and has_split)


def _find_source_dir_inside(search_root: str) -> Optional[str]:
    if _source_has_required_structure(search_root):
        return search_root
    for root, dirs, _ in os.walk(search_root):
        depth = os.path.relpath(root, search_root).count(os.sep)
        if depth > 6:
            dirs[:] = []
            continue
        if _source_has_required_structure(root):
            return root
    return None


def ensure_source_dir() -> str:
    folder_candidates = [
        os.path.join(BASE_DIR, SOURCE_FOLDER_NAME),
        os.path.join(FALLBACK_DIR, SOURCE_FOLDER_NAME),
    ]
    for folder in folder_candidates:
        if _source_has_required_structure(folder):
            return folder

    archive_candidates = [
        os.path.join(BASE_DIR, SOURCE_ARCHIVE_NAME),
        os.path.join(FALLBACK_DIR, SOURCE_ARCHIVE_NAME),
    ]
    archive_path = None
    for candidate in archive_candidates:
        if os.path.exists(candidate) and _archive_has_required_structure(candidate):
            archive_path = candidate
            break

    if archive_path is None:
        raise FileNotFoundError(
            "Required Step 2 source archive/folder was not found or did not contain both locked baseline-only and dynamic PROM-expanded model artifacts. "
            f"Expected archive name: {SOURCE_ARCHIVE_NAME}"
        )

    extract_root = os.path.join(BASE_DIR, "_step2_hospital_internal_validation_source")
    if os.path.isdir(extract_root):
        shutil.rmtree(extract_root)
    os.makedirs(extract_root, exist_ok=True)
    print(f"Extracting source archive: {archive_path}")
    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(extract_root)

    detected = _find_source_dir_inside(extract_root)
    if detected is None:
        raise FileNotFoundError("The extracted archive did not contain the expected Step 2 model structure.")
    return detected


def find_locked_artifact(source_dir: str, model_kind: str) -> str:
    if model_kind == "dynamic":
        preferred_roots = ["dynamic_PROM_expanded"]
    elif model_kind == "baseline":
        preferred_roots = ["baseline_only", "baseline"]
    else:
        raise ValueError(model_kind)

    selected = []
    for root_name in preferred_roots:
        root = os.path.join(source_dir, root_name)
        if not os.path.isdir(root):
            continue
        for folder_name in os.listdir(root):
            folder_path = os.path.join(root, folder_name)
            if not os.path.isdir(folder_path) or not folder_name.startswith("top01_"):
                continue
            art_dir = os.path.join(folder_path, "model_artifacts")
            if not os.path.isdir(art_dir):
                continue
            for fn in os.listdir(art_dir):
                if fn.endswith("_lightgbm_pipeline_artifact_Final.joblib"):
                    selected.append(os.path.join(art_dir, fn))

    if len(selected) != 1:
        raise RuntimeError(f"Could not identify exactly one locked {model_kind} LightGBM artifact. Found {len(selected)}.")
    return selected[0]


def load_split_assignment(source_dir: str) -> pd.DataFrame:
    workbooks = []
    for root, _, files in os.walk(source_dir):
        for fn in files:
            if fn.endswith(".xlsx"):
                workbooks.append(os.path.join(root, fn))

    for xlsx in workbooks:
        try:
            xl = pd.ExcelFile(xlsx)
            if "split_assignment" not in xl.sheet_names:
                continue
            split = pd.read_excel(xlsx, sheet_name="split_assignment")
            split.columns = [str(c).strip() for c in split.columns]
            if GROUP_COL in split.columns and "split" in split.columns:
                return split[[GROUP_COL, "split"]].drop_duplicates()
        except Exception:
            continue
    raise FileNotFoundError("Could not find a workbook containing a valid split_assignment sheet.")


def derive_dynamic_prom_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    required = ["preop_ODI", "postop_ODI", DAYS_BETWEEN_PROM_COL, "postop_ODI_day", TARGET_COL, GROUP_COL]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"Missing required Step 2 columns: {missing}")

    out["preop_ODI"] = pd.to_numeric(out["preop_ODI"], errors="coerce")
    out["postop_ODI"] = pd.to_numeric(out["postop_ODI"], errors="coerce")
    out[DAYS_BETWEEN_PROM_COL] = pd.to_numeric(out[DAYS_BETWEEN_PROM_COL], errors="coerce")
    out["ODI_change"] = out["postop_ODI"] - out["preop_ODI"]
    out[PROM_CHANGE_RATE_COL] = out["ODI_change"] / out[DAYS_BETWEEN_PROM_COL]

    valid_rel = out["preop_ODI"].notna() & out["postop_ODI"].notna() & (out["preop_ODI"] > 0)
    zero_baseline_rel = out["preop_ODI"].notna() & out["postop_ODI"].notna() & out["preop_ODI"].eq(0)
    out[RELATIVE_ODI_MCID_COL] = np.nan
    out.loc[valid_rel, RELATIVE_ODI_MCID_COL] = (
        (out.loc[valid_rel, "preop_ODI"] - out.loc[valid_rel, "postop_ODI"])
        / out.loc[valid_rel, "preop_ODI"]
        >= RELATIVE_PROM_MCID_CUTOFF
    ).astype(int)
    out.loc[zero_baseline_rel, RELATIVE_ODI_MCID_COL] = 0.0

    out[TARGET_COL] = out[TARGET_COL].map(to_binary_target)
    eligibility = (
        out["preop_ODI"].notna()
        & out["postop_ODI"].notna()
        & out[DAYS_BETWEEN_PROM_COL].notna()
        & (out[DAYS_BETWEEN_PROM_COL] > 0)
        & out[TARGET_COL].isin([0.0, 1.0])
    )
    out = out.loc[eligibility].copy()
    out[TARGET_COL] = out[TARGET_COL].astype(int)
    return out.reset_index(drop=True)


def get_test_dataframe(source_dir: str) -> pd.DataFrame:
    csv_path = ensure_input_csv()
    raw = pd.read_csv(csv_path, low_memory=False)
    raw.columns = [str(c).strip() for c in raw.columns]
    work = derive_dynamic_prom_features(raw)
    split = load_split_assignment(source_dir)
    work = work.merge(split, on=GROUP_COL, how="left")
    if work["split"].isna().any():
        n_missing = int(work["split"].isna().sum())
        raise RuntimeError(f"{n_missing} rows could not be matched to split_assignment by PersonKey.")
    test = work[work["split"].astype(str).eq("test")].reset_index(drop=True)
    if test.empty:
        raise RuntimeError("No held-out test rows were found after applying split_assignment.")
    return test


def predict_locked_artifact(artifact_path: str, test: pd.DataFrame) -> Tuple[Dict[str, Any], np.ndarray]:
    artifact = joblib.load(artifact_path)
    features = list(artifact["features"])
    pre = artifact["preprocessor"]
    model = artifact["model"]
    calibrator = artifact["calibrator"]
    X = test[features].copy()
    Xenc = pre.transform(X)
    p_raw = model.predict_proba(Xenc)[:, 1]
    p_cal = np.clip(calibrator.predict(p_raw), 0, 1)
    return artifact, p_cal


# ============================================================
# Metrics and bootstrap
# ============================================================

def safe_ap(y: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    if len(y) == 0 or len(np.unique(y)) < 2:
        return np.nan
    return float(average_precision_score(y, p))


def paired_delta_ap_bootstrap(y: np.ndarray, p_base: np.ndarray, p_expanded: np.ndarray, n_boot: int, seed: int) -> Dict[str, Any]:
    y = np.asarray(y).astype(int)
    p_base = np.asarray(p_base).astype(float)
    p_expanded = np.asarray(p_expanded).astype(float)
    n = len(y)

    ap_base = safe_ap(y, p_base)
    ap_expanded = safe_ap(y, p_expanded)
    delta = ap_expanded - ap_base if np.isfinite(ap_base) and np.isfinite(ap_expanded) else np.nan

    if n == 0 or len(np.unique(y)) < 2:
        return {
            "AP_baseline": ap_base,
            "AP_dynamic_PROM_expanded": ap_expanded,
            "delta_AP": delta,
            "delta_AP_CI_lower": np.nan,
            "delta_AP_CI_upper": np.nan,
            "delta_AP_p_value": np.nan,
            "bootstrap_valid_iterations": 0,
        }

    rng = np.random.default_rng(seed)
    boot = []
    idx = np.arange(n)
    for _ in range(n_boot):
        b = rng.choice(idx, size=n, replace=True)
        if len(np.unique(y[b])) < 2:
            continue
        b_base = safe_ap(y[b], p_base[b])
        b_exp = safe_ap(y[b], p_expanded[b])
        if np.isfinite(b_base) and np.isfinite(b_exp):
            boot.append(b_exp - b_base)

    boot = np.asarray(boot, dtype=float)
    if len(boot) == 0:
        ci_l = ci_u = p_val = np.nan
    else:
        ci_l, ci_u = np.percentile(boot, [2.5, 97.5])
        p_val = 2.0 * min(np.mean(boot <= 0), np.mean(boot >= 0))
        p_val = min(float(p_val), 1.0)

    return {
        "AP_baseline": ap_base,
        "AP_dynamic_PROM_expanded": ap_expanded,
        "delta_AP": float(delta) if np.isfinite(delta) else np.nan,
        "delta_AP_CI_lower": float(ci_l) if np.isfinite(ci_l) else np.nan,
        "delta_AP_CI_upper": float(ci_u) if np.isfinite(ci_u) else np.nan,
        "delta_AP_p_value": p_val,
        "bootstrap_valid_iterations": int(len(boot)),
    }



# ============================================================
# SHAP helpers
# ============================================================

def raw_feature_numeric_values(X_raw: pd.DataFrame, feature: str) -> pd.Series:
    if feature in CONTINUOUS_FEATURES:
        s = pd.to_numeric(X_raw[feature].map(clean_scalar), errors="coerce")
    elif feature in BINARY_FEATURES:
        tmp_pre = Step2Preprocessor([], [], [], [], {})
        s = X_raw[feature].map(lambda z: tmp_pre._parse_binary(z, feature)).astype(float)
    elif feature in ORDINAL_FEATURES:
        tmp_pre = Step2Preprocessor([], [], [], [], {})
        s = X_raw[feature].map(lambda z: tmp_pre._parse_ordinal(z, feature)).astype(float)
    else:
        vals = X_raw[feature].map(lambda z: str(clean_scalar(z)) if not pd.isna(clean_scalar(z)) else "Missing")
        cats = {v: i for i, v in enumerate(sorted(vals.dropna().unique()))}
        s = vals.map(cats).astype(float)
    if s.isna().any():
        med = s.median(skipna=True)
        s = s.fillna(0.0 if pd.isna(med) else med)
    return s.astype(float)


def encoded_to_group_mapping(pre: Step2Preprocessor, features: List[str]) -> Dict[str, List[int]]:
    encoded_names = list(pre.output_feature_names)
    mapping: Dict[str, List[int]] = {}
    for feature in features:
        idx = []
        if feature in encoded_names:
            idx.append(encoded_names.index(feature))
        if feature in NOMINAL_FEATURES:
            prefix = feature + "_"
            idx.extend([i for i, name in enumerate(encoded_names) if name.startswith(prefix)])
        if idx:
            mapping[feature] = sorted(set(idx))
    return mapping


def compute_grouped_shap(artifact: Dict[str, Any], X_raw: pd.DataFrame, y_true: np.ndarray, p_calibrated: np.ndarray) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    features = list(artifact["features"])
    pre = artifact["preprocessor"]
    model = artifact["model"]

    X_model = X_raw[features].copy().reset_index(drop=True)
    y_true = np.asarray(y_true).astype(int)
    p_calibrated = np.asarray(p_calibrated).astype(float)

    Xenc = pre.transform(X_model)
    encoded_names = list(pre.output_feature_names)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(Xenc)
    shap_values = shap_values[-1] if isinstance(shap_values, list) else shap_values
    shap_values = np.asarray(shap_values)
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, -1]

    mapping = encoded_to_group_mapping(pre, features)
    grouped_values = {}
    grouped_data = {}
    importance_rows = []

    for raw_feature, ids in mapping.items():
        display = pretty_feature_name(raw_feature)
        grouped_values[display] = shap_values[:, ids].sum(axis=1)
        grouped_data[display] = raw_feature_numeric_values(X_model, raw_feature).values

    shap_df = pd.DataFrame(grouped_values)
    data_df = pd.DataFrame(grouped_data)
    total_abs = float(np.abs(shap_df.values).mean(axis=0).sum()) if shap_df.shape[1] else 0.0

    for raw_feature, ids in mapping.items():
        display = pretty_feature_name(raw_feature)
        mean_abs = float(np.abs(shap_df[display]).mean())
        importance_rows.append({
            "raw_feature": raw_feature,
            "display_feature": display,
            "mean_abs_SHAP": mean_abs,
            "percent_of_grouped_SHAP": 100.0 * mean_abs / total_abs if total_abs > 0 else np.nan,
            "n_encoded_columns_grouped": len(ids),
            "encoded_columns": "; ".join([encoded_names[i] for i in ids]),
        })

    imp_df = pd.DataFrame(importance_rows).sort_values("mean_abs_SHAP", ascending=False).reset_index(drop=True)
    shap_df.insert(0, "__row_id__", np.arange(len(shap_df)))
    shap_df.insert(1, "__y_true__", y_true)
    shap_df.insert(2, "__p_calibrated__", p_calibrated)
    return shap_df, data_df, imp_df


def subgroup_importance(shap_sub: pd.DataFrame, imp_all: pd.DataFrame) -> pd.DataFrame:
    feature_rows = []
    features = [f for f in imp_all["display_feature"].tolist() if f in shap_sub.columns]
    total_abs = float(np.abs(shap_sub[features].values).mean(axis=0).sum()) if features else 0.0
    raw_lookup = dict(zip(imp_all["display_feature"], imp_all["raw_feature"]))
    enc_lookup = dict(zip(imp_all["display_feature"], imp_all["encoded_columns"])) if "encoded_columns" in imp_all.columns else {}

    for display in features:
        mean_abs = float(np.abs(shap_sub[display]).mean())
        feature_rows.append({
            "raw_feature": raw_lookup.get(display, display),
            "display_feature": display,
            "mean_abs_SHAP": mean_abs,
            "percent_of_grouped_SHAP": 100.0 * mean_abs / total_abs if total_abs > 0 else np.nan,
            "encoded_columns": enc_lookup.get(display, ""),
        })
    return pd.DataFrame(feature_rows).sort_values("mean_abs_SHAP", ascending=False).reset_index(drop=True)


def save_beeswarm(shap_df: pd.DataFrame, data_df: pd.DataFrame, imp_df: pd.DataFrame, path: str, title: str) -> None:
    ordered = [f for f in imp_df["display_feature"].tolist() if f in shap_df.columns and f in data_df.columns]
    ordered = ordered[: min(SHAP_BEESWARM_MAX_DISPLAY, len(ordered))]
    if not ordered:
        return

    plt.figure(figsize=(11.0, max(8.8, 0.32 * len(ordered) + 2)))
    try:
        shap.summary_plot(
            shap_df[ordered].values,
            features=data_df[ordered],
            feature_names=ordered,
            max_display=len(ordered),
            show=False,
            plot_size=None,
            color=SHAP_COLOR_CMAP,
        )
    except TypeError:
        shap.summary_plot(
            shap_df[ordered].values,
            features=data_df[ordered],
            feature_names=ordered,
            max_display=len(ordered),
            show=False,
            plot_size=None,
        )
    plt.title(title, fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()


# ============================================================
# Main analysis
# ============================================================

def make_zip(src_dir: str, zip_path: str) -> None:
    if os.path.exists(zip_path):
        os.remove(zip_path)
    tmp_zip = zip_path + ".tmp"
    if os.path.exists(tmp_zip):
        os.remove(tmp_zip)
    with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=ZIP_COMPRESSION_LEVEL) as zf:
        for root, _, files in os.walk(src_dir):
            for fn in files:
                full = os.path.join(root, fn)
                rel = os.path.relpath(full, os.path.dirname(src_dir))
                zf.write(full, rel)
    os.replace(tmp_zip, zip_path)



def find_existing_column(columns: List[str], candidates: List[str], required: bool = True) -> Optional[str]:
    lookup = {str(c).strip().lower(): c for c in columns}
    for cand in candidates:
        key = cand.strip().lower()
        if key in lookup:
            return lookup[key]
    if required:
        raise ValueError(f"None of the requested columns was found: {candidates}")
    return None


def assign_hospital_strata(test: pd.DataFrame, y_test: np.ndarray) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, str, Optional[str]]:
    hospital_id_col = find_existing_column(test.columns.tolist(), HOSPITAL_ID_CANDIDATES, required=True)
    hospital_label_col = find_existing_column(test.columns.tolist(), HOSPITAL_LABEL_CANDIDATES, required=False)

    work = test[[GROUP_COL, TARGET_COL, hospital_id_col]].copy()
    work["__row_index_in_test__"] = np.arange(len(work))
    work["hospital_id"] = work[hospital_id_col].map(clean_scalar)
    work["hospital_id"] = work["hospital_id"].astype("object")

    if hospital_label_col:
        work["hospital_label_source"] = test[hospital_label_col].map(clean_scalar).astype("object")
    else:
        work["hospital_label_source"] = work["hospital_id"]

    missing_hospital = int(work["hospital_id"].isna().sum())
    work = work.dropna(subset=["hospital_id"]).copy()
    work["hospital_id"] = work["hospital_id"].astype(str)
    work["hospital_label_source"] = work["hospital_label_source"].fillna(work["hospital_id"]).astype(str)
    work[TARGET_COL] = np.asarray(y_test)[work["__row_index_in_test__"].to_numpy()].astype(int)

    hospital_counts = (
        work.groupby("hospital_id", dropna=False)
        .agg(
            n=(TARGET_COL, "size"),
            events=(TARGET_COL, "sum"),
            hospital_label_source=("hospital_label_source", "first"),
        )
        .reset_index()
    )
    hospital_counts["prevalence"] = hospital_counts["events"] / hospital_counts["n"]
    hospital_counts["meets_individual_reporting_criteria"] = (
        hospital_counts["n"].ge(MIN_HOSPITAL_TEST_ROWS)
        & hospital_counts["events"].ge(MIN_HOSPITAL_EVENTS)
    )

    eligible = hospital_counts[hospital_counts["meets_individual_reporting_criteria"]].copy()
    eligible = eligible.sort_values(["events", "n", "hospital_id"], ascending=[False, False, True]).reset_index(drop=True)
    eligible["hospital_stratum"] = [f"Hospital {i:02d}" for i in range(1, len(eligible) + 1)]

    stratum_map = dict(zip(eligible["hospital_id"], eligible["hospital_stratum"]))
    work["hospital_stratum"] = work["hospital_id"].map(stratum_map).fillna(LOWER_VOLUME_STRATUM_LABEL)
    work["hospital_stratum_type"] = np.where(
        work["hospital_stratum"].eq(LOWER_VOLUME_STRATUM_LABEL),
        "pooled_lower_volume",
        "individual_hospital",
    )

    crosswalk = hospital_counts.merge(
        eligible[["hospital_id", "hospital_stratum"]],
        on="hospital_id",
        how="left",
    )
    crosswalk["hospital_stratum"] = crosswalk["hospital_stratum"].fillna(LOWER_VOLUME_STRATUM_LABEL)
    crosswalk["minimum_test_rows_required"] = MIN_HOSPITAL_TEST_ROWS
    crosswalk["minimum_events_required"] = MIN_HOSPITAL_EVENTS
    crosswalk["missing_hospital_rows_in_test"] = missing_hospital
    crosswalk = crosswalk.sort_values(["hospital_stratum", "events", "n"], ascending=[True, False, False]).reset_index(drop=True)

    audit = pd.DataFrame([
        {"item": "hospital_id_column", "value": hospital_id_col, "note": ""},
        {"item": "hospital_label_column", "value": hospital_label_col if hospital_label_col else "none", "note": ""},
        {"item": "test_rows_without_hospital_id", "value": missing_hospital, "note": "excluded from hospital-stratified estimates"},
        {"item": "unique_hospitals_in_test", "value": int(hospital_counts["hospital_id"].nunique()), "note": ""},
        {"item": "minimum_test_rows_for_individual_hospital", "value": int(MIN_HOSPITAL_TEST_ROWS), "note": ""},
        {"item": "minimum_events_for_individual_hospital", "value": int(MIN_HOSPITAL_EVENTS), "note": ""},
        {"item": "number_of_individual_hospital_strata", "value": int(eligible.shape[0]), "note": ""},
        {"item": "pooled_stratum_label", "value": LOWER_VOLUME_STRATUM_LABEL, "note": ""},
    ])

    return work, crosswalk, audit, hospital_id_col, hospital_label_col



# ============================================================
# Hospital-held-out validation (true hospital-level split)
# ============================================================

HOSPITAL_HELDOUT_DIR = os.path.join(BASE_DIR, "Step2_ODI_hospital-held-out validation")
HOSPITAL_HELDOUT_TABLE_DIR = os.path.join(HOSPITAL_HELDOUT_DIR, "tables")
HOSPITAL_HELDOUT_PLOT_DIR = os.path.join(HOSPITAL_HELDOUT_DIR, "plots")
HOSPITAL_HELDOUT_AUDIT_DIR = os.path.join(HOSPITAL_HELDOUT_DIR, "audit")
HOSPITAL_HELDOUT_TEST_FRACTION = 0.20
HOSPITAL_HELDOUT_CAL_FRACTION_WITHIN_DEVELOPMENT = 0.20
HOSPITAL_HELDOUT_MIN_TEST_EVENTS = 5
HOSPITAL_HELDOUT_MIN_CAL_EVENTS = 3
HOSPITAL_HELDOUT_MIN_TRAIN_EVENTS = 10
HOSPITAL_HELDOUT_SPLIT_ATTEMPTS = 2000


def _hho_safe_roc_auc(y: np.ndarray, p: np.ndarray) -> float:
    try:
        from sklearn.metrics import roc_auc_score
        y = np.asarray(y).astype(int)
        p = np.asarray(p).astype(float)
        if len(y) == 0 or len(np.unique(y)) < 2:
            return np.nan
        return float(roc_auc_score(y, p))
    except Exception:
        return np.nan


def _hho_brier(y: np.ndarray, p: np.ndarray) -> float:
    try:
        from sklearn.metrics import brier_score_loss
        y = np.asarray(y).astype(int)
        p = np.asarray(p).astype(float)
        if len(y) == 0:
            return np.nan
        return float(brier_score_loss(y, p))
    except Exception:
        return np.nan


def _hho_ece(y: np.ndarray, p: np.ndarray, n_bins: int = 10) -> float:
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    if len(y) == 0:
        return np.nan
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    total = len(y)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        if hi == 1.0:
            m = (p >= lo) & (p <= hi)
        else:
            m = (p >= lo) & (p < hi)
        if not np.any(m):
            continue
        ece += (np.sum(m) / total) * abs(float(np.mean(y[m])) - float(np.mean(p[m])))
    return float(ece)


def _hho_group_metrics(y: np.ndarray, p_base: np.ndarray, p_expanded: np.ndarray) -> Dict[str, Any]:
    y = np.asarray(y).astype(int)
    p_base = np.asarray(p_base).astype(float)
    p_expanded = np.asarray(p_expanded).astype(float)
    ap_base = safe_ap(y, p_base)
    ap_exp = safe_ap(y, p_expanded)
    roc_base = _hho_safe_roc_auc(y, p_base)
    roc_exp = _hho_safe_roc_auc(y, p_expanded)
    return {
        "n": int(len(y)),
        "events": int(np.sum(y)),
        "event_rate": float(np.mean(y)) if len(y) else np.nan,
        "baseline_AP": ap_base,
        "dynamic_PROM_expanded_AP": ap_exp,
        "delta_AP": float(ap_exp - ap_base) if np.isfinite(ap_base) and np.isfinite(ap_exp) else np.nan,
        "baseline_ROC_AUC": roc_base,
        "dynamic_PROM_expanded_ROC_AUC": roc_exp,
        "delta_ROC_AUC": float(roc_exp - roc_base) if np.isfinite(roc_base) and np.isfinite(roc_exp) else np.nan,
        "baseline_Brier": _hho_brier(y, p_base),
        "dynamic_PROM_expanded_Brier": _hho_brier(y, p_expanded),
        "baseline_ECE_10bins": _hho_ece(y, p_base, n_bins=10),
        "dynamic_PROM_expanded_ECE_10bins": _hho_ece(y, p_expanded, n_bins=10),
    }


def _hho_find_hospital_column(df: pd.DataFrame) -> Tuple[str, Optional[str]]:
    hospital_id_col = find_existing_column(df.columns.tolist(), HOSPITAL_ID_CANDIDATES, required=True)
    hospital_label_col = find_existing_column(df.columns.tolist(), HOSPITAL_LABEL_CANDIDATES, required=False)
    return hospital_id_col, hospital_label_col


def _hho_choose_hospital_split(df: pd.DataFrame, hospital_col: str, target_col: str, seed: int) -> Tuple[Dict[str, str], pd.DataFrame]:
    work = df[[hospital_col, target_col]].copy()
    work[hospital_col] = work[hospital_col].map(clean_scalar)
    work = work.dropna(subset=[hospital_col]).copy()
    work[hospital_col] = work[hospital_col].astype(str)
    stats = work.groupby(hospital_col, dropna=False).agg(n=(target_col, "size"), events=(target_col, "sum")).reset_index()
    hospital_ids = stats[hospital_col].astype(str).to_numpy()
    if len(hospital_ids) < 3:
        raise RuntimeError("Hospital-held-out validation requires at least 3 hospitals with usable data.")

    rng = np.random.default_rng(seed)
    n_hosp = len(hospital_ids)
    n_test = max(1, int(round(HOSPITAL_HELDOUT_TEST_FRACTION * n_hosp)))
    best = None
    best_score = -np.inf

    def _split_counts(split_map: Dict[str, str]) -> Dict[str, Tuple[int, int, int]]:
        tmp = stats.copy()
        tmp["split"] = tmp[hospital_col].astype(str).map(split_map)
        out = {}
        for s in ["train", "calibration", "hospital_heldout_test"]:
            d = tmp[tmp["split"].eq(s)]
            out[s] = (int(len(d)), int(d["n"].sum()), int(d["events"].sum()))
        return out

    for attempt in range(HOSPITAL_HELDOUT_SPLIT_ATTEMPTS):
        perm = rng.permutation(hospital_ids)
        test_ids = set(perm[:n_test])
        remain = np.array([h for h in perm if h not in test_ids], dtype=object)
        n_cal = max(1, int(round(HOSPITAL_HELDOUT_CAL_FRACTION_WITHIN_DEVELOPMENT * len(remain))))
        cal_ids = set(remain[:n_cal])
        train_ids = set([h for h in remain if h not in cal_ids])
        split_map = {h: "train" for h in train_ids}
        split_map.update({h: "calibration" for h in cal_ids})
        split_map.update({h: "hospital_heldout_test" for h in test_ids})
        counts = _split_counts(split_map)
        train_events = counts["train"][2]
        cal_events = counts["calibration"][2]
        test_events = counts["hospital_heldout_test"][2]
        train_n = counts["train"][1]
        cal_n = counts["calibration"][1]
        test_n = counts["hospital_heldout_test"][1]
        ok = (
            train_events >= HOSPITAL_HELDOUT_MIN_TRAIN_EVENTS
            and cal_events >= HOSPITAL_HELDOUT_MIN_CAL_EVENTS
            and test_events >= HOSPITAL_HELDOUT_MIN_TEST_EVENTS
            and train_n > 0 and cal_n > 0 and test_n > 0
        )
        score = min(train_events / max(HOSPITAL_HELDOUT_MIN_TRAIN_EVENTS, 1), cal_events / max(HOSPITAL_HELDOUT_MIN_CAL_EVENTS, 1), test_events / max(HOSPITAL_HELDOUT_MIN_TEST_EVENTS, 1))
        if score > best_score:
            best = (split_map, counts, attempt, ok)
            best_score = score
        if ok:
            break
    else:
        split_map, counts, attempt, ok = best

    split_map, counts, attempt, ok = best
    split_audit_rows = []
    for split_name, (n_hospitals, n_patients, n_events) in counts.items():
        split_audit_rows.append({
            "split": split_name,
            "n_hospitals": n_hospitals,
            "n_patients": n_patients,
            "events": n_events,
            "selection_attempt": int(attempt),
            "met_minimum_event_criteria": bool(ok),
            "minimum_train_events": HOSPITAL_HELDOUT_MIN_TRAIN_EVENTS,
            "minimum_calibration_events": HOSPITAL_HELDOUT_MIN_CAL_EVENTS,
            "minimum_test_events": HOSPITAL_HELDOUT_MIN_TEST_EVENTS,
        })
    return split_map, pd.DataFrame(split_audit_rows)


def _hho_preprocessor_for_features(features: List[str]) -> Step2Preprocessor:
    return Step2Preprocessor(
        continuous_features=[f for f in CONTINUOUS_FEATURES if f in features],
        binary_features=[f for f in BINARY_FEATURES if f in features],
        ordinal_features=[f for f in ORDINAL_FEATURES if f in features],
        nominal_features=[f for f in NOMINAL_FEATURES if f in features],
        preferred_nominal_levels=PREFERRED_NOMINAL_LEVELS,
    )


def _hho_sanitize_lgbm_params(source_model: Any, seed: int) -> Dict[str, Any]:
    from lightgbm import LGBMClassifier
    allowed = set(LGBMClassifier().get_params().keys())
    try:
        raw = dict(source_model.get_params())
    except Exception:
        raw = {}
    params = {k: v for k, v in raw.items() if k in allowed}
    if not params:
        params = {
            "n_estimators": 500,
            "learning_rate": 0.03,
            "num_leaves": 31,
            "subsample": 0.80,
            "colsample_bytree": 0.80,
            "min_child_samples": 40,
        }
    params["random_state"] = seed
    params["n_jobs"] = -1
    params["objective"] = "binary"
    params["verbose"] = -1
    return params


def _hho_fit_predict_model(train_df: pd.DataFrame, cal_df: pd.DataFrame, test_df: pd.DataFrame, features: List[str], source_model: Any, seed: int) -> Tuple[np.ndarray, Dict[str, Any]]:
    from sklearn.isotonic import IsotonicRegression
    from lightgbm import LGBMClassifier

    pre = _hho_preprocessor_for_features(features)
    X_train = train_df[features].copy()
    X_cal = cal_df[features].copy()
    X_test = test_df[features].copy()
    y_train = train_df[TARGET_COL].astype(int).to_numpy()
    y_cal = cal_df[TARGET_COL].astype(int).to_numpy()

    pre.fit(X_train)
    X_train_enc = pre.transform(X_train)
    X_cal_enc = pre.transform(X_cal)
    X_test_enc = pre.transform(X_test)

    params = _hho_sanitize_lgbm_params(source_model, seed=seed)
    if "scale_pos_weight" not in params or params.get("scale_pos_weight") in [None, 1, 1.0]:
        pos = max(int(y_train.sum()), 1)
        neg = max(int(len(y_train) - y_train.sum()), 1)
        params["scale_pos_weight"] = float(neg / pos)

    model = LGBMClassifier(**params)
    model.fit(X_train_enc, y_train)
    p_cal_raw = model.predict_proba(X_cal_enc)[:, 1]
    p_test_raw = model.predict_proba(X_test_enc)[:, 1]

    if len(np.unique(y_cal)) < 2:
        p_test = np.full_like(p_test_raw, fill_value=float(np.mean(y_train)), dtype=float)
        calibration_method = "constant_train_event_rate_due_to_single_class_calibration_split"
    else:
        calibrator = IsotonicRegression(out_of_bounds="clip")
        calibrator.fit(p_cal_raw, y_cal)
        p_test = np.clip(calibrator.predict(p_test_raw), 0.0, 1.0)
        calibration_method = "isotonic_regression_fit_in_hospital_calibration_split"

    model_audit = {
        "n_train": int(len(train_df)),
        "events_train": int(y_train.sum()),
        "n_calibration": int(len(cal_df)),
        "events_calibration": int(y_cal.sum()),
        "n_test": int(len(test_df)),
        "events_test": int(test_df[TARGET_COL].astype(int).sum()),
        "n_encoded_features": int(X_train_enc.shape[1]),
        "calibration_method": calibration_method,
        "lgbm_params": json.dumps(json_native(params), sort_keys=True),
    }
    return p_test, model_audit


def save_hospital_heldout_delta_plot(metrics_df: pd.DataFrame, path: str, title: str) -> None:
    if metrics_df.empty or "delta_AP" not in metrics_df.columns:
        return
    fig, ax = plt.subplots(figsize=(6.8, 4.6))
    vals = [float(metrics_df["baseline_AP"].iloc[0]), float(metrics_df["dynamic_PROM_expanded_AP"].iloc[0])]
    labels = ["Baseline", "Dynamic ODI"]
    ax.bar(labels, vals)
    ax.set_ylabel("Average precision")
    ax.set_title(title, fontweight="bold")
    delta = float(metrics_df["delta_AP"].iloc[0])
    ax.text(0.5, max(vals) * 1.02 if max(vals) > 0 else 0.01, f"ΔAP = {delta:+.4f}", ha="center", va="bottom", fontsize=10)
    ax.set_ylim(0, max(vals) * 1.20 if max(vals) > 0 else 0.1)
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def run_hospital_heldout_validation_step2(source_dir: str, baseline_artifact_path: str, dynamic_artifact_path: str) -> str:
    for d in [HOSPITAL_HELDOUT_DIR, HOSPITAL_HELDOUT_TABLE_DIR, HOSPITAL_HELDOUT_PLOT_DIR, HOSPITAL_HELDOUT_AUDIT_DIR]:
        os.makedirs(d, exist_ok=True)

    raw_csv = ensure_input_csv()
    raw = pd.read_csv(raw_csv, low_memory=False)
    raw.columns = [str(c).strip() for c in raw.columns]
    data = derive_dynamic_prom_features(raw)
    hospital_id_col, hospital_label_col = _hho_find_hospital_column(data)
    data = data.copy()
    data["__hospital_id__"] = data[hospital_id_col].map(clean_scalar)
    data = data.dropna(subset=["__hospital_id__"]).copy()
    data["__hospital_id__"] = data["__hospital_id__"].astype(str)

    split_map, split_audit = _hho_choose_hospital_split(data, "__hospital_id__", TARGET_COL, seed=RANDOM_STATE + 9001)
    data["hospital_heldout_split"] = data["__hospital_id__"].map(split_map)
    train_df = data[data["hospital_heldout_split"].eq("train")].reset_index(drop=True)
    cal_df = data[data["hospital_heldout_split"].eq("calibration")].reset_index(drop=True)
    test_df = data[data["hospital_heldout_split"].eq("hospital_heldout_test")].reset_index(drop=True)

    baseline_artifact = joblib.load(baseline_artifact_path)
    dynamic_artifact = joblib.load(dynamic_artifact_path)
    baseline_features = list(baseline_artifact["features"])
    dynamic_features = list(dynamic_artifact["features"])

    p_base, base_audit = _hho_fit_predict_model(
        train_df=train_df,
        cal_df=cal_df,
        test_df=test_df,
        features=baseline_features,
        source_model=baseline_artifact["model"],
        seed=RANDOM_STATE + 9101,
    )
    p_dyn, dyn_audit = _hho_fit_predict_model(
        train_df=train_df,
        cal_df=cal_df,
        test_df=test_df,
        features=dynamic_features,
        source_model=dynamic_artifact["model"],
        seed=RANDOM_STATE + 9201,
    )

    y_test = test_df[TARGET_COL].astype(int).to_numpy()
    metrics = _hho_group_metrics(y_test, p_base, p_dyn)
    metrics.update({
        "analysis": "hospital-held-out validation",
        "model_comparison": "dynamic_PROM_expanded_vs_baseline",
        "hospital_split_design": "hospitals assigned exclusively to train, calibration, or held-out validation",
        "n_train_hospitals": int(train_df["__hospital_id__"].nunique()),
        "n_calibration_hospitals": int(cal_df["__hospital_id__"].nunique()),
        "n_heldout_test_hospitals": int(test_df["__hospital_id__"].nunique()),
    })
    metrics_df = pd.DataFrame([metrics])
    metrics_df.to_csv(os.path.join(HOSPITAL_HELDOUT_TABLE_DIR, "Step2_hospital-held-out_metrics.csv"), index=False)

    pred_cols = [GROUP_COL, TARGET_COL, "__hospital_id__", "hospital_heldout_split"]
    if hospital_label_col and hospital_label_col in test_df.columns:
        pred_cols.append(hospital_label_col)
    predictions = test_df[pred_cols].copy()
    predictions["p_baseline_hospital_heldout"] = p_base
    predictions["p_dynamic_PROM_expanded_hospital_heldout"] = p_dyn
    predictions.to_csv(os.path.join(HOSPITAL_HELDOUT_TABLE_DIR, "Step2_hospital-held-out_predictions.csv"), index=False)

    hospital_counts = (
        data.groupby(["hospital_heldout_split", "__hospital_id__"], dropna=False)
        .agg(n=(TARGET_COL, "size"), events=(TARGET_COL, "sum"))
        .reset_index()
        .sort_values(["hospital_heldout_split", "events", "n"], ascending=[True, False, False])
    )
    hospital_counts.to_csv(os.path.join(HOSPITAL_HELDOUT_AUDIT_DIR, "Step2_hospital-held-out_hospital_counts.csv"), index=False)
    split_audit.to_csv(os.path.join(HOSPITAL_HELDOUT_AUDIT_DIR, "Step2_hospital-held-out_split_audit.csv"), index=False)
    pd.DataFrame([{**base_audit, "model": "baseline"}, {**dyn_audit, "model": "dynamic_PROM_expanded"}]).to_csv(
        os.path.join(HOSPITAL_HELDOUT_AUDIT_DIR, "Step2_hospital-held-out_model_audit.csv"), index=False
    )

    plot_path = os.path.join(HOSPITAL_HELDOUT_PLOT_DIR, "Step2_hospital-held-out_AP_comparison.png")
    save_hospital_heldout_delta_plot(metrics_df, plot_path, "Step 2 hospital-held-out validation")

    manifest = {
        "analysis": "Step 2 hospital-held-out validation",
        "input_csv": raw_csv,
        "source_dir": source_dir,
        "baseline_artifact_path": baseline_artifact_path,
        "dynamic_artifact_path": dynamic_artifact_path,
        "hospital_id_column": hospital_id_col,
        "hospital_label_column": hospital_label_col if hospital_label_col else "none",
        "refit_within_hospital_development_set": True,
        "calibration": "isotonic regression fitted in hospital-level calibration split",
        "heldout_unit": "hospital",
        "test_hospitals_seen_during_training_or_calibration": False,
        "threshold_reoptimization": False,
        "output_metrics": ["AP", "delta_AP", "ROC_AUC", "Brier", "ECE_10bins"],
    }
    with open(os.path.join(HOSPITAL_HELDOUT_DIR, "run_manifest.json"), "w") as f:
        json.dump(json_native(manifest), f, indent=2, sort_keys=True)

    summary_xlsx = os.path.join(HOSPITAL_HELDOUT_DIR, "Step2_hospital-held-out_validation_summary.xlsx")
    with pd.ExcelWriter(summary_xlsx, engine="openpyxl") as writer:
        metrics_df.to_excel(writer, sheet_name="metrics", index=False)
        predictions.to_excel(writer, sheet_name="heldout_predictions", index=False)
        split_audit.to_excel(writer, sheet_name="split_audit", index=False)
        hospital_counts.to_excel(writer, sheet_name="hospital_counts", index=False)
        pd.DataFrame([{**base_audit, "model": "baseline"}, {**dyn_audit, "model": "dynamic_PROM_expanded"}]).to_excel(writer, sheet_name="model_audit", index=False)

    hho_zip_path = os.path.join(BASE_DIR, "Step2_ODI_hospital-held-out_validation.zip")
    make_zip(HOSPITAL_HELDOUT_DIR, hho_zip_path)
    return hho_zip_path


def main() -> None:
    print("=" * 100)
    print("Step 2 hospital-level internal validation with locked LightGBM models")
    print("=" * 100)

    source_dir = ensure_source_dir()
    print("Source run:", source_dir)

    test = get_test_dataframe(source_dir)
    y_test = test[TARGET_COL].astype(int).to_numpy()
    print(f"Held-out test set: rows={len(test):,}, events={int(y_test.sum()):,}, prevalence={np.mean(y_test):.4%}")

    baseline_artifact_path = find_locked_artifact(source_dir, "baseline")
    dynamic_artifact_path = find_locked_artifact(source_dir, "dynamic")
    print("Locked baseline artifact:", baseline_artifact_path)
    print("Locked dynamic PROM-expanded artifact:", dynamic_artifact_path)

    baseline_artifact, p_baseline = predict_locked_artifact(baseline_artifact_path, test)
    dynamic_artifact, p_dynamic = predict_locked_artifact(dynamic_artifact_path, test)

    hospital_work, crosswalk, hospital_audit, hospital_id_col, hospital_label_col = assign_hospital_strata(test, y_test)

    predictions = hospital_work[[GROUP_COL, TARGET_COL, "__row_index_in_test__", "hospital_id", "hospital_label_source", "hospital_stratum", "hospital_stratum_type"]].copy()
    predictions["p_baseline_calibrated"] = p_baseline[predictions["__row_index_in_test__"].to_numpy()]
    predictions["p_dynamic_PROM_expanded_calibrated"] = p_dynamic[predictions["__row_index_in_test__"].to_numpy()]
    predictions.to_csv(os.path.join(TABLE_DIR, "Step2_ODI_paired_test_predictions_with_hospital_strata.csv"), index=False)
    crosswalk.to_csv(os.path.join(AUDIT_DIR, "Step2_ODI_hospital_strata_crosswalk.csv"), index=False)

    strata_order = (
        predictions.groupby(["hospital_stratum", "hospital_stratum_type"], dropna=False)
        .agg(n=(TARGET_COL, "size"), events=(TARGET_COL, "sum"))
        .reset_index()
        .sort_values(["hospital_stratum"], ascending=True)
    )
    ordered = strata_order["hospital_stratum"].tolist()
    if LOWER_VOLUME_STRATUM_LABEL in ordered:
        ordered = [s for s in ordered if s != LOWER_VOLUME_STRATUM_LABEL] + [LOWER_VOLUME_STRATUM_LABEL]

    delta_rows = []
    audit_rows = [
        {"item": "test_rows", "value": int(len(test)), "note": "held-out Step 2 test set before excluding missing hospital IDs"},
        {"item": "test_events", "value": int(y_test.sum()), "note": ""},
        {"item": "test_prevalence", "value": float(np.mean(y_test)), "note": ""},
        {"item": "rows_in_hospital_validation", "value": int(len(predictions)), "note": "held-out test rows with hospital assignment"},
    ]
    audit_rows.extend(hospital_audit.to_dict("records"))

    for i, stratum in enumerate(ordered, start=1):
        sub = predictions[predictions["hospital_stratum"].astype(str).eq(str(stratum))].copy()
        y_sub = sub[TARGET_COL].astype(int).to_numpy()
        p_base_sub = sub["p_baseline_calibrated"].astype(float).to_numpy()
        p_dyn_sub = sub["p_dynamic_PROM_expanded_calibrated"].astype(float).to_numpy()
        metrics = paired_delta_ap_bootstrap(
            y_sub,
            p_base_sub,
            p_dyn_sub,
            n_boot=N_BOOTSTRAPS,
            seed=RANDOM_STATE + 100 * i,
        )
        stratum_type = sub["hospital_stratum_type"].iloc[0] if len(sub) else "unknown"
        row = {
            "subgroup_type": "hospital_internal_validation",
            "hospital_stratum": str(stratum),
            "hospital_stratum_type": stratum_type,
            "n": int(len(sub)),
            "events": int(y_sub.sum()) if len(y_sub) else 0,
            "prevalence": float(np.mean(y_sub)) if len(y_sub) else np.nan,
        }
        row.update(metrics)
        delta_rows.append(row)
        audit_rows.append({"item": f"{safe_filename(stratum)}_n", "value": int(len(sub)), "note": str(stratum)})
        audit_rows.append({"item": f"{safe_filename(stratum)}_events", "value": int(y_sub.sum()) if len(y_sub) else 0, "note": str(stratum)})

    delta_df = pd.DataFrame(delta_rows)
    delta_path = os.path.join(TABLE_DIR, "Step2_ODI_hospital_delta_AP.csv")
    delta_df.to_csv(delta_path, index=False)

    audit_df = pd.DataFrame(audit_rows)
    audit_df.to_csv(os.path.join(AUDIT_DIR, "Step2_ODI_hospital_internal_validation_audit.csv"), index=False)

    print("Computing SHAP values for the locked dynamic PROM-expanded model...")
    shap_df_all, data_df_all, imp_df_all = compute_grouped_shap(dynamic_artifact, test, y_test, p_dynamic)
    shap_df_all["__row_index_in_test__"] = np.arange(len(shap_df_all))
    data_df_all["__row_index_in_test__"] = np.arange(len(data_df_all))

    stratum_lookup = predictions[["__row_index_in_test__", "hospital_stratum", "hospital_stratum_type"]].copy()
    shap_df_all = shap_df_all.merge(stratum_lookup, on="__row_index_in_test__", how="inner", validate="one_to_one")
    data_df_all = data_df_all.merge(stratum_lookup, on="__row_index_in_test__", how="inner", validate="one_to_one")

    shap_df_all.to_csv(os.path.join(SHAP_DIR, "Step2_grouped_SHAP_values_all_hospital_rows_Final.csv"), index=False)
    data_df_all.to_csv(os.path.join(SHAP_DIR, "Step2_grouped_SHAP_feature_values_all_hospital_rows_Final.csv"), index=False)
    imp_df_all.to_csv(os.path.join(SHAP_DIR, "Step2_grouped_SHAP_importance_all_test_Final.csv"), index=False)

    shap_manifest_rows = []
    for _, r in delta_df.iterrows():
        stratum = str(r["hospital_stratum"])
        stratum_type = str(r["hospital_stratum_type"])
        mask = shap_df_all["hospital_stratum"].astype(str).eq(stratum)
        shap_sub = shap_df_all.loc[mask].drop(columns=["hospital_stratum", "hospital_stratum_type"], errors="ignore").reset_index(drop=True)
        data_sub = data_df_all.loc[mask].drop(columns=["hospital_stratum", "hospital_stratum_type"], errors="ignore").reset_index(drop=True)
        if len(shap_sub) == 0:
            shap_manifest_rows.append({
                "subgroup_type": "hospital_internal_validation",
                "hospital_stratum": stratum,
                "hospital_stratum_type": stratum_type,
                "n_shap_rows": 0,
                "events_in_shap_rows": 0,
                "beeswarm_plot": "",
                "top_5_SHAP_features": "",
                "status": "no_rows",
            })
            continue

        safe_label = safe_filename(stratum)
        subgroup_dir = os.path.join(SHAP_DIR, safe_label)
        os.makedirs(subgroup_dir, exist_ok=True)

        imp_sub = subgroup_importance(shap_sub, imp_df_all)
        shap_sub.to_csv(os.path.join(subgroup_dir, f"grouped_SHAP_values_{safe_label}.csv"), index=False)
        data_sub.to_csv(os.path.join(subgroup_dir, f"grouped_SHAP_feature_values_{safe_label}.csv"), index=False)
        imp_sub.to_csv(os.path.join(subgroup_dir, f"grouped_SHAP_importance_{safe_label}.csv"), index=False)

        plot_path = os.path.join(PLOT_DIR, f"SHAP_beeswarm_{safe_label}.png")
        save_beeswarm(
            shap_df=shap_sub,
            data_df=data_sub,
            imp_df=imp_sub,
            path=plot_path,
            title=f"Step 2 hospital internal validation: {stratum}",
        )
        shap_manifest_rows.append({
            "subgroup_type": "hospital_internal_validation",
            "hospital_stratum": stratum,
            "hospital_stratum_type": stratum_type,
            "n_shap_rows": int(len(shap_sub)),
            "events_in_shap_rows": int(shap_sub["__y_true__"].sum()) if "__y_true__" in shap_sub.columns else np.nan,
            "beeswarm_plot": plot_path,
            "top_5_SHAP_features": "; ".join(imp_sub["display_feature"].head(5).tolist()),
            "status": "created",
        })

    shap_manifest = pd.DataFrame(shap_manifest_rows)
    shap_manifest.to_csv(os.path.join(TABLE_DIR, "Step2_ODI_hospital_SHAP_beeswarm_manifest.csv"), index=False)

    run_manifest = {
        "analysis": "Step 2 ODI hospital-level internal validation",
        "model": "locked LightGBM",
        "source_dir": source_dir,
        "baseline_artifact_path": baseline_artifact_path,
        "dynamic_artifact_path": dynamic_artifact_path,
        "refit": False,
        "retune": False,
        "recalibrate": False,
        "threshold_reoptimization": False,
        "reported_metrics": ["delta_AP"],
        "reported_plots": ["SHAP beeswarm"],
        "hospital_id_column": hospital_id_col,
        "hospital_label_column": hospital_label_col if hospital_label_col else "none",
        "hospital_id_candidates": HOSPITAL_ID_CANDIDATES,
        "individual_hospital_minimum_test_rows": MIN_HOSPITAL_TEST_ROWS,
        "individual_hospital_minimum_events": MIN_HOSPITAL_EVENTS,
        "pooled_stratum_label": LOWER_VOLUME_STRATUM_LABEL,
        "bootstrap_iterations_for_delta_AP_CI": N_BOOTSTRAPS,
        "test_rows": int(len(test)),
        "test_events": int(y_test.sum()),
        "test_prevalence": float(np.mean(y_test)),
        "timestamp_unix": time.time(),
    }
    with open(os.path.join(OUTPUT_DIR, "run_manifest.json"), "w") as f:
        json.dump(json_native(run_manifest), f, indent=2, sort_keys=True)

    summary_xlsx = os.path.join(OUTPUT_DIR, "Step2_ODI_Hospital_InternalValidation_DeltaAP_SHAP_summary.xlsx")
    with pd.ExcelWriter(summary_xlsx, engine="openpyxl") as writer:
        delta_df.to_excel(writer, sheet_name="delta_AP", index=False)
        shap_manifest.to_excel(writer, sheet_name="SHAP_beeswarm_manifest", index=False)
        audit_df.to_excel(writer, sheet_name="audit", index=False)
        crosswalk.to_excel(writer, sheet_name="hospital_strata_crosswalk", index=False)
        imp_df_all.to_excel(writer, sheet_name="SHAP_importance_all_test", index=False)

    zip_path = os.path.join(BASE_DIR, "Step2_ODI_Hospital_InternalValidation_DeltaAP_SHAP.zip")
    make_zip(OUTPUT_DIR, zip_path)

    print("\nRunning hospital-held-out validation...")
    hospital_heldout_zip_path = run_hospital_heldout_validation_step2(source_dir, baseline_artifact_path, dynamic_artifact_path)

    print("\nDONE")
    print("Output folder:", OUTPUT_DIR)
    print("Delta AP table:", delta_path)
    print("Summary Excel:", summary_xlsx)
    print("ZIP:", zip_path)
    print("Hospital-held-out ZIP:", hospital_heldout_zip_path)

    if CREATE_COLAB_DOWNLOAD_LINK:
        try:
            from IPython.display import HTML, display
            display(HTML(
                f'<p><b>Step 2 hospital-level internal validation output archives are ready.</b></p>'
                f'<p><a href="/files{zip_path}" download>Download hospital-stratified ZIP archive</a></p>'
                f'<p><a href="/files{hospital_heldout_zip_path}" download>Download hospital-held-out ZIP archive</a></p>'
                f'<p>Hospital-stratified path: <code>{zip_path}</code></p>'
                f'<p>Hospital-held-out path: <code>{hospital_heldout_zip_path}</code></p>'
            ))
        except Exception as e:
            print("Download link display skipped:", repr(e))

    if AUTO_DOWNLOAD_ZIP:
        try:
            from google.colab import files
            files.download(zip_path)
            files.download(hospital_heldout_zip_path)
        except Exception as e:
            print("Automatic download skipped:", repr(e))


if __name__ == "__main__":
    main()


#**Survival Analysis**

In [ ]:
# -*- coding: utf-8 -*-
"""
Step 2 dynamic PROM survival analysis
=====================================

This script performs supportive time-to-event analyses for delayed lumbar
reoperation after the postoperative 3-month landmark. PROM-derived risk strata
are defined using prespecified SHAP-informed thresholds. Kaplan-Meier curves,
log-rank tests, and adjusted Cox proportional hazards models are generated
using 10 clinically prespecified parsimonious baseline covariates.
"""

# ============================================================
# Imports
# ============================================================

import os
import re
import sys
import json
import zipfile
import shutil
import warnings
import subprocess
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    from lifelines import CoxPHFitter, KaplanMeierFitter
    from lifelines.statistics import logrank_test, proportional_hazard_test
    from lifelines.plotting import add_at_risk_counts
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lifelines"])
    from lifelines import CoxPHFitter, KaplanMeierFitter
    from lifelines.statistics import logrank_test, proportional_hazard_test
    from lifelines.plotting import add_at_risk_counts

from matplotlib import pyplot as plt
from matplotlib.offsetbox import AnchoredOffsetbox, VPacker, HPacker, TextArea, DrawingArea
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")

# ============================================================
# Configuration
# ============================================================

BASE_DIR = "/content"
FALLBACK_DIR = "/mnt/data"

INPUT_CANDIDATES = [
    "Step 2_ODI_Cohort.csv",
    "Step2_ODI_Cohort.csv",
    "Step 2_ODI.csv",
    "Step2_ODI.csv",
    "Step_2_ODI_Cohort.csv",
    "Step_2_ODI.csv",
]

OUTPUT_DIR = os.path.join(BASE_DIR, "Step2_ODI_Survival_KM_Cox_BinaryOnly_Final")
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
AUDIT_DIR = os.path.join(OUTPUT_DIR, "audit")
for d in [OUTPUT_DIR, PLOT_DIR, TABLE_DIR, AUDIT_DIR]:
    os.makedirs(d, exist_ok=True)

ZIP_COMPRESSION_LEVEL = 1
AUTO_DOWNLOAD_ZIP = True
CREATE_COLAB_DOWNLOAD_LINK = True

# Landmark/horizon definition
LANDMARK_DAY = 90.0
MAX_FOLLOWUP_DAY = 365.0
MAX_TIME_AFTER_LANDMARK = MAX_FOLLOWUP_DAY - LANDMARK_DAY

TARGET_COL = "final_reop_step2"
GROUP_COL = "PersonKey"

# Unpenalized Cox model with robust variance estimation.
COX_PENALIZER = 0.0
COX_L1_RATIO = 0.0
MIN_EVENTS_PER_PARAMETER_WARNING = 10.0

# ============================================================
# Prespecified SHAP-informed thresholds
# ============================================================

THRESHOLDS = {
    "ODI_change": -9.47,
    "postop_ODI_day": 76.99,
    "postop_ODI": 30.11,
    "preop_ODI": 40.60,
    "ODI_change_rate": -0.27,
}

RELATIVE_ODI_MCID_THRESHOLD_FRACTION = 0.30
RELATIVE_ODI_MCID_STATUS_COL = "relative_ODI_MCID_achieved"
RELATIVE_ODI_MCID_CANDIDATES = [
    "relative_ODI_MCID_achieved",
    "relative_ODI_MCID",
    "relative_ODI_MCID_status",
    "ODI_relative_MCID",
    "ODI_relative_MCID_status",
    "ODI_relative_MCID_achieved",
    "relative_odi_mcid",
    "relative_odi_mcid_status",
    "relative_odi_mcid_achieved",
    "relative_MCID",
    "relative_MCID_status",
    "MCID_relative_ODI",
    "MCID_relative_ODI_status",
]

# ODI-related predictors used for Step 2 survival analyses.
# The five numeric ODI-related variables are evaluated using binary
# SHAP-informed thresholds and continuous scaled terms. Relative ODI MCID
# status is evaluated as a binary clinical recovery-status variable.
FOCAL_FEATURES = [
    "preop_ODI",
    "postop_ODI",
    "ODI_change",
    "ODI_change_rate",
    "relative_ODI_MCID",
    "postop_ODI_day",
]

FOCAL_LABELS = {
    "preop_ODI": "Preoperative ODI",
    "postop_ODI": "Postoperative ODI",
    "ODI_change": "Change in ODI",
    "ODI_change_rate": "ODI change rate",
    "relative_ODI_MCID": "Relative ODI MCID status",
    "postop_ODI_day": "Timing of postoperative ODI assessment",
}

FOCAL_BIN_LABELS = {
    "preop_ODI": "Preoperative ODI ≥ 40.60 vs < 40.60",
    "postop_ODI": "Postoperative ODI ≥ 30.11 vs < 30.11",
    "ODI_change": "Change in ODI ≥ -9.47 vs < -9.47",
    "ODI_change_rate": "ODI change rate ≥ -0.27 vs < -0.27",
    "relative_ODI_MCID": "Relative ODI MCID not achieved vs achieved",
    "postop_ODI_day": "Timing of postoperative ODI assessment ≥ 76.99 vs < 76.99",
}

# ============================================================
# Parsimonious baseline covariates
# ============================================================

# Cox adjustment is intentionally restricted to 10 clinically prespecified
# covariates. These columns are the final adjustment terms used in all Cox
# analyses, before adding the ODI-derived predictor(s).
BASELINE_FEATURES = [
    "age_per_10y",
    "male_sex",
    "bmi_per_5",
    "asa_ordinal",
    "diabetes_ordinal",
    "current_smoking",
    "deformity_instability_dx",
    "stenosis_or_radicular_dx",
    "number_levels_ordinal",
    "fusion_or_instrumentation_complexity",
]

PARSIMONIOUS_COVARIATE_LABELS = {
    "age_per_10y": "Age, per 10 years",
    "male_sex": "Male sex",
    "bmi_per_5": "BMI, per 5 kg/m²",
    "asa_ordinal": "ASA physical status, ordinal",
    "diabetes_ordinal": "Diabetes status, ordinal",
    "current_smoking": "Current smoking",
    "deformity_instability_dx": "Deformity or instability diagnosis",
    "stenosis_or_radicular_dx": "Stenosis or radiculopathy diagnosis",
    "number_levels_ordinal": "Number of operated levels, ordinal",
    "fusion_or_instrumentation_complexity": "Fusion/instrumentation complexity",
}

MISSING_STRINGS = {"", " ", "na", "n/a", "nan", "none", "null", ".", "missing", "<na>"}

BINARY_MAPS = {
    "sex": {"female": 0, "f": 0, "male": 1, "m": 1},
    "ethnicity": {"non-hispanic": 0, "non hispanic": 0, "hispanic": 1},
    "cancer_status": {"no cancer": 0, "no": 0, "none": 0, "any cancer": 1, "yes": 1, "cancer": 1},
    "institution_type": {"hospital": 0, "non-hospital": 1, "non hospital": 1, "nonhospital": 1},
    "operative_region_extent": {
        "lumbar only": 0,
        "extended_region_involvement": 1,
        "extended region involvement": 1,
        "extended": 1,
    },
}

ORDINAL_MAPS = {
    "diabetes_status": {
        "no": 0, "none": 0, "0": 0,
        "without comp": 1, "without complication": 1, "without complications": 1, "1": 1,
        "with comp": 2, "with complication": 2, "with complications": 2, "2": 2,
    },
    "institution_size": {
        "between 1-99 beds": 0, "1-99": 0,
        "between 100-399 beds": 1, "100-399": 1,
        ">= 400 beds": 2, ">=400 beds": 2, ">=400": 2, ">= 400": 2,
    },
    "asa": {"1": 1, "i": 1, "2": 2, "ii": 2, "3": 3, "iii": 3, "4": 4, "iv": 4, ">=4": 4, ">= 4": 4, "5": 4, "v": 4},
    "number_operated_levels": {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4, ">=4": 4, ">= 4": 4, "5": 4, "6": 4, "7": 4, "8": 4, "9": 4, "10": 4},
}

FEATURE_LABELS = {
    "finaldx_degenerative": "Degenerative diagnosis",
    "finaldx_radicular": "Radiculopathy diagnosis",
    "finaldx_stenosis": "Spinal stenosis diagnosis",
    "finaldx_deformity_instability": "Deformity or instability diagnosis",
    "finaldx_other_diagnosis": "Other lumbar diagnosis",
    "age": "Age",
    "bmi": "BMI",
    "sex": "Male sex",
    "race": "Race",
    "ethnicity": "Hispanic ethnicity",
    "cancer_status": "Any cancer",
    "chronic_pulmonary_disease": "Chronic pulmonary disease",
    "congestive_heart_failure": "Congestive heart failure",
    "connective_tissue_rheumatic_disease": "Connective tissue/rheumatic disease",
    "diabetes_status": "Diabetes status",
    "myocardial_infarction": "Myocardial infarction",
    "renal_disease": "Renal disease",
    "institution_type": "Non-hospital institution",
    "institution_size": "Institution size",
    "institution_region": "Institution region",
    "asa": "ASA physical status",
    "payer_status": "Primary payer",
    "PatTobaccoUse": "Tobacco use",
    "alif_llif": "Anterior/lateral lumbar interbody fusion",
    "corpectomy": "Corpectomy",
    "discectomy": "Discectomy",
    "foraminotomy": "Foraminotomy",
    "instrumentation": "Instrumentation",
    "laminectomy_posterior_decompression": "Posterior decompression or laminectomy",
    "pelvic_fixation": "Pelvic fixation",
    "plf": "Posterolateral fusion",
    "tlif_plif": "Posterior/transforaminal lumbar interbody fusion",
    "other_lumbar_procedure": "Other lumbar procedure",
    "number_operated_levels": "Number of operated levels",
    "operative_region_extent": "Extended operative region",
}

# Candidate columns for event time and follow-up/censoring time.
REOP_TIME_CANDIDATES = [
    "reoptime", "reop_time", "reoperation_time", "time_to_reoperation",
    "time_to_reoperation_days", "days_to_reoperation", "reoperation_days",
    "days_from_index_to_reoperation", "reop_days", "reoperation_time_days",
]
FOLLOWUP_TIME_CANDIDATES = [
    "followup_days", "follow_up_days", "last_followup_days", "last_follow_up_days",
    "time_to_last_followup_days", "days_to_last_followup", "days_to_last_contact",
    "days_from_index_to_last_followup", "followup_time_days", "censoring_time_days",
]

# ============================================================
# Utility functions
# ============================================================

def clean_scalar(x: Any) -> Any:
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().replace("≥", ">=")
        return np.nan if s.lower() in MISSING_STRINGS else s
    return x


def norm_text(x: Any) -> Optional[str]:
    x = clean_scalar(x)
    if pd.isna(x):
        return None
    return str(x).strip().replace("≥", ">=").lower()


def to_binary_target(x: Any) -> float:
    sx = norm_text(x)
    if sx is None:
        return np.nan
    if sx in {"1", "1.0", "yes", "y", "true", "t"}:
        return 1.0
    if sx in {"0", "0.0", "no", "n", "false", "f"}:
        return 0.0
    try:
        v = float(sx)
        return float(v) if v in (0.0, 1.0) else np.nan
    except Exception:
        return np.nan


def safe_filename(x: str) -> str:
    x = str(x).replace("≥", "ge").replace("≤", "le").replace("<", "lt").replace(">", "gt")
    x = re.sub(r"[^A-Za-z0-9_.-]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x[:180] if x else "file"


def find_input_csv() -> str:
    for root in [BASE_DIR, FALLBACK_DIR]:
        for name in INPUT_CANDIDATES:
            p = os.path.join(root, name)
            if os.path.exists(p):
                if root == FALLBACK_DIR:
                    dst = os.path.join(BASE_DIR, name)
                    if not os.path.exists(dst):
                        shutil.copy2(p, dst)
                    return dst
                return p
    raise FileNotFoundError(
        "Could not find Step 2 ODI cohort CSV. Expected one of: "
        + ", ".join(INPUT_CANDIDATES)
    )


def find_existing_column(columns: List[str], candidates: List[str]) -> Optional[str]:
    lookup = {str(c).lower(): str(c) for c in columns}
    for c in candidates:
        if c in columns:
            return c
        if c.lower() in lookup:
            return lookup[c.lower()]
    return None


def find_relative_odi_mcid_column(columns: List[str]) -> Optional[str]:
    """
    Find the dataset column containing relative ODI MCID status.

    First uses the prespecified candidate list. If the exact column name differs
    across exported datasets, falls back to a conservative fuzzy search requiring
    MCID plus ODI and/or relative wording in the column name.
    """
    exact = find_existing_column(columns, RELATIVE_ODI_MCID_CANDIDATES)
    if exact is not None:
        return exact

    fuzzy = []
    for c in columns:
        cl = str(c).lower()
        compact = re.sub(r"[^a-z0-9]+", "", cl)
        has_mcid = "mcid" in cl or "mcid" in compact
        has_odi = "odi" in cl or "odi" in compact
        has_relative = "relative" in cl or "rel" in cl or "relative" in compact
        if has_mcid and (has_odi or has_relative):
            fuzzy.append(str(c))

    # Prefer columns that look like status/achieved indicators rather than
    # raw threshold constants or notes.
    priority_words = ["achieved", "status", "met", "responder", "response", "relative"]
    fuzzy = sorted(
        fuzzy,
        key=lambda col: (
            -sum(word in col.lower() for word in priority_words),
            len(str(col)),
            str(col).lower(),
        ),
    )
    return fuzzy[0] if fuzzy else None


def parse_binary_value(x: Any, feature: str) -> float:
    sx = norm_text(x)
    if sx is None:
        return np.nan
    if feature in BINARY_MAPS and sx in BINARY_MAPS[feature]:
        return float(BINARY_MAPS[feature][sx])
    if sx in {"1", "1.0", "yes", "y", "true", "t", "present", "positive"}:
        return 1.0
    if sx in {"0", "0.0", "no", "n", "false", "f", "absent", "negative"}:
        return 0.0
    try:
        v = float(sx)
        return float(v) if v in (0.0, 1.0) else np.nan
    except Exception:
        return np.nan


def parse_relative_odi_mcid_achieved(x: Any) -> float:
    """
    Parse relative ODI MCID status as 1 = achieved, 0 = not achieved.

    Handles numeric 0/1 columns as well as common text labels such as
    achieved/not achieved, met/not met, yes/no, and true/false.
    """
    sx = norm_text(x)
    if sx is None:
        return np.nan

    achieved_values = {
        "1", "1.0", "yes", "y", "true", "t", "achieved", "mcid achieved",
        "met", "mcid met", "responder", "response", "improved", "improvement achieved",
    }
    not_achieved_values = {
        "0", "0.0", "no", "n", "false", "f", "not achieved", "mcid not achieved",
        "not met", "mcid not met", "nonresponder", "non-responder",
        "no response", "not improved", "improvement not achieved",
    }

    if sx in achieved_values:
        return 1.0
    if sx in not_achieved_values:
        return 0.0

    try:
        v = float(sx)
        if v in (0.0, 1.0):
            return float(v)
    except Exception:
        pass

    return np.nan


def parse_ordinal_value(x: Any, feature: str) -> float:
    sx = norm_text(x)
    if sx is None:
        return np.nan
    if feature in ORDINAL_MAPS and sx in ORDINAL_MAPS[feature]:
        return float(ORDINAL_MAPS[feature][sx])
    try:
        v = float(sx)
        if feature == "asa":
            return float(min(max(int(round(v)), 1), 4))
        if feature == "number_operated_levels":
            return float(min(max(int(round(v)), 0), 4))
        if feature == "diabetes_status":
            return float(min(max(int(round(v)), 0), 2))
        if feature == "institution_size":
            return float(min(max(int(round(v)), 0), 2))
        return float(v)
    except Exception:
        return np.nan


def parse_current_smoking(x: Any) -> float:
    sx = norm_text(x)
    if sx is None:
        return np.nan
    if sx in {"current", "current smoker", "yes", "1", "1.0", "active"}:
        return 1.0
    if sx in {"never", "former", "no", "0", "0.0", "unknown/not reported/multiple", "unknown", "not reported", "multiple"}:
        return 0.0
    try:
        v = float(sx)
        return 1.0 if v == 1.0 else 0.0 if v == 0.0 else np.nan
    except Exception:
        return np.nan


def canonical_nominal(feature: str, x: Any) -> Any:
    x = clean_scalar(x)
    if pd.isna(x):
        return "Missing"
    s = str(x).strip()
    sl = s.lower()
    if feature == "race":
        if sl == "white":
            return "White"
        if sl == "black":
            return "Black"
        return "Other"
    if feature == "institution_region":
        return {
            "south": "South",
            "north east": "North East",
            "northeast": "North East",
            "north-east": "North East",
            "west": "West",
            "midwest": "Midwest",
            "mid west": "Midwest",
        }.get(sl, s)
    if feature == "payer_status":
        if sl == "medicare":
            return "Medicare"
        if sl in {"commercial/private", "commercial", "private", "commercial private"}:
            return "Commercial/Private"
        if sl in {"medicaid/public/government", "medicaid", "public", "government", "public/government"}:
            return "Medicaid/Public/Government"
        return "Other"
    if feature == "PatTobaccoUse":
        if sl == "never":
            return "Never"
        if sl == "former":
            return "Former"
        if sl == "current":
            return "Current"
        return "Unknown/Not reported/Multiple"
    return s


def add_dynamic_prom_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    required = ["preop_ODI", "postop_ODI"]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"Missing required ODI columns: {missing}")

    out["preop_ODI"] = pd.to_numeric(out["preop_ODI"], errors="coerce")
    out["postop_ODI"] = pd.to_numeric(out["postop_ODI"], errors="coerce")

    if "ODI_change" not in out.columns:
        out["ODI_change"] = out["postop_ODI"] - out["preop_ODI"]
    else:
        out["ODI_change"] = pd.to_numeric(out["ODI_change"], errors="coerce")

    if "days_between_PROMs" in out.columns:
        out["days_between_PROMs"] = pd.to_numeric(out["days_between_PROMs"], errors="coerce")

    if "ODI_change_rate" not in out.columns:
        if "days_between_PROMs" not in out.columns:
            raise ValueError("ODI_change_rate is missing and days_between_PROMs is unavailable.")
        out["ODI_change_rate"] = out["ODI_change"] / out["days_between_PROMs"]
    else:
        out["ODI_change_rate"] = pd.to_numeric(out["ODI_change_rate"], errors="coerce")

    if "postop_ODI_day" not in out.columns:
        if "days_between_PROMs" in out.columns:
            out["postop_ODI_day"] = pd.to_numeric(out["days_between_PROMs"], errors="coerce")
        else:
            raise ValueError("postop_ODI_day is missing and days_between_PROMs is unavailable.")
    else:
        out["postop_ODI_day"] = pd.to_numeric(out["postop_ODI_day"], errors="coerce")

    relative_mcid_col = find_relative_odi_mcid_column(out.columns.tolist())
    if relative_mcid_col is not None:
        # Dataset column is interpreted as 1 = relative ODI MCID achieved,
        # 0 = not achieved. Common text labels are also supported.
        out[RELATIVE_ODI_MCID_STATUS_COL] = out[relative_mcid_col].map(parse_relative_odi_mcid_achieved)
    else:
        # Fallback: derive relative ODI MCID status from preoperative and postoperative ODI.
        # Relative MCID achieved is defined as at least 30% improvement:
        # (preoperative ODI - postoperative ODI) / preoperative ODI >= 0.30.
        preop = pd.to_numeric(out["preop_ODI"], errors="coerce")
        postop = pd.to_numeric(out["postop_ODI"], errors="coerce")
        relative_improvement = (preop - postop) / preop.replace(0, np.nan)
        out[RELATIVE_ODI_MCID_STATUS_COL] = np.where(
            relative_improvement >= RELATIVE_ODI_MCID_THRESHOLD_FRACTION,
            1.0,
            0.0,
        )
        out.loc[relative_improvement.isna(), RELATIVE_ODI_MCID_STATUS_COL] = np.nan

    # Alias used by FOCAL_FEATURES for complete-case filtering and plotting loops.
    # Cox/KM models use the derived binary contrast relative_ODI_MCID_not_achieved.
    out["relative_ODI_MCID"] = out[RELATIVE_ODI_MCID_STATUS_COL]

    return out


def prepare_survival_data(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    audit_rows = []
    work = df.copy()
    work.columns = [str(c).strip() for c in work.columns]

    if TARGET_COL not in work.columns:
        raise ValueError(f"Target column {TARGET_COL} not found.")

    work = add_dynamic_prom_features(work)
    work[TARGET_COL] = work[TARGET_COL].map(to_binary_target)

    reop_time_col = find_existing_column(work.columns.tolist(), REOP_TIME_CANDIDATES)
    followup_col = find_existing_column(work.columns.tolist(), FOLLOWUP_TIME_CANDIDATES)

    audit_rows.append({"item": "input_rows", "value": int(len(work)), "note": ""})
    audit_rows.append({"item": "target_column", "value": TARGET_COL, "note": ""})
    audit_rows.append({"item": "reoperation_time_column", "value": reop_time_col if reop_time_col else "NOT_FOUND", "note": ""})
    audit_rows.append({"item": "followup_time_column", "value": followup_col if followup_col else "NOT_FOUND", "note": ""})

    if reop_time_col is None and work[TARGET_COL].fillna(0).sum() > 0:
        raise ValueError(
            "A reoperation time column is required for survival analysis among event-positive patients. "
            f"Tried: {REOP_TIME_CANDIDATES}"
        )

    if reop_time_col is not None:
        work["_reop_time_from_index"] = pd.to_numeric(work[reop_time_col], errors="coerce")
    else:
        work["_reop_time_from_index"] = np.nan

    if followup_col is not None:
        work["_followup_time_from_index"] = pd.to_numeric(work[followup_col], errors="coerce")
    else:
        work["_followup_time_from_index"] = MAX_FOLLOWUP_DAY
        audit_rows.append({
            "item": "censoring_assumption",
            "value": f"non-events censored at day {MAX_FOLLOWUP_DAY}",
            "note": "No follow-up time column was detected. Confirm this is appropriate for the registry-defined 1-year outcome.",
        })

    # Landmark eligibility.
    target_valid = work[TARGET_COL].isin([0.0, 1.0])
    complete_prom = work[FOCAL_FEATURES].notna().all(axis=1)

    before_landmark_event = (
        work[TARGET_COL].eq(1.0)
        & work["_reop_time_from_index"].notna()
        & (work["_reop_time_from_index"] <= LANDMARK_DAY)
    )

    missing_event_time = work[TARGET_COL].eq(1.0) & work["_reop_time_from_index"].isna()

    eligible = target_valid & complete_prom & (~before_landmark_event) & (~missing_event_time)

    audit_rows.extend([
        {"item": "invalid_or_missing_target_rows", "value": int((~target_valid).sum()), "note": ""},
        {"item": "rows_missing_any_focal_ODI_feature", "value": int((~complete_prom).sum()), "note": ""},
        {"item": "events_at_or_before_landmark_excluded", "value": int(before_landmark_event.sum()), "note": f"landmark day = {LANDMARK_DAY}"},
        {"item": "event_positive_rows_missing_reoperation_time_excluded", "value": int(missing_event_time.sum()), "note": ""},
    ])

    surv = work.loc[eligible].copy()

    # Event definition: delayed reoperation from day 91 to 365.
    event_in_window = (
        surv[TARGET_COL].eq(1.0)
        & surv["_reop_time_from_index"].notna()
        & (surv["_reop_time_from_index"] > LANDMARK_DAY)
        & (surv["_reop_time_from_index"] <= MAX_FOLLOWUP_DAY)
    )

    surv["event"] = event_in_window.astype(int)

    # Duration after landmark.
    duration = np.where(
        surv["event"].eq(1),
        surv["_reop_time_from_index"] - LANDMARK_DAY,
        np.minimum(surv["_followup_time_from_index"], MAX_FOLLOWUP_DAY) - LANDMARK_DAY,
    )
    surv["duration_after_landmark"] = duration
    surv = surv[np.isfinite(surv["duration_after_landmark"]) & (surv["duration_after_landmark"] > 0)].copy()
    surv["duration_after_landmark"] = np.clip(surv["duration_after_landmark"], 1e-3, MAX_TIME_AFTER_LANDMARK)

    audit_rows.extend([
        {"item": "landmark_eligible_rows", "value": int(len(surv)), "note": ""},
        {"item": "delayed_reoperation_events_day91_to_day365", "value": int(surv["event"].sum()), "note": ""},
        {"item": "censored_rows", "value": int((1 - surv["event"]).sum()), "note": ""},
        {"item": "event_prevalence_among_landmark_eligible", "value": float(surv["event"].mean()), "note": ""},
        {"item": "median_duration_after_landmark", "value": float(surv["duration_after_landmark"].median()), "note": "days after postoperative day 90"},
        {"item": "max_duration_after_landmark", "value": float(surv["duration_after_landmark"].max()), "note": "days after postoperative day 90"},
    ])

    return surv.reset_index(drop=True), pd.DataFrame(audit_rows)


# ============================================================
# Feature engineering for Cox
# ============================================================

def add_threshold_variables(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["preop_ODI_ge_40_60"] = (pd.to_numeric(out["preop_ODI"], errors="coerce") >= THRESHOLDS["preop_ODI"]).astype(float)
    out["postop_ODI_ge_30_11"] = (pd.to_numeric(out["postop_ODI"], errors="coerce") >= THRESHOLDS["postop_ODI"]).astype(float)
    out["ODI_change_ge_minus_9_47"] = (pd.to_numeric(out["ODI_change"], errors="coerce") >= THRESHOLDS["ODI_change"]).astype(float)
    out["ODI_change_rate_ge_minus_0_27"] = (pd.to_numeric(out["ODI_change_rate"], errors="coerce") >= THRESHOLDS["ODI_change_rate"]).astype(float)
    out["relative_ODI_MCID_not_achieved"] = 1.0 - pd.to_numeric(out[RELATIVE_ODI_MCID_STATUS_COL], errors="coerce")
    out["postop_ODI_day_ge_76_99"] = (pd.to_numeric(out["postop_ODI_day"], errors="coerce") >= THRESHOLDS["postop_ODI_day"]).astype(float)

    return out


FOCAL_DUMMY_MAP = {
    "preop_ODI": "preop_ODI_ge_40_60",
    "postop_ODI": "postop_ODI_ge_30_11",
    "ODI_change": "ODI_change_ge_minus_9_47",
    "ODI_change_rate": "ODI_change_rate_ge_minus_0_27",
    "relative_ODI_MCID": "relative_ODI_MCID_not_achieved",
    "postop_ODI_day": "postop_ODI_day_ge_76_99",
}

FOCAL_DUMMY_LABELS = {
    "preop_ODI_ge_40_60": "Preoperative ODI ≥ 40.60 vs < 40.60",
    "postop_ODI_ge_30_11": "Postoperative ODI ≥ 30.11 vs < 30.11",
    "ODI_change_ge_minus_9_47": "Change in ODI ≥ -9.47 vs < -9.47",
    "ODI_change_rate_ge_minus_0_27": "ODI change rate ≥ -0.27 vs < -0.27",
    "relative_ODI_MCID_not_achieved": "Relative ODI MCID not achieved vs achieved",
    "postop_ODI_day_ge_76_99": "Timing of postoperative ODI assessment ≥ 76.99 vs < 76.99",
}

FOCAL_STRATUM_LABELS = {
    "preop_ODI": {
        0: "Preoperative ODI < 40.60",
        1: "Preoperative ODI ≥ 40.60",
    },
    "postop_ODI": {
        0: "Postoperative ODI < 30.11",
        1: "Postoperative ODI ≥ 30.11",
    },
    "ODI_change": {
        0: "Change in ODI < -9.47",
        1: "Change in ODI ≥ -9.47",
    },
    "ODI_change_rate": {
        0: "ODI change rate < -0.27",
        1: "ODI change rate ≥ -0.27",
    },
    "relative_ODI_MCID": {
        0: "Relative ODI MCID achieved",
        1: "Relative ODI MCID not achieved",
    },
    "postop_ODI_day": {
        0: "Timing of postoperative ODI assessment < 76.99",
        1: "Timing of postoperative ODI assessment ≥ 76.99",
    },
}


def stratum_labels_for_dummy(group_col: str, fallback_label: str) -> Tuple[str, str]:
    # Direct lookup first so model-specific labels such as
    # "Combined Cox risk score < median (x.xxx)" are used when available.
    if group_col in FOCAL_STRATUM_LABELS:
        labels = FOCAL_STRATUM_LABELS.get(group_col, {})
        return labels.get(0, fallback_label + " < threshold"), labels.get(1, fallback_label + " ≥ threshold")

    for feature, dummy in FOCAL_DUMMY_MAP.items():
        if dummy == group_col:
            labels = FOCAL_STRATUM_LABELS.get(feature, {})
            return labels.get(0, fallback_label + " < threshold"), labels.get(1, fallback_label + " ≥ threshold")

    if group_col == "combined_cox_risk_group":
        return "Combined Cox risk score < median", "Combined Cox risk score ≥ median"

    return fallback_label + " < threshold", fallback_label + " ≥ threshold"




def display_label_for_encoded_term(feature: str, encoded_term: str) -> str:
    """
    DISPLAY-ONLY relabeling.

    This function does not change the Cox design matrix or any coding used for
    model fitting. It only changes labels shown in tables/forest plots.
    """
    level = encoded_term.replace(feature + "_", "")

    if feature == "institution_size":
        if level == "1":
            return "Institution size: Between 1-99"
        return f"Institution size: {level}"

    return f"{FEATURE_LABELS.get(feature, feature)}: {level}"


def mode_or_default(s: pd.Series, default: float = 0.0) -> float:
    mode = s.mode(dropna=True)
    if not mode.empty:
        return float(mode.iloc[0])
    if s.notna().any():
        return float(s.median())
    return float(default)


def build_baseline_design(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, str]]:
    """
    Build the clinically prespecified 10-covariate parsimonious adjustment set.

    The returned design matrix contains exactly the 10 baseline covariate terms
    listed in BASELINE_FEATURES before any ODI-derived focal predictor is added.
    Ordinal variables are modeled as single ordered numeric terms to keep the
    Cox adjustment set parsimonious.
    """
    d = add_threshold_variables(df)
    design = pd.DataFrame(index=d.index)
    label_map: Dict[str, str] = {}

    def add_col(name: str, values: pd.Series, default: float = 0.0):
        s = pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan)
        fill = mode_or_default(s, default=default)
        design[name] = s.fillna(fill).astype(float)
        label_map[name] = PARSIMONIOUS_COVARIATE_LABELS.get(name, name)

    if "age" in d.columns:
        add_col("age_per_10y", pd.to_numeric(d["age"], errors="coerce") / 10.0)
    else:
        add_col("age_per_10y", pd.Series(np.nan, index=d.index))

    if "sex" in d.columns:
        add_col("male_sex", d["sex"].map(lambda z: parse_binary_value(z, "sex")))
    else:
        add_col("male_sex", pd.Series(np.nan, index=d.index))

    if "bmi" in d.columns:
        add_col("bmi_per_5", pd.to_numeric(d["bmi"], errors="coerce") / 5.0)
    else:
        add_col("bmi_per_5", pd.Series(np.nan, index=d.index))

    if "asa" in d.columns:
        add_col("asa_ordinal", d["asa"].map(lambda z: parse_ordinal_value(z, "asa")), default=2.0)
    else:
        add_col("asa_ordinal", pd.Series(np.nan, index=d.index), default=2.0)

    if "diabetes_status" in d.columns:
        add_col("diabetes_ordinal", d["diabetes_status"].map(lambda z: parse_ordinal_value(z, "diabetes_status")))
    else:
        add_col("diabetes_ordinal", pd.Series(np.nan, index=d.index))

    if "PatTobaccoUse" in d.columns:
        add_col("current_smoking", d["PatTobaccoUse"].map(parse_current_smoking))
    else:
        add_col("current_smoking", pd.Series(np.nan, index=d.index))

    if "finaldx_deformity_instability" in d.columns:
        add_col("deformity_instability_dx", d["finaldx_deformity_instability"].map(lambda z: parse_binary_value(z, "finaldx_deformity_instability")))
    else:
        add_col("deformity_instability_dx", pd.Series(np.nan, index=d.index))

    stenosis_radicular_sources = [c for c in ["finaldx_stenosis", "finaldx_radicular"] if c in d.columns]
    if stenosis_radicular_sources:
        vals = [d[c].map(lambda z, col=c: parse_binary_value(z, col)) for c in stenosis_radicular_sources]
        arr = pd.concat(vals, axis=1)
        collapsed = arr.max(axis=1, skipna=True)
        collapsed[arr.isna().all(axis=1)] = np.nan
        add_col("stenosis_or_radicular_dx", collapsed)
    else:
        add_col("stenosis_or_radicular_dx", pd.Series(np.nan, index=d.index))

    if "number_operated_levels" in d.columns:
        add_col("number_levels_ordinal", d["number_operated_levels"].map(lambda z: parse_ordinal_value(z, "number_operated_levels")), default=1.0)
    else:
        add_col("number_levels_ordinal", pd.Series(np.nan, index=d.index), default=1.0)

    complexity_cols = [c for c in ["instrumentation", "alif_llif", "plf", "tlif_plif", "pelvic_fixation", "corpectomy"] if c in d.columns]
    if complexity_cols:
        vals = [d[c].map(lambda z, col=c: parse_binary_value(z, col)) for c in complexity_cols]
        arr = pd.concat(vals, axis=1)
        collapsed = arr.max(axis=1, skipna=True)
        collapsed[arr.isna().all(axis=1)] = np.nan
        add_col("fusion_or_instrumentation_complexity", collapsed)
    else:
        add_col("fusion_or_instrumentation_complexity", pd.Series(np.nan, index=d.index))

    # Enforce column order and exactly 10 baseline terms.
    design = design[BASELINE_FEATURES].copy()
    label_map = {c: PARSIMONIOUS_COVARIATE_LABELS.get(c, c) for c in BASELINE_FEATURES}
    return design, label_map


def add_focal_design(design: pd.DataFrame, df: pd.DataFrame, focal_features: List[str]) -> Tuple[pd.DataFrame, Dict[str, str]]:
    d = add_threshold_variables(df)
    out = design.copy()
    labels = {}
    for f in focal_features:
        dummy = FOCAL_DUMMY_MAP[f]
        out[dummy] = d[dummy].astype(float)
        labels[dummy] = FOCAL_DUMMY_LABELS[dummy]
    return out, labels



def drop_unstable_columns(X: pd.DataFrame, y: pd.Series, protected_cols: List[str]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    audit = []
    keep_cols = []
    protected = set(protected_cols)

    for col in X.columns:
        s = pd.to_numeric(X[col], errors="coerce")
        nunique = int(s.nunique(dropna=True))
        col_sum = float(s.sum(skipna=True))
        event_sum = float(s[y.eq(1)].sum(skipna=True))
        nonevent_sum = float(s[y.eq(0)].sum(skipna=True))

        reason = ""
        keep = True

        if nunique <= 1:
            keep = False
            reason = "zero_variance"
        elif col not in protected:
            # Drop non-focal columns with complete separation or impossible support.
            if event_sum == 0 or nonevent_sum == 0:
                keep = False
                reason = "complete_or_near_separation_non_focal"

        if keep or col in protected:
            keep_cols.append(col)
            action = "kept"
        else:
            action = "dropped"

        audit.append({
            "covariate": col,
            "nunique": nunique,
            "sum": col_sum,
            "event_sum": event_sum,
            "nonevent_sum": nonevent_sum,
            "action": action,
            "reason": reason,
            "protected_focal": col in protected,
        })

    return X[keep_cols].copy(), pd.DataFrame(audit)


def fit_cox_model(
    df: pd.DataFrame,
    covariates: pd.DataFrame,
    focal_cols: List[str],
    model_name: str,
) -> Tuple[CoxPHFitter, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    y = df["event"].astype(int)
    X = covariates.copy()
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)

    # Median/mode fill for any residual missingness after deterministic parsing.
    for col in X.columns:
        if X[col].isna().any():
            mode = X[col].mode(dropna=True)
            fill = float(mode.iloc[0]) if not mode.empty else 0.0
            X[col] = X[col].fillna(fill)

    X, sparse_audit = drop_unstable_columns(X, y, protected_cols=focal_cols)

    cox_df = pd.concat(
        [
            df[["duration_after_landmark", "event"]].reset_index(drop=True),
            X.reset_index(drop=True),
        ],
        axis=1,
    )

    cph = CoxPHFitter(penalizer=COX_PENALIZER, l1_ratio=COX_L1_RATIO)
    cph.fit(
        cox_df,
        duration_col="duration_after_landmark",
        event_col="event",
        robust=True,
        show_progress=False,
    )

    summary = cph.summary.reset_index().rename(columns={"covariate": "term"})
    if "index" in summary.columns and "term" not in summary.columns:
        summary = summary.rename(columns={"index": "term"})

    summary["model_name"] = model_name
    summary["HR"] = np.exp(summary["coef"])
    summary["HR_lower_95"] = np.exp(summary["coef lower 95%"])
    summary["HR_upper_95"] = np.exp(summary["coef upper 95%"])
    summary["n"] = int(len(df))
    summary["events"] = int(df["event"].sum())
    summary["n_parameters"] = int(len(X.columns))
    summary["events_per_parameter"] = float(df["event"].sum() / max(len(X.columns), 1))
    summary["cox_penalizer"] = COX_PENALIZER

    try:
        ph = proportional_hazard_test(cph, cox_df, time_transform="rank")
        ph_df = ph.summary.reset_index().rename(columns={"index": "term"})
        ph_df["model_name"] = model_name
    except Exception as e:
        ph_df = pd.DataFrame([{
            "model_name": model_name,
            "term": "PH_TEST_FAILED",
            "test_statistic": np.nan,
            "p": np.nan,
            "error": repr(e),
        }])

    epv = pd.DataFrame([{
        "model_name": model_name,
        "n": int(len(df)),
        "events": int(df["event"].sum()),
        "n_parameters_after_dropping_unstable_columns": int(len(X.columns)),
        "events_per_parameter": float(df["event"].sum() / max(len(X.columns), 1)),
        "warning_events_per_parameter_lt_10": bool((df["event"].sum() / max(len(X.columns), 1)) < MIN_EVENTS_PER_PARAMETER_WARNING),
        "cox_penalizer": COX_PENALIZER,
        "cox_l1_ratio": COX_L1_RATIO,
    }])

    return cph, summary, ph_df, pd.concat([sparse_audit.assign(model_name=model_name), epv], ignore_index=True, sort=False)


# ============================================================
# Plotting
# ============================================================

def add_logrank_caption_with_line_keys(
    ax,
    p_value: float,
    label0: str,
    label1: str,
    n0: int,
    e0: int,
    n1: int,
    e1: int,
    loc: str = "lower left",
):
    """
    Add one caption box containing:
      - log-rank p-value
      - blue line key for stratum 0
      - orange line key for stratum 1

    This replaces the separate legend so the figure has only one caption.
    """
    def line_row(color: str, text_value: str):
        da = DrawingArea(28, 10, 0, 0)
        da.add_artist(Line2D([2, 26], [5, 5], color=color, linewidth=2.4))
        ta = TextArea(text_value, textprops={"fontsize": 9})
        return HPacker(children=[da, ta], align="center", pad=0, sep=5)

    p_text = "NA" if not np.isfinite(p_value) else f"{p_value:.3g}"
    title = TextArea(f"Log-rank p = {p_text}", textprops={"fontsize": 9})
    row0 = line_row("tab:blue", f"{label0}")
    row1 = line_row("tab:orange", f"{label1}")

    box = VPacker(children=[title, row0, row1], align="left", pad=0, sep=2)
    anchored = AnchoredOffsetbox(
        loc=loc,
        child=box,
        pad=0.35,
        borderpad=0.5,
        frameon=True,
    )
    anchored.patch.set_boxstyle("round,pad=0.35")
    anchored.patch.set_facecolor("white")
    anchored.patch.set_edgecolor("gray")
    anchored.patch.set_alpha(0.92)
    ax.add_artist(anchored)
    return anchored



def plot_forest(df: pd.DataFrame, label_col: str, path: str, title: str):
    d = df.copy()
    d = d.replace([np.inf, -np.inf], np.nan)
    d = d.dropna(subset=["HR", "HR_lower_95", "HR_upper_95"])
    if d.empty:
        return

    d = d.iloc[::-1].reset_index(drop=True)
    fig_h = max(4.8, 0.62 * len(d) + 1.8)
    fig, ax = plt.subplots(figsize=(15.2, fig_h))

    y = np.arange(len(d))
    hr = d["HR"].astype(float).to_numpy()
    lo = d["HR_lower_95"].astype(float).to_numpy()
    hi = d["HR_upper_95"].astype(float).to_numpy()

    significant = (lo > 1.0) | (hi < 1.0)

    for yi, h, l, u, is_sig in zip(y, hr, lo, hi, significant):
        ax.errorbar(
            [h],
            [yi],
            xerr=np.array([[h - l], [u - h]]),
            fmt="o",
            color="#2563EB",
            ecolor="#2563EB",
            capsize=5 if is_sig else 4,
            markersize=9.0 if is_sig else 7.0,
            elinewidth=2.8 if is_sig else 1.6,
            markeredgewidth=1.8 if is_sig else 1.2,
        )

    ax.axvline(1.0, color="gray", linestyle="--", linewidth=1.2)
    ax.set_xscale("log")
    ax.set_yticks(y)
    ax.set_yticklabels(d[label_col].tolist(), fontsize=15)
    ax.tick_params(axis="x", labelsize=13)
    ax.tick_params(axis="y", pad=10)
    ax.set_xlabel("Hazard ratio (log scale)", fontsize=15)
    ax.set_title(title, fontweight="bold", fontsize=18, pad=12)

    x_left = max(min(float(np.nanmin(lo)) * 0.8, 0.8), 0.05)
    x_max = max(float(np.nanmax(hi)), 1.1)
    x_text = x_max * 1.14
    x_right = x_max * 3.8
    for i, row in d.iterrows():
        is_sig = (float(row["HR_lower_95"]) > 1.0) or (float(row["HR_upper_95"]) < 1.0)
        txt = f"{row['HR']:.2f} ({row['HR_lower_95']:.2f}–{row['HR_upper_95']:.2f}), p={row['p']:.3g}"
        ax.text(
            x_text,
            i,
            txt,
            va="center",
            ha="left",
            fontsize=13,
            fontweight="bold" if is_sig else "normal",
            clip_on=False,
        )

    ax.set_xlim(left=x_left, right=x_right)
    ax.grid(axis="x", alpha=0.25)
    fig.subplots_adjust(left=0.38, right=0.92, top=0.88, bottom=0.16)
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_km_for_binary(df: pd.DataFrame, group_col: str, group_label: str, path: str, title: str):
    fig, ax = plt.subplots(figsize=(8.8, 6.0))

    km0 = KaplanMeierFitter()
    km1 = KaplanMeierFitter()

    mask0 = df[group_col].eq(0)
    mask1 = df[group_col].eq(1)

    label0, label1 = stratum_labels_for_dummy(group_col, group_label)

    lr = logrank_test(
        df.loc[mask0, "duration_after_landmark"],
        df.loc[mask1, "duration_after_landmark"],
        event_observed_A=df.loc[mask0, "event"],
        event_observed_B=df.loc[mask1, "event"],
    )

    km0.fit(df.loc[mask0, "duration_after_landmark"], event_observed=df.loc[mask0, "event"], label=label0)
    km1.fit(df.loc[mask1, "duration_after_landmark"], event_observed=df.loc[mask1, "event"], label=label1)

    # Draw curves without matplotlib's separate legend.
    # The single caption below contains the color keys and log-rank details.
    km0.plot_survival_function(ax=ax, ci_show=True, legend=False, color="tab:blue")
    km1.plot_survival_function(ax=ax, ci_show=True, legend=False, color="tab:orange")

    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Days from postoperative day 90 landmark to 1-year follow-up")
    ax.set_ylabel("Probability of no reoperation")
    ax.set_xlim(0, MAX_TIME_AFTER_LANDMARK)
    ax.grid(alpha=0.2)

    add_logrank_caption_with_line_keys(
        ax=ax,
        p_value=float(lr.p_value),
        label0=label0,
        label1=label1,
        n0=int(mask0.sum()),
        e0=int(df.loc[mask0, "event"].sum()),
        n1=int(mask1.sum()),
        e1=int(df.loc[mask1, "event"].sum()),
        loc="lower left",
    )

    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    return {
        "group_col": group_col,
        "group_label": group_label,
        "stratum_0_label": label0,
        "stratum_1_label": label1,
        "n_stratum_0": int(mask0.sum()),
        "events_stratum_0": int(df.loc[mask0, "event"].sum()),
        "n_stratum_1": int(mask1.sum()),
        "events_stratum_1": int(df.loc[mask1, "event"].sum()),
        "logrank_test_statistic": float(lr.test_statistic),
        "logrank_p": float(lr.p_value),
        "plot_path": path,
    }


def plot_cumulative_for_binary(df: pd.DataFrame, group_col: str, group_label: str, path: str, title: str):
    fig, ax = plt.subplots(figsize=(8.8, 6.0))

    label0, label1 = stratum_labels_for_dummy(group_col, group_label)
    masks = [
        (0, label0, df[group_col].eq(0), "tab:blue"),
        (1, label1, df[group_col].eq(1), "tab:orange"),
    ]

    lr = logrank_test(
        df.loc[masks[0][2], "duration_after_landmark"],
        df.loc[masks[1][2], "duration_after_landmark"],
        event_observed_A=df.loc[masks[0][2], "event"],
        event_observed_B=df.loc[masks[1][2], "event"],
    )

    for value, label, mask, color in masks:
        km = KaplanMeierFitter()
        km.fit(df.loc[mask, "duration_after_landmark"], event_observed=df.loc[mask, "event"], label=label)
        surv = km.survival_function_
        ci = km.confidence_interval_
        t = surv.index.values
        cum = 1.0 - surv.iloc[:, 0].values
        ax.plot(t, cum, label=label, color=color)

        try:
            lower_surv = ci.iloc[:, 0].values
            upper_surv = ci.iloc[:, 1].values
            ax.fill_between(t, 1.0 - upper_surv, 1.0 - lower_surv, alpha=0.15, color=color)
        except Exception:
            pass

    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Days from postoperative day 90 landmark to 1-year follow-up")
    ax.set_ylabel("Cumulative probability of reoperation")
    ax.set_xlim(0, MAX_TIME_AFTER_LANDMARK)
    ax.grid(alpha=0.2)

    # Cumulative-risk curves start near zero in the lower-left, so the caption is
    # placed in the upper-left to avoid overlapping the curves.
    add_logrank_caption_with_line_keys(
        ax=ax,
        p_value=float(lr.p_value),
        label0=label0,
        label1=label1,
        n0=int(masks[0][2].sum()),
        e0=int(df.loc[masks[0][2], "event"].sum()),
        n1=int(masks[1][2].sum()),
        e1=int(df.loc[masks[1][2], "event"].sum()),
        loc="upper left",
    )

    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_combined_risk_km_and_cumulative(df: pd.DataFrame, cph: CoxPHFitter, cox_design: pd.DataFrame, out_prefix: str) -> pd.DataFrame:
    # Use the model's linear predictor to create low/high adjusted-risk groups.
    score = cph.predict_log_partial_hazard(cox_design).astype(float).values
    cutoff = float(np.nanmedian(score))
    tmp = df.copy()
    tmp["combined_cox_risk_group"] = np.where(score >= cutoff, 1, 0)

    # Update exact plot labels for this model-specific median cutoff.
    global FOCAL_STRATUM_LABELS
    FOCAL_STRATUM_LABELS["combined_cox_risk_group"] = {
        0: f"Combined Cox risk score < median ({cutoff:.3f})",
        1: f"Combined Cox risk score ≥ median ({cutoff:.3f})",
    }

    km_path = out_prefix + "_KM.png"
    cum_path = out_prefix + "_cumulative_reoperation.png"

    row = plot_km_for_binary(
        tmp,
        "combined_cox_risk_group",
        "Combined 6-feature Cox risk score",
        km_path,
        "Kaplan–Meier analysis by combined Cox risk score",
    )
    plot_cumulative_for_binary(
        tmp,
        "combined_cox_risk_group",
        "Combined 6-feature Cox risk score",
        cum_path,
        "Cumulative reoperation risk by combined Cox risk score",
    )
    row["cumulative_plot_path"] = cum_path
    row["risk_score_median_cutoff"] = cutoff
    return pd.DataFrame([row])


# ============================================================
# Main analyses
# ============================================================

def make_survival_threshold_plots(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    d = add_threshold_variables(df)

    for f in FOCAL_FEATURES:
        dummy = FOCAL_DUMMY_MAP[f]
        label = FOCAL_LABELS[f]
        km_path = os.path.join(PLOT_DIR, f"KM_{safe_filename(label)}_Final.png")
        cum_path = os.path.join(PLOT_DIR, f"Cumulative_reoperation_{safe_filename(label)}_Final.png")

        row = plot_km_for_binary(
            d,
            dummy,
            label,
            km_path,
            f"Kaplan–Meier analysis: {label}",
        )
        plot_cumulative_for_binary(
            d,
            dummy,
            label,
            cum_path,
            f"Cumulative reoperation risk: {label}",
        )
        row["cumulative_plot_path"] = cum_path
        row["threshold"] = THRESHOLDS.get(f, "not achieved")
        row["comparison"] = FOCAL_DUMMY_LABELS[dummy]
        rows.append(row)

    return pd.DataFrame(rows)


def run_single_feature_models(df: pd.DataFrame, baseline_design: pd.DataFrame, baseline_labels: Dict[str, str]):
    """
    Run separate adjusted Cox models using binary SHAP-informed or recovery-status
    ODI-derived predictors only.

    Each model contains the same baseline covariates plus exactly one binary
    ODI-derived predictor. Continuous versions of preoperative ODI,
    postoperative ODI, ODI change, ODI change rate, and timing of postoperative
    ODI assessment are intentionally not modeled.
    """
    all_summaries = []
    all_ph = []
    all_audit = []
    focal_rows = []

    for f in FOCAL_FEATURES:
        dummy = FOCAL_DUMMY_MAP[f]
        model_name = f"Baseline + {FOCAL_LABELS[f]} (binary threshold/status)"
        design, focal_labels = add_focal_design(baseline_design, df, [f])
        focal_cols = [dummy]
        focal_term = dummy
        comparison = FOCAL_DUMMY_LABELS[dummy]

        cph, summary, ph, audit = fit_cox_model(
            df=df,
            covariates=design,
            focal_cols=focal_cols,
            model_name=model_name,
        )

        summary["PROM_modeling_form"] = "binary"
        ph["PROM_modeling_form"] = "binary"
        audit["PROM_modeling_form"] = "binary"

        summary["display_label"] = summary["term"].map(lambda x: focal_labels.get(x, baseline_labels.get(x, x)))
        all_summaries.append(summary)
        all_ph.append(ph)
        all_audit.append(audit)

        row = summary[summary["term"].eq(focal_term)].copy()
        if not row.empty:
            row["feature"] = f
            row["feature_label"] = FOCAL_LABELS[f]
            row["PROM_modeling_form"] = "binary"
            row["comparison"] = comparison
            focal_rows.append(row)

    all_summary_df = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()
    all_ph_df = pd.concat(all_ph, ignore_index=True) if all_ph else pd.DataFrame()
    all_audit_df = pd.concat(all_audit, ignore_index=True) if all_audit else pd.DataFrame()
    focal_df = pd.concat(focal_rows, ignore_index=True) if focal_rows else pd.DataFrame()

    if not focal_df.empty:
        forest_df = focal_df[
            [
                "feature",
                "feature_label",
                "PROM_modeling_form",
                "comparison",
                "model_name",
                "term",
                "HR",
                "HR_lower_95",
                "HR_upper_95",
                "p",
                "n",
                "events",
                "events_per_parameter",
                "cox_penalizer",
            ]
        ].copy()

        feature_order = {f: i for i, f in enumerate(FOCAL_FEATURES)}
        forest_df["feature_order"] = forest_df["feature"].map(feature_order)
        forest_df = forest_df.sort_values(["feature_order"]).drop(columns=["feature_order"])
        forest_df = forest_df.reset_index(drop=True)

        forest_path = os.path.join(PLOT_DIR, "Forest_single_feature_6_binary_adjusted_Cox_Final.png")
        plot_forest(
            forest_df.rename(columns={"comparison": "label"}),
            label_col="label",
            path=forest_path,
            title="Adjusted Cox models: each binary ODI-derived predictor added separately",
        )
        forest_df["forest_plot_path"] = forest_path
    else:
        forest_df = pd.DataFrame()

    return all_summary_df, all_ph_df, all_audit_df, forest_df

def run_combined_feature_model(df: pd.DataFrame, baseline_design: pd.DataFrame, baseline_labels: Dict[str, str]):
    focal_dummies = [FOCAL_DUMMY_MAP[f] for f in FOCAL_FEATURES]
    design, focal_labels = add_focal_design(baseline_design, df, FOCAL_FEATURES)

    cph, summary, ph, audit = fit_cox_model(
        df=df,
        covariates=design,
        focal_cols=focal_dummies,
        model_name="Baseline + all six ODI-related features",
    )

    display_labels = {}
    display_labels.update(baseline_labels)
    display_labels.update(focal_labels)
    summary["display_label"] = summary["term"].map(lambda x: display_labels.get(x, x))

    focal_summary = summary[summary["term"].isin(focal_dummies)].copy()
    focal_summary["comparison"] = focal_summary["term"].map(FOCAL_DUMMY_LABELS)

    forest_path = os.path.join(PLOT_DIR, "Forest_combined_6_ODI_features_adjusted_Cox_Final.png")
    if not focal_summary.empty:
        plot_forest(
            focal_summary.rename(columns={"comparison": "label"}),
            label_col="label",
            path=forest_path,
            title="Adjusted Cox model: all six ODI-related features together",
        )
        focal_summary["forest_plot_path"] = forest_path

    # Create combined Cox risk-score KM/cumulative curves using the same design columns retained by cph.
    retained_cols = list(cph.params_.index)
    cox_design = pd.concat(
        [
            df[["duration_after_landmark", "event"]].reset_index(drop=True),
            design[retained_cols].reset_index(drop=True),
        ],
        axis=1,
    )
    combined_risk_plots = plot_combined_risk_km_and_cumulative(
        df=df.reset_index(drop=True),
        cph=cph,
        cox_design=cox_design,
        out_prefix=os.path.join(PLOT_DIR, "Combined_6_feature_Cox_risk_score_Final"),
    )

    significant_summary = summary[
        (summary["p"] < 0.05)
        & summary["HR"].notna()
        & summary["HR_lower_95"].notna()
        & summary["HR_upper_95"].notna()
    ].copy()

    significant_forest_path = os.path.join(PLOT_DIR, "Forest_combined_6_ODI_features_all_significant_covariates_Cox_Final.png")
    if not significant_summary.empty:
        significant_summary = significant_summary.sort_values("p").reset_index(drop=True)
        plot_forest(
            significant_summary.rename(columns={"display_label": "label"}),
            label_col="label",
            path=significant_forest_path,
            title="Adjusted Cox model: significant covariates and ODI predictors",
        )
        significant_summary["forest_plot_path"] = significant_forest_path

    return summary, ph, audit, focal_summary, combined_risk_plots, significant_summary


def save_threshold_audit(df: pd.DataFrame) -> pd.DataFrame:
    d = add_threshold_variables(df)
    rows = []

    threshold_pairs = {
        "preop_ODI": ("preop_ODI_ge_40_60", "Preoperative ODI", THRESHOLDS["preop_ODI"]),
        "postop_ODI": ("postop_ODI_ge_30_11", "Postoperative ODI", THRESHOLDS["postop_ODI"]),
        "ODI_change": ("ODI_change_ge_minus_9_47", "Change in ODI", THRESHOLDS["ODI_change"]),
        "ODI_change_rate": ("ODI_change_rate_ge_minus_0_27", "ODI change rate", THRESHOLDS["ODI_change_rate"]),
        "relative_ODI_MCID": ("relative_ODI_MCID_not_achieved", "Relative ODI MCID status", "not achieved vs achieved"),
        "postop_ODI_day": ("postop_ODI_day_ge_76_99", "Timing of postoperative ODI assessment", THRESHOLDS["postop_ODI_day"]),
    }

    for raw, (dummy, label, threshold) in threshold_pairs.items():
        for value, group_name in [(0, "below_threshold"), (1, "at_or_above_threshold")]:
            m = d[dummy].eq(value)
            rows.append({
                "feature": raw,
                "feature_label": label,
                "threshold": threshold,
                "stratum": group_name,
                "n": int(m.sum()),
                "events": int(d.loc[m, "event"].sum()),
                "event_rate": float(d.loc[m, "event"].mean()) if m.sum() else np.nan,
                "median_duration_after_landmark": float(d.loc[m, "duration_after_landmark"].median()) if m.sum() else np.nan,
            })

    return pd.DataFrame(rows)


def main():
    print("=" * 100)
    print("Step 2 Dynamic PROM Survival Analysis: Kaplan-Meier + Cox, binary focal predictors only")
    print("=" * 100)

    input_csv = find_input_csv()
    print("Input:", input_csv)

    raw = pd.read_csv(input_csv, low_memory=False)
    raw.columns = [str(c).strip() for c in raw.columns]

    surv, eligibility_audit = prepare_survival_data(raw)
    surv = add_threshold_variables(surv)

    print(f"Landmark-eligible Step 2 cohort: n={len(surv):,}, events={int(surv['event'].sum()):,}, prevalence={surv['event'].mean():.3%}")

    threshold_audit = save_threshold_audit(surv)
    threshold_audit.to_csv(os.path.join(AUDIT_DIR, "threshold_strata_event_audit_Final.csv"), index=False)
    eligibility_audit.to_csv(os.path.join(AUDIT_DIR, "landmark_censoring_eligibility_audit_Final.csv"), index=False)

    baseline_design, baseline_labels = build_baseline_design(surv)

    # KM and cumulative plots for each focal ODI feature.
    km_summary = make_survival_threshold_plots(surv)
    km_summary.to_csv(os.path.join(TABLE_DIR, "KM_logrank_summary_by_ODI_feature_Final.csv"), index=False)

    # Single-feature adjusted Cox models.
    single_summary, single_ph, single_audit, single_forest = run_single_feature_models(
        df=surv,
        baseline_design=baseline_design,
        baseline_labels=baseline_labels,
    )

    # Combined adjusted Cox model.
    combined_summary, combined_ph, combined_audit, combined_focal, combined_risk_plots, combined_significant = run_combined_feature_model(
        df=surv,
        baseline_design=baseline_design,
        baseline_labels=baseline_labels,
    )

    # Multicollinearity / correlation audit for focal threshold variables.
    focal_dummies = [FOCAL_DUMMY_MAP[f] for f in FOCAL_FEATURES]
    corr = surv[focal_dummies].corr()
    corr.to_csv(os.path.join(AUDIT_DIR, "focal_ODI_threshold_variable_correlation_Final.csv"))

    # Save CSV tables.
    single_summary.to_csv(os.path.join(TABLE_DIR, "Cox_single_feature_all_covariates_Final.csv"), index=False)
    single_forest.to_csv(os.path.join(TABLE_DIR, "Cox_single_feature_6_binary_focal_HR_Final.csv"), index=False)
    single_ph.to_csv(os.path.join(TABLE_DIR, "Schoenfeld_PH_tests_single_feature_models_Final.csv"), index=False)
    single_audit.to_csv(os.path.join(AUDIT_DIR, "Cox_single_feature_sparse_EPV_audit_Final.csv"), index=False)

    combined_summary.to_csv(os.path.join(TABLE_DIR, "Cox_combined_6_features_all_covariates_Final.csv"), index=False)
    combined_focal.to_csv(os.path.join(TABLE_DIR, "Cox_combined_6_features_focal_HR_Final.csv"), index=False)
    combined_significant.to_csv(os.path.join(TABLE_DIR, "Cox_combined_6_features_significant_all_terms_Final.csv"), index=False)
    combined_ph.to_csv(os.path.join(TABLE_DIR, "Schoenfeld_PH_test_combined_6_features_Final.csv"), index=False)
    combined_audit.to_csv(os.path.join(AUDIT_DIR, "Cox_combined_6_features_sparse_EPV_audit_Final.csv"), index=False)
    combined_risk_plots.to_csv(os.path.join(TABLE_DIR, "KM_logrank_combined_Cox_risk_score_Final.csv"), index=False)

    settings = {
        "landmark_day": LANDMARK_DAY,
        "max_followup_day": MAX_FOLLOWUP_DAY,
        "time_scale": "days after postoperative day 90",
        "cox_penalizer": COX_PENALIZER,
        "cox_l1_ratio": COX_L1_RATIO,
        "focal_features": FOCAL_FEATURES,
        "thresholds": THRESHOLDS,
        "relative_ODI_MCID_status_column": RELATIVE_ODI_MCID_STATUS_COL,
        "baseline_features": BASELINE_FEATURES,
        "n_baseline_covariates": len(BASELINE_FEATURES),
        "baseline_covariate_set": "10 clinically prespecified parsimonious covariates",
        "note": (
            "All Cox analyses use binary ODI-derived predictors only. "
            "Single-feature Cox analyses include six separate models: five binary SHAP-threshold ODI-related predictors "
            "and relative ODI MCID status. Continuous versions of preoperative ODI, postoperative ODI, ODI change, "
            "ODI change rate, and timing of postoperative ODI assessment are not modeled."
        ),
    }
    with open(os.path.join(AUDIT_DIR, "analysis_settings_Final.json"), "w") as f:
        json.dump(settings, f, indent=2, sort_keys=True)

    # Summary workbook.
    summary_xlsx = os.path.join(OUTPUT_DIR, "Step2_ODI_Survival_KM_Cox_BinaryOnly_summary_Final.xlsx")
    with pd.ExcelWriter(summary_xlsx, engine="openpyxl") as writer:
        eligibility_audit.to_excel(writer, sheet_name="eligibility_censoring_audit", index=False)
        threshold_audit.to_excel(writer, sheet_name="threshold_strata_audit", index=False)
        km_summary.to_excel(writer, sheet_name="KM_logrank_ODI_features", index=False)
        single_forest.to_excel(writer, sheet_name="single_feature_6_binary_HR", index=False)
        combined_focal.to_excel(writer, sheet_name="combined_6_focal_HR", index=False)
        combined_significant.to_excel(writer, sheet_name="combined_6_significant", index=False)
        combined_risk_plots.to_excel(writer, sheet_name="combined_risk_KM", index=False)
        single_ph.to_excel(writer, sheet_name="PH_single_models", index=False)
        combined_ph.to_excel(writer, sheet_name="PH_combined_model", index=False)
        pd.DataFrame([settings]).to_excel(writer, sheet_name="analysis_settings", index=False)
        corr.reset_index().to_excel(writer, sheet_name="focal_correlation", index=False)

    # ZIP all outputs.
    zip_path = os.path.join(BASE_DIR, "Step2_ODI_Survival_KM_Cox_BinaryOnly_Final.zip")
    tmp_zip = zip_path + ".tmp"
    for p in [zip_path, tmp_zip]:
        if os.path.exists(p):
            os.remove(p)

    with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=ZIP_COMPRESSION_LEVEL) as z:
        for root, _, files in os.walk(OUTPUT_DIR):
            for fn in files:
                full = os.path.join(root, fn)
                rel = os.path.relpath(full, os.path.dirname(OUTPUT_DIR))
                z.write(full, rel)
    os.replace(tmp_zip, zip_path)

    print("\nDONE")
    print("Output folder:", OUTPUT_DIR)
    print("Summary Excel:", summary_xlsx)
    print("ZIP:", zip_path)

    if CREATE_COLAB_DOWNLOAD_LINK:
        try:
            from IPython.display import HTML, display
            display(HTML(
                f'<p><b>Step 2 survival outputs are ready.</b></p>'
                f'<p><a href="/files{zip_path}" download>Click here to download the ZIP archive</a></p>'
                f'<p>Path: <code>{zip_path}</code></p>'
            ))
        except Exception as e:
            print("Download link display skipped:", repr(e))

    if AUTO_DOWNLOAD_ZIP:
        try:
            from google.colab import files
            files.download(zip_path)
        except Exception as e:
            print("Automatic download skipped:", repr(e))


if __name__ == "__main__":
    main()